# Muni Characteristic Factor Notebook

## Model Overview

**Instrumented Principal Components Analysis (IPCA)** — Kelly, Pruitt & Su (2019), applied to the **municipal bond** universe.

Each model observation is one bond $i$ on one date $t+1$, with **lagged** static characteristics from date $t$:

$$r_{i,t+1} = z_{i,t}' \Gamma f_{t+1} + \varepsilon_{i,t+1}$$

- $r_{i,t+1}$: return for muni bond $i$ at time $t+1$
- $z_{i,t}$: $L \times 1$ vector of observable characteristics for bond $i$ at time $t$ (lagged, no look-ahead)
- $\Gamma$: $L \times K$ mapping from characteristics to factor loadings
- $f_{t+1}$: $K \times 1$ latent factor realizations
- $\beta_{i,t} = z_{i,t}' \Gamma$: time-varying factor loadings per bond-date

### Static characteristics used in this model

Per `muni_characteristic_buckets.md`, we build the following static, security-level characteristics:

| Characteristic | Type | Source | Bucketing |
|----------------|------|--------|-----------|
| **Market loading** | intercept | — | `market_fv = 1` for every bond |
| **Rating** | categorical → one-hot | `muni_offline.muni_algo_trade_track.final_comp_rating`, fallback `comp_rating` | direct composite rating label |
| **Coupon (cpn)** | categorical → one-hot | `muni.Product.cpn` | coupon buckets `3 / 4 / 5 / 6 / 7+` |
| **Call group** | categorical → one-hot | `muni.Product.callable`, `muni.Product.nxt_call_dt` | `No Call / 0-3m / 3-6m / … / 10y` |
| **State group** | categorical → one-hot | `muni.Product.state_code` | `CA / NY / TX / IL / Other States` |
| **Tenor group** | categorical → one-hot | `muni.Product.maturity` → `tenor_years` | `1y / 2y / … / 5-7y / … / 21y+` |

> These are **static** characteristics (they describe the security, not the market state). Numeric coupon and tenor could alternatively be z-scored; here we follow the muni bucket taxonomy and one-hot encode each group, dropping a reference category to stay full-rank against the intercept.

### Pipeline
1. Data ingestion from Deephaven: `muni.ICEEval` daily marks + `muni.Product` metadata + algo-track composite ratings, restricted to cusips active in `muni_offline.muni_algo_trade_track` from `2026-04-01`
2. Bond-date enrichment, tenor / call / coupon / state / rating derivation
3. Daily return computation + equal-weight muni market series
4. Static characteristic construction (one-hot buckets)
5. Monthly expanding validation for factor-count selection (primary); optional weekly August walk-forward as robustness
6. Full-sample refit, Γ review, core R² decomposition, and residual diagnostics

### Reproducibility note

Before sharing this notebook as research evidence, run a clean restart-and-run-all or export a fresh executed HTML. The cached outputs are useful for review, but a clean run is the only defensible reproducibility artifact.
---

In [ ]:
# Settings required to establish connection and import necessary modules

import os
os.environ['DH_ENV'] = 'FICC'
os.environ['DH_HEAP_SIZE'] = '16'

from jupyter_tools.deephaven import *
from deephaven_ipywidgets import DeephavenWidget

import pandas as pd
import numpy as np
import time
from contextlib import contextmanager
from pathlib import Path
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.3f}'.format


def format_elapsed(seconds: float) -> str:
    """Compact elapsed-time formatter for notebook runtime logs."""
    seconds = float(seconds)
    if seconds < 60:
        return f"{seconds:.2f}s"
    minutes, rem = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {rem:05.2f}s"
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(minutes)}m {rem:05.2f}s"


def log_runtime(message: str, start: float | None = None) -> None:
    """Print a timestamped runtime message; include elapsed time when start is supplied."""
    stamp = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    if start is None:
        print(f"[{stamp}] {message}")
    else:
        print(f"[{stamp}] {message} | elapsed {format_elapsed(time.perf_counter() - start)}")


@contextmanager
def timed_block(label: str):
    """Context manager for timing long notebook operations."""
    start = time.perf_counter()
    log_runtime(f"START {label}")
    try:
        yield
    finally:
        log_runtime(f"END {label}", start)

Opening...
['', '-DQueryTable.enableParallelSnapshot=false', '-DSystemicObjectTracker.enabled=true', '-DDatabase.useLocationCaches=false']
Waiting for query "Python Console 2026-09-17 20:05:06.702160" to be ready
2026-09-17 20:05:07.386570 Python Console 2026-09-17 20:05:06.702160 : Query Status:  PQS_CONNECTING 
2026-09-17 20:05:07.408000 Python Console 2026-09-17 20:05:06.702160 : Query Status:  PQS_ACQUIRING_WORKER 
2026-09-17 20:05:07.437844 Python Console 2026-09-17 20:05:06.702160 : Query Status:  PQS_ACQUIRING_WORKER Creating new worker: worker_63a25b21
2026-09-17 20:05:14.710806 Python Console 2026-09-17 20:05:06.702160 : Query Status:  PQS_EXECUTING 
Connecting to new query "Python Console 2026-09-17 20:05:06.702160", ProcessInfoId="63a25b21-51b8-421c-a313-8570706e3eba"


## 1. Data Ingestion (Deephaven)

Three inputs per bond-date, confirmed against the FICC muni namespace:

1. **Returns** — `muni.ICEEval` daily evaluated marks (`mean_evaluation_clean` price, `mean_yield`, `cusip_cins`).
2. **Static product characteristics** — `muni.Product` (coupon `cpn`, `state_code`, `callable`/`nxt_call_dt`, `maturity`).
3. **Composite rating bucket** — latest `muni_offline.muni_algo_trade_track.final_comp_rating`, with `comp_rating` retained as the fallback rating source.

The research window starts on `2026-04-01`. The universe is restricted to cusips active in `muni_offline.muni_algo_trade_track`, then semi-joined to `muni.ICEEval` to keep the panel tractable while preserving the algo-traded muni universe.

The raw pulled `panel_slim` dataframe is cached locally in `panel_slim_cache.pkl`. On rerun, the Deephaven query starts from the latest cached date, re-pulls that overlap date, then concatenates and de-duplicates by `date` and `cusip` so only new or revised daily rows are added to the local cache.

In [ ]:
# Determine the incremental Deephaven pull window from the local raw-panel cache,
# then push it into the Deephaven session so the %%deephaven_cell query can use it.
# The Deephaven cell runs in a separate (remote) namespace, so the value must be
# injected via run_script rather than shared as an ordinary local variable.
from pathlib import Path

base_cutoff_date = '2026-04-01'
PANEL_CACHE_PATH = Path('panel_slim_cache.pkl')
cutoff_date = base_cutoff_date

if PANEL_CACHE_PATH.exists():
    try:
        _cached = pd.read_pickle(PANEL_CACHE_PATH)
        if isinstance(_cached, pd.DataFrame) and not _cached.empty and 'date' in _cached.columns:
            _dates = pd.to_datetime(_cached['date'], errors='coerce').dropna()
            if not _dates.empty:
                _refresh_from = max(_dates.max().normalize() - pd.Timedelta(days=1), pd.Timestamp(base_cutoff_date))
                cutoff_date = str(_refresh_from.date())
                print(f"Found local panel cache through {_dates.max().date()}; refreshing from {cutoff_date}")
            else:
                print("Local panel cache has no parseable dates; using base cutoff")
        else:
            print("Local panel cache is empty or invalid; using base cutoff")
    except Exception as exc:
        print(f"WARNING: could not inspect local panel cache {PANEL_CACHE_PATH}: {exc}; using base cutoff")
    finally:
        globals().pop('_cached', None)
        globals().pop('_dates', None)
else:
    print(f"No local panel cache found; pulling from {cutoff_date}")

# Inject cutoff_date into the Deephaven server session if it is already connected.
# On a cold kernel the session may not exist yet; the ingestion cell then falls
# back to the full research window (safe, since there is nothing to append to).
try:
    DeephavenSession.session.run_script(f"cutoff_date = '{cutoff_date}'")
    print(f"Injected cutoff_date='{cutoff_date}' into the Deephaven session.")
except Exception as exc:
    print(f"Deephaven session not ready to receive cutoff_date ({exc}); "
          f"ingestion cell will default to the full window until the session is live.")

Found local panel cache through 2026-09-15; refreshing from 2026-09-14
Injected cutoff_date='2026-09-14' into the Deephaven session.


In [ ]:
%%deephaven_cell

import time as _time

# This cell runs inside the remote Deephaven session, a separate namespace from
# the local kernel. cutoff_date may have been injected by the preceding local
# cell; default to the full research window when it has not.
try:
    cutoff_date
except NameError:
    cutoff_date = '2026-04-01'

_cell_start = _time.perf_counter()
print("[runtime] START Deephaven ingestion: ICEEval + algo universe + Product join")
print(f"cutoff_date: {cutoff_date}")

# Restrict the universe to the desk's algo-traded bonds and carry the algo-track
# composite rating fields used by the model.
algo_track_ref = (
    db.historical_table("muni_offline", "muni_algo_trade_track")
      .last_by(['cusip'])
      .view(['cusip', 'final_comp_rating', 'comp_rating'])
)
universe_cusips = algo_track_ref.select_distinct(['cusip'])

# Muni EOD price source (daily evaluated close): muni.ICEEval
#   price = mean_evaluation_clean, yield = mean_yield, cusip = cusip_cins
pricing_tb = (
    db.historical_table("muni", "ICEEval")
      .where(f"date >= `{cutoff_date}`")
      .update_view(['cusip = cusip_cins'])
      .where_in(universe_cusips, cols=['cusip'])   # semi-join to algo universe
      .last_by(['date', 'cusip'])                  # one evaluation per bond per day
)

# Product is a static reference table (one row per cusip).
product = (
    db.historical_table("muni", "Product")
      .last_by(['cusip'])
)

# Static product characteristics and algo-track composite ratings joined onto
# the daily price panel by CUSIP.
pricing_enriched = (
    pricing_tb.natural_join(
        product,
        on=['cusip=cusip'],
        joins=['cpn', 'state_code', 'callable', 'nxt_call_dt', 'maturity'],
    )
    .natural_join(
        algo_track_ref,
        on=['cusip=cusip'],
        joins=['final_comp_rating', 'comp_rating'],
    )
)

# Keep only the columns the model needs, to keep the pandas pull light.
panel_slim = pricing_enriched.view([
    'date', 'cusip',
    'price = mean_evaluation_clean', 'mkt_yield = mean_yield',
    'cpn', 'state_code', 'callable', 'nxt_call_dt', 'maturity',
    'final_comp_rating', 'comp_rating',
])

print("algo universe cusips:", universe_cusips.size)
print("algo rating ref size:", algo_track_ref.size)
print("panel_slim incremental size:", panel_slim.size)
print(f"[runtime] END Deephaven ingestion | elapsed {_time.perf_counter() - _cell_start:.2f}s")
DeephavenWidget(panel_slim)

[runtime] START Deephaven ingestion: ICEEval + algo universe + Product join
cutoff_date: 2026-09-14
('algo universe cusips:', 255400)
('algo rating ref size:', 255400)
('panel_slim incremental size:', 718178)
[runtime] END Deephaven ingestion | elapsed 48.06s



DeephavenWidget(height=600, iframe_url='https://nppra52a0006.wellsfargo.com:24115/iframe/widget/?name=_dw_var_…



Completed on FICC Deephaven worker


In [ ]:
# Convert DH object to pandas and maintain a local raw-panel cache.
PANEL_CACHE_PATH = Path('panel_slim_cache.pkl')
PANEL_CACHE_KEYS = ['date', 'cusip']
PANEL_CACHE_COLUMNS = [
    'date', 'cusip', 'price', 'mkt_yield',
    'cpn', 'state_code', 'callable', 'nxt_call_dt', 'maturity',
    'final_comp_rating', 'comp_rating',
]

with timed_block("Pull incremental panel_slim to pandas"):
    panel_incremental = DeephavenSession.session.open_table('panel_slim').to_arrow().to_pandas()

panel_incremental['date'] = pd.to_datetime(panel_incremental['date'], errors='coerce')
panel_incremental = panel_incremental.reindex(columns=PANEL_CACHE_COLUMNS)
print(f"panel_slim incremental pull: {panel_incremental.shape}")

if PANEL_CACHE_PATH.exists():
    with timed_block("Load local panel_slim cache"):
        try:
            cached_panel = pd.read_pickle(PANEL_CACHE_PATH)
        except Exception as exc:
            print(f"WARNING: could not load {PANEL_CACHE_PATH}: {exc}; rebuilding cache from incremental pull")
            cached_panel = pd.DataFrame(columns=PANEL_CACHE_COLUMNS)
    if not cached_panel.empty:
        cached_panel['date'] = pd.to_datetime(cached_panel['date'], errors='coerce')
        cached_panel = cached_panel.reindex(columns=PANEL_CACHE_COLUMNS)
else:
    cached_panel = pd.DataFrame(columns=PANEL_CACHE_COLUMNS)

with timed_block("Merge incremental pull into local panel cache"):
    before_rows = len(cached_panel)
    data = pd.concat([cached_panel, panel_incremental], ignore_index=True)
    data = data.dropna(subset=PANEL_CACHE_KEYS)
    data = data.drop_duplicates(PANEL_CACHE_KEYS, keep='last').sort_values(PANEL_CACHE_KEYS).reset_index(drop=True)
    data.to_pickle(PANEL_CACHE_PATH)

new_or_revised_rows = len(data) - before_rows
print(f"local panel cache: {before_rows:,} → {len(data):,} rows ({new_or_revised_rows:+,} net)")
print(f"cache date range: {data['date'].min().date()} → {data['date'].max().date()}")
print(f"columns: {list(data.columns)}")
display(data.head(3))

[2026-09-17 20:06:30] START Pull incremental panel_slim to pandas
[2026-09-17 20:06:34] END Pull incremental panel_slim to pandas | elapsed 4.52s
panel_slim incremental pull: (718178, 11)
[2026-09-17 20:06:34] START Load local panel_slim cache
[2026-09-17 20:06:51] END Load local panel_slim cache | elapsed 17.12s
[2026-09-17 20:06:52] START Merge incremental pull into local panel cache
[2026-09-17 20:08:08] END Merge incremental pull into local panel cache | elapsed 1m 16.22s
local panel cache: 26,519,201 → 26,760,668 rows (+241,467 net)
cache date range: 2026-04-01 → 2026-09-16
columns: ['date', 'cusip', 'price', 'mkt_yield', 'cpn', 'state_code', 'callable', 'nxt_call_dt', 'maturity', 'final_comp_rating', 'comp_rating']


,date,cusip,price,mkt_yield,cpn,state_code,callable,nxt_call_dt,maturity,final_comp_rating,comp_rating
0,2026-04-01,0004162A4,113.383,2.316,5.0,CA,N,,1943323200000,NaN,AA
1,2026-04-01,0004162B2,115.041,2.422,5.0,CA,N,,1974945600000,NaN,AA
2,2026-04-01,0004162C0,116.517,2.518,5.0,CA,N,,2006481600000,NaN,AA


## 2. Bucket Derivation

Static characteristics are built from point-in-time fields in `muni.Product` plus the latest composite rating fields carried on `muni_offline.muni_algo_trade_track`, then lagged by one observed row per bond before modeling.

### Source fields and parsing

- Trade date: `date`
- Maturity: `maturity` (mixed format: epoch-millis or date string)
- Coupon: `cpn`
- Callability: `callable`, next call date `nxt_call_dt`
- State: `state_code`
- Rating bucket source:
  - Primary: `final_comp_rating` from `muni_algo_trade_track`
  - Fallback: `comp_rating` from `muni_algo_trade_track`

Datetime parsing uses `_parse_mixed_datetime`:

1. Try numeric parse and interpret as epoch milliseconds.
2. For rows still missing, parse as date strings.
3. Unparseable values stay `NaT`.

### Exact bucket rules

#### Tenor bucket (`tenor_bucket`)

- Compute `trade_year = year(date)`
- Compute `maturity_year = year(parsed maturity)`
- `tenor_years = max(1, maturity_year - trade_year)`
- Map to labels:
  - `<= 1` -> `1y`
  - `<= 2` -> `2y`
  - `<= 3` -> `3y`
  - `<= 4` -> `4y`
  - `<= 7` -> `5-7y`
  - `<= 9` -> `8-9y`
  - `<= 15` -> `10-15y`
  - `<= 20` -> `16-20y`
  - `> 20` -> `21y+`
- Missing tenor -> `MISSING`

#### Coupon bucket (`coupon_bucket`)

- Parse `cpn` to numeric; unparseable -> missing
- Map:
  - `< 3.5` -> `3`
  - `3.5 <= c < 4.5` -> `4`
  - `4.5 <= c < 5.5` -> `5`
  - `5.5 <= c < 6.5` -> `6`
  - `>= 6.5` -> `7+`
- Missing coupon -> `MISSING`

#### Call bucket (`call_bucket`)

- `is_callable = callable in {Y, YES, TRUE, 1, T}` after trim + upper
- `call_days = (parsed nxt_call_dt - date).days`
- If not callable, missing `call_days`, or `call_days < 0` -> `No Call`
- Otherwise map:
  - `<= 90` -> `0-3m`
  - `<= 180` -> `3-6m`
  - `<= 270` -> `6-9m`
  - `<= 365` -> `9-12m`
  - `<= 730` -> `1y`
  - `<= 1095` -> `2y`
  - `<= 1460` -> `3y`
  - `<= 1825` -> `4y`
  - `<= 2555` -> `5-7y`
  - `<= 3285` -> `8-9y`
  - `> 3285` -> `10y`

#### State bucket (`state_bucket`)

- Normalize: strip + uppercase
- Map major states directly:
  - `CA` -> `CA`
  - `NY` -> `NY`
  - `TX` -> `TX`
  - `IL` -> `IL`
- All other values, including missing -> `Other States`

#### Rating bucket (`rating_grade`)

1. Normalize `final_comp_rating` and `comp_rating`: trim, uppercase, remove embedded spaces.
2. Use normalized `final_comp_rating` when available.
3. If `final_comp_rating` is missing, use normalized `comp_rating`.
4. If both are missing, assign `NR`.

### Modeling note

These bucket labels are later one-hot encoded, with one reference category dropped per characteristic family to avoid exact multicollinearity in the IPCA characteristic matrix.

In [ ]:
_bucket_cell_start = time.perf_counter()
log_runtime("START bucket derivation")

price_col = 'price'   # muni.ICEEval mean_evaluation_clean

df = data.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['trade_year'] = df['date'].dt.year


def _parse_mixed_datetime(s):
    """Parse a column that may hold epoch-millis strings and/or date strings."""
    as_num = pd.to_numeric(s, errors='coerce')
    out = pd.to_datetime(as_num, unit='ms', errors='coerce')   # epoch-ms path
    need = out.isna()
    if need.any():
        out.loc[need] = pd.to_datetime(
            s[need].replace('', np.nan), errors='coerce')       # date-string path
    return out

# ── TENOR ────────────────────────────────────────────────────────────────
_step_start = time.perf_counter()
maturity_dt = _parse_mixed_datetime(df['maturity'])
df['maturity_year'] = maturity_dt.dt.year
df['tenor_years'] = (df['maturity_year'] - df['trade_year']).clip(lower=1)


def tenor_bucket(y):
    if pd.isna(y):
        return 'MISSING'
    y = float(y)
    if y <= 1:  return '1y'
    if y <= 2:  return '2y'
    if y <= 3:  return '3y'
    if y <= 4:  return '4y'
    if y <= 7:  return '5-7y'
    if y <= 9:  return '8-9y'
    if y <= 15: return '10-15y'
    if y <= 20: return '16-20y'
    return '21y+'


df['tenor_bucket'] = df['tenor_years'].apply(tenor_bucket)
log_runtime("Derived tenor buckets", _step_start)

# ── COUPON ───────────────────────────────────────────────────────────────
_step_start = time.perf_counter()
df['coupon'] = pd.to_numeric(df['cpn'], errors='coerce')


def coupon_bucket(c):
    if pd.isna(c):
        return 'MISSING'
    c = float(c)
    if c < 3.5: return '3'
    if c < 4.5: return '4'
    if c < 5.5: return '5'
    if c < 6.5: return '6'
    return '7+'


df['coupon_bucket'] = df['coupon'].apply(coupon_bucket)
log_runtime("Derived coupon buckets", _step_start)

# ── CALL ─────────────────────────────────────────────────────────────────
_step_start = time.perf_counter()
df['is_callable'] = df['callable'].astype('string').str.strip().str.upper().isin(['Y', 'YES', 'TRUE', '1', 'T'])
next_call_dt = _parse_mixed_datetime(df['nxt_call_dt'])
df['call_days'] = (next_call_dt - df['date']).dt.days


def call_bucket(row):
    if not row['is_callable'] or pd.isna(row['call_days']) or row['call_days'] < 0:
        return 'No Call'
    d = row['call_days']
    if d <= 90:   return '0-3m'
    if d <= 180:  return '3-6m'
    if d <= 270:  return '6-9m'
    if d <= 365:  return '9-12m'
    if d <= 730:  return '1y'
    if d <= 1095: return '2y'
    if d <= 1460: return '3y'
    if d <= 1825: return '4y'
    if d <= 2555: return '5-7y'
    if d <= 3285: return '8-9y'
    return '10y'


df['call_bucket'] = df.apply(call_bucket, axis=1)
log_runtime("Derived call buckets", _step_start)

# ── STATE ────────────────────────────────────────────────────────────────
_step_start = time.perf_counter()


def state_bucket(s):
    if pd.isna(s):
        return 'Other States'
    s = str(s).strip().upper()
    return s if s in ('CA', 'NY', 'TX', 'IL') else 'Other States'


df['state_bucket'] = df['state_code'].apply(state_bucket)
log_runtime("Derived state buckets", _step_start)

# ── RATING (algo-track final composite rating with strategy fallback) ─────
_step_start = time.perf_counter()
MISSING_RATING_VALUES = ['', 'NAN', 'NONE', 'NULL', '<NA>']


def _clean_rating_bucket(col):
    out = df[col].astype('string').str.strip().str.upper().str.replace(' ', '', regex=False)
    return out.mask(out.isna() | out.isin(MISSING_RATING_VALUES))


final_rating = _clean_rating_bucket('final_comp_rating')
fallback_rating = _clean_rating_bucket('comp_rating')
df['rating_source'] = np.where(
    final_rating.notna(),
    'final_comp_rating',
    np.where(fallback_rating.notna(), 'comp_rating', 'missing'),
)
df['rating_grade'] = final_rating.fillna(fallback_rating).fillna('NR').astype(str)
log_runtime("Derived rating buckets from algo-track composite rating", _step_start)

print("Bucket value counts:")
for col in ['tenor_bucket', 'coupon_bucket', 'call_bucket', 'state_bucket', 'rating_grade']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

print("\nrating_source:")
print(df['rating_source'].value_counts(dropna=False))

log_runtime("END bucket derivation", _bucket_cell_start)

[2026-09-17 20:08:08] START bucket derivation
[2026-09-17 20:08:48] Derived tenor buckets | elapsed 29.91s
[2026-09-17 20:09:05] Derived coupon buckets | elapsed 17.12s
[2026-09-17 20:13:05] Derived call buckets | elapsed 4m 00.40s
[2026-09-17 20:13:17] Derived state buckets | elapsed 12.19s
[2026-09-17 20:13:32] Derived rating buckets from algo-track composite rating | elapsed 14.70s
Bucket value counts:

tenor_bucket:
tenor_bucket
10-15y     5902953
5-7y       5641538
1y         3191237
8-9y       3159612
2y         2432115
3y         2262112
4y         2241240
16-20y     1608786
21y+        318066
MISSING       3009
Name: count, dtype: int64

coupon_bucket:
coupon_bucket
5          17102142
4           6091106
3           3349925
6            162677
7+            51809
MISSING        3009
Name: count, dtype: int64

call_bucket:
call_bucket
No Call    10502374
8-9y        2724310
5-7y        2722632
1y          2502616
2y          1433001
3y          1360001
4y          1358562
9-12m

## 3. Return Computation

Daily **clean-price evaluated-mark** return:

$$r_{i,t} = \dfrac{P_{i,t}}{P_{i,t-1}} - 1$$

Scope caveat: this is not a total-return, excess-return, yield-change, duration-adjusted, or curve-hedged return. That is acceptable for evaluated-mark fair-value / de-smoothing diagnostics, but the results should not be described as broad muni return alpha without adding accrual, carry, duration, curve, and benchmark controls.

Missing returns are filled with **0** (no observed mark move = no clean-price return), not forward-filled, to avoid stale-price leakage. We also build an equal-weight muni market series for diagnostics.

In [ ]:
with timed_block("Return computation and equal-weight market series"):
    df = df.sort_values(["cusip", "date"]).copy()

    # Prior price per bond; coerce to plain float so downstream numpy math is safe.
    df["prev_price"] = df.groupby("cusip")[price_col].shift(1)
    df["prev_price"] = pd.to_numeric(df["prev_price"], errors="coerce").astype("float64")
    df["daily_return"] = pd.to_numeric(df[price_col], errors="coerce").astype("float64") / df["prev_price"] - 1

    # Treat divide-by-zero / non-positive prior prices as no observation so they
    # do not leak ±inf into the panel and corrupt panel-level metrics.
    df["daily_return"] = df["daily_return"].replace([np.inf, -np.inf], np.nan)
    df.loc[df["prev_price"] <= 0, "daily_return"] = np.nan

    df["daily_return_filled"] = df["daily_return"].fillna(0.0).astype("float64")

    # Equal-weight muni market return series (diagnostic)
    market_return_ew = (
        df.groupby("date")
          .agg(market_return_ew_real=("daily_return", "mean"),
               market_return_ew_filled=("daily_return_filled", "mean"),
               n_bonds_market=("cusip", "nunique"))
          .reset_index()
    )
    market_return_ew["market_return_ew"] = (
        market_return_ew["market_return_ew_real"].fillna(market_return_ew["market_return_ew_filled"])
    )
    df = df.merge(market_return_ew, on="date", how="left")

n_total = len(df)
n_null = int(df["daily_return"].isna().sum())
n_bad_prev = int((df["prev_price"] <= 0).sum())
_filled_np = df["daily_return_filled"].to_numpy(dtype="float64")
print(f"Total obs: {n_total:,}")
print(f"Real returns: {n_total - n_null:,} ({100*(n_total-n_null)/max(n_total,1):.1f}%)")
print(f"Zero-filled (null incl. bad prev price): {n_null:,} ({100*n_null/max(n_total,1):.1f}%)")
print(f"Non-positive prev_price rows: {n_bad_prev:,}")
print(f"Non-finite filled returns remaining: {int((~np.isfinite(_filled_np)).sum()):,}")
print("\nReturn stats (filled):")
print(df["daily_return_filled"].describe())

[2026-09-17 20:13:35] START Return computation and equal-weight market series


## 4. Static Characteristic Construction

One-hot encode each bucket group and drop a reference category per group (to stay full-rank against the `market_fv` intercept):

- Rating → drop `AAA` (highest quality reference)
- Coupon → drop `5` (modal muni coupon)
- Call → drop `No Call`
- State → drop `Other States`
- Tenor → drop `1y`

In [ ]:
with timed_block("Static characteristic one-hot construction"):
    def create_factor_onehot_encoding(frame: pd.DataFrame, factor_name: str) -> pd.DataFrame:
        """One-hot encode a categorical column; safe to rerun."""
        existing = [c for c in frame.columns if c.startswith(f'{factor_name}_')]
        if existing:
            frame = frame.drop(columns=existing)
        frame[factor_name] = frame[factor_name].astype("object").where(frame[factor_name].notna(), "MISSING")
        dummies = pd.get_dummies(frame[factor_name], prefix=factor_name).astype(float)
        return pd.concat([frame, dummies], axis=1)

    # ── 4a. Market intercept ────────────────────────────────────────────────
    df["market_fv"] = 1.0

    # ── 4b–4f. One-hot encode each bucket group ─────────────────────────────
    group_cols = {
        'rating_grade':  'AAA',            # (column, reference category to drop)
        'coupon_bucket': '5',
        'call_bucket':   'No Call',
        'state_bucket':  'Other States',
        'tenor_bucket':  '1y',
    }

    characteristics = ['market_fv']
    for col, ref in group_cols.items():
        factor_start = time.perf_counter()
        df = create_factor_onehot_encoding(df, col)
        dummy_cols = [c for c in df.columns if c.startswith(f'{col}_')]
        ref_col = f'{col}_{ref}'
        if ref_col in dummy_cols:
            dummy_cols = [c for c in dummy_cols if c != ref_col]
            print(f"{col}: dropped reference '{ref}' → {len(dummy_cols)} dummies")
        else:
            print(f"{col}: reference '{ref}' not present → {len(dummy_cols)} dummies")
        characteristics += dummy_cols
        log_runtime(f"Encoded {col}", factor_start)

print(f"\nTotal characteristics (L): {len(characteristics)}")
print(characteristics)

[2026-09-16 20:36:05] START Static characteristic one-hot construction
rating_grade: dropped reference 'AAA' → 17 dummies
[2026-09-16 20:36:08] Encoded rating_grade | elapsed 3.07s
coupon_bucket: dropped reference '5' → 5 dummies
[2026-09-16 20:36:09] Encoded coupon_bucket | elapsed 1.53s
call_bucket: dropped reference 'No Call' → 11 dummies
[2026-09-16 20:36:12] Encoded call_bucket | elapsed 2.96s
state_bucket: dropped reference 'Other States' → 3 dummies
[2026-09-16 20:36:15] Encoded state_bucket | elapsed 2.93s
tenor_bucket: dropped reference '1y' → 9 dummies
[2026-09-16 20:36:18] Encoded tenor_bucket | elapsed 2.79s
[2026-09-16 20:36:18] END Static characteristic one-hot construction | elapsed 13.31s

Total characteristics (L): 46
['market_fv', 'rating_grade_A', 'rating_grade_A+', 'rating_grade_A-', 'rating_grade_AA', 'rating_grade_AA+', 'rating_grade_AA-', 'rating_grade_B', 'rating_grade_B+', 'rating_grade_B-', 'rating_grade_BB', 'rating_grade_BB+', 'rating_grade_BB-', 'rating_gra

In [ ]:
# ── IPCA (Kelly, Pruitt & Su 2019) — inlined so the remote kernel can use it ──
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm


class IPCA(BaseEstimator, TransformerMixin):
    """Instrumented Principal Components Analysis with an unbalanced-panel mask."""

    def __init__(self, n_factors=1, max_iter=100, tol=1e-5):
        self.n_factors = n_factors
        self.max_iter = max_iter
        self.tol = tol
        self.Gamma = None
        self.Factors = None

    def fit(self, R, Z, obs_mask=None, gamma_init=None, verbose=True):
        fit_start = time.perf_counter()
        N, T = R.shape
        _, L, _ = Z.shape
        K = self.n_factors
        if verbose:
            print(f"[IPCA] START fit | K={K}, N={N:,}, T={T:,}, L={L:,}, max_iter={self.max_iter}")
        if obs_mask is None:
            obs_mask = np.ones((N, T))
        self._obs_mask = obs_mask
        if verbose:
            print(f"[IPCA] observed entries: {int(obs_mask.sum()):,} / {N*T:,} ({100*obs_mask.sum()/max(N*T,1):.1f}%)")

        if gamma_init is not None:
            # Warm start: reuse a supplied Gamma (e.g. previous CV fold) and skip SVD.
            Gamma_init = np.asarray(gamma_init, dtype=float)
            if Gamma_init.shape != (L, K):
                raise ValueError(f"gamma_init shape {Gamma_init.shape} does not match (L, K)=({L}, {K}).")
        else:
            managed_portfolios = np.zeros((L, T))
            for t in range(T):
                mask_t = obs_mask[:, t] > 0
                if mask_t.sum() == 0:
                    continue
                managed_portfolios[:, t] = Z[mask_t, :, t].T @ R[mask_t, t] / mask_t.sum()
            try:
                U, S, Vt = np.linalg.svd(managed_portfolios @ managed_portfolios.T)
                Gamma_init = U[:, :K]
            except np.linalg.LinAlgError:
                Gamma_init = np.random.randn(L, K)
        self.Gamma = self._orthogonalize_gamma(Gamma_init)

        loss_history = []
        iterator = tqdm(range(self.max_iter), desc="IPCA Iteration") if verbose else range(self.max_iter)
        for i in iterator:
            iter_start = time.perf_counter()

            # Step 1: Estimate Factors given Gamma
            F_new = np.zeros((T, K))
            for t in range(T):
                mask_t = obs_mask[:, t] > 0
                if mask_t.sum() < K:
                    continue
                Zt = Z[mask_t, :, t]
                beta_t = Zt @ self.Gamma
                BtB = beta_t.T @ beta_t
                BtR = beta_t.T @ R[mask_t, t]
                F_new[t, :] = np.linalg.solve(BtB + 1e-10 * np.eye(K), BtR)

            # Step 2: Estimate Gamma given Factors.
            # Vectorized normal equations via the Kronecker mixed-product identity:
            #   LHS += kron(f_t f_t', Z_t'Z_t)      RHS += kron(f_t, Z_t'R_t)
            # Numerically identical to the naive kron design but avoids the large
            # (n_obs x L*K) matrix, so it scales as O(n_obs*L^2) per date.
            LHS = np.zeros((L * K, L * K))
            RHS = np.zeros(L * K)
            for t in range(T):
                mask_t = obs_mask[:, t] > 0
                if mask_t.sum() == 0:
                    continue
                Zt = Z[mask_t, :, t]
                ft = F_new[t, :]
                ZtZ = Zt.T @ Zt
                ZtR = Zt.T @ R[mask_t, t]
                LHS += np.kron(np.outer(ft, ft), ZtZ)
                RHS += np.kron(ft, ZtR)
            gamma_vec = np.linalg.solve(LHS + 1e-10 * np.eye(L * K), RHS)
            Gamma_new = gamma_vec.reshape((L, K), order='F')

            Gamma_new_ortho = self._orthogonalize_gamma(Gamma_new)
            R_qr = Gamma_new_ortho.T @ Gamma_new
            self.Gamma = Gamma_new_ortho
            self.Factors = F_new @ R_qr.T

            loss = self._calculate_loss(R, Z)
            loss_history.append(loss)
            iter_elapsed = format_elapsed(time.perf_counter() - iter_start) if 'format_elapsed' in globals() else f"{time.perf_counter() - iter_start:.2f}s"
            if verbose and (i == 0 or (i + 1) % 5 == 0):
                print(f"[IPCA] iter {i+1:03d} | loss={loss:.6e} | iter_elapsed={iter_elapsed}")
            if i > 0:
                prev = loss_history[-2]
                rel_change = abs(prev - loss) / max(abs(prev), 1e-12)
                if rel_change < self.tol:
                    if verbose:
                        print(f"Converged at iteration {i+1} with loss {loss:.6e} "
                              f"(relative change {rel_change:.2e})")
                    break
        elapsed = format_elapsed(time.perf_counter() - fit_start) if 'format_elapsed' in globals() else f"{time.perf_counter() - fit_start:.2f}s"
        if verbose:
            print(f"[IPCA] END fit | K={K} | elapsed {elapsed}")
        return self

    def _orthogonalize_gamma(self, Gamma):
        Q, R = np.linalg.qr(Gamma)
        sign_diag = np.sign(np.diag(R))
        sign_diag[sign_diag == 0] = 1
        return Q @ np.diag(sign_diag)

    def _calculate_loss(self, R, Z):
        N, T = R.shape
        obs_mask = getattr(self, '_obs_mask', np.ones((N, T)))
        total_loss = 0
        n_obs = 0
        for t in range(T):
            mask_t = obs_mask[:, t] > 0
            if mask_t.sum() == 0:
                continue
            Zt = Z[mask_t, :, t]
            beta_t = Zt @ self.Gamma
            ft = self.Factors[t, :]
            residuals = R[mask_t, t] - beta_t @ ft
            total_loss += np.sum(residuals ** 2)
            n_obs += mask_t.sum()
        return total_loss / max(n_obs, 1)

    def predict_loadings(self, Z):
        if self.Gamma is None:
            raise ValueError("Model has not been fitted yet.")
        if Z.ndim == 3:
            return np.einsum('nlt,lk->nkt', Z, self.Gamma)
        elif Z.ndim == 2:
            return Z @ self.Gamma
        raise ValueError("Z must have 2 or 3 dimensions.")


## 5. Monthly Model Selection (Optional Weekly Walk-Forward)

This section separates **methodology selection** from an **optional production-style weekly scoring** robustness check. The **monthly expanding-window OOS sweep is the primary IPCA validation and `K`-selection evidence**; the weekly August walk-forward is a robustness appendix that is **off by default** (`RUN_WEEKLY_WALK_FORWARD = False`) so reruns do not retrain it.

### Efficient protocol

**Primary validation (always runs, cached):**

1. Start the research panel at `2026-04-01`.
2. Run **monthly** expanding validation folds to choose model settings, instead of re-searching every week.
3. Test candidate factor counts `K = 1, 2, 3, 4, 5` during monthly validation and select `K` from this evidence.

**Optional robustness — off by default (`RUN_WEEKLY_WALK_FORWARD = False`):**

4. Freeze the selected methodology for the test month.
5. During the test month, refit only the selected model each week using data available up to the prior week.
6. Score the next week and store the result.

The monthly recursive OOS sweep is the primary, paper-aligned backward-looking validation; the weekly layer only adds a production-style walk-forward check and is **skipped unless explicitly enabled**, so reruns avoid redundant retraining.

### Monthly selection folds

The selection grid uses lower-frequency validation origins:

| Selection fold | Training data | Validation block |
|---|---|---|
| 1 | Through April | First half of May |
| 2 | Through May | First half of June |
| 3 | Through June | First half of July |

`K`, complexity penalties, and other fixed settings are selected from these historical validation folds only. **August is not used to redesign the methodology.**

### Weekly August test (optional robustness — off by default)

After selecting `K`, the test month can optionally be evaluated in a walk-forward simulation. This layer is **not required** for the IPCA research claim — the monthly recursive OOS above already provides no-look-ahead validation. Enable it only to regenerate the August evidence by setting `RUN_WEEKLY_WALK_FORWARD = True`:

| Test week | Training data | Scored block |
|---|---|---|
| Week 1 | Through July 31 | First August week |
| Week 2 | Through prior August week | Next August week |
| Week 3 | Through prior August week | Next August week |
| Week 4 | Through prior August week | Next August week |
| Week 5 | Through prior August week | 2026-08-31 (final partial week) |

Each week retrains IPCA using all information available at that time, but only for the **selected** model. Candidate `K` values are not re-evaluated weekly during August.

### Fold cache

Completed folds are cached to `ipca_fold_cache.parquet`. The cache key includes the fold type, candidate `K`, train/test dates, characteristics, panel signature, and fit parameters. Cached folds are reused across notebook reruns unless the panel, feature set, or implementation version changes. If the kernel does not have a parquet engine available, the code falls back to `ipca_fold_cache.pkl` so reruns still avoid completed folds.

### Metrics

All R² metrics use the **uncentered** IPCA denominator, i.e. the zero-return benchmark used in the credit notebook and the IPCA paper-style diagnostics:

| Metric | Definition | Interpretation |
|---|---|---|
| **Total R²** | $1 - \frac{\sum_{i,t}(r_{i,t+1}-\hat r_{i,t+1})^2}{\sum_{i,t}r_{i,t+1}^2}$ | Overall panel reconstruction (realized factor) |
| **Predictive R²** | Same form with $\hat f_{t+1}$ replaced by the train-window mean factor $\bar f$ | Could the model forecast *before* seeing the next cross section? |
| **Time Series R²** | Median of per-bond uncentered R² values | Robust single-name time-series fit |
| **Cross Section R²** | Date-average of cross-sectional uncentered R² | Daily relative-value dispersion fit |
| **Relative Pricing Error** | $\frac{\sum_i \alpha_i^2}{\sum_i \bar r_i^2}$, $\alpha_i=\overline{r_i-\hat r_i}$ | Average pricing error; lower is better |
| **$\Gamma$ instability** | $1-\overline{\cos\theta}$ of principal angles between consecutive folds' $\Gamma$ | Subspace drift diagnostic; lower is more stable |

### Choosing $K$

Consistent with the paper's parsimony emphasis (Sections 4.1–4.2), the factor count is a **parsimony** decision read off the whole validation evidence panel — fit (Total / Time-Series / Cross-Section R²), pricing error (RPE), predictive power, and $\Gamma$ stability — **not** a maximisation of incremental cross-sectional fit. The specific tolerance rule below is our own defensible research control, not a requirement from the paper.

The rule is deliberately simple: report the panel by $K$ and take the **smallest $K$ within `SELECTION_TOL = 0.005` of the best validation Total R²** (diminishing returns):

$$K^* = \min\left\{K:\; R^2_{\text{total}}(K) \geq \max_{K'} R^2_{\text{total}}(K') - \delta\right\},\qquad \delta = 0.005 .$$

One metric, one tolerance — no multi-screen thresholds or fallback relaxations. RPE, Predictive R², and $\Gamma$ instability are shown alongside as corroborating evidence. EWMA / weighted-expanding estimation is intentionally deferred for now.

In [ ]:
# ── PANEL CONSTRUCTION ──────────────────────────────────────────────────
_section5_start = time.perf_counter()
log_runtime("START Section 5: monthly selection + weekly walk-forward test")
_panel_start = time.perf_counter()

from pathlib import Path
import json
import pickle

return_col = 'daily_return_filled'
asset_id   = 'cusip'
time_id    = 'date'
market_factor_cols = [c for c in ['market_return_ew', 'market_return_ew_real',
                                  'market_return_ew_filled', 'n_bonds_market']
                      if c in df.columns]

keep_cols = list(dict.fromkeys([asset_id, time_id, return_col] + characteristics + market_factor_cols))
df_model = df[[c for c in keep_cols if c in df.columns]].copy()
df_model[time_id] = pd.to_datetime(df_model[time_id])
df_model = df_model[df_model[time_id] >= pd.Timestamp('2026-04-01')].copy()

min_obs = 10
counts = df_model.groupby(asset_id).size()
df_model = df_model[df_model[asset_id].isin(counts[counts >= min_obs].index)].copy()
print(f"Bonds (>= {min_obs} obs): {df_model[asset_id].nunique()}  |  obs: {len(df_model):,}")

# Lag all characteristics by one observed row per bond.
df_model.sort_values([asset_id, time_id], inplace=True)
df_model[characteristics] = df_model.groupby(asset_id)[characteristics].shift(1)
n_before = len(df_model)
df_model.dropna(subset=characteristics, inplace=True)
print(f"After lagging z by 1 row per bond: {n_before:,} → {len(df_model):,}")

_pivot_start = time.perf_counter()
log_runtime("START panel pivot to R/Z arrays")
R_panel = df_model.pivot(index=asset_id, columns=time_id, values=return_col)
Z_panel = {c: df_model.pivot(index=asset_id, columns=time_id, values=c) for c in characteristics}
obs_mask = R_panel.notna().values.astype(np.float32)
R = np.nan_to_num(R_panel.values).astype(np.float32)
Z = np.nan_to_num(np.array([Z_panel[c].values for c in characteristics], dtype=np.float32).transpose(1, 0, 2))
log_runtime("END panel pivot to R/Z arrays", _pivot_start)

N, T = R.shape
L = Z.shape[1]
panel_dates = pd.DatetimeIndex(pd.to_datetime(R_panel.columns))
print(f"R: ({N}, {T})  |  Z: ({N}, {L}, {T})")
print(f"Date range: {panel_dates.min().date()} → {panel_dates.max().date()}")
print(f"Fill rate: {100*obs_mask.sum()/(N*T):.1f}%")
log_runtime("END model panel construction", _panel_start)

# ── PAPER-STYLE PERFORMANCE METRICS (UNCENTERED DENOMINATORS) ──────────
METRIC_ORDER = ['Total R2', 'Time Series R2', 'Cross Section R2', 'Relative Pricing Error']
ALL_FIT_METRICS = METRIC_ORDER + ['Predictive R2']


def predict_panel_returns(Gamma, Factors, Z_slice, mask_slice):
    """Return fitted returns using fixed Gamma and a T x K factor series."""
    n_assets, _, n_dates = Z_slice.shape
    R_hat = np.zeros((n_assets, n_dates), dtype=np.float64)
    for t in range(n_dates):
        observed = mask_slice[:, t] > 0
        if observed.sum() == 0:
            continue
        beta_t = Z_slice[observed, :, t] @ Gamma
        R_hat[observed, t] = beta_t @ Factors[t, :]
    return R_hat


def estimate_factors_given_gamma(Gamma, R_slice, Z_slice, mask_slice, ridge=1e-10):
    """Estimate test-period factor realizations with train-fitted Gamma held fixed."""
    n_dates = R_slice.shape[1]
    n_factors = Gamma.shape[1]
    factors = np.zeros((n_dates, n_factors), dtype=np.float64)
    for t in range(n_dates):
        observed = mask_slice[:, t] > 0
        if observed.sum() < n_factors:
            continue
        beta_t = Z_slice[observed, :, t] @ Gamma
        lhs = beta_t.T @ beta_t
        rhs = beta_t.T @ R_slice[observed, t]
        factors[t, :] = np.linalg.solve(lhs + ridge * np.eye(n_factors), rhs)
    return factors


def total_r2_panel(R_true, R_hat, mask):
    # Ignore non-finite fitted/actual values so a stray inf/nan cannot collapse
    # the whole panel metric to NaN (matches the nan-skipping cross-section/TS R2).
    valid = (mask > 0) & np.isfinite(R_true) & np.isfinite(R_hat)
    if not np.any(valid):
        return np.nan
    resid = R_true[valid] - R_hat[valid]
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum(R_true[valid] ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan


def median_time_series_r2(R_true, R_hat, mask):
    values = []
    for i in range(R_true.shape[0]):
        observed = mask[i, :] > 0
        if not observed.any():
            continue
        ss_tot_i = np.sum(R_true[i, observed] ** 2)
        if ss_tot_i <= 0:
            continue
        ss_res_i = np.sum((R_true[i, observed] - R_hat[i, observed]) ** 2)
        values.append(1 - ss_res_i / ss_tot_i)
    return np.nanmedian(values) if values else np.nan


def cross_section_r2(R_true, R_hat, mask):
    rows = []
    for t in range(R_true.shape[1]):
        observed = mask[:, t] > 0
        if observed.sum() == 0:
            continue
        ss_tot_t = np.sum(R_true[observed, t] ** 2)
        if ss_tot_t <= 0:
            continue
        ss_res_t = np.sum((R_true[observed, t] - R_hat[observed, t]) ** 2)
        rows.append(1 - ss_res_t / ss_tot_t)
    return np.nanmean(rows) if rows else np.nan


def relative_pricing_error(R_true, R_hat, mask):
    alphas = []
    avg_returns = []
    for i in range(R_true.shape[0]):
        observed = mask[i, :] > 0
        if observed.sum() == 0:
            continue
        alphas.append(np.mean(R_true[i, observed] - R_hat[i, observed]))
        avg_returns.append(np.mean(R_true[i, observed]))
    alphas = np.asarray(alphas, dtype=float)
    avg_returns = np.asarray(avg_returns, dtype=float)
    denom = np.sum(avg_returns ** 2)
    return np.sum(alphas ** 2) / denom if denom > 0 else np.nan


def ipca_performance_metrics(R_true, R_hat, mask):
    return {
        'Total R2': total_r2_panel(R_true, R_hat, mask),
        'Time Series R2': median_time_series_r2(R_true, R_hat, mask),
        'Cross Section R2': cross_section_r2(R_true, R_hat, mask),
        'Relative Pricing Error': relative_pricing_error(R_true, R_hat, mask),
    }


def gamma_subspace_instability(G_prev, G_curr):
    """Subspace drift between two orthonormal Gamma matrices; lower is more stable."""
    if G_prev is None or G_curr is None or G_prev.shape != G_curr.shape:
        return np.nan
    s = np.linalg.svd(G_prev.T @ G_curr, compute_uv=False)
    s = np.clip(s, -1.0, 1.0)
    return float(1.0 - np.mean(s))


# ── FOLD BUILDERS ───────────────────────────────────────────────────────
def make_monthly_selection_folds(dates, train_start='2026-04-01',
                                 first_validation_start='2026-05-01',
                                 validation_days=15, stop_before='2026-08-01'):
    """Lower-frequency validation origins for model-selection CV."""
    dates = pd.DatetimeIndex(pd.to_datetime(dates))
    train_start = pd.Timestamp(train_start)
    validation_start = pd.Timestamp(first_validation_start)
    stop_before = pd.Timestamp(stop_before)
    max_date = dates.max()
    folds = []
    fold = 1
    while validation_start < stop_before and validation_start <= max_date:
        validation_end = min(validation_start + pd.Timedelta(days=validation_days), stop_before)
        train_cols = np.where((dates >= train_start) & (dates < validation_start))[0]
        test_cols = np.where((dates >= validation_start) & (dates < validation_end))[0]
        if len(train_cols) > 0 and len(test_cols) > 0:
            folds.append({
                'fold': fold,
                'fold_kind': 'monthly_selection',
                'train_cols': train_cols,
                'test_cols': test_cols,
                'train_start': dates[train_cols[0]],
                'train_end': dates[train_cols[-1]],
                'test_start': dates[test_cols[0]],
                'test_end': dates[test_cols[-1]],
            })
            fold += 1
        validation_start = validation_start + pd.DateOffset(months=1)
    return folds


def make_weekly_walk_forward_folds(dates, train_start='2026-04-01',
                                   first_test_start='2026-08-01',
                                   stop_before='2026-09-01', test_days=7):
    """Weekly production-style test folds; train expands through the day before each test week."""
    dates = pd.DatetimeIndex(pd.to_datetime(dates))
    train_start = pd.Timestamp(train_start)
    week_start = pd.Timestamp(first_test_start)
    stop_before = pd.Timestamp(stop_before)
    max_date = min(dates.max(), stop_before - pd.Timedelta(days=1))
    folds = []
    fold = 1
    while week_start <= max_date:
        week_end = min(week_start + pd.Timedelta(days=test_days), stop_before)
        train_cols = np.where((dates >= train_start) & (dates < week_start))[0]
        test_cols = np.where((dates >= week_start) & (dates < week_end))[0]
        if len(train_cols) > 0 and len(test_cols) > 0:
            folds.append({
                'fold': fold,
                'fold_kind': 'weekly_test',
                'train_cols': train_cols,
                'test_cols': test_cols,
                'train_start': dates[train_cols[0]],
                'train_end': dates[train_cols[-1]],
                'test_start': dates[test_cols[0]],
                'test_end': dates[test_cols[-1]],
            })
            fold += 1
        week_start = week_end
    return folds


# ── PERSISTED FOLD CACHE ────────────────────────────────────────────────
BUCKET_RULE_VERSION = 'muni_bucket_rules_v1_algo_rating_direct_coupon_tenor_call_state'
FOLD_CACHE_VERSION = 'ipca_monthly_selection_weekly_test_v3_finite_total_r2'
FOLD_CACHE_PATH = Path('ipca_fold_cache.parquet')
FOLD_CACHE_FALLBACK_PATH = Path('ipca_fold_cache.pkl')


def _panel_signature():
    r_obs = R * obs_mask
    return {
        'min_date': str(panel_dates.min().date()),
        'max_date': str(panel_dates.max().date()),
        'n_assets': int(N),
        'n_dates': int(T),
        'n_characteristics': int(L),
        'observed_count': int(obs_mask.sum()),
        'return_sum': float(np.round(np.sum(r_obs), 12)),
        'return_ss': float(np.round(np.sum(r_obs ** 2), 12)),
        'characteristics': list(characteristics),
        'bucket_rule_version': BUCKET_RULE_VERSION,
        'return_col': return_col,
    }


PANEL_SIGNATURE = _panel_signature()


def _json_metric_value(value):
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.datetime64):
        return pd.Timestamp(value).isoformat()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        value = float(value)
        return value if np.isfinite(value) else None
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    if isinstance(value, np.bool_):
        return bool(value)
    if value is None or isinstance(value, (str, int, bool)):
        return value
    return str(value)


def serialize_metrics(metrics):
    return {key: _json_metric_value(value) for key, value in metrics.items()}


def deserialize_metrics(metrics):
    metrics = dict(metrics)
    for col in ['train_start', 'train_end', 'test_start', 'test_end']:
        if metrics.get(col) is not None:
            metrics[col] = pd.Timestamp(metrics[col])
    return metrics


def fold_cache_key(fold_kind, fold, K, max_iter, tol):
    payload = {
        'version': FOLD_CACHE_VERSION,
        'fold_kind': fold_kind,
        'K': int(K),
        'train_start': str(pd.Timestamp(fold['train_start']).date()),
        'train_end': str(pd.Timestamp(fold['train_end']).date()),
        'test_start': str(pd.Timestamp(fold['test_start']).date()),
        'test_end': str(pd.Timestamp(fold['test_end']).date()),
        'max_iter': int(max_iter),
        'tol': float(tol),
        'panel_signature': PANEL_SIGNATURE,
    }
    return json.dumps(payload, sort_keys=True)


def _load_pickle_cache(path=FOLD_CACHE_FALLBACK_PATH):
    if not path.exists():
        return {}
    try:
        with path.open('rb') as fh:
            cache = pickle.load(fh)
        if isinstance(cache, dict):
            print(f"Loaded fallback fold cache: {len(cache):,} entries from {path}")
            return cache
    except Exception as exc:
        print(f"WARNING: could not load fallback fold cache {path}: {exc}")
    return {}


def load_fold_cache(path=FOLD_CACHE_PATH):
    if not path.exists():
        return _load_pickle_cache()
    try:
        cache_df = pd.read_parquet(path)
        cache = {}
        for row in cache_df.to_dict('records'):
            gamma_text = row.get('gamma_json')
            gamma = None
            if gamma_text is not None and not pd.isna(gamma_text):
                gamma = np.asarray(json.loads(gamma_text), dtype=float)
            cache[row['cache_key']] = {
                'metrics': deserialize_metrics(json.loads(row['metrics_json'])),
                'Gamma': gamma,
            }
        print(f"Loaded fold cache: {len(cache):,} entries from {path}")
        return cache
    except Exception as exc:
        print(f"WARNING: could not load parquet fold cache {path}: {exc}")
        return _load_pickle_cache()


def save_fold_cache(cache, path=FOLD_CACHE_PATH):
    rows = []
    for key, entry in cache.items():
        gamma = entry.get('Gamma')
        rows.append({
            'cache_key': str(key),
            'metrics_json': json.dumps(serialize_metrics(entry['metrics']), sort_keys=True),
            'gamma_json': json.dumps(np.asarray(gamma, dtype=float).tolist()) if gamma is not None else None,
        })
    cache_df = pd.DataFrame(rows)
    try:
        tmp_path = path.with_name(path.name + '.tmp')
        cache_df.to_parquet(tmp_path, index=False)
        tmp_path.replace(path)
        print(f"Saved fold cache: {len(cache):,} entries to {path}")
    except Exception as exc:
        print(f"WARNING: could not save parquet fold cache {path}: {exc}")
        with FOLD_CACHE_FALLBACK_PATH.open('wb') as fh:
            pickle.dump(cache, fh, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Saved fallback fold cache: {len(cache):,} entries to {FOLD_CACHE_FALLBACK_PATH}")


fold_cache = load_fold_cache()


# ── FIT / SCORE HELPERS ─────────────────────────────────────────────────
def fit_and_score_fold(R, Z, obs_mask, fold, K, max_iter=50, tol=1e-5,
                       gamma_init=None, verbose=False):
    fold_start = time.perf_counter()
    tr, te = fold['train_cols'], fold['test_cols']
    R_tr, Z_tr, m_tr = R[:, tr], Z[:, :, tr], obs_mask[:, tr]
    R_te, Z_te, m_te = R[:, te], Z[:, :, te], obs_mask[:, te]

    if verbose:
        print(
            f"[FIT] START {fold['fold_kind']} K={K} fold={fold['fold']:02d} | "
            f"train {fold['train_start'].date()}..{fold['train_end'].date()} "
            f"({len(tr)} dates, {int(m_tr.sum()):,} obs) | "
            f"test {fold['test_start'].date()}..{fold['test_end'].date()} "
            f"({len(te)} dates, {int(m_te.sum()):,} obs)"
        )

    fit_start = time.perf_counter()
    model = IPCA(n_factors=K, max_iter=max_iter, tol=tol)
    model.fit(R_tr, Z_tr, obs_mask=m_tr, gamma_init=gamma_init, verbose=False)
    fit_elapsed = time.perf_counter() - fit_start

    factor_start = time.perf_counter()
    F_te = estimate_factors_given_gamma(model.Gamma, R_te, Z_te, m_te)
    R_hat_te = predict_panel_returns(model.Gamma, F_te, Z_te, m_te)
    f_bar = model.Factors.mean(axis=0) if model.Factors is not None else np.zeros(K)
    F_pred = np.tile(f_bar, (len(te), 1))
    R_hat_pred = predict_panel_returns(model.Gamma, F_pred, Z_te, m_te)
    factor_elapsed = time.perf_counter() - factor_start

    score_start = time.perf_counter()
    metrics = ipca_performance_metrics(R_te, R_hat_te, m_te)
    metrics['Predictive R2'] = total_r2_panel(R_te, R_hat_pred, m_te)
    score_elapsed = time.perf_counter() - score_start
    fold_elapsed = time.perf_counter() - fold_start

    metrics.update({
        'fold_kind': fold['fold_kind'],
        'K': K,
        'fold': fold['fold'],
        'train_start': fold['train_start'],
        'train_end': fold['train_end'],
        'test_start': fold['test_start'],
        'test_end': fold['test_end'],
        'train_dates': len(tr),
        'test_dates': len(te),
        'train_obs': int(m_tr.sum()),
        'test_obs': int(m_te.sum()),
        'fit_seconds': fit_elapsed,
        'factor_score_seconds': factor_elapsed,
        'metric_seconds': score_elapsed,
        'fold_seconds': fold_elapsed,
    })
    return metrics, model.Gamma


def fit_and_score_cached(R, Z, obs_mask, fold, K, max_iter=50, tol=1e-5,
                         gamma_init=None, use_cache=True):
    key = fold_cache_key(fold['fold_kind'], fold, K, max_iter, tol)
    if use_cache and key in fold_cache:
        entry = fold_cache[key]
        metrics = dict(entry['metrics'])
        gamma = np.asarray(entry['Gamma'], dtype=float) if entry.get('Gamma') is not None else None
        metrics['cache_hit'] = True
        return metrics, gamma

    metrics, gamma = fit_and_score_fold(
        R, Z, obs_mask, fold, K, max_iter=max_iter, tol=tol,
        gamma_init=gamma_init, verbose=False,
    )
    metrics['cache_hit'] = False
    if use_cache:
        fold_cache[key] = {
            'metrics': dict(metrics),
            'Gamma': np.asarray(gamma, dtype=float),
        }
    return metrics, gamma


def run_fold_sequence(R, Z, obs_mask, folds, K, max_iter=50, tol=1e-5,
                      warm_start=True, use_cache=True):
    rows = []
    prev_gamma = None
    for fold in folds:
        gamma_init = prev_gamma if warm_start else None
        metrics, gamma = fit_and_score_cached(
            R, Z, obs_mask, fold, K, max_iter=max_iter, tol=tol,
            gamma_init=gamma_init, use_cache=use_cache,
        )
        metrics['gamma_instability'] = gamma_subspace_instability(prev_gamma, gamma)
        rows.append(metrics)
        prev_gamma = gamma if gamma is not None else None
        hit = 'hit' if metrics.get('cache_hit') else 'miss'
        print(f"  {fold['fold_kind']} K={K} fold={fold['fold']:02d} "
              f"test={metrics['test_start'].date()}..{metrics['test_end'].date()} "
              f"TotalR2={metrics['Total R2']:.4f} "
              f"CSR2={metrics['Cross Section R2']:.4f} "
              f"RPE={metrics['Relative Pricing Error']:.4f} "
              f"dGamma={metrics['gamma_instability']:.4f} "
              f"cache={hit} elapsed={format_elapsed(metrics['fold_seconds'])}")
    return rows


# ── MONTHLY MODEL SELECTION ─────────────────────────────────────────────

# Clear legacy K-selection globals from older notebook runs.
_stale_k_selection_names = ['BASELINE_K', 'PROMOTION_MARGIN', 'LAMBDA_RPE', 'LAMBDA_INSTABILITY', 'LAMBDA_COMPLEXITY', 'score_df', 'best_by_score', 'baseline_score', 'challenger_score']
for _stale_name in _stale_k_selection_names:
    globals().pop(_stale_name, None)
del _stale_k_selection_names, _stale_name

MAX_K = 5
CANDIDATE_K = tuple(range(1, MAX_K + 1))
CV_MAX_ITER = 50
TRAIN_START = '2026-04-01'
FINAL_TEST_START = '2026-08-01'
FINAL_TEST_STOP_BEFORE = '2026-09-01'
MONTHLY_VALIDATION_DAYS = 15
SELECTION_TOL = 0.005

# Set True to force a full K-selection resweep even if the cached methodology
# signature below still matches. A genuine methodology change (characteristics,
# K grid, windows, tolerance, fold-cache version) auto-triggers a resweep anyway;
# this flag is only for manually forcing a rerun on an unchanged methodology.
RETRAIN_K_SELECTION = False
SELECTION_SUMMARY_CACHE_PATH = Path('ipca_selection_summary_cache.pkl')


def _selection_methodology_signature():
    """Signature of settings that should force a K-selection resweep.

    Deliberately excludes panel date range / observed-count / return-sum
    (PANEL_SIGNATURE) so a daily incremental data pull does not by itself
    invalidate the monthly selection sweep; only a methodology change does.
    """
    return {
        'characteristics': list(characteristics),
        'bucket_rule_version': BUCKET_RULE_VERSION,
        'fold_cache_version': FOLD_CACHE_VERSION,
        'candidate_K': list(CANDIDATE_K),
        'cv_max_iter': int(CV_MAX_ITER),
        'train_start': str(TRAIN_START),
        'monthly_validation_days': int(MONTHLY_VALIDATION_DAYS),
        'final_test_start': str(FINAL_TEST_START),
        'selection_tol': float(SELECTION_TOL),
    }


def load_selection_summary_cache(path=SELECTION_SUMMARY_CACHE_PATH):
    if not path.exists():
        return None
    try:
        with path.open('rb') as fh:
            entry = pickle.load(fh)
        if isinstance(entry, dict) and 'signature' in entry:
            return entry
    except Exception as exc:
        print(f"WARNING: could not load selection summary cache {path}: {exc}")
    return None


def save_selection_summary_cache(signature, selected_K, monthly_selection_folds_df,
                                 path=SELECTION_SUMMARY_CACHE_PATH):
    entry = {
        'signature': signature,
        'selected_K': int(selected_K),
        'monthly_selection_folds_df': monthly_selection_folds_df,
    }
    tmp_path = path.with_name(path.name + '.tmp')
    with tmp_path.open('wb') as fh:
        pickle.dump(entry, fh, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_path.replace(path)
    print(f"Saved K-selection summary cache to {path}")


_selection_start = time.perf_counter()
selection_folds = make_monthly_selection_folds(
    panel_dates,
    train_start=TRAIN_START,
    first_validation_start='2026-05-01',
    validation_days=MONTHLY_VALIDATION_DAYS,
    stop_before=FINAL_TEST_START,
)
print(f"Monthly selection folds: {len(selection_folds)}")
if selection_folds:
    print(f"First selection fold: train {selection_folds[0]['train_start'].date()} → {selection_folds[0]['train_end'].date()}, "
          f"validate {selection_folds[0]['test_start'].date()} → {selection_folds[0]['test_end'].date()}")
    print(f"Last selection fold: train {selection_folds[-1]['train_start'].date()} → {selection_folds[-1]['train_end'].date()}, "
          f"validate {selection_folds[-1]['test_start'].date()} → {selection_folds[-1]['test_end'].date()}")
else:
    raise ValueError("No monthly selection folds available. Check FINAL_TEST_START and panel dates.")

_selection_signature = _selection_methodology_signature()
_cached_selection = load_selection_summary_cache()
_selection_cache_hit = (
    not RETRAIN_K_SELECTION
    and _cached_selection is not None
    and _cached_selection.get('signature') == _selection_signature
)

if _selection_cache_hit:
    print("K-selection sweep: SKIPPED (cached, methodology unchanged)")
    selected_K = int(_cached_selection['selected_K'])
    monthly_selection_folds_df = _cached_selection['monthly_selection_folds_df']
    print(f"Loaded cached selected_K={selected_K} from {SELECTION_SUMMARY_CACHE_PATH}")
else:
    _retrain_reason = ('forced (RETRAIN_K_SELECTION=True)' if RETRAIN_K_SELECTION
                       else ('no cached summary' if _cached_selection is None
                             else 'methodology signature changed'))
    print(f"K-selection sweep: RUNNING ({_retrain_reason})")

    from joblib import Parallel, delayed

    def _run_k_selection(K):
        k_start = time.perf_counter()
        print(f"\n[MONTHLY SELECTION] START K={K}")
        rows = run_fold_sequence(
            R, Z, obs_mask, selection_folds, K,
            max_iter=CV_MAX_ITER, tol=1e-5, warm_start=True, use_cache=True,
        )
        print(f"[MONTHLY SELECTION] END K={K} | elapsed {format_elapsed(time.perf_counter() - k_start)}")
        return rows

    # Independent across K (folds *within* a K stay sequential for warm-start);
    # threads suffice since the IPCA fit's hot path is numpy linalg (releases the GIL).
    _k_selection_results = Parallel(n_jobs=min(len(CANDIDATE_K), 5), prefer='threads')(
        delayed(_run_k_selection)(K) for K in CANDIDATE_K
    )
    selection_rows = [row for rows in _k_selection_results for row in rows]
    monthly_selection_folds_df = pd.DataFrame(selection_rows)

    # ── K SELECTION: PARSIMONY OVER THE VALIDATION FIT PANEL ─────────────
    # Following the paper, the factor count is a parsimony call read off the
    # whole evidence panel (fit, pricing error, predictive power, Gamma
    # stability), not a maximization of incremental cross-sectional fit.
    # Rule: smallest K whose validation Total R2 is within SELECTION_TOL of
    # the best (diminishing returns).
    _selection_mean = monthly_selection_folds_df.groupby('K')[METRIC_ORDER].mean()
    _best_total = _selection_mean['Total R2'].max()
    eligible_K = [int(k) for k in _selection_mean.index
                  if _selection_mean.loc[k, 'Total R2'] >= _best_total - SELECTION_TOL]
    selected_K = min(eligible_K)

    save_selection_summary_cache(_selection_signature, selected_K, monthly_selection_folds_df)

val_folds = monthly_selection_folds_df
cv_summary = monthly_selection_folds_df.groupby('K')[METRIC_ORDER].agg(['mean', 'std'])
summary_mean = monthly_selection_folds_df.groupby('K')[METRIC_ORDER].mean()
summary_std = monthly_selection_folds_df.groupby('K')['Total R2'].std().rename('Total R2 std')
extra_mean = monthly_selection_folds_df.groupby('K')[['Predictive R2', 'gamma_instability']].mean()
selection_table = summary_mean.join(summary_std).join(extra_mean)
_best_total = selection_table['Total R2'].max()
selection_table['selected'] = selection_table.index.astype(int) == selected_K

selection_format = {col: '{:.4f}' for col in [
    'Total R2', 'Time Series R2', 'Cross Section R2', 'Relative Pricing Error',
    'Total R2 std', 'Predictive R2', 'gamma_instability'] if col in selection_table.columns}
print("Monthly validation metrics by K (evidence panel; RPE lower is better):")
display(selection_table.style.format(selection_format))
print(f"Selected K = {selected_K}: smallest K within {SELECTION_TOL:.3f} of the best validation "
      f"Total R2 ({_best_total:.4f}); factors beyond K={selected_K} add < {SELECTION_TOL:.3f} to Total R2.")
log_runtime("END monthly model-selection sweep", _selection_start)

# ── WEEKLY WALK-FORWARD FINAL TEST ──────────────────────────────────────
# Weekly August walk-forward is OPTIONAL robustness (production-style refit each
# week). Its fold cache is keyed by the panel signature, so it re-trains on every
# data change. The monthly expanding OOS above is the primary IPCA validation, so
# this is OFF by default; set True to regenerate the August evidence (it also seeds
# the Section 18 attention query with the first weekly-fold Gamma).
RUN_WEEKLY_WALK_FORWARD = False

_test_start = time.perf_counter()
weekly_folds = make_weekly_walk_forward_folds(
    panel_dates,
    train_start=TRAIN_START,
    first_test_start=FINAL_TEST_START,
    stop_before=FINAL_TEST_STOP_BEFORE,
    test_days=7,
)
print(f"\nWeekly final-test folds from {FINAL_TEST_START} to before {FINAL_TEST_STOP_BEFORE}: {len(weekly_folds)}")
if weekly_folds:
    print(f"First test fold: train {weekly_folds[0]['train_start'].date()} → {weekly_folds[0]['train_end'].date()}, "
          f"test {weekly_folds[0]['test_start'].date()} → {weekly_folds[0]['test_end'].date()}")
    print(f"Last test fold: train {weekly_folds[-1]['train_start'].date()} → {weekly_folds[-1]['train_end'].date()}, "
          f"test {weekly_folds[-1]['test_start'].date()} → {weekly_folds[-1]['test_end'].date()}")

if RUN_WEEKLY_WALK_FORWARD and weekly_folds:
    weekly_test_rows = run_fold_sequence(
        R, Z, obs_mask, weekly_folds, selected_K,
        max_iter=CV_MAX_ITER, tol=1e-5, warm_start=True, use_cache=True,
    )
else:
    if weekly_folds:
        print("Weekly walk-forward: SKIPPED (RUN_WEEKLY_WALK_FORWARD=False; optional robustness). "
              "Set RUN_WEEKLY_WALK_FORWARD=True to regenerate the August weekly evidence.")
    weekly_test_rows = []
weekly_test_folds_df = pd.DataFrame(weekly_test_rows)
test_folds = weekly_test_folds_df
log_runtime("END weekly final-test scoring", _test_start)

# Keep the names expected by the review snapshot / downstream cells.
cv_folds = pd.concat([monthly_selection_folds_df, weekly_test_folds_df], ignore_index=True)
runtime_summary = cv_folds.groupby(['fold_kind', 'K'])[['fit_seconds', 'factor_score_seconds', 'metric_seconds', 'fold_seconds']].agg(['mean', 'sum'])
cache_summary = cv_folds.groupby(['fold_kind', 'cache_hit']).size().rename('fold_count').reset_index()
print("\nRuntime summary by fold kind and K:")
display(runtime_summary.style.format('{:.2f}'))
print("\nFold cache summary:")
display(cache_summary)

if not weekly_test_folds_df.empty:
    test_report = weekly_test_folds_df[ALL_FIT_METRICS].mean()
    print(f"\n[FINAL TEST] August walk-forward, selected K={selected_K}, "
          f"{len(weekly_test_folds_df)} week(s), methodology frozen:")
    for name, val in test_report.items():
        print(f"    {name:<24s} {val:.4f}")
else:
    print("\n[FINAL TEST] No August weekly test folds available yet.")

fold_total = weekly_test_folds_df.pivot_table(index='test_start', columns='K', values='Total R2') if not weekly_test_folds_df.empty else pd.DataFrame()
print("\nWeekly final-test Total R2 by selected K:")
display(fold_total.style.format('{:.4f}') if not fold_total.empty else fold_total)

# ── PLOTS ───────────────────────────────────────────────────────────────
_plot_start = time.perf_counter()
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, metric in zip(axes.ravel(), METRIC_ORDER):
    for K in CANDIDATE_K:
        sub = monthly_selection_folds_df[monthly_selection_folds_df['K'] == K].sort_values('test_start')
        ax.plot(sub['test_start'], sub[metric], marker='o', label=f'Monthly select K={K}')
    if not weekly_test_folds_df.empty:
        sub = weekly_test_folds_df.sort_values('test_start')
        ax.plot(sub['test_start'], sub[metric], marker='X', linestyle='--', color='black',
                label=f'Aug test K={selected_K}')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.axvline(pd.Timestamp(FINAL_TEST_START), color='red', linestyle='--', alpha=0.6)
    ax.set_title(f'OOS {metric}')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    if metric == 'Relative Pricing Error':
        ax.set_ylabel('Lower is better')
    elif metric == 'Time Series R2':
        ax.set_ylabel('Median uncentered R²')
    else:
        ax.set_ylabel('Uncentered R²')
axes[0, 0].legend()
plt.tight_layout(); plt.show()
log_runtime("END Section 5 plotting", _plot_start)

save_fold_cache(fold_cache)
log_runtime("END Section 5: monthly selection + weekly walk-forward test", _section5_start)


[2026-09-17 20:54:33] START Section 5: monthly selection + weekly walk-forward test
Bonds (>= 10 obs): 235722  |  obs: 26,711,439
After lagging z by 1 row per bond: 26,711,439 → 26,475,717
[2026-09-17 20:55:14] START panel pivot to R/Z arrays
[2026-09-17 20:59:00] END panel pivot to R/Z arrays | elapsed 3m 45.78s
R: (235722, 116)  |  Z: (235722, 46, 116)
Date range: 2026-04-02 → 2026-09-16
Fill rate: 96.8%
[2026-09-17 20:59:00] END model panel construction | elapsed 4m 26.52s
Loaded fold cache: 103 entries from ipca_fold_cache.parquet
Monthly selection folds: 3
First selection fold: train 2026-04-02 → 2026-04-30, validate 2026-05-01 → 2026-05-15
Last selection fold: train 2026-04-02 → 2026-06-30, validate 2026-07-01 → 2026-07-15
K-selection sweep: SKIPPED (cached, methodology unchanged)
Loaded cached selected_K=3 from ipca_selection_summary_cache.pkl
Monthly validation metrics by K (evidence panel; RPE lower is better):


<HTML output>

,Total R2,Time Series R2,Cross Section R2,Relative Pricing Error,Total R2 std,Predictive R2,gamma_instability,selected
K,,,,,,,,
1,0.6580,0.7194,0.3916,0.4472,0.1148,-0.0639,0.0090,False
2,0.6666,0.7609,0.4079,0.4353,0.1146,-0.0671,0.0692,False
3,0.6706,0.7725,0.4146,0.4195,0.1125,-0.0673,0.1158,True
4,0.6734,0.7752,0.4196,0.4155,0.1117,-0.0670,0.1304,False
5,0.6751,0.7827,0.4233,0.4143,0.1119,-0.0671,0.1883,False


Selected K = 3: smallest K within 0.005 of the best validation Total R2 (0.6751); factors beyond K=3 add < 0.005 to Total R2.
[2026-09-17 20:59:00] END monthly model-selection sweep | elapsed 0.02s

Weekly final-test folds from 2026-08-01 to before 2026-09-01: 5
First test fold: train 2026-04-02 → 2026-07-31, test 2026-08-03 → 2026-08-07
Last test fold: train 2026-04-02 → 2026-08-28, test 2026-08-31 → 2026-08-31
Weekly walk-forward: SKIPPED (RUN_WEEKLY_WALK_FORWARD=False; optional robustness). Set RUN_WEEKLY_WALK_FORWARD=True to regenerate the August weekly evidence.
[2026-09-17 20:59:00] END weekly final-test scoring | elapsed 0.00s

Runtime summary by fold kind and K:


<HTML output>

...

,fold_kind,cache_hit,fold_count
0,monthly_selection,True,15


[FINAL TEST] No August weekly test folds available yet.

Weekly final-test Total R2 by selected K:


""


<Figure: muni_characteristic_factor_export_14_8.png (image not included in markdown export)>

[2026-09-17 20:59:00] END Section 5 plotting | elapsed 0.63s
Saved fold cache: 103 entries to ipca_fold_cache.parquet
[2026-09-17 20:59:00] END Section 5: monthly selection + weekly walk-forward test | elapsed 4m 27.30s


In [ ]:
# Full-sample selected-model cache.
# This avoids refitting Section 6 when the panel, selected K, and fit settings are unchanged.
FULL_MODEL_CACHE_VERSION = 'ipca_full_sample_selected_model_v1'
FULL_MODEL_CACHE_PATH = Path('ipca_full_model_cache.pkl')


def full_model_cache_key(K, max_iter, tol):
    payload = {
        'version': FULL_MODEL_CACHE_VERSION,
        'K': int(K),
        'max_iter': int(max_iter),
        'tol': float(tol),
        'panel_signature': PANEL_SIGNATURE,
    }
    return json.dumps(payload, sort_keys=True)


def load_full_model_cache(path=FULL_MODEL_CACHE_PATH):
    if not path.exists():
        return {}
    try:
        with path.open('rb') as fh:
            cache = pickle.load(fh)
        if isinstance(cache, dict):
            print(f"Loaded full-model cache: {len(cache):,} entries from {path}")
            return cache
    except Exception as exc:
        print(f"WARNING: could not load full-model cache {path}: {exc}")
    return {}


def save_full_model_cache(cache, path=FULL_MODEL_CACHE_PATH):
    tmp_path = path.with_name(path.name + '.tmp')
    with tmp_path.open('wb') as fh:
        pickle.dump(cache, fh, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_path.replace(path)
    print(f"Saved full-model cache: {len(cache):,} entries to {path}")


def fit_or_load_full_ipca_model(R, Z, obs_mask, K, max_iter=100, tol=1e-5, use_cache=True):
    key = full_model_cache_key(K, max_iter, tol)
    cache = load_full_model_cache() if use_cache else {}
    if use_cache and key in cache:
        entry = cache[key]
        model = IPCA(n_factors=K, max_iter=max_iter, tol=tol)
        model.Gamma = np.asarray(entry['Gamma'], dtype=float)
        model.Factors = np.asarray(entry['Factors'], dtype=float)
        model._obs_mask = obs_mask
        print(f"Loaded cached full-sample IPCA model | K={K}, max_iter={max_iter}, tol={tol}")
        return model, True

    model = IPCA(n_factors=K, max_iter=max_iter, tol=tol)
    model.fit(R, Z, obs_mask=obs_mask)
    if use_cache:
        cache[key] = {
            'Gamma': np.asarray(model.Gamma, dtype=float),
            'Factors': np.asarray(model.Factors, dtype=float),
        }
        save_full_model_cache(cache)
    return model, False

## 6. Selected Model Fit, R² Decomposition, and Residual Diagnostics

After monthly validation selects $K$, refit the selected model on the full Apr-forward sample for research diagnostics.

This section keeps the three R² views separate:

- **Total R²**: realized latent factors, panel-level reconstruction.
- **Time Series R²**: median per-bond uncentered reconstruction R².
- **Cross Section R²**: average daily relative-value fit.

It also reports **Relative Pricing Error** and a predictive variant where realized factors are replaced by the full-sample factor mean $\hat\lambda$. The predictive variant is a much harder hurdle; a large Total-vs-Predictive gap means the model explains realized shocks better than it forecasts unconditional expected returns.

> **Paper alignment (corporate-bond paper, Section 2.3, Eq. 6).** The relative pricing error here is computed **exactly** as $\displaystyle \text{RPE}=\frac{\sum_i \alpha_i^2}{\sum_i \bar r_i^2}$ with $\alpha_i=\overline{r_{i}-\hat r_{i}}$ and $\bar r_i=\overline{r_i}$ (function `relative_pricing_error`) — **not** an observation-level RMSE and **not** $\sqrt{1-R^2}$. The paper's cross-sectional *pricing error* measures unexplained **average** returns, i.e. $\alpha_i=\bar r_i-\overline{\beta_{i}'\hat\lambda}$, so it corresponds to the **predictive-fit** RPE row (`Lambda mean / Predictive fit`). The `Realized factors / Total fit` RPE uses contemporaneous realized-factor residuals and is a fit statistic, not the $\alpha$ pricing test. (Note: the $K$-selection CV in Section 5 screens on the realized-factor RPE.)

In [ ]:
# ── FULL-SAMPLE REFIT WITH WEEKLY-CV SELECTED K ─────────────────────────
_section6_start = time.perf_counter()
log_runtime("START Section 6: selected model fit and diagnostics")

n_factors = int(globals().get('selected_K', 3))
FULL_MODEL_MAX_ITER = 100
FULL_MODEL_TOL = 1e-5
print(f"Loading or fitting full-sample IPCA with K={n_factors}...")

_refit_start = time.perf_counter()
ipca_model, full_model_cache_hit = fit_or_load_full_ipca_model(
    R,
    Z,
    obs_mask,
    n_factors,
    max_iter=FULL_MODEL_MAX_ITER,
    tol=FULL_MODEL_TOL,
    use_cache=True,
)
cache_status = 'cache hit' if full_model_cache_hit else 'cache miss; fitted and saved'
log_runtime(f"END full-sample IPCA load/refit ({cache_status})", _refit_start)

gamma_df = pd.DataFrame(
    ipca_model.Gamma, index=characteristics,
    columns=[f'Factor_{k+1}' for k in range(n_factors)]
)
factors_df = pd.DataFrame(
    ipca_model.Factors, index=R_panel.columns,
    columns=[f'Factor_{k+1}' for k in range(n_factors)]
)

print("Estimated Γ (characteristic → factor loading):")
display(gamma_df.style.bar(align='mid', color=['#d65f5f', '#5fba7d']))

_plot_start = time.perf_counter()
fig, ax = plt.subplots(figsize=(14, 5))
for k in range(n_factors):
    ax.plot(factors_df.index, factors_df[f'Factor_{k+1}'], label=f'Factor {k+1}')
ax.axvline(pd.Timestamp('2026-08-01'), color='red', linestyle='--', alpha=0.7, label='August OOS test start')
ax.set_title(f'Estimated Latent Muni Factors (K={n_factors})')
ax.set_xlabel('Date'); ax.set_ylabel('Factor Value')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
log_runtime("END factor time-series plot", _plot_start)

# ── TOTAL VS PREDICTIVE FIT ─────────────────────────────────────────────
_fit_eval_start = time.perf_counter()
R_hat_total = predict_panel_returns(ipca_model.Gamma, ipca_model.Factors, Z, obs_mask)
lambda_hat = ipca_model.Factors.mean(axis=0)
R_hat_predictive = predict_panel_returns(
    ipca_model.Gamma,
    np.tile(lambda_hat.reshape(1, -1), (T, 1)),
    Z,
    obs_mask,
)

fit_rows = []
for label, R_hat in [
    ('Realized factors / Total fit', R_hat_total),
    ('Lambda mean / Predictive fit', R_hat_predictive),
]:
    model_eval_start = time.perf_counter()
    metrics = ipca_performance_metrics(R, R_hat, obs_mask)
    resid = np.where(obs_mask > 0, R - R_hat, np.nan)
    finite = resid[np.isfinite(resid)]
    metrics.update({
        'RMSE': np.sqrt(np.nanmean(finite ** 2)),
        'MAE': np.nanmean(np.abs(finite)),
        'Residual Mean': np.nanmean(finite),
        'Residual Std': np.nanstd(finite),
    })
    fit_rows.append(pd.Series(metrics, name=label))
    log_runtime(f"Computed core metrics: {label}", model_eval_start)

core_fit_summary = pd.DataFrame(fit_rows)
print('Core fit summary (uncentered R² denominators; RPE lower is better):')
display(core_fit_summary.style.format({
    'Total R2': '{:.4f}',
    'Time Series R2': '{:.4f}',
    'Cross Section R2': '{:.4f}',
    'Relative Pricing Error': '{:.4f}',
    'RMSE': '{:.6f}',
    'MAE': '{:.6f}',
    'Residual Mean': '{:.6f}',
    'Residual Std': '{:.6f}',
}))
log_runtime("END total vs predictive fit evaluation", _fit_eval_start)

lambda_df = pd.DataFrame({
    'lambda_hat': lambda_hat,
    'factor_mean': ipca_model.Factors.mean(axis=0),
    'factor_vol': ipca_model.Factors.std(axis=0),
    'daily_sharpe': np.divide(
        ipca_model.Factors.mean(axis=0),
        ipca_model.Factors.std(axis=0),
        out=np.zeros(n_factors),
        where=ipca_model.Factors.std(axis=0) > 0,
    ),
}, index=[f'Factor_{k+1}' for k in range(n_factors)])
print('\nFactor mean / vol diagnostics:')
display(lambda_df.style.format('{:.6f}'))

# ── DATE-LEVEL AND BOND-LEVEL DIAGNOSTICS ───────────────────────────────
_diag_start = time.perf_counter()
mask_bool = obs_mask > 0
resid_total = np.where(mask_bool, R - R_hat_total, np.nan)
resid_predictive = np.where(mask_bool, R - R_hat_predictive, np.nan)


def _single_slice_r2(y, y_hat, mask):
    return total_r2_panel(y, y_hat, mask)


date_rows = []
_date_diag_start = time.perf_counter()
for t, date in enumerate(R_panel.columns):
    observed = mask_bool[:, t]
    if not observed.any():
        continue
    date_rows.append({
        'date': date,
        'n_obs': int(observed.sum()),
        'Total R2': _single_slice_r2(R[:, [t]], R_hat_total[:, [t]], obs_mask[:, [t]]),
        'Predictive R2': _single_slice_r2(R[:, [t]], R_hat_predictive[:, [t]], obs_mask[:, [t]]),
        'Total RMSE': np.sqrt(np.nanmean(resid_total[observed, t] ** 2)),
        'Predictive RMSE': np.sqrt(np.nanmean(resid_predictive[observed, t] ** 2)),
        'Avg Total Residual': np.nanmean(resid_total[observed, t]),
    })
date_fit_df = pd.DataFrame(date_rows).set_index('date')
log_runtime("Computed date-level diagnostics", _date_diag_start)

bond_rows = []
_bond_diag_start = time.perf_counter()
for i, asset in enumerate(R_panel.index):
    observed = mask_bool[i, :]
    if observed.sum() < 5:
        continue
    bond_rows.append({
        asset_id: asset,
        'n_obs': int(observed.sum()),
        'Total R2': _single_slice_r2(R[[i], :], R_hat_total[[i], :], obs_mask[[i], :]),
        'Predictive R2': _single_slice_r2(R[[i], :], R_hat_predictive[[i], :], obs_mask[[i], :]),
        'Total RMSE': np.sqrt(np.nanmean(resid_total[i, observed] ** 2)),
        'Predictive RMSE': np.sqrt(np.nanmean(resid_predictive[i, observed] ** 2)),
        'Alpha / Avg Residual': np.nanmean(resid_total[i, observed]),
        'Avg Return': np.nanmean(R[i, observed]),
    })
bond_fit_df = pd.DataFrame(bond_rows).set_index(asset_id)
log_runtime("Computed bond-level diagnostics", _bond_diag_start)

# ── CAUSAL RESIDUAL STATE (point-in-time standardized residual) ─────
# Replace the full-history average residual as the rich/cheap signal with a
# point-in-time standardized residual z_{i,t} = (eps_{i,t} - trailing_mean) /
# (trailing_std + delta), where the normalization uses only residuals observed
# strictly before t (shift(1)); the current residual enters the numerator.
RESID_Z_WIN, RESID_Z_MINP = 20, 5
_resid_by_date = pd.DataFrame(resid_total.T, index=R_panel.columns, columns=R_panel.index)  # dates × bonds
_resid_vol = _resid_by_date.std(axis=0)                       # per-bond residual vol (skipna)
# Residual-scale floor: a low percentile of the cross-sectional residual-vol distribution,
# so tiny-vol (stale/smoothed) bonds do not explode the standardized residual.
RESID_Z_DELTA = float(_resid_vol.quantile(0.10))
_z_mean_trail = _resid_by_date.shift(1).rolling(RESID_Z_WIN, min_periods=RESID_Z_MINP).mean()
_z_std_trail = _resid_by_date.shift(1).rolling(RESID_Z_WIN, min_periods=RESID_Z_MINP).std()
_resid_z_panel = (_resid_by_date - _z_mean_trail) / _z_std_trail.clip(lower=RESID_Z_DELTA)
# Point-in-time state as of the panel end: last available standardized residual per bond.
_latest_resid_z = _resid_z_panel.ffill().iloc[-1]

# Signal eligibility: a bond may carry a signal only with enough residual history,
# a well-defined residual vol, and a recent-enough last mark (staleness gate).
SIGNAL_MIN_OBS, SIGNAL_MAX_STALENESS_D = 20, 10
SIGNAL_MIN_RESID_VOL = RESID_Z_DELTA                          # drop the stale low-vol tail from the signal
_last_obs_pos = np.where(mask_bool.any(axis=1),
                         mask_bool.shape[1] - 1 - np.argmax(mask_bool[:, ::-1], axis=1), -1)
_panel_end = R_panel.columns[-1]
_last_obs_date = pd.to_datetime(pd.Series(
    [R_panel.columns[p] if p >= 0 else pd.NaT for p in _last_obs_pos], index=R_panel.index))
_staleness_days = (_panel_end - _last_obs_date).dt.days

bond_fit_df['Resid Z (PIT)'] = _latest_resid_z.reindex(bond_fit_df.index)
bond_fit_df['Resid Vol'] = _resid_vol.reindex(bond_fit_df.index)
bond_fit_df['Staleness (days)'] = _staleness_days.reindex(bond_fit_df.index)
bond_fit_df['signal_eligible'] = (
    (bond_fit_df['n_obs'] >= SIGNAL_MIN_OBS)
    & bond_fit_df['Resid Vol'].notna() & (bond_fit_df['Resid Vol'] >= SIGNAL_MIN_RESID_VOL)
    & (bond_fit_df['Staleness (days)'] <= SIGNAL_MAX_STALENESS_D)
    & bond_fit_df['Resid Z (PIT)'].notna()
)
print(f"Signal-eligible bonds: {int(bond_fit_df['signal_eligible'].sum()):,} / "
      f"{len(bond_fit_df):,} (>= {SIGNAL_MIN_OBS} obs, resid vol >= {SIGNAL_MIN_RESID_VOL:.2e}, "
      f"<= {SIGNAL_MAX_STALENESS_D}d stale, valid PIT z)")
del _resid_by_date, _z_mean_trail, _z_std_trail, _resid_z_panel

print('\nDate-level fit summary:')
display(date_fit_df[['Total R2', 'Predictive R2', 'Total RMSE', 'Predictive RMSE', 'Avg Total Residual']].describe().T.style.format('{:.4f}'))

# Uncentered per-bond R² has a heavy left tail for near-zero-return bonds, so the
# mean is not meaningful; report a robust median / IQR / P10-P90 summary instead.
print('\nBond-level fit summary (robust; median / IQR / P10-P90 — mean omitted due to uncentered R² left tail):')
_bond_cols = ['Total R2', 'Predictive R2', 'Total RMSE', 'Predictive RMSE', 'Alpha / Avg Residual']
_bond_robust_summary = bond_fit_df[_bond_cols].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).T
_bond_robust_summary.columns = ['P10', 'P25', 'Median', 'P75', 'P90']
_bond_robust_summary.insert(0, 'count', bond_fit_df[_bond_cols].count())
_bond_robust_summary['min'] = bond_fit_df[_bond_cols].min()
_bond_robust_summary['max'] = bond_fit_df[_bond_cols].max()
_bond_robust_summary = _bond_robust_summary[['count', 'min', 'P10', 'P25', 'Median', 'P75', 'P90', 'max']]
_bond_robust_fmt = {'count': '{:,.0f}'}
_bond_robust_fmt.update({c: '{:.4f}' for c in ['min', 'P10', 'P25', 'Median', 'P75', 'P90', 'max']})
display(_bond_robust_summary.style.format(_bond_robust_fmt))

# Share of bonds with a well-behaved fit, since the raw min/max are dominated by
# a few near-zero-return names.
_bond_total_r2 = bond_fit_df['Total R2']
print(f"Bonds with Total R2 >= 0: {int((_bond_total_r2 >= 0).sum()):,} / {int(_bond_total_r2.count()):,} "
      f"({100 * (_bond_total_r2 >= 0).mean():.1f}%); "
      f"Total R2 >= 0.5: {100 * (_bond_total_r2 >= 0.5).mean():.1f}%")

print('\nLargest point-in-time standardized residuals among signal-eligible bonds '
      '(causal rich/cheap state; avg residual shown for reference):')
_rich_cheap_start = time.perf_counter()
_eligible_fit = bond_fit_df[bond_fit_df['signal_eligible']]
if _eligible_fit.empty:                                   # safety: never leave the ranking empty
    _eligible_fit = bond_fit_df[bond_fit_df['Resid Z (PIT)'].notna()]
rich_cheap = _eligible_fit.assign(
    abs_resid_z=_eligible_fit['Resid Z (PIT)'].abs()).sort_values('abs_resid_z', ascending=False)
display(rich_cheap[['n_obs', 'Resid Z (PIT)', 'Staleness (days)', 'Alpha / Avg Residual',
                    'Avg Return', 'Total R2', 'Total RMSE']].head(20).style.format({
    'n_obs': '{:,.0f}', 'Resid Z (PIT)': '{:.3f}', 'Staleness (days)': '{:.0f}',
    'Alpha / Avg Residual': '{:.6f}', 'Avg Return': '{:.6f}',
    'Total R2': '{:.4f}', 'Total RMSE': '{:.6f}'}))
log_runtime("Computed causal rich-cheap residual ranking", _rich_cheap_start)
log_runtime("END residual diagnostic tables", _diag_start)

# ── VISUAL DIAGNOSTICS ──────────────────────────────────────────────────
_plot_start = time.perf_counter()
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(date_fit_df.index, date_fit_df['Total R2'], label='Total R2', linewidth=1.8)
axes[0, 0].plot(date_fit_df.index, date_fit_df['Predictive R2'], label='Predictive R2', linewidth=1.8, color='tab:red')
axes[0, 0].axhline(0, color='black', linewidth=0.6)
axes[0, 0].axvline(pd.Timestamp('2026-08-01'), color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Daily Return Fit: Total vs Predictive R2')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

axes[0, 1].hist(bond_fit_df['Total R2'].clip(lower=-1).replace([np.inf, -np.inf], np.nan).dropna(), bins=80, alpha=0.7, label='Total R2 (clipped ≥ -1)')
axes[0, 1].hist(bond_fit_df['Predictive R2'].clip(lower=-1).replace([np.inf, -np.inf], np.nan).dropna(), bins=80, alpha=0.5, label='Predictive R2 (clipped ≥ -1)')
axes[0, 1].axvline(0, color='black', linewidth=0.6)
axes[0, 1].set_title('Bond-Level R2 Distribution')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

sample_total = resid_total[np.isfinite(resid_total)]
sample_predictive = resid_predictive[np.isfinite(resid_predictive)]
if sample_total.size > 500_000:
    rng = np.random.default_rng(7)
    sample_total = rng.choice(sample_total, size=500_000, replace=False)
if sample_predictive.size > 500_000:
    rng = np.random.default_rng(11)
    sample_predictive = rng.choice(sample_predictive, size=500_000, replace=False)
axes[1, 0].hist(sample_total, bins=120, alpha=0.7, label='Total residual')
axes[1, 0].hist(sample_predictive, bins=120, alpha=0.45, label='Predictive residual')
axes[1, 0].axvline(0, color='black', linewidth=0.6)
axes[1, 0].set_title('Residual Distribution')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(date_fit_df.index, date_fit_df['Avg Total Residual'], linewidth=1.5)
axes[1, 1].axhline(0, color='black', linewidth=0.6)
axes[1, 1].axvline(pd.Timestamp('2026-08-01'), color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Average Cross-Sectional Residual Over Time')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()
log_runtime("END residual diagnostic plotting", _plot_start)

print('\nInterpretation checks:')
print(f"  Total R2:          {core_fit_summary.loc['Realized factors / Total fit', 'Total R2']:.4f}")
print(f"  Time Series R2:    {core_fit_summary.loc['Realized factors / Total fit', 'Time Series R2']:.4f}")
print(f"  Cross Section R2:  {core_fit_summary.loc['Realized factors / Total fit', 'Cross Section R2']:.4f}")
print(f"  RPE:               {core_fit_summary.loc['Realized factors / Total fit', 'Relative Pricing Error']:.4f}")
print(f"  Predictive R2 gap: {core_fit_summary.loc['Realized factors / Total fit', 'Total R2'] - core_fit_summary.loc['Lambda mean / Predictive fit', 'Total R2']:.4f}")

log_runtime("END Section 6: selected model fit and diagnostics", _section6_start)

[2026-09-17 21:02:27] START Section 6: selected model fit and diagnostics
Loading or fitting full-sample IPCA with K=3...
Loaded full-model cache: 3 entries from ipca_full_model_cache.pkl
[IPCA] START fit | K=3, N=235,722, T=116, L=46, max_iter=100
[IPCA] observed entries: 26,475,716 / 27,343,752 (96.8%)


IPCA Iteration:   1%|          | 1/100 [01:31<2:31:42, 91.94s/it]

[IPCA] iter 001 | loss=8.673935e-07 | iter_elapsed=1m 31.94s


IPCA Iteration:   5%|▌         | 5/100 [07:38<2:25:00, 91.58s/it]

[IPCA] iter 005 | loss=8.625085e-07 | iter_elapsed=1m 32.09s


IPCA Iteration:   6%|▌         | 6/100 [10:41<2:47:30, 106.92s/it]

Converged at iteration 7 with loss 8.624971e-07 (relative change 3.17e-06)
[IPCA] END fit | K=3 | elapsed 10m 57.08s
Saved full-model cache: 4 entries to ipca_full_model_cache.pkl
[2026-09-17 21:13:24] END full-sample IPCA load/refit (cache miss; fitted and saved) | elapsed 10m 57.08s
Estimated Γ (characteristic → factor loading):


<HTML output>

,Factor_1,Factor_2,Factor_3
market_fv,-0.010519,-0.062167,-0.128207
rating_grade_A,-0.002986,0.013358,-0.000839
rating_grade_A+,-0.002463,0.008998,0.010589
rating_grade_A-,-0.007895,0.019905,0.020629
rating_grade_AA,-0.005501,0.005299,0.008360
rating_grade_AA+,-0.001664,0.000924,-0.003621
rating_grade_AA-,-0.005763,0.008074,0.019838
rating_grade_B,-0.042378,0.088992,-0.172446
rating_grade_B+,-0.025338,-0.121265,-0.210057
rating_grade_B-,0.150585,0.068791,0.372146


<Figure: muni_characteristic_factor_export_17_9.png (image not included in markdown export)>

[2026-09-17 21:13:24] END factor time-series plot | elapsed 0.21s
[2026-09-17 21:14:45] Computed core metrics: Realized factors / Total fit | elapsed 11.64s
[2026-09-17 21:14:57] Computed core metrics: Lambda mean / Predictive fit | elapsed 11.49s
Core fit summary (uncentered R² denominators; RPE lower is better):


...

<HTML output>

,n_obs,Resid Z (PIT),Staleness (days),Alpha / Avg Residual,Avg Return,Total R2,Total RMSE
cusip,,,,,,,
119677RF1,116,-62.531,0,-0.000164,-0.000417,0.5320,0.001549
960225BY5,116,-54.096,0,-0.000213,-0.000592,0.4294,0.002769
59447UAN0,116,-48.864,0,-0.000057,-0.000452,0.8239,0.000997
265543RL4,116,-41.533,0,-0.000313,-0.000561,0.0214,0.005134
447290CP0,116,41.143,0,0.000113,-0.000204,0.6102,0.001243
59447UAL4,116,-38.467,0,-0.000098,-0.000399,0.8519,0.000706
09005PDG2,116,-33.071,0,-0.000131,-0.000275,0.6459,0.000700
432884NL4,116,31.760,0,0.000259,0.000072,0.1667,0.001250
132286CC7,116,-31.521,0,-0.000061,-0.000441,0.8998,0.000689


[2026-09-17 21:15:56] Computed causal rich-cheap residual ranking | elapsed 0.11s
[2026-09-17 21:15:56] END residual diagnostic tables | elapsed 58.98s


<Figure: muni_characteristic_factor_export_17_21.png (image not included in markdown export)>

[2026-09-17 21:15:57] END residual diagnostic plotting | elapsed 1.25s

Interpretation checks:
  Total R2:          0.7658
  Time Series R2:    0.7818
  Cross Section R2:  0.4639
  RPE:               0.1147
  Predictive R2 gap: 0.7371
[2026-09-17 21:15:57] END Section 6: selected model fit and diagnostics | elapsed 13m 30.25s


In [ ]:
# Quant research review snapshot
# Reads already-computed variables; does not refit IPCA or reload Deephaven data.

import numpy as np
import pandas as pd

required = [
    "data", "df_model", "R", "Z", "obs_mask", "panel_dates", "characteristics",
    "cv_folds", "summary_mean", "selection_table", "fold_total", "selected_K",
    "core_fit_summary", "lambda_df", "date_fit_df", "bond_fit_df", "rich_cheap",
    "gamma_df", "CANDIDATE_K", "weekly_folds",
]
missing = [name for name in required if name not in globals()]

if missing:
    print("Review snapshot could not run because the current kernel is missing these variables:")
    print(missing)
    print("Run Cells 1-16 first, then rerun this review cell.")
else:
    observed_entries = int(obs_mask.sum())
    total_slots = int(obs_mask.size)
    fill_rate = observed_entries / max(total_slots, 1)
    total_fit = core_fit_summary.loc['Realized factors / Total fit']
    predictive_fit = core_fit_summary.loc['Lambda mean / Predictive fit']
    selected_cv = summary_mean.loc[selected_K]
    date_desc = date_fit_df[['Total R2', 'Predictive R2', 'Total RMSE', 'Predictive RMSE', 'Avg Total Residual']].describe().T
    bond_quant = bond_fit_df[['Total R2', 'Predictive R2', 'Total RMSE', 'Predictive RMSE', 'Alpha / Avg Residual']].quantile([0.10, 0.50, 0.90]).T

    print("=== REVIEW SNAPSHOT ===")
    print(f"raw_panel_rows={len(data):,}")
    print(f"model_rows={len(df_model):,}")
    print(f"assets={R.shape[0]:,}")
    print(f"dates={R.shape[1]:,}")
    print(f"characteristics={len(characteristics):,}")
    print(f"observed_entries={observed_entries:,}")
    print(f"fill_rate={fill_rate:.6f}")
    print(f"date_window={pd.Timestamp(panel_dates.min()).date()}..{pd.Timestamp(panel_dates.max()).date()}")
    print(f"weekly_folds={len(weekly_folds):,}")
    print(f"candidate_K={CANDIDATE_K}")
    print(f"selected_K={selected_K}")

    print("\n=== CV_MEAN_BY_K ===")
    print(summary_mean.round(6).to_string())
    print("\n=== SELECTION_TABLE ===")
    print(selection_table.round(6).to_string())
    print("\n=== FOLD_TOTAL_R2 ===")
    print(fold_total.round(6).to_string())
    print("\n=== SELECTED_CV ===")
    print(selected_cv.round(6).to_string())

    print("\n=== CORE_FIT_SUMMARY ===")
    print(core_fit_summary.round(6).to_string())
    print("\n=== TOTAL_VS_PREDICTIVE_GAP ===")
    for metric in ['Total R2', 'Time Series R2', 'Cross Section R2', 'Relative Pricing Error']:
        print(f"{metric}: total={total_fit[metric]:.6f}, predictive={predictive_fit[metric]:.6f}, gap={total_fit[metric] - predictive_fit[metric]:.6f}")

    print("\n=== FACTOR_MEAN_VOL ===")
    print(lambda_df.round(6).to_string())
    print("\n=== DATE_DIAGNOSTIC_MEAN_MIN_MAX ===")
    print(date_desc[['mean', 'min', '50%', 'max']].round(6).to_string())
    print("\n=== BOND_DIAGNOSTIC_P10_MEDIAN_P90 (mean omitted; uncentered R² left tail) ===")
    bond_compact = bond_quant.rename(columns={0.10: 'P10', 0.50: 'Median', 0.90: 'P90'})[['P10', 'Median', 'P90']]
    print(bond_compact.round(6).to_string())
    _bond_total_r2 = bond_fit_df['Total R2']
    print(f"bond_total_r2_share>=0={100 * (_bond_total_r2 >= 0).mean():.1f}%  share>=0.5={100 * (_bond_total_r2 >= 0.5).mean():.1f}%")

    print("\n=== TOP_5_ABS_RESID_Z_PIT (causal rich/cheap) ===")
    print(rich_cheap[['n_obs', 'Resid Z (PIT)', 'Alpha / Avg Residual', 'Avg Return', 'Total R2', 'Total RMSE']].head(5).round(6).to_string())

    print("\n=== TOP_GAMMA_LOADINGS ===")
    for factor in gamma_df.columns:
        ranked = gamma_df[factor].reindex(gamma_df[factor].abs().sort_values(ascending=False).index).head(5)
        print(f"{factor}: " + "; ".join(f"{idx}={val:.6f}" for idx, val in ranked.items()))

=== REVIEW SNAPSHOT ===
raw_panel_rows=26,760,668
model_rows=26,475,717
assets=235,722
dates=116
characteristics=46
observed_entries=26,475,716
fill_rate=0.968255
date_window=2026-04-02..2026-09-16
weekly_folds=5
candidate_K=(1, 2, 3, 4, 5)
selected_K=3

=== CV_MEAN_BY_K ===
   Total R2  Time Series R2  Cross Section R2  Relative Pricing Error
K                                                                    
1     0.658           0.719             0.392                   0.447
2     0.667           0.761             0.408                   0.435
3     0.671           0.773             0.415                   0.419
4     0.673           0.775             0.420                   0.415
5     0.675           0.783             0.423                   0.414

=== SELECTION_TABLE ===
   Total R2  Time Series R2  Cross Section R2  Relative Pricing Error  Total R2 std  Predictive R2  gamma_instability  selected
K                                                                                

## 7. Factor-Mimicking Portfolios

The IPCA fit in Section 6 already estimates each period's factor return by the cross-sectional GLS projection that solves the per-date least-squares problem

$$f_t \;=\; \arg\min_{f}\; \sum_{i \in \mathcal{O}_t}\big(r_{i,t} - \beta_{i,t-1}'\,f\big)^2 \;=\; \big(B_{t-1}'B_{t-1}\big)^{-1}B_{t-1}'\,r_t , \qquad B_{t-1} = Z_{t-1}\,\Gamma ,$$

where $Z_{t-1}$ are the **lagged** characteristics (shifted one observed row per bond in Section 5), so the loadings $B_{t-1}$ are known before $r_t$ is realised. This section re-expresses that same solution as an **explicit portfolio of the underlying bonds** and adds a ridge control on the inversion:

$$f_t \;=\; W_{t-1}\, r_t ,\qquad W_{t-1} \;=\; \big(B_{t-1}'B_{t-1} + \lambda I_K\big)^{-1}B_{t-1}' \;\in\; \mathbb{R}^{K\times N_t}.$$

- $r_t$: $N_t \times 1$ realised returns of the bonds observed at $t$ (the set $\mathcal{O}_t$).
- $B_{t-1}=Z_{t-1}\Gamma$: $N_t \times K$ **predetermined** factor loadings built from the fitted $\Gamma$.
- $W_{t-1}$: $K \times N_t$ weight matrix — **row $k$ is the portfolio for factor $k$**, so $f_{k,t}$ is the return of a tradeable long/short book of bonds.
- $\lambda \ge 0$: ridge hyperparameter controlling how aggressively the $K\times K$ exposure matrix $B_{t-1}'B_{t-1}$ is inverted.

### Methodology

1. **Reuse the fitted model.** Hold $\Gamma =$ `ipca_model.Gamma` fixed; do **not** refit. Form $B_{t-1}=Z_{t-1}\Gamma$ per date from the observed bonds only (via `obs_mask`).
2. **Build the weights.** $W_{t-1} = (B_{t-1}'B_{t-1}+\lambda I)^{-1}B_{t-1}'$; the mimicking factor return is $f_t = W_{t-1}\, r_t$.
3. **Ridge as inversion control.** $\lambda \to 0$ recovers the IPCA factors (Section 6 uses $\lambda=10^{-10}$ internally); larger $\lambda$ shrinks the weights, lowering leverage and turnover at the cost of tracking the raw factors less tightly.
4. **Mimicking property.** By construction $W_{t-1}B_{t-1} = (B_{t-1}'B_{t-1}+\lambda I)^{-1}B_{t-1}'B_{t-1} \approx I_K$: each portfolio carries unit loading on its own factor and $\approx 0$ on the others.

### Conventions and scope

- **Weights:** the raw projection weights are primary (so $f_t$ reconciles with the IPCA factors); a **gross-leverage $= 1$** normalisation ($w \mapsto w/\sum_i|w_i|$) is reported alongside for interpretability.
- **Timing:** weights use predetermined $B_{t-1}$, so the portfolio for date $t$ is formed from information available at $t-1$.
- **Scope:** these are **analytical long/short** portfolios from the closed-form projection. A constrained, long-only *investable* implementation (munis are hard to short) is deferred to future work.

In [ ]:
# ── 7.1 BUILD FACTOR-MIMICKING PORTFOLIO WEIGHTS AND RECONSTRUCT FACTORS ──
_section7_start = time.perf_counter()
log_runtime("START Section 7: factor-mimicking portfolios")

FMP_PRIMARY_RIDGE = 1e-10  # matches the ridge inside the IPCA cross-sectional solve
_fmp_factor_cols = [f'Factor_{k+1}' for k in range(n_factors)]
_fmp_cusips = R_panel.index


def build_mimicking_portfolios(Gamma, Z, R, obs_mask, ridge, weight_dates=None):
    """Explicit IPCA factor-mimicking portfolios: F_t = W_t R_t, W_t = (B'B + ridge*I)^-1 B'.

    Returns the mimicking factor returns (T x K), a per-date/per-factor construction
    record frame, and the full weight matrices for any dates in `weight_dates`.
    """
    n_dates = Z.shape[2]
    K = Gamma.shape[1]
    fmp_factors = np.full((n_dates, K), np.nan)
    records = []
    saved_weights = {}
    weight_dates = {int(t) for t in (weight_dates or [])}
    prev_w = [None] * K  # previous-date weight Series per factor, keyed by cusip
    with timed_block("factor-mimicking weight construction"):
        for t in range(n_dates):
            observed = obs_mask[:, t] > 0
            n_obs = int(observed.sum())
            if n_obs < K:
                prev_w = [None] * K
                continue
            beta_t = Z[observed, :, t] @ Gamma                          # N_obs x K predetermined loadings
            gram = beta_t.T @ beta_t                                    # K x K exposure matrix
            W_t = np.linalg.solve(gram + ridge * np.eye(K), beta_t.T)   # K x N_obs portfolio weights
            fmp_factors[t, :] = W_t @ R[observed, t]
            cusips_t = _fmp_cusips[observed]
            if t in weight_dates:
                saved_weights[t] = pd.DataFrame(W_t.T, index=cusips_t, columns=_fmp_factor_cols)
            for k in range(K):
                w = W_t[k, :]
                gross = float(np.abs(w).sum())
                w_series = pd.Series(w, index=cusips_t)
                if prev_w[k] is not None and gross > 0:
                    idx = prev_w[k].index.union(w_series.index)
                    turnover = 0.5 * float(np.abs(w_series.reindex(idx).fillna(0.0)
                                                  - prev_w[k].reindex(idx).fillna(0.0)).sum())
                else:
                    turnover = np.nan
                prev_w[k] = w_series
                hhi = float(np.sum((np.abs(w) / gross) ** 2)) if gross > 0 else np.nan
                top5 = float(np.sort(np.abs(w))[::-1][:5].sum() / gross) if gross > 0 else np.nan
                records.append({
                    'date': panel_dates[t], 'factor': _fmp_factor_cols[k],
                    'n_positions': n_obs, 'gross_leverage': gross, 'net_exposure': float(w.sum()),
                    'n_long': int((w > 0).sum()), 'n_short': int((w < 0).sum()),
                    'hhi': hhi, 'top5_weight_share': top5,
                    'effective_breadth': (1.0 / hhi) if gross > 0 else np.nan,
                    'turnover': turnover,
                })
    return fmp_factors, pd.DataFrame(records), saved_weights


_last_t = int(len(panel_dates) - 1)
fmp_factors, fmp_construction_df, fmp_weight_snapshots = build_mimicking_portfolios(
    ipca_model.Gamma, Z, R, obs_mask, FMP_PRIMARY_RIDGE, weight_dates=[_last_t],
)
fmp_factors_df = pd.DataFrame(fmp_factors, index=panel_dates, columns=_fmp_factor_cols)

# Reconciliation: F_t = W_t R_t should reproduce the IPCA factors up to convergence tolerance.
_recon_rows = []
for k, col in enumerate(_fmp_factor_cols):
    a = ipca_model.Factors[:, k].astype(float)
    b = fmp_factors[:, k].astype(float)
    valid = np.isfinite(a) & np.isfinite(b)
    _recon_rows.append({
        'factor': col,
        'corr_with_ipca': np.corrcoef(a[valid], b[valid])[0, 1],
        'max_abs_diff': float(np.max(np.abs(a[valid] - b[valid]))),
        'rmse': float(np.sqrt(np.mean((a[valid] - b[valid]) ** 2))),
    })
fmp_reconciliation = pd.DataFrame(_recon_rows).set_index('factor')
print(f"Reconstructed F_t = W_t R_t at ridge={FMP_PRIMARY_RIDGE:.0e} "
      f"(matches IPCA factors up to convergence tolerance):")
display(fmp_reconciliation.style.format({'corr_with_ipca': '{:.6f}', 'max_abs_diff': '{:.3e}', 'rmse': '{:.3e}'}))

[2026-09-17 21:17:07] START Section 7: factor-mimicking portfolios
[2026-09-17 21:17:07] START factor-mimicking weight construction
[2026-09-17 21:18:10] END factor-mimicking weight construction | elapsed 1m 02.98s
Reconstructed F_t = W_t R_t at ridge=1e-10 (matches IPCA factors up to convergence tolerance):


<HTML output>

,corr_with_ipca,max_abs_diff,rmse
factor,,,
Factor_1,1.000000,1.542e-06,5.503e-07
Factor_2,1.000000,7.097e-07,1.944e-07
Factor_3,0.999994,9.925e-06,3.375e-06


### 7.2 Mimicking-Property Verification

Because the weights invert the bonds' own exposure matrix, each portfolio should load one-for-one on its target factor and negligibly on the others:

$$W_{t-1}B_{t-1} \;=\; \big(B_{t-1}'B_{t-1}+\lambda I_K\big)^{-1}B_{t-1}'B_{t-1} \;\xrightarrow[\lambda \to 0]{}\; I_K .$$

Averaging $W_{t-1}B_{t-1}$ across dates, the diagonal should sit at $\approx 1$ and the off-diagonals at $\approx 0$, confirming the portfolios are genuine factor-mimicking books rather than incidental combinations.

In [ ]:
# ── 7.2 MIMICKING PROPERTY: average W_t B_t across dates ─────────────────
_wb_sum = np.zeros((n_factors, n_factors))
_wb_count = 0
_wb_diag = []
with timed_block("mimicking-property W_tB_t check"):
    for t in range(len(panel_dates)):
        observed = obs_mask[:, t] > 0
        if int(observed.sum()) < n_factors:
            continue
        beta_t = Z[observed, :, t] @ ipca_model.Gamma
        gram = beta_t.T @ beta_t
        W_t = np.linalg.solve(gram + FMP_PRIMARY_RIDGE * np.eye(n_factors), beta_t.T)
        WB = W_t @ beta_t
        _wb_sum += WB
        _wb_count += 1
        _wb_diag.append(np.diag(WB))

fmp_WB_mean = pd.DataFrame(_wb_sum / max(_wb_count, 1), index=_fmp_factor_cols, columns=_fmp_factor_cols)
print(f"Mean W_tB_t across {_wb_count} dates (diagonal ≈ 1, off-diagonal ≈ 0):")
display(fmp_WB_mean.style.format('{:.4f}').background_gradient(cmap='RdBu', vmin=-1, vmax=1))

_wb_diag = np.array(_wb_diag)
print("Own-factor loading (diagonal) across dates — min / mean / max:")
for k, col in enumerate(_fmp_factor_cols):
    print(f"  {col}: {_wb_diag[:, k].min():.4f} / {_wb_diag[:, k].mean():.4f} / {_wb_diag[:, k].max():.4f}")

[2026-09-17 21:18:18] START mimicking-property W_tB_t check
[2026-09-17 21:18:53] END mimicking-property W_tB_t check | elapsed 35.41s
Mean W_tB_t across 116 dates (diagonal ≈ 1, off-diagonal ≈ 0):


<HTML output>

,Factor_1,Factor_2,Factor_3
Factor_1,1.0000,0.0000,-0.0000
Factor_2,0.0000,1.0000,0.0000
Factor_3,-0.0000,0.0000,1.0000


Own-factor loading (diagonal) across dates — min / mean / max:
  Factor_1: 1.0000 / 1.0000 / 1.0000
  Factor_2: 1.0000 / 1.0000 / 1.0000
  Factor_3: 1.0000 / 1.0000 / 1.0000


### 7.3 Portfolio Construction Diagnostics

For each date $t$ and factor $k$ with weight vector $w = W_{t-1}^{(k)}$ over the $N_t$ observed bonds:

$$\text{gross leverage} = \sum_i |w_i|,\quad \text{net exposure} = \sum_i w_i,\quad \text{HHI} = \sum_i \Big(\tfrac{|w_i|}{\sum_j|w_j|}\Big)^2,\quad \text{breadth} = \tfrac{1}{\text{HHI}} .$$

Turnover between consecutive dates uses the union of held bonds, zero-filling entries and exits:

$$\text{turnover}_t = \tfrac{1}{2}\sum_i \big|w_{i,t} - w_{i,t-1}\big| .$$

Because the raw weights are unnormalised, gross leverage reports the natural scale of each factor's book; the median / IQR summary is robust to the occasional high-leverage date when the cross-sectional exposure matrix is near-singular. A **gross-leverage $= 1$** view rescales each date's book to $w/\sum_i|w_i|$ for interpretability.

In [ ]:
# ── 7.3 CONSTRUCTION DIAGNOSTICS: robust summary + gross=1 normalized view ─
_fmp_metric_cols = ['gross_leverage', 'net_exposure', 'n_long', 'n_short',
                    'hhi', 'top5_weight_share', 'effective_breadth', 'turnover']
fmp_construction_summary = fmp_construction_df.groupby('factor')[_fmp_metric_cols].median()
print("Per-factor construction diagnostics (median across dates):")
display(fmp_construction_summary.style.format('{:,.4f}'))

# Robust distribution of the scale-sensitive metrics.
for _metric in ['gross_leverage', 'turnover']:
    _dist = fmp_construction_df.groupby('factor')[_metric].quantile([0.10, 0.50, 0.90]).unstack()
    _dist.columns = ['P10', 'Median', 'P90']
    print(f"\n{_metric} — P10 / Median / P90 by factor:")
    display(_dist.style.format('{:,.4f}'))

# Gross-leverage = 1 normalized factor returns: F_norm = F / gross.
fmp_gross = fmp_construction_df.pivot(index='date', columns='factor', values='gross_leverage')[_fmp_factor_cols]
fmp_factors_norm_df = fmp_factors_df / fmp_gross.reindex(fmp_factors_df.index)
print("\nGross-leverage=1 normalized factor returns — daily mean / vol / annualized Sharpe:")
_norm_stats = pd.DataFrame({
    'daily_mean': fmp_factors_norm_df.mean(),
    'daily_vol': fmp_factors_norm_df.std(),
    'ann_sharpe': np.sqrt(252) * fmp_factors_norm_df.mean() / fmp_factors_norm_df.std(),
})
display(_norm_stats.style.format('{:.6f}'))

Per-factor construction diagnostics (median across dates):


<HTML output>

,gross_leverage,net_exposure,n_long,n_short,hhi,top5_weight_share,effective_breadth,turnover
factor,,,,,,,,
Factor_1,3.4616,-3.3909,"10,166.5000","218,876.0000",0.0000,0.0001,"178,139.6464",0.0021
Factor_2,4.3284,-0.4720,"68,512.5000","161,577.0000",0.0000,0.0001,"154,088.2162",0.0030
Factor_3,3.8511,-0.5490,"81,792.5000","147,280.5000",0.0000,0.0001,"162,508.9739",0.0032


gross_leverage — P10 / Median / P90 by factor:


<HTML output>

...

<HTML output>

,P10,Median,P90
factor,,,
Factor_1,0.0001,0.0021,0.0076
Factor_2,0.0003,0.0030,0.0090
Factor_3,0.0006,0.0032,0.0161


Gross-leverage=1 normalized factor returns — daily mean / vol / annualized Sharpe:


<HTML output>

,daily_mean,daily_vol,ann_sharpe
Factor_1,0.000355,0.001785,3.153129
Factor_2,-0.000092,0.000568,-2.573234
Factor_3,0.000045,0.000254,2.789384


### 7.4 Ridge Sensitivity — $\lambda$ as Inversion Control

Rewriting the ridge relative to the scale of the exposure matrix, $\lambda = c\cdot \operatorname{tr}(B_{t-1}'B_{t-1})/K$, the dimensionless knob $c$ controls how aggressively $B_{t-1}'B_{t-1}$ is inverted. Small $c$ tracks the IPCA factors tightly but can produce high leverage and turnover when the exposure matrix is near-singular; larger $c$ shrinks the weights toward a damped, lower-turnover book at the cost of tracking error against the raw factors.

In [ ]:
# ── 7.4 RIDGE SWEEP: relative ridge c controls inversion aggressiveness ───
def _relative_ridge_sweep(Gamma, Z, R, obs_mask, c_grid):
    K = Gamma.shape[1]
    n_dates = Z.shape[2]
    ipca_F = ipca_model.Factors
    rows = []
    for c in c_grid:
        F = np.full((n_dates, K), np.nan)
        gross = np.full((n_dates, K), np.nan)
        prev_w = [None] * K
        turn = [[] for _ in range(K)]
        for t in range(n_dates):
            observed = obs_mask[:, t] > 0
            if int(observed.sum()) < K:
                prev_w = [None] * K
                continue
            beta_t = Z[observed, :, t] @ Gamma
            gram = beta_t.T @ beta_t
            ridge_t = c * np.trace(gram) / K
            W_t = np.linalg.solve(gram + ridge_t * np.eye(K), beta_t.T)
            F[t, :] = W_t @ R[observed, t]
            cusips_t = _fmp_cusips[observed]
            for k in range(K):
                gross[t, k] = np.abs(W_t[k, :]).sum()
                ws = pd.Series(W_t[k, :], index=cusips_t)
                if prev_w[k] is not None:
                    idx = prev_w[k].index.union(ws.index)
                    turn[k].append(0.5 * float(np.abs(ws.reindex(idx).fillna(0.0)
                                                      - prev_w[k].reindex(idx).fillna(0.0)).sum()))
                prev_w[k] = ws
        corrs = []
        for k in range(K):
            v = np.isfinite(F[:, k]) & np.isfinite(ipca_F[:, k])
            corrs.append(np.corrcoef(ipca_F[v, k], F[v, k])[0, 1])
        rows.append({
            'c': c,
            'mean_tracking_corr': float(np.nanmean(corrs)),
            'mean_gross_leverage': float(np.nanmean(gross)),
            'mean_turnover': float(np.nanmean([np.mean(tk) for tk in turn if tk])),
        })
    return pd.DataFrame(rows).set_index('c')


FMP_RIDGE_GRID = [1e-8, 1e-6, 1e-4, 1e-3, 1e-2, 1e-1]
with timed_block("ridge sensitivity sweep"):
    fmp_ridge_sweep = _relative_ridge_sweep(ipca_model.Gamma, Z, R, obs_mask, FMP_RIDGE_GRID)
print("Ridge sweep (relative ridge c = λ·K / tr(B'B)); higher c → lower leverage/turnover, lower tracking:")
display(fmp_ridge_sweep.style.format({'mean_tracking_corr': '{:.6f}',
                                      'mean_gross_leverage': '{:,.2f}', 'mean_turnover': '{:,.4f}'}))

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(fmp_ridge_sweep.index, fmp_ridge_sweep['mean_tracking_corr'], 'o-', color='tab:blue', label='Tracking corr vs IPCA')
ax1.set_xscale('log'); ax1.set_xlabel('relative ridge c (log scale)')
ax1.set_ylabel('Mean tracking corr', color='tab:blue'); ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(fmp_ridge_sweep.index, fmp_ridge_sweep['mean_gross_leverage'], 's--', color='tab:red', label='Mean gross leverage')
ax2.set_yscale('log'); ax2.set_ylabel('Mean gross leverage (log)', color='tab:red'); ax2.tick_params(axis='y', labelcolor='tab:red')
ax1.set_title('Ridge λ: tracking vs leverage tradeoff')
fig.tight_layout(); plt.show()

[2026-09-17 21:19:08] START ridge sensitivity sweep
[2026-09-17 21:25:25] END ridge sensitivity sweep | elapsed 6m 17.13s
Ridge sweep (relative ridge c = λ·K / tr(B'B)); higher c → lower leverage/turnover, lower tracking:


<HTML output>

,mean_tracking_corr,mean_gross_leverage,mean_turnover
c,,,
0.000000,0.999998,3.88,0.0047
0.000001,0.999998,3.88,0.0047
0.000100,0.999998,3.88,0.0047
0.001000,0.999998,3.88,0.0047
0.010000,0.999970,3.84,0.0047
0.100000,0.996949,3.47,0.0040


<Figure: muni_characteristic_factor_export_26_2.png (image not included in markdown export)>

    


### 7.5 Factor-Portfolio Performance

Treating each mimicking portfolio's daily return $f_{k,t}$ as an investable series, the cumulative return and annualised statistics (252 trading days) are

$$\text{cumulative}_k = \prod_t (1 + f_{k,t}) - 1,\qquad \text{Sharpe}_k = \sqrt{252}\,\frac{\overline{f_k}}{\operatorname{std}(f_k)} .$$

Raw projection weights drive the primary series, so returns match the IPCA factors. The mimicking property makes the *loadings* orthonormal ($\Gamma'\Gamma = I$, and $W_{t-1}B_{t-1}=I_K$ cross-sectionally); the realised factor **return** series are not forced to be uncorrelated, so the correlation matrix below is a genuine time-series diagnostic rather than an identity.

In [ ]:
# ── 7.5 PERFORMANCE: cumulative returns, annualized stats, correlation ────
_ANN = 252
fmp_perf = pd.DataFrame({
    'daily_mean': fmp_factors_df.mean(),
    'daily_vol': fmp_factors_df.std(),
    'ann_return': _ANN * fmp_factors_df.mean(),
    'ann_vol': np.sqrt(_ANN) * fmp_factors_df.std(),
    'ann_sharpe': np.sqrt(_ANN) * fmp_factors_df.mean() / fmp_factors_df.std(),
})
print("Factor-mimicking portfolio performance (raw weights, annualized with 252 days):")
display(fmp_perf.style.format({'daily_mean': '{:.6f}', 'daily_vol': '{:.6f}',
                               'ann_return': '{:.4f}', 'ann_vol': '{:.4f}', 'ann_sharpe': '{:.3f}'}))

print("\nRealized factor-return correlation matrix (loadings are orthonormal; return series need not be):")
display(fmp_factors_df.corr().style.format('{:.4f}').background_gradient(cmap='RdBu', vmin=-1, vmax=1))

fmp_cumulative = (1 + fmp_factors_df.fillna(0.0)).cumprod() - 1
fig, ax = plt.subplots(figsize=(14, 5))
for col in _fmp_factor_cols:
    ax.plot(fmp_cumulative.index, fmp_cumulative[col], label=col)
ax.axvline(pd.Timestamp('2026-08-01'), color='red', linestyle='--', alpha=0.7, label='August OOS test start')
ax.axhline(0, color='black', linewidth=0.6)
ax.set_title('Cumulative Factor-Mimicking Portfolio Returns (raw weights)')
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative return')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

Factor-mimicking portfolio performance (raw weights, annualized with 252 days):


<HTML output>

,daily_mean,daily_vol,ann_return,ann_vol,ann_sharpe
Factor_1,0.001232,0.006192,0.3106,0.0983,3.159
Factor_2,-0.000393,0.002453,-0.0991,0.0389,-2.546
Factor_3,0.000170,0.000980,0.0429,0.0156,2.758


Realized factor-return correlation matrix (loadings are orthonormal; return series need not be):


<HTML output>

,Factor_1,Factor_2,Factor_3
Factor_1,1.0000,-0.8768,0.8012
Factor_2,-0.8768,1.0000,-0.7402
Factor_3,0.8012,-0.7402,1.0000


<Figure: muni_characteristic_factor_export_28_4.png (image not included in markdown export)>

    


### 7.6 Latest-Date Holdings

The most recent date's weights show the actual bonds each factor is long and short, joined to their (lagged) rating, tenor, coupon, call, and state buckets for interpretation. These are the raw analytical weights $W_{t-1}^{(k)}$ — negative entries are shorts, which a long-only investable build would later remove or constrain.

In [ ]:
# ── 7.6 LATEST-DATE TOP HOLDINGS PER FACTOR ──────────────────────────────
_latest_date = panel_dates[_last_t]
_latest_weights = fmp_weight_snapshots[_last_t]
_bucket_groups = {
    'rating': 'rating_grade_', 'coupon': 'coupon_bucket_', 'call': 'call_bucket_',
    'state': 'state_bucket_', 'tenor': 'tenor_bucket_',
}
_latest_desc = df_model[df_model[time_id] == _latest_date].set_index(asset_id)
_desc_cols = {}
for _label, _prefix in _bucket_groups.items():
    _grp = [c for c in characteristics if c.startswith(_prefix)]
    if _grp:
        _desc_cols[_label] = _latest_desc[_grp].idxmax(axis=1).str.replace(_prefix, '', regex=False)
_latest_desc_df = pd.DataFrame(_desc_cols)

print(f"Latest-date holdings ({_latest_date.date()}) — top 5 long / top 5 short per factor:")
for col in _fmp_factor_cols:
    w = _latest_weights[col].sort_values(ascending=False)
    top = pd.concat([w.head(5), w.tail(5)])
    holdings = _latest_desc_df.reindex(top.index).assign(weight=top.values)
    holdings = holdings[['weight'] + list(_desc_cols)]
    print(f"\n{col} (gross={np.abs(_latest_weights[col]).sum():,.2f}, net={_latest_weights[col].sum():.4f}):")
    display(holdings.style.format({'weight': '{:.6f}'}))

log_runtime("END Section 7: factor-mimicking portfolios", _section7_start)

Latest-date holdings (2026-09-16) — top 5 long / top 5 short per factor:

Factor_1 (gross=3.48, net=-3.3969):


<HTML output>

,weight,rating,coupon,call,state,tenor
cusip,,,,,,
3042434U9,0.000016,NR,4,0-3m,CA,10-15y
495062KR1,0.000016,NR,4,0-3m,CA,10-15y
60534WEW4,0.000016,NR,4,0-3m,CA,10-15y
605313AG0,0.000016,NR,4,0-3m,CA,10-15y
915801CT9,0.000016,NR,4,0-3m,CA,10-15y
64990FM43,-0.000048,AA+,3,5-7y,NY,21y+
91514AKM6,-0.000049,A,3,4y,TX,21y+
91514ALD5,-0.000050,A,3,5-7y,TX,21y+
5905455Z8,-0.000052,AA-,3,10y,CA,21y+


Factor_2 (gross=4.30, net=-0.4415):


<HTML output>

,weight,rating,coupon,call,state,tenor
cusip,,,,,,
448474E44,0.000078,AA-,4,10y,TX,21y+
120827EM8,0.000078,A+,4,10y,CA,21y+
542433A23,0.000077,AA-,4,10y,CA,21y+
542433B71,0.000077,AA-,4,10y,CA,21y+
724581ZH9,0.000077,AA-,4,10y,CA,21y+
60636APH0,-0.000030,B+,4,6-9m,CA,5-7y
6460663U2,-0.000031,BB,3,9-12m,CA,5-7y
315000CB3,-0.000033,BB,3,6-9m,CA,5-7y
60636APG2,-0.000034,B+,3,6-9m,CA,4y


Factor_3 (gross=3.81, net=-0.4733):


<HTML output>

,weight,rating,coupon,call,state,tenor
cusip,,,,,,
646066Z78,0.000084,B-,4,9-12m,CA,10-15y
5465892Z4,0.000054,AA-,4,0-3m,CA,10-15y
590047BK1,0.000054,AA-,4,0-3m,CA,10-15y
107304EW3,0.000054,AA-,4,0-3m,CA,10-15y
469063AV3,0.000054,AA-,4,0-3m,CA,10-15y
414019DW6,-0.000089,A,3,10y,TX,21y+
880081VJ0,-0.000089,A,3,10y,TX,21y+
513174V33,-0.000089,A,3,10y,TX,21y+
737577NM4,-0.000089,A,3,10y,TX,21y+


[2026-09-17 21:26:34] END Section 7: factor-mimicking portfolios | elapsed 9m 27.22s


## 8. Residual (Idiosyncratic) Portfolios

Every bond's return splits into a **systematic** part spanned by the factor loadings and an **idiosyncratic residual**. Using the factor-mimicking weights $W^F_{t-1}=(B_{t-1}'B_{t-1}+\lambda I)^{-1}B_{t-1}'$ from Section 7, the systematic (factor-replicated) return is

$$\widehat{R}_t \;=\; B_{t-1}\,f_t \;=\; B_{t-1}W^F_{t-1}\,R_t \;=\; H_{t-1}R_t, \qquad H_{t-1} \equiv B_{t-1}W^F_{t-1}\in\mathbb{R}^{N_t\times N_t},$$

so the residual is the return with its factor exposure stripped out:

$$\varepsilon_t \;=\; R_t - \widehat{R}_t \;=\; \big(I - B_{t-1}W^F_{t-1}\big)R_t \;=\; W^{\varepsilon}_{t-1}R_t, \qquad \boxed{\,W^{\varepsilon}_{t-1} \;=\; I - B_{t-1}W^F_{t-1}\,}.$$

**The proposed math is correct.** $W^{\varepsilon}_{t-1}$ is the $N_t\times N_t$ *residual-maker* (annihilator) matrix, and each of its rows is a tradable portfolio: **row $i$ holds bond $i$ and shorts the factor-replicating hedge $B_{i,t-1}W^F_{t-1}$**, so $\varepsilon_{i,t}$ is the return of a factor-neutral long/short book.

### Properties (at $\lambda\to 0$)

- **Projection.** $H_{t-1}$ is the orthogonal projection onto the column space of $B_{t-1}$; $W^{\varepsilon}_{t-1}=I-H_{t-1}$ is idempotent and symmetric with rank $N_t-K$.
- **Factor-neutral.** Residuals are orthogonal to the loadings and carry no factor return: $B_{t-1}'\varepsilon_t = 0$ and $W^F_{t-1}\varepsilon_t = 0$.
- **Trace / leverage.** $\operatorname{tr}(H_{t-1}) = K$, so the average self-exposure $h_{ii}=B_{i,t-1}(B_{t-1}'B_{t-1})^{-1}B_{i,t-1}'$ is $K/N_t$: with $K=3$ factors and tens of thousands of bonds, each residual book is $\approx$ long its own bond plus a small diffuse hedge.

### Methodology and scope

1. Reuse the fitted $\Gamma$ and the Section 7 weights $W^F_{t-1}$ at the same primary ridge.
2. Form residual returns $\varepsilon_t = R_t - B_{t-1}f_t$ per date (never materialise the $N_t\times N_t$ matrix).
3. Verify factor-neutrality ($B'\varepsilon\approx0$, $W^F\varepsilon\approx0$, $\operatorname{tr}(H)=K$).
4. Inspect the residual-book anatomy (self-weight, hedge leverage) and a concrete latest-date example.

These are the same **analytical long/short** portfolios as Section 7; a constrained, long-only investable build is deferred to future work.

In [ ]:
# ── 8.1 RESIDUAL RETURNS AND FACTOR-NEUTRALITY CHECKS ────────────────────
_section8_start = time.perf_counter()
log_runtime("START Section 8: residual (idiosyncratic) portfolios")

RESID_RIDGE = float(globals().get('FMP_PRIMARY_RIDGE', 1e-10))

resid_panel = np.full_like(R, np.nan, dtype=np.float64)   # epsilon_t = (I - B W^F) R_t
_neutral_rows = []
_trace_H = []
with timed_block("residual construction + neutrality checks"):
    for t in range(len(panel_dates)):
        observed = obs_mask[:, t] > 0
        if int(observed.sum()) < n_factors:
            continue
        beta_t = Z[observed, :, t] @ ipca_model.Gamma          # N_obs x K
        A = np.linalg.inv(beta_t.T @ beta_t + RESID_RIDGE * np.eye(n_factors))
        W_f = A @ beta_t.T                                     # K x N_obs  (factor-mimicking)
        f_t = W_f @ R[observed, t]                             # K          (factor return)
        eps_t = R[observed, t] - beta_t @ f_t                 # N_obs      residual returns
        resid_panel[observed, t] = eps_t
        _neutral_rows.append({
            'date': panel_dates[t],
            'max_abs_B_eps': float(np.max(np.abs(beta_t.T @ eps_t))),     # B'ε ≈ 0
            'max_abs_Wf_eps': float(np.max(np.abs(W_f @ eps_t))),         # Wᶠε ≈ 0
            'resid_var_share': float(np.sum(eps_t ** 2) / max(np.sum(R[observed, t] ** 2), 1e-300)),
        })
        _trace_H.append(float(np.einsum('nk,kl,nl->', beta_t, A, beta_t)))  # tr(H) without the N×N matrix

resid_neutrality_df = pd.DataFrame(_neutral_rows).set_index('date')
print(f"Residual factor-neutrality across {len(resid_neutrality_df)} dates "
      f"(B'ε and Wᶠε should be ≈ 0; resid_var_share = idiosyncratic fraction of return variance):")
display(resid_neutrality_df.describe().loc[['mean', 'min', 'max']].T
        .style.format({'mean': '{:.3e}', 'min': '{:.3e}', 'max': '{:.3e}'}))
print(f"\ntr(H_t) across dates — should equal K={n_factors}: "
      f"min={np.min(_trace_H):.4f}, mean={np.mean(_trace_H):.4f}, max={np.max(_trace_H):.4f}")

_eps_flat = resid_panel[np.isfinite(resid_panel)]
print(f"\nResidual return pool: {_eps_flat.size:,} bond-days | "
      f"mean={_eps_flat.mean():.3e}, std={_eps_flat.std():.3e}, "
      f"median idiosyncratic variance share={resid_neutrality_df['resid_var_share'].median():.3f}")

[2026-09-17 21:27:09] START Section 8: residual (idiosyncratic) portfolios
[2026-09-17 21:27:09] START residual construction + neutrality checks
[2026-09-17 21:27:59] END residual construction + neutrality checks | elapsed 50.38s
Residual factor-neutrality across 116 dates (B'ε and Wᶠε should be ≈ 0; resid_var_share = idiosyncratic fraction of return variance):


<HTML output>

,mean,min,max
max_abs_B_eps,3.853e-13,9.856e-15,2.324e-12
max_abs_Wf_eps,3.078e-17,1.193e-18,1.649e-16
resid_var_share,5.361e-01,1.095e-01,9.868e-01


tr(H_t) across dates — should equal K=3: min=3.0000, mean=3.0000, max=3.0000

Residual return pool: 26,475,717 bond-days | mean=-1.849e-05, std=9.285e-04, median idiosyncratic variance share=0.502


### 8.2 Residual-Book Anatomy

Each residual portfolio (row $i$ of $W^{\varepsilon}_{t-1}$) is **long bond $i$ with weight $1-h_{ii}$** and short the factor hedge $-B_{i,t-1}W^F_{t-1}$, where the self-exposure is

$$h_{ii} \;=\; B_{i,t-1}\big(B_{t-1}'B_{t-1}+\lambda I\big)^{-1}B_{i,t-1}' \in [0,1].$$

Because $\sum_i h_{ii}=\operatorname{tr}(H_{t-1})=K$, most bonds have tiny $h_{ii}$ (self-weight $\approx 1$) and a small, diffuse hedge; a large $h_{ii}$ flags a bond whose return is almost fully spanned by the factors, leaving little idiosyncratic content. Row-wise hedge leverage is sampled to avoid forming the $N_t\times N_t$ matrix.

In [ ]:
# ── 8.2 RESIDUAL-BOOK ANATOMY: self-exposure and hedge leverage ──────────
_hii_rows = []
_rng = np.random.default_rng(17)
_hedge_blocks = []
with timed_block("residual-book anatomy"):
    for t in range(len(panel_dates)):
        observed = obs_mask[:, t] > 0
        n_obs = int(observed.sum())
        if n_obs < n_factors:
            continue
        beta_t = Z[observed, :, t] @ ipca_model.Gamma
        A = np.linalg.inv(beta_t.T @ beta_t + RESID_RIDGE * np.eye(n_factors))
        hii = np.einsum('nk,kl,nl->n', beta_t, A, beta_t)      # self-exposure per bond (diag of H)
        _hii_rows.append(pd.Series(hii, dtype=float).describe()[['mean', '50%', 'max']].rename(panel_dates[t]))
        # Sampled residual books, vectorized (H is symmetric, so column j = row j of H).
        sample = _rng.choice(n_obs, size=min(100, n_obs), replace=False)
        H_cols = (beta_t @ A) @ beta_t[sample].T               # N_obs x n_sample
        diag = H_cols[sample, np.arange(sample.size)]          # H_jj for sampled j
        self_w = 1.0 - diag
        hedge_gross = np.abs(H_cols).sum(axis=0) - np.abs(diag)
        _hedge_blocks.append(np.column_stack([self_w, hedge_gross, np.abs(self_w) + hedge_gross]))

hii_daily = pd.DataFrame(_hii_rows)
hii_daily.columns = ['mean_hii', 'median_hii', 'max_hii']
hedge_sample_df = pd.DataFrame(np.vstack(_hedge_blocks), columns=['self_weight', 'hedge_gross', 'book_gross'])
print("Self-exposure h_ii across bonds (per-date mean / median / max, pooled over dates):")
display(hii_daily.describe().loc[['mean', 'min', 'max']].T.style.format('{:.6f}'))
print(f"\nResidual-book leverage over {len(hedge_sample_df):,} sampled bond×date books "
      f"(self weight ≈ 1, hedge small and diffuse):")
_hedge_q = hedge_sample_df.quantile([0.10, 0.50, 0.90]).T
_hedge_q.columns = ['P10', 'Median', 'P90']
display(_hedge_q.style.format('{:.4f}'))

[2026-09-17 21:28:26] START residual-book anatomy
[2026-09-17 21:29:23] END residual-book anatomy | elapsed 56.68s
Self-exposure h_ii across bonds (per-date mean / median / max, pooled over dates):


<HTML output>

,mean,min,max
mean_hii,0.000013,0.000013,0.000013
median_hii,0.000011,0.000011,0.000012
max_hii,0.000123,0.000120,0.000126


Residual-book leverage over 11,600 sampled bond×date books (self weight ≈ 1, hedge small and diffuse):


<HTML output>

,P10,Median,P90
self_weight,1.0000,1.0000,1.0000
hedge_gross,0.6642,1.2656,1.7956
book_gross,1.6642,2.2656,2.7956


### 8.3 A Concrete Residual Portfolio

To make "each residual is a tradable portfolio" explicit, take the top rich/cheap bond from Section 6 (largest $|$point-in-time standardized residual$|$) and show its residual book on a recent date: a weight $\approx +1$ on itself and the offsetting factor hedge $-B_{i,t-1}W^F_{t-1}$ that neutralises its rating / tenor / coupon exposure. By construction the book's factor exposure $B_{t-1}'w^{\varepsilon}$ is $\approx 0$.

In [ ]:
# ── 8.3 CONCRETE RESIDUAL PORTFOLIO FOR A TOP RICH/CHEAP BOND ────────────
_target_cusip = rich_cheap.index[0]
_target_row = R_panel.index.get_loc(_target_cusip)
_t = _last_t
while _t >= 0 and not (obs_mask[_target_row, _t] > 0):   # most recent date the bond trades
    _t -= 1
_date = panel_dates[_t]
observed = obs_mask[:, _t] > 0
cusips_t = R_panel.index[observed]
beta_t = Z[observed, :, _t] @ ipca_model.Gamma
A = np.linalg.inv(beta_t.T @ beta_t + RESID_RIDGE * np.eye(n_factors))
j = int(np.where(cusips_t == _target_cusip)[0][0])
_hedge = (beta_t @ A) @ beta_t[j]                        # row j of H_t over observed bonds
resid_book = pd.Series(-_hedge, index=cusips_t)
resid_book.iloc[j] += 1.0                                # e_j - H_j  →  residual portfolio

_groups = {'rating': 'rating_grade_', 'coupon': 'coupon_bucket_', 'call': 'call_bucket_',
           'state': 'state_bucket_', 'tenor': 'tenor_bucket_'}
_desc = df_model[df_model[time_id] == _date].set_index(asset_id)
_desc_cols = {}
for _lab, _pre in _groups.items():
    _g = [c for c in characteristics if c.startswith(_pre)]
    if _g:
        _desc_cols[_lab] = _desc[_g].idxmax(axis=1).str.replace(_pre, '', regex=False)
_desc_df = pd.DataFrame(_desc_cols)

print(f"Residual portfolio for {_target_cusip} on {_date.date()} "
      f"(self weight={resid_book[_target_cusip]:.6f}, "
      f"book gross={np.abs(resid_book).sum():.4f}, net={resid_book.sum():.6f}):")
_show = pd.concat([resid_book.loc[[_target_cusip]],
                   resid_book.drop(_target_cusip).sort_values(key=np.abs, ascending=False).head(8)])
_book = _desc_df.reindex(_show.index).assign(weight=_show.values)
_book = _book[['weight'] + list(_desc_cols)]
display(_book.style.format({'weight': '{:.6f}'}))

_book_beta = resid_book.values @ beta_t                  # factor exposure of the residual book ≈ 0
print(f"Residual book factor exposure Bᵀwᵉ (≈ 0): "
      f"{np.array2string(_book_beta, precision=2, floatmode='maxprec')}")

log_runtime("END Section 8: residual (idiosyncratic) portfolios", _section8_start)

Residual portfolio for 119677RF1 on 2026-09-16 (self weight=0.999988, book gross=2.2949, net=0.108210):


<HTML output>

,weight,rating,coupon,call,state,tenor
cusip,,,,,,
119677RF1,0.999988,NR,4,5-7y,NY,5-7y
414019DW6,0.000023,A,3,10y,TX,21y+
234650QA0,0.000023,A,3,10y,TX,21y+
014393G30,0.000023,A,3,10y,TX,21y+
59785MBP2,0.000023,A,3,10y,TX,21y+
737577NM4,0.000023,A,3,10y,TX,21y+
880081VJ0,0.000023,A,3,10y,TX,21y+
234650QB8,0.000023,A,3,10y,TX,21y+
234650PZ6,0.000023,A,3,10y,TX,21y+


Residual book factor exposure Bᵀwᵉ (≈ 0): [-1.24e-15 -1.61e-15  1.99e-15]
[2026-09-17 21:29:33] END Section 8: residual (idiosyncratic) portfolios | elapsed 2m 24.41s


### 8.4 Latent Representation Diagnostics

Before forecasting residuals, inspect whether the IPCA representation creates economically sensible muni neighborhoods. These plots and tables are **EDA / model interpretation**, not alpha features.

We look at:

1. **IPCA loading space**: $x_i = z_{i,t-1}'\Gamma$ at the latest model date. PCA / t-SNE / UMAP should cluster bonds with similar rating, tenor, call, state, coupon, and liquidity profiles.
2. **Nearest-neighbor peers**: for selected bonds, report nearest neighbors in loading space, with rating / tenor / call descriptors, distance, residual correlation, and MSRB executable markout similarity where available.
3. **Residual-regime clusters**: cluster the loading space and summarize residual AR(1), residual volatility, trade coverage, and executable-opportunity performance by cluster.
4. **Executable-opportunity heatmaps**: visualize where the signal works by trade freshness, tenor, side, size, and print-order context.

For market-making review, the neighbor tables and cluster-level diagnostics are more important than the t-SNE picture: they directly answer, "Does the model find sensible comparable bonds?"

In [ ]:
# ── 8.4.1 LATENT DIAGNOSTIC DATASET ─────────────────────────────────────
_section84_start = time.perf_counter()
log_runtime("START Section 8.4: latent representation diagnostics")

LATENT_SAMPLE_N = 8_000
LATENT_RANDOM_SEED = 42

with timed_block("build latest loading-space diagnostics"):
    _latest_obs = obs_mask[:, _last_t] > 0
    _latest_cusips = R_panel.index[_latest_obs].astype(str)
    beta_latest = Z[_latest_obs, :, _last_t].astype(float) @ ipca_model.Gamma
    beta_cols = [f'beta_{k+1}' for k in range(beta_latest.shape[1])]
    beta_latest_df = pd.DataFrame(beta_latest, index=_latest_cusips, columns=beta_cols)

    _desc_base = (df.groupby('cusip', sort=False)
                    .agg(rating_grade=('rating_grade', 'last'),
                         tenor_bucket=('tenor_bucket', 'last'),
                         coupon_bucket=('coupon_bucket', 'last'),
                         call_bucket=('call_bucket', 'last'),
                         state_bucket=('state_bucket', 'last')))
    _desc_base.index = _desc_base.index.astype(str)

    _resid_stats = pd.DataFrame({
        'resid_vol': np.nanstd(resid_panel, axis=1),
        'resid_obs': np.isfinite(resid_panel).sum(axis=1),
    }, index=R_panel.index.astype(str))
    _resid_stats['resid_lag1_corr'] = np.nan
    for _i, _cusip in enumerate(R_panel.index.astype(str)):
        _x, _y = resid_panel[_i, :-1], resid_panel[_i, 1:]
        _m = np.isfinite(_x) & np.isfinite(_y)
        if _m.sum() >= 20 and np.std(_x[_m]) > 0 and np.std(_y[_m]) > 0:
            _resid_stats.loc[_cusip, 'resid_lag1_corr'] = np.corrcoef(_x[_m], _y[_m])[0, 1]

    latent_diag_df = beta_latest_df.join(_desc_base, how='left').join(_resid_stats, how='left')
    latent_diag_df['call_status'] = np.where(latent_diag_df['call_bucket'].astype(str).eq('No Call'), 'noncall', 'callable')
    latent_diag_df['tenor_coarse'] = latent_diag_df['tenor_bucket'].map(lambda x: _coarse_tenor_label(x) if '_coarse_tenor_label' in globals() else str(x))
    latent_diag_df['liq_vol_quintile'] = pd.qcut(latent_diag_df['resid_vol'].rank(method='first'), 5, labels=[f'L{i}' for i in range(1, 6)])

    _sample_n = min(LATENT_SAMPLE_N, len(latent_diag_df))
    latent_sample_df = latent_diag_df.sample(_sample_n, random_state=LATENT_RANDOM_SEED).copy()
    _X_latent = latent_sample_df[beta_cols].to_numpy(dtype=float)
    _X_latent = (_X_latent - _X_latent.mean(axis=0)) / np.maximum(_X_latent.std(axis=0), 1e-12)

print(f"Latest-date loading vectors: {len(latent_diag_df):,} bonds; plot sample={len(latent_sample_df):,}")
print("Loading-space descriptor coverage:")
display(latent_diag_df[['rating_grade', 'tenor_bucket', 'call_status', 'state_bucket', 'liq_vol_quintile']].describe().T[['unique', 'top', 'freq']])


[2026-09-17 21:29:59] START Section 8.4: latent representation diagnostics
[2026-09-17 21:29:59] START build latest loading-space diagnostics
[2026-09-17 21:31:14] END build latest loading-space diagnostics | elapsed 1m 15.71s
Latest-date loading vectors: 223,246 bonds; plot sample=8,000
Loading-space descriptor coverage:


,unique,top,freq
rating_grade,18,AA,52248
tenor_bucket,9,10-15y,50848
call_status,2,callable,137755
state_bucket,4,Other States,142483
liq_vol_quintile,5,L1,44650


In [ ]:
# ── 8.4.2 PCA / t-SNE / UMAP OF IPCA LOADING SPACE (PLOTLY) ─────────────
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px

with timed_block("latent loading-space embeddings"):
    pca2 = PCA(n_components=2, random_state=LATENT_RANDOM_SEED)
    _pca_xy = pca2.fit_transform(_X_latent)
    latent_sample_df['pca_x'] = _pca_xy[:, 0]
    latent_sample_df['pca_y'] = _pca_xy[:, 1]

    tsne = TSNE(n_components=2, perplexity=40, init='pca', learning_rate='auto',
                random_state=LATENT_RANDOM_SEED, max_iter=750)
    _tsne_xy = tsne.fit_transform(_X_latent)
    latent_sample_df['tsne_x'] = _tsne_xy[:, 0]
    latent_sample_df['tsne_y'] = _tsne_xy[:, 1]

    try:
        import umap
        reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.05,
                            random_state=LATENT_RANDOM_SEED)
        _umap_xy = reducer.fit_transform(_X_latent)
        latent_sample_df['umap_x'] = _umap_xy[:, 0]
        latent_sample_df['umap_y'] = _umap_xy[:, 1]
        _has_umap = True
    except Exception as exc:
        _has_umap = False
        print(f"UMAP skipped ({exc}). Install umap-learn for this optional view.")

print(f"PCA explained variance ratio: {pca2.explained_variance_ratio_[0]:.3f}, {pca2.explained_variance_ratio_[1]:.3f}")

_plot_cols = ['rating_grade', 'tenor_bucket', 'call_status', 'state_bucket', 'liq_vol_quintile', 'resid_lag1_corr', 'resid_vol']
_hover_cols = [c for c in _plot_cols if c in latent_sample_df.columns]
_plot_df = latent_sample_df.reset_index().rename(columns={'index': 'cusip'})

_embed_specs = [('pca_x', 'pca_y', 'PCA'), ('tsne_x', 'tsne_y', 't-SNE')]
if _has_umap:
    _embed_specs.append(('umap_x', 'umap_y', 'UMAP'))

_color_specs = [
    ('rating_grade', 'Rating'),
    ('tenor_coarse', 'Tenor'),
    ('call_status', 'Call status'),
    ('liq_vol_quintile', 'Residual-vol quintile'),
]

for _xcol, _ycol, _embed_name in _embed_specs:
    for _color_col, _label in _color_specs:
        fig = px.scatter(
            _plot_df,
            x=_xcol,
            y=_ycol,
            color=_color_col,
            hover_data=['cusip'] + [c for c in _hover_cols if c != _color_col],
            title=f'{_embed_name} IPCA loading space colored by {_label}',
            opacity=0.65,
            render_mode='webgl',
            height=620,
        )
        fig.update_traces(marker={'size': 4})
        fig.update_layout(legend_title_text=_label, xaxis_visible=False, yaxis_visible=False)
        fig.show()


[2026-09-17 21:31:55] START latent loading-space embeddings
UMAP skipped (No module named 'umap'). Install umap-learn for this optional view.
[2026-09-17 21:32:04] END latent loading-space embeddings | elapsed 8.99s
PCA explained variance ratio: 0.487, 0.294


In [ ]:
# ── 8.4.3 NEAREST-NEIGHBOR PEER ANALYSIS ────────────────────────────────
from sklearn.neighbors import NearestNeighbors

with timed_block("nearest-neighbor peer analysis"):
    _beta_full = latent_diag_df[beta_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    _X_full = _beta_full.to_numpy(dtype=float)
    _X_full = (_X_full - _X_full.mean(axis=0)) / np.maximum(_X_full.std(axis=0), 1e-12)
    _nn = NearestNeighbors(n_neighbors=21, metric='euclidean')
    _nn.fit(_X_full)
    _nn_index = _beta_full.index.to_numpy(dtype=str)
    _row_lookup = pd.Series(np.arange(len(R_panel.index), dtype=np.int64), index=R_panel.index.astype(str))

    if 'executable_opps' in globals():
        _exec_by_cusip = (executable_opps.assign(cusip=lambda x: x['cusip'].astype(str),
                                                 markout_bps=lambda x: 1e4 * x['executable_ret'])
                          .groupby('cusip')
                          .agg(exec_markout_bps=('markout_bps', 'mean'),
                               exec_obs=('markout_bps', 'size')))
    else:
        _exec_by_cusip = pd.DataFrame(columns=['exec_markout_bps', 'exec_obs'])

    _target_candidates = []
    if 'rich_cheap' in globals():
        _target_candidates.extend([str(c) for c in rich_cheap.index if str(c) in _beta_full.index])
    _target_candidates.extend([str(c) for c in latent_diag_df.sample(20, random_state=7).index if str(c) in _beta_full.index])
    _targets = list(dict.fromkeys(_target_candidates))[:3]

    _peer_rows = []
    for _target in _targets:
        _pos = int(np.where(_nn_index == _target)[0][0])
        _dist, _idx = _nn.kneighbors(_X_full[[_pos]], n_neighbors=21)
        _rank = 0
        _target_desc = latent_diag_df.loc[_target]
        _target_exec = _exec_by_cusip['exec_markout_bps'].get(_target, np.nan)
        for _d, _j in zip(_dist.ravel(), _idx.ravel()):
            _nbr = _nn_index[_j]
            if _nbr == _target:
                continue
            _rank += 1
            _nbr_desc = latent_diag_df.loc[_nbr]
            _corr = np.nan
            if _target in _row_lookup.index and _nbr in _row_lookup.index:
                _ti, _ni = int(_row_lookup.loc[_target]), int(_row_lookup.loc[_nbr])
                _x, _y = resid_panel[_ti], resid_panel[_ni]
                _m = np.isfinite(_x) & np.isfinite(_y)
                if _m.sum() >= 20 and np.std(_x[_m]) > 0 and np.std(_y[_m]) > 0:
                    _corr = float(np.corrcoef(_x[_m], _y[_m])[0, 1])
            _nbr_exec = _exec_by_cusip['exec_markout_bps'].get(_nbr, np.nan)
            _peer_rows.append({
                'target_cusip': _target,
                'target_rating': _target_desc.get('rating_grade'),
                'target_tenor': _target_desc.get('tenor_bucket'),
                'target_call': _target_desc.get('call_status'),
                'neighbor_rank': _rank,
                'neighbor_cusip': _nbr,
                'distance': float(_d),
                'neighbor_rating': _nbr_desc.get('rating_grade'),
                'neighbor_tenor': _nbr_desc.get('tenor_bucket'),
                'neighbor_call': _nbr_desc.get('call_status'),
                'neighbor_state': _nbr_desc.get('state_bucket'),
                'resid_corr': _corr,
                'target_exec_markout_bps': _target_exec,
                'neighbor_exec_markout_bps': _nbr_exec,
                'abs_markout_diff_bps': abs(_target_exec - _nbr_exec) if np.isfinite(_target_exec) and np.isfinite(_nbr_exec) else np.nan,
                'neighbor_exec_obs': _exec_by_cusip['exec_obs'].get(_nbr, np.nan),
            })
            if _rank >= 20:
                break

nearest_peer_df = pd.DataFrame(_peer_rows)
peer_quality_summary = (nearest_peer_df.assign(
    same_rating=lambda x: x['target_rating'].astype(str) == x['neighbor_rating'].astype(str),
    same_tenor=lambda x: x['target_tenor'].astype(str) == x['neighbor_tenor'].astype(str),
    same_call=lambda x: x['target_call'].astype(str) == x['neighbor_call'].astype(str),
).groupby('target_cusip')
 .agg(mean_distance=('distance', 'mean'),
      median_resid_corr=('resid_corr', 'median'),
      same_rating_share=('same_rating', 'mean'),
      same_tenor_share=('same_tenor', 'mean'),
      same_call_share=('same_call', 'mean'),
      median_abs_markout_diff_bps=('abs_markout_diff_bps', 'median')))

print("Nearest-neighbor quality summary (20 nearest loading-space peers per target):")
display(peer_quality_summary.style.format({
    'mean_distance': '{:.4f}', 'median_resid_corr': '{:.3f}',
    'same_rating_share': '{:.1%}', 'same_tenor_share': '{:.1%}', 'same_call_share': '{:.1%}',
    'median_abs_markout_diff_bps': '{:.2f}',
}))

for _target in _targets:
    print(f"\nNearest peers for {_target}:")
    _cols = ['neighbor_rank', 'neighbor_cusip', 'distance', 'neighbor_rating', 'neighbor_tenor',
             'neighbor_call', 'neighbor_state', 'resid_corr', 'neighbor_exec_markout_bps', 'neighbor_exec_obs']
    display(nearest_peer_df[nearest_peer_df['target_cusip'] == _target][_cols].head(10).style.format({
        'distance': '{:.4f}', 'resid_corr': '{:.3f}', 'neighbor_exec_markout_bps': '{:.2f}',
        'neighbor_exec_obs': '{:,.0f}',
    }))


[2026-09-16 20:54:31] START nearest-neighbor peer analysis
[2026-09-16 20:54:32] END nearest-neighbor peer analysis | elapsed 0.89s
Nearest-neighbor quality summary (20 nearest loading-space peers per target):


<HTML output>

,mean_distance,median_resid_corr,same_rating_share,same_tenor_share,same_call_share,median_abs_markout_diff_bps
target_cusip,,,,,,
300767QD5,0.0000,0.245,100.0%,100.0%,100.0%,nan
890210RZ2,0.0000,0.125,100.0%,100.0%,100.0%,nan
890210SB4,0.0000,0.497,100.0%,100.0%,100.0%,nan


Nearest peers for 890210RZ2:


<HTML output>

...

<HTML output>

,neighbor_rank,neighbor_cusip,distance,neighbor_rating,neighbor_tenor,neighbor_call,neighbor_state,resid_corr,neighbor_exec_markout_bps,neighbor_exec_obs
20,1,64966SAW6,0.0000,AA,16-20y,callable,NY,0.284,nan,nan
21,2,59260XCD8,0.0000,AA,16-20y,callable,NY,0.187,nan,nan
22,3,295084YQ4,0.0000,AA,16-20y,callable,NY,0.513,nan,nan
23,4,242145WT3,0.0000,AA,16-20y,callable,NY,0.367,nan,nan
24,5,076316QR2,0.0000,AA,16-20y,callable,NY,0.491,nan,nan
25,6,076316QT8,0.0000,AA,16-20y,callable,NY,0.349,nan,nan
26,7,68277DKT0,0.0000,AA,16-20y,callable,NY,0.529,nan,nan
27,8,64966SCG9,0.0000,AA,16-20y,callable,NY,0.372,nan,nan
28,9,682539ZB0,0.0000,AA,16-20y,callable,NY,0.512,nan,nan
29,10,63165T6Y6,0.0000,AA,16-20y,callable,NY,0.342,nan,nan


Nearest peers for 300767QD5:


<HTML output>

,neighbor_rank,neighbor_cusip,distance,neighbor_rating,neighbor_tenor,neighbor_call,neighbor_state,resid_corr,neighbor_exec_markout_bps,neighbor_exec_obs
40,1,079491XZ4,0.0000,AA+,16-20y,callable,Other States,0.632,nan,nan
41,2,099140GE9,0.0000,AA+,16-20y,callable,Other States,0.569,nan,nan
42,3,031231D91,0.0000,AA+,16-20y,callable,Other States,-0.126,nan,nan
43,4,044037CX5,0.0000,AA+,16-20y,callable,Other States,0.006,nan,nan
44,5,187882RF4,0.0000,AA+,16-20y,callable,Other States,-0.046,nan,nan
45,6,010734VL2,0.0000,AA+,16-20y,callable,Other States,0.065,nan,nan
46,7,192792TK5,0.0000,AA+,16-20y,callable,Other States,0.632,nan,nan
47,8,092347KA0,0.0000,AA+,16-20y,callable,Other States,0.243,nan,nan
48,9,14984PGS8,0.0000,AA+,16-20y,callable,Other States,-0.078,nan,nan
49,10,206400WA7,0.0000,AA+,16-20y,callable,Other States,0.688,nan,nan


In [ ]:
# ── 8.4.4 LOADING-SPACE CLUSTERS AND RESIDUAL REGIMES (PLOTLY) ──────────
from sklearn.cluster import MiniBatchKMeans
import plotly.express as px

LATENT_N_CLUSTERS = 12

with timed_block("loading-space cluster diagnostics"):
    _cluster_X = latent_diag_df[beta_cols].replace([np.inf, -np.inf], np.nan).dropna()
    _cluster_vals = _cluster_X.to_numpy(dtype=float)
    _cluster_vals = (_cluster_vals - _cluster_vals.mean(axis=0)) / np.maximum(_cluster_vals.std(axis=0), 1e-12)
    _kmeans = MiniBatchKMeans(n_clusters=LATENT_N_CLUSTERS, random_state=LATENT_RANDOM_SEED,
                              batch_size=4096, n_init='auto')
    _cluster_labels = _kmeans.fit_predict(_cluster_vals)
    latent_diag_df['latent_cluster'] = np.nan
    latent_diag_df.loc[_cluster_X.index, 'latent_cluster'] = _cluster_labels
    latent_diag_df['latent_cluster'] = latent_diag_df['latent_cluster'].astype('Int64')
    latent_sample_df['latent_cluster'] = latent_diag_df.loc[latent_sample_df.index, 'latent_cluster'].to_numpy()
    latent_sample_df['latent_cluster_label'] = 'C' + latent_sample_df['latent_cluster'].astype(str)

    def _safe_mode(s):
        m = s.dropna().astype(str).mode()
        return m.iat[0] if not m.empty else None

    _cluster_summary = []
    for _cluster, _g in latent_diag_df.dropna(subset=['latent_cluster']).groupby('latent_cluster'):
        _cluster_summary.append({
            'cluster': int(_cluster),
            'n_bonds': len(_g),
            'top_rating': _safe_mode(_g['rating_grade']),
            'top_tenor': _safe_mode(_g['tenor_bucket']),
            'top_call': _safe_mode(_g['call_status']),
            'top_state': _safe_mode(_g['state_bucket']),
            'median_resid_vol': _g['resid_vol'].median(),
            'median_resid_lag1': _g['resid_lag1_corr'].median(),
            'mean_resid_lag1': _g['resid_lag1_corr'].mean(),
            'median_obs': _g['resid_obs'].median(),
        })
    latent_cluster_summary = pd.DataFrame(_cluster_summary).sort_values('cluster').set_index('cluster')

    if 'executable_opps' in globals():
        _cluster_map = latent_diag_df['latent_cluster'].dropna().astype(int)
        _exec_cluster = executable_opps.assign(cusip=lambda x: x['cusip'].astype(str))
        _exec_cluster['latent_cluster'] = _exec_cluster['cusip'].map(_cluster_map)
        _exec_cluster = _exec_cluster.dropna(subset=['latent_cluster']).copy()
        _exec_cluster['latent_cluster'] = _exec_cluster['latent_cluster'].astype(int)
        _exec_cluster_summary = (_exec_cluster.groupby('latent_cluster')
                                 .agg(exec_rows=('executable_ret', 'size'),
                                      mean_exec_ret_bps=('executable_ret', lambda s: 1e4 * s.mean()),
                                      median_exec_hold_days=('exec_hold_days', 'median'),
                                      buy_share=('opportunity_side', lambda s: (s == 'buy').mean()))
                                 .rename_axis('cluster'))
        latent_cluster_summary = latent_cluster_summary.join(_exec_cluster_summary, how='left')

print("Loading-space clusters: residual persistence / liquidity / optional MSRB executable context")
_cluster_summary_format = {
    'n_bonds': '{:,.0f}', 'median_resid_vol': '{:.5f}', 'median_resid_lag1': '{:.3f}',
    'mean_resid_lag1': '{:.3f}', 'median_obs': '{:,.0f}', 'exec_rows': '{:,.0f}',
    'mean_exec_ret_bps': '{:.2f}', 'median_exec_hold_days': '{:.1f}', 'buy_share': '{:.1%}',
}
display(latent_cluster_summary.style.format({
    col: fmt for col, fmt in _cluster_summary_format.items()
    if col in latent_cluster_summary.columns
}))

_cluster_plot_df = latent_sample_df.reset_index().rename(columns={'index': 'cusip'})
fig = px.scatter(
    _cluster_plot_df,
    x='pca_x',
    y='pca_y',
    color='latent_cluster_label',
    hover_data=['cusip', 'rating_grade', 'tenor_bucket', 'call_status', 'state_bucket', 'resid_lag1_corr', 'resid_vol'],
    title='PCA IPCA loading space colored by latent cluster',
    opacity=0.7,
    render_mode='webgl',
    height=650,
)
fig.update_traces(marker={'size': 4})
fig.update_layout(legend_title_text='Cluster', xaxis_visible=False, yaxis_visible=False)
fig.show()

_cluster_bar_df = latent_cluster_summary.reset_index()
_cluster_bar_hover = [
    col for col in [
        'n_bonds', 'top_rating', 'top_call', 'top_state', 'median_resid_vol',
        'exec_rows', 'mean_exec_ret_bps',
    ]
    if col in _cluster_bar_df.columns
]
fig = px.bar(
    _cluster_bar_df,
    x='cluster',
    y='median_resid_lag1',
    color='top_tenor',
    hover_data=_cluster_bar_hover,
    title='Cluster-level median residual AR(1)',
    height=520,
)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(xaxis_title='Latent cluster', yaxis_title='Median per-bond lag-1 residual corr')
fig.show()


[2026-09-17 12:51:21] START loading-space cluster diagnostics
[2026-09-17 12:51:22] END loading-space cluster diagnostics | elapsed 0.59s
Loading-space clusters: residual persistence / liquidity / optional MSRB executable context


<HTML output>

,n_bonds,top_rating,top_tenor,top_call,top_state,median_resid_vol,median_resid_lag1,mean_resid_lag1,median_obs
cluster,,,,,,,,,
0,"14,394",NR,5-7y,callable,Other States,0.00100,0.250,0.241,115
1,"36,369",AA+,5-7y,noncall,Other States,0.00032,0.139,0.137,115
2,"17,948",AA,10-15y,callable,Other States,0.00080,0.035,0.031,115
3,"12,880",AA,10-15y,callable,Other States,0.00129,0.261,0.234,115
4,"10,264",NR,1y,callable,Other States,0.00035,0.275,0.239,115
5,"37,722",NR,1y,noncall,Other States,0.00025,0.235,0.216,115
6,"10,899",AA,16-20y,callable,Other States,0.00113,0.111,0.114,115
7,"33,453",AA,5-7y,noncall,Other States,0.00050,0.085,0.096,115
8,"5,987",AA,16-20y,callable,Other States,0.00193,0.214,0.205,115


<HTML output>

<HTML output>

In [ ]:
# ── 8.4.5 EXECUTABLE-OPPORTUNITY HEATMAPS BY CONTEXT (PLOTLY) ───────────
import plotly.express as px


def _slice_rank_ic(frame, signal_col='alpha_signal', target_col='executable_ret', min_n=200):
    d = frame[[signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < min_n or d[signal_col].std() == 0 or d[target_col].std() == 0:
        return np.nan
    return float(d[signal_col].corr(d[target_col], method='spearman'))


def _slice_spread_bps(frame, signal_col='alpha_signal', target_col='executable_ret', n_bins=10, min_n=200):
    d = frame[[signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < max(min_n, n_bins * 20):
        return np.nan
    dec = pd.qcut(d[signal_col].rank(method='first'), n_bins, labels=False)
    prof = d.groupby(dec, observed=False)[target_col].mean()
    if 0 not in prof.index or n_bins - 1 not in prof.index:
        return np.nan
    return float(1e4 * (prof.loc[n_bins - 1] - prof.loc[0]))


def _plotly_heatmap(df, title, color_label, zmin=None, zmax=None, text_fmt='.3f'):
    if df.empty:
        print(f"{title}: not available")
        return
    _plot = df.copy()
    _text = _plot.map(lambda x: '' if not np.isfinite(x) else format(float(x), text_fmt))
    fig = px.imshow(
        _plot,
        text_auto=False,
        color_continuous_scale='RdBu_r',
        zmin=zmin,
        zmax=zmax,
        aspect='auto',
        title=title,
        height=max(360, 40 * len(_plot.index) + 220),
    )
    fig.update_traces(text=_text.to_numpy(), texttemplate='%{text}', textfont={'size': 11})
    fig.update_layout(coloraxis_colorbar_title=color_label, xaxis_title='', yaxis_title='')
    fig.show()

if 'executable_opps' not in globals():
    print("Executable heatmaps skipped: run Section 15 first to create executable_opps.")
else:
    with timed_block("executable opportunity heatmaps"):
        _heat = executable_opps.copy()
        _heat['cusip'] = _heat['cusip'].astype(str)
        if 'latent_cluster' in latent_diag_df.columns:
            _heat['latent_cluster'] = _heat['cusip'].map(latent_diag_df['latent_cluster'].dropna().astype(int))
        _heat['freshness'] = pd.cut(_heat['signal_to_trade_lag'], bins=[0, 1, 2, 5, 20],
                                    labels=['1d', '2d', '3-5d', '6-20d'])

        def _grid(row_col, col_col, value_func):
            rows = []
            for (_r, _c), _g in _heat.groupby([row_col, col_col], observed=False):
                rows.append({row_col: _r, col_col: _c, 'value': value_func(_g), 'n': len(_g)})
            return pd.DataFrame(rows).pivot(index=row_col, columns=col_col, values='value')

        heat_side_size_ic = _grid('opportunity_side', 'size_bucket', _slice_rank_ic)
        heat_side_size_spread = _grid('opportunity_side', 'size_bucket', _slice_spread_bps)
        heat_side_print_ic = _grid('opportunity_side', 'print_order', _slice_rank_ic)
        heat_cluster_side_ic = (_grid('latent_cluster', 'opportunity_side', _slice_rank_ic)
                                if 'latent_cluster' in _heat.columns else pd.DataFrame())

    _plotly_heatmap(heat_side_size_ic, 'Rank IC by side x size', 'Rank IC', zmin=-0.08, zmax=0.08, text_fmt='.3f')
    _plotly_heatmap(heat_side_size_spread, 'D10-D1 bps by side x size', 'bps', zmin=-50, zmax=50, text_fmt='.1f')
    _plotly_heatmap(heat_side_print_ic, 'Rank IC by side x first/sub print', 'Rank IC', zmin=-0.08, zmax=0.08, text_fmt='.3f')
    _plotly_heatmap(heat_cluster_side_ic, 'Rank IC by latent cluster x side', 'Rank IC', zmin=-0.08, zmax=0.08, text_fmt='.3f')


Executable heatmaps skipped: run Section 15 first to create executable_opps.


In [ ]:
# ── 8.4.6 LATENT REPRESENTATION DIAGNOSTIC VERDICT ─────────────────────
print("=" * 78)
print("SECTION 8.4 VERDICT — latent representation diagnostics")
print("=" * 78)
print("1. Loading space is economically organized, especially by tenor and call status.")
print("   Rating is less cleanly separated, suggesting the current static feature set mainly")
print("   captures structure / maturity / optionality rather than fine credit neighborhoods.")
print("2. Nearest-neighbor peers are descriptor-coherent, but many distances are exactly zero")
print("   because one-hot static buckets create coarse identical loading vectors.")
print("   Dynamic features are needed for sharper peer ranking within those buckets.")
print("3. Cluster-level residual AR(1) varies materially across latent neighborhoods, which")
print("   supports conditional persistence / freshness modeling before LongConv.")
if 'executable_opps' in globals():
    print("4. Executable heatmaps show side asymmetry: sell opportunities carry more stable rank IC")
    print("   than buy opportunities, and latent clusters can flip sign by side.")
    print("   This supports separate customer-buy/customer-sell calibration models.")
else:
    print("4. Run Section 15 to add executable-opportunity heatmaps by side, size, print order, and cluster.")
print("=" * 78)
log_runtime(f"END Section 8.4 (elapsed {time.perf_counter() - _section84_start:.1f}s)")


SECTION 8.4 VERDICT — latent representation diagnostics
1. Loading space is economically organized, especially by tenor and call status.
   Rating is less cleanly separated, suggesting the current static feature set mainly
   captures structure / maturity / optionality rather than fine credit neighborhoods.
2. Nearest-neighbor peers are descriptor-coherent, but many distances are exactly zero
   because one-hot static buckets create coarse identical loading vectors.
   Dynamic features are needed for sharper peer ranking within those buckets.
3. Cluster-level residual AR(1) varies materially across latent neighborhoods, which
   supports conditional persistence / freshness modeling before LongConv.
4. Run Section 15 to add executable-opportunity heatmaps by side, size, print order, and cluster.
[2026-09-17 12:51:33] END Section 8.4 (elapsed 57516.0s)


## 9. Residual Alpha Validation (Diagnostic Pre-Gate)

Section 6 showed strong **in-sample** fit (full-sample Total R² ≈ 0.77) but weak **predictive** fit (Predictive R² ≈ 0.03), and Section 8 confirmed the residuals $\varepsilon_{i,t}$ carry roughly **half** the cross-sectional return variance. This section tests whether those residuals contain temporal structure — persistence, mean reversion, or momentum — or are approximately white noise.

The input is the residual panel `resid_panel` ($\varepsilon_{i,t}$, bond x date) from Section 8.

> **Diagnostic scope.** This residual panel comes from the full-sample Section 6 IPCA refit. Treat Sections 9-16 as exploratory diagnostics until the result is reproduced with a fully point-in-time chain: train $\Gamma$ before date $t$, compute residual as-of $t$, fit AR only on earlier residuals, then score future MSRB prints.

### Method (adapted to the 110-date sample)

- **Horizons** are scaled to the short sample: forward residual sums over $h \in \{1,3,5\}$ trading days; autocorrelation lags $1..20$; AR orders $\{1,3,5\}$.
- **Features** per bond-date: $
arepsilon_t$, forward sums $\sum_{k=1}^{h}
arepsilon_{i,t+k}$, trailing MAs (5/20/60), 60-day vol, and z-score $
arepsilon_t/\operatorname{std}(
arepsilon_{t-60:t})$.
- **Signals** A-E: $
arepsilon_t$, $-
arepsilon_t$, z-score, MA20, MA60.
- **IC** = cross-sectional daily correlation of signal vs future residual, averaged over dates; t-stat from the daily-IC series (Pearson and Spearman rank IC).
- **Universe:** bonds with $\ge 60$ observed days. **AR** benchmarks are pooled panel regressions with a time-based out-of-sample split.

### Stages

1. Simple residual signals → IC, rank IC, t-stat, monthly stability, hit rate.
2. Decile portfolios → D10−D1 spread and monotonicity.
3. Z-score bucket response → persistence vs mean-reversion shape.
4. Residual autocorrelation → temporal-structure diagnostic.
5. AR(1/3/5) OOS benchmarks → baseline any later challenger must beat.

### Diagnostic decision gate

1. **Residual structure:** best $|	ext{rank IC}| \ge 0.01$, monthly IC sign-stable, and monotonic decile spread.
2. **Autocorrelation:** $|
ho_1| \ge 0.02$.
3. **AR power:** best AR out-of-sample $R^2 > 0$.

A pass here means continue to stale-mark, transaction-cost, and MSRB executable-opportunity tests. It does not approve LongConv or a trading strategy; later sections supersede this diagnostic gate.

> Caveat: 110 daily dates is short. Overlapping $h>1$ horizons inflate t-stats and lag-20 / AR estimates are noisy; results are treated as **indicative**, not definitive.

In [ ]:
# ── 9.1 RESIDUAL FEATURE PANEL ──────────────────────────────────────────
from scipy.stats import rankdata

_section9_start = time.perf_counter()
log_runtime("START Section 9: residual alpha validation")

# Config (scaled to the ~110-date sample).
FWD_HORIZONS = [1, 3, 5]
MA_WINDOWS = [5, 20, 60]
VOL_WINDOW = 60
ZSCORE_WINDOW = 60
AUTOCORR_MAX_LAG = 20
AR_ORDERS = [1, 3, 5]
MIN_OBS_UNIVERSE = 60
N_DECILES = 10
ANN = 252
RANK_IC_GATE = 0.01

_obs_all = np.isfinite(resid_panel)
_uni = _obs_all.sum(axis=1) >= MIN_OBS_UNIVERSE
rv_cusips = R_panel.index[_uni]
rv_dates = panel_dates
Rp = resid_panel[_uni]                     # N_uni x T residual matrix (NaN where unobserved)
N_uni, T_rv = Rp.shape
print(f"Residual universe: {N_uni:,} bonds with >= {MIN_OBS_UNIVERSE} obs "
      f"(of {resid_panel.shape[0]:,}); dates={T_rv}")

# Cumulative sums / counts → all trailing and forward windows in O(1) per date.
_obs_u = np.isfinite(Rp)
_Rz = np.where(_obs_u, Rp, 0.0)
_C1 = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_Rz, axis=1)], axis=1)
_C2 = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_Rz * _Rz, axis=1)], axis=1)
_Cn = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_obs_u.astype(float), axis=1)], axis=1)
_tt = np.arange(T_rv)


def _trailing_mean(w, min_p):
    lo = np.clip(_tt + 1 - w, 0, T_rv); hi = _tt + 1
    s = _C1[:, hi] - _C1[:, lo]; n = _Cn[:, hi] - _Cn[:, lo]
    return np.where(n >= min_p, s / np.where(n == 0, 1, n), np.nan)


def _trailing_std(w, min_p):
    lo = np.clip(_tt + 1 - w, 0, T_rv); hi = _tt + 1
    s = _C1[:, hi] - _C1[:, lo]; s2 = _C2[:, hi] - _C2[:, lo]; n = _Cn[:, hi] - _Cn[:, lo]
    nn = np.where(n == 0, 1, n)
    var = np.clip(s2 / nn - (s / nn) ** 2, 0, None)
    return np.where(n >= min_p, np.sqrt(var), np.nan)


def _forward_sum(h):
    lo = _tt + 1; hi = np.clip(_tt + 1 + h, 0, T_rv)
    s = _C1[:, hi] - _C1[:, lo]; n = _Cn[:, hi] - _Cn[:, lo]
    return np.where(n == h, s, np.nan)     # require a full h-observation forward window


rv_fwd = {h: _forward_sum(h) for h in FWD_HORIZONS}
_ma = {w: _trailing_mean(w, max(3, w // 3)) for w in MA_WINDOWS}
_vol = _trailing_std(VOL_WINDOW, 20)
with np.errstate(divide='ignore', invalid='ignore'):
    rv_zscore = np.where(_vol > 0, Rp / _vol, np.nan)

# Signals A–E (each N_uni x T), defined only where the anchor residual is observed.
rv_signals = {
    'Residual (A)': np.where(_obs_u, Rp, np.nan),
    'Neg-Residual (B)': np.where(_obs_u, -Rp, np.nan),
    'Residual Z (C)': rv_zscore,
    'MA20 (D)': _ma[20],
    'MA60 (E)': _ma[60],
}
rv_month = pd.PeriodIndex(rv_dates, freq='M')
print(f"Signals built: {list(rv_signals)} | horizons={FWD_HORIZONS} | "
      f"forward-return coverage h=1: {int(np.isfinite(rv_fwd[1]).sum()):,} bond-days")
del _Rz, _C1, _C2, _Cn


[2026-09-15 18:00:38] START Section 9: residual alpha validation
Residual universe: 229,857 bonds with >= 60 obs (of 235,722); dates=114
Signals built: ['Residual (A)', 'Neg-Residual (B)', 'Residual Z (C)', 'MA20 (D)', 'MA60 (E)'] | horizons=[1, 3, 5] | forward-return coverage h=1: 25,580,488 bond-days


### 9.2 Stage 1 — Simple Residual Signals (IC)

For each signal and horizon $h$, compute the daily cross-sectional Information Coefficient and average it over dates:

$$\text{IC}_t = \operatorname{corr}_i\big(\text{signal}_{i,t},\, \textstyle\sum_{k=1}^{h}\varepsilon_{i,t+k}\big),\qquad \overline{\text{IC}} = \tfrac{1}{T}\sum_t \text{IC}_t,\qquad t\text{-stat} = \sqrt{T}\,\overline{\text{IC}}/\operatorname{std}(\text{IC}_t).$$

Both Pearson IC and Spearman **rank IC** are reported, plus a pooled hit rate ($\operatorname{sign}(\text{signal})=\operatorname{sign}(\text{future residual})$) and monthly rank-IC stability. A **negative** IC on the raw residual signal indicates **mean reversion** (high residual → negative future residual); the $-\varepsilon_t$ signal flips the sign so a positive IC there is the tradable reversion signal.

In [ ]:
# ── 9.2 STAGE 1: CROSS-SECTIONAL IC OF SIMPLE RESIDUAL SIGNALS ──────────
def _pearson(a, b):
    if a.size < 10 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _daily_ic(sig, fwd, spearman=False):
    ics = np.full(T_rv, np.nan)
    for t in range(T_rv):
        a, b = sig[:, t], fwd[:, t]
        m = np.isfinite(a) & np.isfinite(b)
        if m.sum() < 10:
            continue
        av, bv = a[m], b[m]
        if spearman:
            av, bv = rankdata(av), rankdata(bv)
        ics[t] = _pearson(av, bv)
    return ics


def _ic_tstat(x):
    x = x[np.isfinite(x)]
    return float(np.sqrt(x.size) * x.mean() / x.std()) if x.size > 2 and x.std() > 0 else np.nan


_ic_rows = []
rv_daily_ic = {}
with timed_block("Stage 1 cross-sectional IC"):
    for sname, sig in rv_signals.items():
        for h in FWD_HORIZONS:
            fwd = rv_fwd[h]
            p_ic = _daily_ic(sig, fwd, spearman=False)
            s_ic = _daily_ic(sig, fwd, spearman=True)
            rv_daily_ic[(sname, h, 'pearson')] = p_ic
            rv_daily_ic[(sname, h, 'spearman')] = s_ic
            m = np.isfinite(sig) & np.isfinite(fwd) & (sig != 0) & (fwd != 0)
            hit = float(np.mean(np.sign(sig[m]) == np.sign(fwd[m]))) if m.any() else np.nan
            _ic_rows.append({
                'signal': sname, 'horizon': f'{h}d',
                'pearson_IC': np.nanmean(p_ic), 'pearson_t': _ic_tstat(p_ic),
                'rank_IC': np.nanmean(s_ic), 'rank_IC_t': _ic_tstat(s_ic),
                'hit_rate': hit, 'n_dates': int(np.isfinite(s_ic).sum()),
            })

ic_summary = pd.DataFrame(_ic_rows).set_index(['signal', 'horizon'])
print("Stage 1 — cross-sectional IC (daily IC averaged over dates; t from daily-IC series):")
print("  (h>1 windows overlap and inflate t-stats; treat as indicative on this short sample.)")
display(ic_summary.style.format({'pearson_IC': '{:.4f}', 'pearson_t': '{:.2f}',
                                 'rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}',
                                 'hit_rate': '{:.3f}', 'n_dates': '{:.0f}'}))

# Monthly rank-IC stability at h=1.
monthly_ic_df = pd.DataFrame({
    sname: pd.Series(rv_daily_ic[(sname, 1, 'spearman')], index=rv_month).groupby(level=0).mean()
    for sname in rv_signals
})
print("\nMonthly mean rank-IC (h=1) by signal — sign-stability check:")
display(monthly_ic_df.style.format('{:.4f}').background_gradient(cmap='RdBu', vmin=-0.1, vmax=0.1))


[2026-09-15 18:00:43] START Stage 1 cross-sectional IC
[2026-09-15 18:02:32] END Stage 1 cross-sectional IC | elapsed 1m 48.67s
Stage 1 — cross-sectional IC (daily IC averaged over dates; t from daily-IC series):
  (h>1 windows overlap and inflate t-stats; treat as indicative on this short sample.)


<HTML output>

<HTML output>

Monthly mean rank-IC (h=1) by signal — sign-stability check:


<HTML output>

,Residual (A),Neg-Residual (B),Residual Z (C),MA20 (D),MA60 (E)
date,,,,,
2026-04,0.2277,-0.2277,0.2533,-0.0116,0.2344
2026-05,0.2233,-0.2233,0.2212,0.0033,0.0193
2026-06,0.0610,-0.0610,0.0784,0.0639,0.0824
2026-07,0.2892,-0.2892,0.2642,0.1999,0.1638
2026-08,0.2615,-0.2615,0.2449,-0.0014,0.0694
2026-09,0.3001,-0.3001,0.3016,0.3194,0.3233


### 9.3 Stage 2 — Decile Portfolio Tests

Each date, rank bonds by the signal into deciles D1…D10 (equal-weight) and average the realised future residual per decile, pooled over dates. A monotone profile — increasing (momentum) or decreasing (reversion) — is stronger evidence than the raw D10−D1 spread. The annualised spread Sharpe uses the daily D10−D1 series ($\sqrt{252/h}$ scaling for the $h$-day horizon).

In [ ]:
# ── 9.3 STAGE 2: DECILE PORTFOLIOS ──────────────────────────────────────
def _decile_profile(sig, fwd, n_bins=N_DECILES):
    sums = np.zeros(n_bins); counts = np.zeros(n_bins); spread = []
    for t in range(T_rv):
        a, b = sig[:, t], fwd[:, t]
        m = np.isfinite(a) & np.isfinite(b)
        if m.sum() < n_bins * 5:
            continue
        av, bv = a[m], b[m]
        ranks = rankdata(av, method='average')
        d = np.clip(((ranks - 1) / len(ranks) * n_bins).astype(int), 0, n_bins - 1)
        for g in range(n_bins):
            gb = bv[d == g]
            if gb.size:
                sums[g] += gb.sum(); counts[g] += gb.size
        if (d == n_bins - 1).any() and (d == 0).any():
            spread.append(bv[d == n_bins - 1].mean() - bv[d == 0].mean())
    avg = np.where(counts > 0, sums / np.where(counts == 0, 1, counts), np.nan)
    return avg, np.array(spread)


_decile_h = 5
decile_tbl = {}
_spread_rows = []
with timed_block("Stage 2 decile portfolios"):
    for sname, sig in rv_signals.items():
        decile_tbl[sname], _ = _decile_profile(sig, rv_fwd[_decile_h])
        for h in FWD_HORIZONS:
            _, sp = _decile_profile(sig, rv_fwd[h])
            sh = (float(np.sqrt(ANN / h) * np.nanmean(sp) / np.nanstd(sp))
                  if np.isfinite(sp).sum() > 2 and np.nanstd(sp) > 0 else np.nan)
            _spread_rows.append({'signal': sname, 'horizon': f'{h}d',
                                 'D10_minus_D1': np.nanmean(sp), 'spread_sharpe': sh})

decile_profile_df = pd.DataFrame(decile_tbl, index=[f'D{i+1}' for i in range(N_DECILES)])
spread_df = pd.DataFrame(_spread_rows).set_index(['signal', 'horizon'])
print(f"Stage 2 — decile-average future {_decile_h}d residual (equal-weight, pooled over dates):")
display(decile_profile_df.style.format('{:.6f}').background_gradient(cmap='RdBu', axis=0))
print("\nD10 − D1 spread and annualized spread Sharpe by horizon:")
display(spread_df.style.format({'D10_minus_D1': '{:.6f}', 'spread_sharpe': '{:.2f}'}))

fig, ax = plt.subplots(figsize=(11, 5))
for sname in rv_signals:
    ax.plot(range(1, N_DECILES + 1), decile_profile_df[sname], marker='o', label=sname)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xlabel('Signal decile (D1 → D10)'); ax.set_ylabel(f'Avg future {_decile_h}d residual')
ax.set_title('Decile monotonicity: signal vs future residual'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


[2026-09-15 18:02:32] START Stage 2 decile portfolios


[2026-09-15 18:03:50] END Stage 2 decile portfolios | elapsed 1m 18.67s
Stage 2 — decile-average future 5d residual (equal-weight, pooled over dates):


<HTML output>

,Residual (A),Neg-Residual (B),Residual Z (C),MA20 (D),MA60 (E)
D1,-0.000610,0.000542,-0.000486,-0.000262,-0.000331
D2,-0.000443,0.000363,-0.000490,-0.000136,-0.000250
D3,-0.000327,0.000133,-0.000332,-0.000217,-0.000260
D4,-0.000225,0.000009,-0.000250,-0.000175,-0.000146
D5,-0.000163,-0.000091,-0.000188,-0.000104,-0.000120
D6,-0.000091,-0.000163,-0.000068,-0.000089,-0.000077
D7,0.000009,-0.000225,0.000074,-0.000050,-0.000000
D8,0.000133,-0.000327,0.000275,0.000044,0.000089
D9,0.000363,-0.000443,0.000513,0.000140,0.000139
D10,0.000542,-0.000610,0.000251,0.000045,0.000257


D10 − D1 spread and annualized spread Sharpe by horizon:


<HTML output>

<HTML output>

<Figure: muni_characteristic_factor_export_49_5.png (image not included in markdown export)>

    


### 9.4 Stage 3 — Residual Z-Score Bucket Response

Pool all bond-dates, bucket by current residual z-score ($<-3,\,-3..-2,\,\dots,\,>3$), and average the future residual in each bucket. A **downward** slope (high z → negative future residual) is mean reversion; an **upward** slope is momentum; a flat line is no structure.

In [ ]:
# ── 9.4 STAGE 3: RESIDUAL Z-SCORE BUCKET RESPONSE ───────────────────────
_z_edges = [-np.inf, -3, -2, -1, 0, 1, 2, 3, np.inf]
_z_labels = ['<-3', '-3..-2', '-2..-1', '-1..0', '0..1', '1..2', '2..3', '>3']
_bucket_cols = []
for h in FWD_HORIZONS:
    m = np.isfinite(rv_zscore) & np.isfinite(rv_fwd[h])
    zc = pd.cut(rv_zscore[m], bins=_z_edges, labels=_z_labels)
    _bucket_cols.append(pd.Series(rv_fwd[h][m]).groupby(zc, observed=False).mean().rename(f'{h}d'))
zbucket_df = pd.concat(_bucket_cols, axis=1)
_zc_all = pd.cut(rv_zscore[np.isfinite(rv_zscore)], bins=_z_edges, labels=_z_labels)
zbucket_df['count'] = _zc_all.value_counts().reindex(_z_labels)
print("Stage 3 — average future residual by current z-score bucket (downward slope → mean reversion):")
_zfmt = {f'{h}d': '{:.6f}' for h in FWD_HORIZONS}
_zfmt['count'] = '{:,.0f}'
display(zbucket_df.style.format(_zfmt))

fig, ax = plt.subplots(figsize=(11, 5))
_xmid = list(range(len(_z_labels)))
for h in FWD_HORIZONS:
    ax.plot(_xmid, zbucket_df[f'{h}d'], marker='o', label=f'future {h}d')
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xticks(_xmid); ax.set_xticklabels(_z_labels, rotation=45)
ax.set_xlabel('Current residual z-score bucket'); ax.set_ylabel('Avg future residual')
ax.set_title('Z-score bucket vs future residual'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


Stage 3 — average future residual by current z-score bucket (downward slope → mean reversion):


<HTML output>

,1d,3d,5d,count
<-3,-0.000277,-0.000305,-0.000555,"336,160"
-3..-2,-0.000380,-0.000581,-0.000701,"572,230"
-2..-1,-0.000218,-0.000422,-0.000511,"2,065,318"
-1..0,-0.000105,-0.000212,-0.000268,"8,512,010"
0..1,0.000082,0.000143,0.000185,"7,420,851"
1..2,0.000246,0.000484,0.000386,"1,721,016"
2..3,0.000409,0.000492,0.000597,"523,238"
>3,0.000259,-0.000084,0.000107,"286,646"


<Figure: muni_characteristic_factor_export_51_2.png (image not included in markdown export)>

    


### 9.5 Stage 4 — Residual Autocorrelation Structure

Pool across bonds and compute the residual autocorrelation at each lag $k=1..20$:

$$\rho_k = \operatorname{corr}\big(\varepsilon_{i,t},\, \varepsilon_{i,t+k}\big)\quad\text{over all valid } (i,t).$$

- **Flat** ($\rho_k \approx 0$): no temporal structure → residuals ≈ white noise.
- **Negative short lag**: mean reversion.
- **Decaying positive**: momentum / persistence.

In [ ]:
# ── 9.5 STAGE 4: RESIDUAL AUTOCORRELATION STRUCTURE ─────────────────────
_ac = []
_per_bond_lag1 = []
with timed_block("Stage 4 autocorrelation"):
    for k in range(1, AUTOCORR_MAX_LAG + 1):
        a = Rp[:, :T_rv - k].ravel(); b = Rp[:, k:].ravel()
        m = np.isfinite(a) & np.isfinite(b)
        _ac.append(_pearson(a[m], b[m]) if m.sum() > 100 else np.nan)
    # Per-bond lag-1 autocorrelation (averaged) for the decision gate.
    for i in range(N_uni):
        a = Rp[i, :-1]; b = Rp[i, 1:]
        m = np.isfinite(a) & np.isfinite(b)
        if m.sum() >= 20 and np.std(a[m]) > 0 and np.std(b[m]) > 0:
            _per_bond_lag1.append(np.corrcoef(a[m], b[m])[0, 1])

autocorr_df = pd.DataFrame({'lag': range(1, AUTOCORR_MAX_LAG + 1), 'autocorr': _ac}).set_index('lag')
_mean_bond_lag1 = float(np.mean(_per_bond_lag1)) if _per_bond_lag1 else np.nan
print("Stage 4 — pooled residual autocorrelation by lag (across all bonds):")
display(autocorr_df.T.style.format('{:.4f}'))
print(f"Pooled lag-1 autocorr: {autocorr_df.loc[1, 'autocorr']:.4f} | "
      f"mean per-bond lag-1 autocorr ({len(_per_bond_lag1):,} bonds): {_mean_bond_lag1:.4f}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(autocorr_df.index, autocorr_df['autocorr'], color='tab:purple', alpha=0.75)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Lag k (trading days)'); ax.set_ylabel(r'corr($\varepsilon_t,\ \varepsilon_{t+k}$)')
ax.set_title('Residual autocorrelation (negative lag-1 → mean reversion)'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


[2026-09-15 18:03:54] START Stage 4 autocorrelation
[2026-09-15 18:04:26] END Stage 4 autocorrelation | elapsed 31.99s
Stage 4 — pooled residual autocorrelation by lag (across all bonds):


<HTML output>

lag,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
autocorr,0.1754,0.0440,0.0012,0.0146,0.0301,0.0265,0.0164,-0.0110,-0.0381,0.0297,-0.0035,0.0187,-0.0059,-0.0405,-0.0455,0.0372,0.0399,0.0278,-0.0309,-0.0085


Pooled lag-1 autocorr: 0.1754 | mean per-bond lag-1 autocorr (229,857 bonds): 0.1714


<Figure: muni_characteristic_factor_export_53_3.png (image not included in markdown export)>

    


### 9.6 Stage 5 — AR Benchmarks (the bar LongConv must beat)

Fit **pooled panel** AR($p$) models that predict the next-day residual from its own lags,

$$\varepsilon_{i,t+1} = a + \sum_{l=0}^{p-1} b_l\,\varepsilon_{i,t-l} + u_{i,t+1},\qquad p\in\{1,3,5\},$$

pooling all bonds and estimating by OLS on a **time-based train window** (first 70% of dates), then forecasting the held-out tail. Report out-of-sample $R^2$ (uncentered, vs the zero-return benchmark), cross-sectional IC / rank IC of the forecast, the decile D10−D1 spread, and its annualised Sharpe. These are the baselines a LongConv model must **materially** beat to be worth building.

In [ ]:
# ── 9.6 STAGE 5: POOLED AR BENCHMARKS (OOS) ─────────────────────────────
_split_t = int(np.floor(T_rv * 0.7))
print(f"AR OOS split at date index {_split_t} ({rv_dates[_split_t].date()}): "
      f"train t<{_split_t}, test t>={_split_t}")


def _build_ar(Rp, p):
    X_parts, y_parts, t_parts = [], [], []
    for tt in range(p, T_rv):
        y = Rp[:, tt]
        lags = np.column_stack([Rp[:, tt - 1 - l] for l in range(p)])   # lag1..lagp
        v = np.isfinite(y) & np.all(np.isfinite(lags), axis=1)
        if v.any():
            X_parts.append(lags[v]); y_parts.append(y[v]); t_parts.append(np.full(int(v.sum()), tt))
    return np.vstack(X_parts), np.concatenate(y_parts), np.concatenate(t_parts)


def _ar_eval(Rp, p):
    X, y, tv = _build_ar(Rp, p)
    tr, te = tv < _split_t, tv >= _split_t
    Xtr = np.column_stack([np.ones(int(tr.sum())), X[tr]])
    Xte = np.column_stack([np.ones(int(te.sum())), X[te]])
    beta, *_ = np.linalg.lstsq(Xtr, y[tr], rcond=None)
    yhat, yte, tte = Xte @ beta, y[te], tv[te]
    oos_r2 = 1.0 - np.sum((yte - yhat) ** 2) / np.sum(yte ** 2)     # uncentered, vs zero
    ics, rics, spreads = [], [], []
    for tt in np.unique(tte):
        mk = tte == tt
        s, aret = yhat[mk], yte[mk]
        if s.size < N_DECILES * 5:
            continue
        ics.append(_pearson(s, aret)); rics.append(_pearson(rankdata(s), rankdata(aret)))
        ranks = rankdata(s, method='average')
        d = np.clip(((ranks - 1) / len(ranks) * N_DECILES).astype(int), 0, N_DECILES - 1)
        spreads.append(aret[d == N_DECILES - 1].mean() - aret[d == 0].mean())
    sp = np.array(spreads)
    sharpe = (float(np.sqrt(ANN) * np.nanmean(sp) / np.nanstd(sp))
              if np.isfinite(sp).sum() > 2 and np.nanstd(sp) > 0 else np.nan)
    return {'AR_order': p, 'oos_R2': oos_r2, 'IC': np.nanmean(ics), 'rank_IC': np.nanmean(rics),
            'D10_minus_D1': np.nanmean(sp), 'spread_sharpe': sharpe,
            'train_obs': int(tr.sum()), 'test_obs': int(te.sum())}


ar_rows = []
with timed_block("Stage 5 pooled AR benchmarks"):
    for p in AR_ORDERS:
        ar_rows.append(_ar_eval(Rp, p))
ar_summary = pd.DataFrame(ar_rows).set_index('AR_order')
print("Stage 5 — pooled AR(p) out-of-sample (predict next-day residual):")
display(ar_summary.style.format({'oos_R2': '{:.4f}', 'IC': '{:.4f}', 'rank_IC': '{:.4f}',
                                 'D10_minus_D1': '{:.6f}', 'spread_sharpe': '{:.2f}',
                                 'train_obs': '{:,.0f}', 'test_obs': '{:,.0f}'}))


AR OOS split at date index 79 (2026-07-27): train t<79, test t>=79
[2026-09-15 18:04:26] START Stage 5 pooled AR benchmarks
[2026-09-15 18:04:49] END Stage 5 pooled AR benchmarks | elapsed 22.51s
Stage 5 — pooled AR(p) out-of-sample (predict next-day residual):


<HTML output>

,oos_R2,IC,rank_IC,D10_minus_D1,spread_sharpe,train_obs,test_obs
AR_order,,,,,,,
1,0.0377,0.1716,0.2600,0.000948,8.72,"17,736,091","7,838,792"
3,0.0140,0.1263,0.2064,0.000661,6.01,"17,276,363","7,838,783"
5,0.0118,0.1214,0.1952,0.000624,5.67,"16,816,637","7,838,775"


### 9.7 Model Comparison and Diagnostic Decision Gate

The comparison table collects IC, rank IC, D10-D1 spread, and Sharpe at $h=1$ for every simple residual signal and AR benchmark. LongConv is not approved here. The gate only evaluates whether the residual panel has enough temporal structure to justify the next diagnostic layer:

1. **Residual structure** - best $|\text{rank IC}| \ge 0.01$, monthly IC sign-stable, and monotone deciles.
2. **Autocorrelation** - $|\rho_1| \ge 0.02$.
3. **AR power** - best AR out-of-sample $R^2 > 0$.

A pass here means **continue to stale-mark, cost, and MSRB validation**. It is superseded by Sections 11, 15, 16, and 18. Simple signals are parameter-free diagnostics on the full-sample residual panel; AR rows use a time split but still inherit the full-sample residual construction caveat.

In [ ]:
# -- 9.7 MODEL COMPARISON AND DIAGNOSTIC DECISION GATE -------------------
_cmp_rows = []
for sname in rv_signals:
    _, sp = _decile_profile(rv_signals[sname], rv_fwd[1])
    sh = (float(np.sqrt(ANN) * np.nanmean(sp) / np.nanstd(sp))
          if np.isfinite(sp).sum() > 2 and np.nanstd(sp) > 0 else np.nan)
    _cmp_rows.append({'model': sname,
                      'IC': np.nanmean(rv_daily_ic[(sname, 1, 'pearson')]),
                      'rank_IC': np.nanmean(rv_daily_ic[(sname, 1, 'spearman')]),
                      'D10_minus_D1': np.nanmean(sp), 'sharpe': sh,
                      'basis': 'diagnostic full-sample residual'})
for p in AR_ORDERS:
    r = ar_summary.loc[p]
    _cmp_rows.append({'model': f'AR({p})', 'IC': r['IC'], 'rank_IC': r['rank_IC'],
                      'D10_minus_D1': r['D10_minus_D1'], 'sharpe': r['spread_sharpe'],
                      'basis': 'OOS split; full-sample residual'})
_cmp_rows.append({'model': 'LongConv', 'IC': np.nan, 'rank_IC': np.nan,
                  'D10_minus_D1': np.nan, 'sharpe': np.nan, 'basis': 'not approved here'})
comparison_df = pd.DataFrame(_cmp_rows).set_index('model')
print("=== RESIDUAL STRUCTURE MODEL COMPARISON (h=1; diagnostic residuals) ===")
display(comparison_df.style.format({'IC': '{:.4f}', 'rank_IC': '{:.4f}',
                                    'D10_minus_D1': '{:.6f}', 'sharpe': '{:.2f}'}, na_rep='--'))

_best_rank_ic = float(np.nanmax([abs(np.nanmean(rv_daily_ic[(s, 1, 'spearman')])) for s in rv_signals]))
_best_sign_stab = float(monthly_ic_df.apply(
    lambda c: abs(np.sign(c.dropna()).mean()) if c.notna().any() else 0.0).max())
_lag1 = float(autocorr_df.loc[1, 'autocorr'])
_mono = float(max(abs(_pearson(rankdata(np.arange(N_DECILES)), rankdata(decile_profile_df[s].values)))
                  for s in rv_signals if np.isfinite(decile_profile_df[s].values).all()))
_best_ar_r2 = float(ar_summary['oos_R2'].max())

cond1 = (_best_rank_ic >= RANK_IC_GATE) and (_best_sign_stab >= 0.6) and (_mono >= 0.7)
cond2 = abs(_lag1) >= 0.02
cond3 = _best_ar_r2 > 0
print("\n=== DIAGNOSTIC DECISION GATE ===")
print(f"  1 residual structure: best|rankIC|={_best_rank_ic:.4f} (>={RANK_IC_GATE}), "
      f"monthly sign-stability={_best_sign_stab:.2f} (>=0.60), decile monotonicity={_mono:.2f} (>=0.70) -> {'PASS' if cond1 else 'FAIL'}")
print(f"  2 temporal structure: |lag-1 autocorr|={abs(_lag1):.4f} (>=0.02) -> {'PASS' if cond2 else 'FAIL'}")
print(f"  3 AR OOS power      : best AR OOS R2={_best_ar_r2:.4f} (>0) -> {'PASS' if cond3 else 'FAIL'}")
_verdict = ('CONTINUE to stale-mark, cost, and MSRB validation' if (cond1 and cond2 and cond3)
            else 'STOP complex-model work for now; residuals look too weak on this diagnostic gate')
print(f"\n  VERDICT: {_verdict}")
print("  NOTE: This gate is superseded by Sections 11, 15, 16, and 18; it does not approve LongConv or production trading.")
log_runtime("END Section 9: residual alpha validation", _section9_start)

=== RESIDUAL ALPHA MODEL COMPARISON (h=1) ===


<HTML output>

,IC,rank_IC,D10_minus_D1,sharpe,basis
model,,,,,
Residual (A),0.1233,0.2193,0.000650,7.39,full-sample
Neg-Residual (B),-0.1233,-0.2193,-0.000650,-7.39,full-sample
Residual Z (C),0.1155,0.2122,0.000440,8.02,full-sample
MA20 (D),0.0222,0.0754,0.000184,2.11,full-sample
MA60 (E),0.0358,0.1089,0.000231,3.02,full-sample
AR(1),0.1716,0.2600,0.000948,8.72,OOS test
AR(3),0.1263,0.2064,0.000661,6.01,OOS test
AR(5),0.1214,0.1952,0.000624,5.67,OOS test
LongConv,—,—,—,—,pending


=== RESEARCH DECISION GATE ===
  1 residual alpha : best|rankIC|=0.2193 (>=0.01), monthly sign-stability=1.00 (>=0.60), decile monotonicity=1.00 (>=0.70) -> PASS
  2 temporal struct: |lag-1 autocorr|=0.1754 (>=0.02) -> PASS
  3 AR OOS power   : best AR OOS R2=0.0377 (>0) -> PASS

  VERDICT: PROCEED to LongConv
[2026-09-15 18:05:08] END Section 9: residual alpha validation | elapsed 4m 30.82s


### 9.8 ARX — Residual Autoregression with Characteristics

Extend the pure AR baseline with **exogenous** static bond characteristics (an ARX model). Predict the **forward residual sum** — never raw returns — from a long residual-lag history plus the current characteristics:

$$\hat{y}_{i,t} = \beta_0 + \sum_{l=0}^{60}\beta_l\,\varepsilon_{i,t-l} + \gamma' z_{i,t},\qquad y_{i,t} = \sum_{k=1}^{H}\varepsilon_{i,t+k},\quad H\in\{5,10\}.$$

- **Residual lags:** $X_t = [\varepsilon_t, \varepsilon_{t-1}, \dots, \varepsilon_{t-60}]$ (61 lags).
- **Characteristics $z_t$:** static one-hot **rating**, **tenor**, **call**, and **coupon** bucket dummies (they are time-invariant per bond, so this adds bucket-level structure the pure AR cannot see).
- **Estimation:** pooled OLS with the same time-based train/test split as Stage 5; AR(1), AR(60), and ARX(lags+chars) are fit on the **identical rows** (bonds with a complete 61-lag window) for an apples-to-apples out-of-sample comparison.

**Question:** do the characteristics add predictive power over AR alone — i.e. is residual drift partly explained by bond type (a sign the K=3 factor model missed structure), or is the whole signal just short-lag autocorrelation? LongConv stays **parked** until ARX shows the characteristics help.

> Caveats: the 61-lag window is data-thin on 110 dates (deep-lag coefficients are noisy), the static-characteristic terms act as **bucket fixed effects**, and — as in Stage 4 — some of the short-lag predictability likely reflects **stale evaluated-mark microstructure** rather than tradable alpha.

In [ ]:
# ── 9.8 ARX: RESIDUAL AUTOREGRESSION + CHARACTERISTICS ──────────────────
from scipy.stats import rankdata

_arx_start = time.perf_counter()
log_runtime("START Section 9.8: ARX residual model")

ARX_MAX_LAG = 60
ARX_TARGETS = [5, 10]
ARX_CHAR_PREFIXES = ('rating_grade_', 'tenor_bucket_', 'call_bucket_', 'coupon_bucket_')

# Self-contained references (robust to kernel state).
_uni_arx = np.isfinite(resid_panel).sum(axis=1) >= MIN_OBS_UNIVERSE
_split_arx = int(np.floor(T_rv * 0.7))


def _pearson_arx(a, b):
    if a.size < 10 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


# Static one-hot characteristics per universe bond (buckets are time-invariant).
_char_idx = [i for i, c in enumerate(characteristics) if c.startswith(ARX_CHAR_PREFIXES)]
_Zc = np.zeros((resid_panel.shape[0], len(_char_idx)), dtype=np.float64)
for _j, _ci in enumerate(_char_idx):
    _Zc[:, _j] = Z[:, _ci, :].max(axis=1)      # 1 if the bond ever carries that bucket
Z_uni_chars = _Zc[_uni_arx]
print(f"ARX design: 61 residual lags + {len(_char_idx)} static char dummies "
      f"(rating/tenor/call/coupon); OOS split at {rv_dates[_split_arx].date()}")

# Forward residual-sum targets on the universe residual matrix.
_obs_u = np.isfinite(Rp)
_Rz = np.where(_obs_u, Rp, 0.0)
_C1 = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_Rz, axis=1)], axis=1)
_Cn = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_obs_u.astype(float), axis=1)], axis=1)


def _fwd_sum(h):
    lo = np.arange(T_rv) + 1; hi = np.clip(np.arange(T_rv) + 1 + h, 0, T_rv)
    s = _C1[:, hi] - _C1[:, lo]; n = _Cn[:, hi] - _Cn[:, lo]
    return np.where(n == h, s, np.nan)


def _build_arx(H):
    fwdH = _fwd_sum(H)
    Xp, yp, tp = [], [], []
    for tt in range(ARX_MAX_LAG, T_rv - H):
        y = fwdH[:, tt]
        lags = np.column_stack([Rp[:, tt - l] for l in range(ARX_MAX_LAG + 1)])   # eps_t .. eps_{t-60}
        v = np.isfinite(y) & np.all(np.isfinite(lags), axis=1)
        if v.any():
            nv = int(v.sum())
            Xp.append(np.hstack([np.ones((nv, 1)), lags[v], Z_uni_chars[v]]))
            yp.append(y[v]); tp.append(np.full(nv, tt))
    return np.vstack(Xp), np.concatenate(yp), np.concatenate(tp)


def _eval_cols(X, y, tv, cols, H):
    tr, te = tv < _split_arx, tv >= _split_arx
    beta, *_ = np.linalg.lstsq(X[tr][:, cols], y[tr], rcond=None)
    yhat, yte, tte = X[te][:, cols] @ beta, y[te], tv[te]
    oos_r2 = 1.0 - np.sum((yte - yhat) ** 2) / np.sum(yte ** 2)
    ics, rics, spreads = [], [], []
    for tt in np.unique(tte):
        mk = tte == tt; s, aret = yhat[mk], yte[mk]
        if s.size < N_DECILES * 5:
            continue
        ics.append(_pearson_arx(s, aret)); rics.append(_pearson_arx(rankdata(s), rankdata(aret)))
        rk = rankdata(s, method='average'); d = np.clip(((rk - 1) / len(rk) * N_DECILES).astype(int), 0, N_DECILES - 1)
        spreads.append(aret[d == N_DECILES - 1].mean() - aret[d == 0].mean())
    sp = np.array(spreads)
    sharpe = (float(np.sqrt(ANN / H) * np.nanmean(sp) / np.nanstd(sp))
              if np.isfinite(sp).sum() > 2 and np.nanstd(sp) > 0 else np.nan)
    return {'oos_R2': oos_r2, 'IC': np.nanmean(ics), 'rank_IC': np.nanmean(rics),
            'D10_minus_D1': np.nanmean(sp), 'spread_sharpe': sharpe}


_n_lag = ARX_MAX_LAG + 1
_model_cols = {'AR(1)': [0, 1], f'AR({ARX_MAX_LAG})': list(range(1 + _n_lag))}
_arx_rows = []
with timed_block("Stage ARX fit/eval"):
    for H in ARX_TARGETS:
        X, y, tv = _build_arx(H)
        cols_map = dict(_model_cols); cols_map['ARX(lags+chars)'] = list(range(X.shape[1]))
        for label, cols in cols_map.items():
            row = _eval_cols(X, y, tv, cols, H)
            row.update({'target': f'{H}d_fwd', 'model': label,
                        'train_obs': int((tv < _split_arx).sum()), 'test_obs': int((tv >= _split_arx).sum())})
            _arx_rows.append(row)
        del X, y, tv

arx_summary = pd.DataFrame(_arx_rows).set_index(['target', 'model'])[
    ['oos_R2', 'IC', 'rank_IC', 'D10_minus_D1', 'spread_sharpe', 'train_obs', 'test_obs']]
print("ARX vs AR benchmarks (predict forward residual sum; identical rows; pooled OOS):")
display(arx_summary.style.format({'oos_R2': '{:.4f}', 'IC': '{:.4f}', 'rank_IC': '{:.4f}',
                                  'D10_minus_D1': '{:.6f}', 'spread_sharpe': '{:.2f}',
                                  'train_obs': '{:,.0f}', 'test_obs': '{:,.0f}'}))

print("\nDoes ARX add value over AR on the same target?")
for H in ARX_TARGETS:
    a1 = arx_summary.loc[(f'{H}d_fwd', 'AR(1)')]
    af = arx_summary.loc[(f'{H}d_fwd', f'AR({ARX_MAX_LAG})')]
    ax = arx_summary.loc[(f'{H}d_fwd', 'ARX(lags+chars)')]
    print(f"  {H}d: OOS R²  AR(1)={a1['oos_R2']:.4f}  AR(60)={af['oos_R2']:.4f}  ARX={ax['oos_R2']:.4f} "
          f"(ARX−AR1 Δ={ax['oos_R2']-a1['oos_R2']:+.4f}) | "
          f"rankIC  AR(1)={a1['rank_IC']:.4f}  ARX={ax['rank_IC']:.4f} (Δ={ax['rank_IC']-a1['rank_IC']:+.4f})")

del _Zc, _Rz, _C1, _Cn
log_runtime("END Section 9.8 ARX residual model", _arx_start)


[2026-09-15 18:05:09] START Section 9.8: ARX residual model
ARX design: 61 residual lags + 42 static char dummies (rating/tenor/call/coupon); OOS split at 2026-07-27
[2026-09-15 18:05:10] START Stage ARX fit/eval
[2026-09-15 18:08:00] END Stage ARX fit/eval | elapsed 2m 50.02s
ARX vs AR benchmarks (predict forward residual sum; identical rows; pooled OOS):


<HTML output>

<HTML output>

Does ARX add value over AR on the same target?
  5d: OOS R²  AR(1)=0.0061  AR(60)=-0.3571  ARX=-0.4272 (ARX−AR1 Δ=-0.4332) | rankIC  AR(1)=0.1653  ARX=-0.2504 (Δ=-0.4157)
  10d: OOS R²  AR(1)=0.0067  AR(60)=-0.6429  ARX=-0.7782 (ARX−AR1 Δ=-0.7849) | rankIC  AR(1)=0.0837  ARX=-0.3431 (Δ=-0.4269)
[2026-09-15 18:08:00] END Section 9.8 ARX residual model | elapsed 2m 51.17s


## 9.9–9.12  Causal OOS Residual-Forecasting Pipeline (redesign)

Sections 9.1–9.8 are **diagnostics** on the full-sample residual. This block turns the residual into an actual **point-in-time, strictly out-of-sample forecast** and a peer-relative score — the object that can be traded and validated against subsequent trades.

Flow: realized residual → point-in-time features → forward label → expanding-window OOS forecast (never full-sample) → peer-relative score → predictive validation. Primary target is the next residual **level** $\hat\varepsilon_{i,t+1\mid t}$; the mean-reversion label $\Delta\hat\varepsilon$ is reported as a secondary test. MSRB next-trade validation is deferred and will reuse Section 12 once the OOS score panel is persisted.

In [ ]:
# ── 9.9 OOS PIPELINE (1/4): CAUSAL FEATURES, PEER BUCKETS, FORWARD LABELS ─
# Point-in-time residual state + strictly-forward labels on the residual universe.
# Features at column t use only residuals observed through t; labels use t+1..t+h only.
_oospipe_start = time.perf_counter()
log_runtime("START 9.9 OOS pipeline substrate")

_obsRp = np.isfinite(Rp)
_ttc = np.arange(T_rv)
rv_month_idx = pd.PeriodIndex(rv_dates, freq='M')

# Signal timing: eps_t is known only after mark r_t at date t -> usable for t+1.
residual_timestamp = rv_dates
signal_available_timestamp = rv_dates            # eps_t usable at end-of-day t

# Static peer buckets (time-invariant metadata), aligned to rv_cusips.
def _coarse_rating9(r):
    if pd.isna(r):
        return 'NR'
    r = str(r).upper()
    for _pre in ('AAA', 'AA', 'BBB', 'BB', 'A', 'B'):
        if r.startswith(_pre):
            return _pre
    if r.startswith(('C', 'D')):
        return 'CCC-'
    return 'NR'

_TEN9 = {'1y': '1-3Y', '2y': '1-3Y', '3y': '1-3Y', '4y': '3-7Y', '5-7y': '3-7Y',
         '8-9y': '8-15Y', '10-15y': '8-15Y', '16-20y': '16Y+', '21y+': '16Y+'}
_meta9 = df.groupby('cusip', sort=False)[['rating_grade', 'tenor_bucket']].first().reindex(rv_cusips)
peer_rating = _meta9['rating_grade'].map(_coarse_rating9).to_numpy()
peer_tenor = _meta9['tenor_bucket'].map(lambda t: _TEN9.get(str(t), 'MISSING')).to_numpy()
if 'is_callable' in df.columns:
    _callm = df.groupby('cusip', sort=False)['is_callable'].first().reindex(rv_cusips)
    peer_call = np.where(_callm.fillna(False).to_numpy().astype(bool), 'callable', 'noncall').astype(object)
elif 'call_bucket' in df.columns:
    peer_call = (df.groupby('cusip', sort=False)['call_bucket'].first()
                 .reindex(rv_cusips).fillna('unknown').astype(str).to_numpy())
else:
    peer_call = np.full(N_uni, 'unknown', dtype=object)
_Ru = R[_uni]; _obsRu = obs_mask[_uni] > 0
_odu = np.maximum(_obsRu.sum(axis=1), 1)
_rmu = (_Ru * _obsRu).sum(axis=1) / _odu
_rvolu = np.sqrt(np.clip((_Ru * _Ru * _obsRu).sum(axis=1) / _odu - _rmu ** 2, 0, None))
peer_liq = np.array([f'L{int(v) + 1}' for v in
                     pd.qcut(pd.Series(_rvolu).rank(method='first'), 5, labels=False)], dtype=object)
peer_bucket = np.array([f'{r}|{t}|{c}|{l}' for r, t, c, l in
                        zip(peer_rating, peer_tenor, peer_call, peer_liq)], dtype=object)
PEER_DIMS = {'rating': peer_rating, 'tenor': peer_tenor, 'call': peer_call, 'liq': peer_liq}
print(f"Peer dims - rating {len(set(peer_rating))} | tenor {len(set(peer_tenor))} | "
      f"call {len(set(peer_call))} | liq {len(set(peer_liq))} | combined cells {len(set(peer_bucket))}")

# Causal trailing stats over the strictly-prior window [t-w, t-1].
_Rz9 = np.where(_obsRp, Rp, 0.0)
_C1 = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_Rz9, axis=1)], axis=1)
_C2 = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_Rz9 * _Rz9, axis=1)], axis=1)
_Cn = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_obsRp.astype(float), axis=1)], axis=1)

def _trail_prior(w, minp):
    lo = np.clip(_ttc - w, 0, T_rv); hi = _ttc
    n = _Cn[:, hi] - _Cn[:, lo]; s = _C1[:, hi] - _C1[:, lo]; s2 = _C2[:, hi] - _C2[:, lo]
    nn = np.where(n == 0, 1.0, n); mean = s / nn
    var = np.clip(s2 / nn - mean * mean, 0, None)
    ok = n >= minp
    return np.where(ok, mean, np.nan), np.where(ok, np.sqrt(var), np.nan)

resid_mean_20, resid_std_20 = _trail_prior(20, 5)
resid_z = np.where(_obsRp, (Rp - resid_mean_20) / np.clip(resid_std_20, RESID_Z_DELTA, None), np.nan)
resid_lag1 = np.concatenate([np.full((N_uni, 1), np.nan), Rp[:, :-1]], axis=1)
resid_history_count = _Cn[:, 1:].copy()
_colidx = np.where(_obsRp, _ttc[None, :], -1)
_prev_obs = np.maximum.accumulate(
    np.concatenate([np.full((N_uni, 1), -1), _colidx[:, :-1]], axis=1), axis=1)
days_since_last_residual = np.where(_prev_obs >= 0, _ttc[None, :] - _prev_obs, np.nan)

# Strictly-forward labels.
def _lead(a, k):
    return np.concatenate([a[:, k:], np.full((N_uni, k), np.nan)], axis=1)

future_resid_1d = _lead(Rp, 1)                       # eps_{t+1}  (PRIMARY target)
future_resid_change_1d = future_resid_1d - Rp        # mean-reversion label
future_resid_5d = _lead(Rp, 5)
future_resid_change_5d = future_resid_5d - Rp
_cov1 = int((np.isfinite(future_resid_1d) & _obsRp).sum())
print(f"Causal features + labels ready; 1d paired coverage {_cov1:,} bond-days; "
      f"resid_z finite {int(np.isfinite(resid_z).sum()):,}")
del _Rz9, _C1, _C2, _Cn
log_runtime("END 9.9 OOS pipeline substrate", _oospipe_start)


In [ ]:
# ── 9.10 OOS PIPELINE (2/4): EXPANDING-WINDOW OOS RESIDUAL FORECASTS ─────
# Strictly OOS monthly expanding folds. Models: zero, pooled AR(1), conditional
# AR(1) (peer-bucketed, shrunk to pooled), ridge panel (resid lags + peer dummies).
# Primary target: next residual LEVEL eps_{t+1}. Persists one row per prediction.
from pathlib import Path
_oosfit_start = time.perf_counter()
log_runtime("START 9.10 expanding-window OOS forecasts")

OOS_BURN_MONTHS = 3
OOS_RIDGE_LAMBDA = 10.0
OOS_SHRINK_KAPPA = float(globals().get('SHRINK_KAPPA', 500.0))
OOS_PANEL_PATH = Path('oos_predictions_cache.parquet')

# AR(1) pairs (i, t): x=eps_t, y=eps_{t+1}, both observed.
_pi, _pt = np.where(_obsRp[:, :-1] & _obsRp[:, 1:])
_x = Rp[_pi, _pt].astype(np.float64)
_y = Rp[_pi, _pt + 1].astype(np.float64)
_bucket_pair = peer_bucket[_pi]
_tgt_month = rv_month_idx[_pt + 1]
_tgt_date = rv_dates.values[_pt + 1]
_sig_date = rv_dates.values[_pt]
_sigma_pair = np.where(np.isfinite(resid_std_20[_pi, _pt]), resid_std_20[_pi, _pt], RESID_Z_DELTA)

# Ridge design: numeric [1, x, lag1, resid_z, ma20_dev] + peer dummies (drop ref level).
def _feat(a):
    v = a[_pi, _pt]
    return np.where(np.isfinite(v), v, 0.0)
_xl1 = _feat(resid_lag1)
_zt = _feat(resid_z)
_ma_dev = np.where(np.isfinite(resid_mean_20[_pi, _pt]), _x - resid_mean_20[_pi, _pt], 0.0)

def _dum(labels):
    u = pd.Index(pd.unique(labels)); code = u.get_indexer(labels)
    M = np.zeros((code.size, max(len(u) - 1, 0)), dtype=np.float32)
    nz = code > 0
    if nz.any():
        M[np.where(nz)[0], code[nz] - 1] = 1.0
    return M

_Xnum = np.column_stack([np.ones_like(_x), _x, _xl1, _zt, _ma_dev]).astype(np.float32)
_Xdum = np.column_stack([_dum(peer_rating[_pi]), _dum(peer_tenor[_pi]),
                         _dum(peer_call[_pi]), _dum(peer_liq[_pi])]).astype(np.float32)
_Xr = np.column_stack([_Xnum, _Xdum]) if _Xdum.size else _Xnum
_pr = _Xr.shape[1]

_months_all = rv_month_idx.unique()
_fold_months = _months_all[OOS_BURN_MONTHS:]
_pb_cats = pd.Index(pd.unique(peer_bucket))
print(f"AR(1) pairs {_x.size:,} | ridge features {_pr} | peer cells {len(_pb_cats)} | "
      f"folds {len(_fold_months)} (monthly expanding, burn-in {OOS_BURN_MONTHS}m)")


def _ols_ar1(x, y):
    n = x.size
    if n < 50:
        return (float(y.mean()) if n else 0.0), 0.0
    sx = x.sum(); sy = y.sum(); den = n * (x * x).sum() - sx * sx
    if den <= 0:
        return float(sy / n), 0.0
    rho = (n * (x * y).sum() - sx * sy) / den
    return float((sy - rho * sx) / n), float(rho)


_MODELS = ['zero', 'pooled_ar1', 'cond_ar1', 'ridge']
_acc = {m: {'sse': 0.0, 'sy2': 0.0, 'n': 0, 'sgc': 0, 'sgn': 0, 'ric': [], 'pic': []} for m in _MODELS}
_parts = []


def _fold_ic(pred, yv, dcodes):
    d = pd.DataFrame({'d': dcodes, 'p': pred, 'y': yv}).groupby('d')
    ric = d.apply(lambda z: z['p'].corr(z['y'], method='spearman') if len(z) >= 20 else np.nan)
    pic = d.apply(lambda z: z['p'].corr(z['y']) if len(z) >= 20 else np.nan)
    return list(ric.to_numpy()), list(pic.to_numpy())


def _tally(name, pred, yv, dcodes):
    if not np.isfinite(pred).all():
        return
    a = _acc[name]
    a['sse'] += float(np.sum((yv - pred) ** 2)); a['sy2'] += float(np.sum(yv ** 2)); a['n'] += int(yv.size)
    nz = (pred != 0) & (yv != 0)
    a['sgc'] += int(np.sum(np.sign(pred[nz]) == np.sign(yv[nz]))); a['sgn'] += int(nz.sum())
    r, p = _fold_ic(pred, yv, dcodes); a['ric'] += r; a['pic'] += p


with timed_block("expanding-window OOS fit/predict"):
    for _m in _fold_months:
        tr = (_tgt_month < _m).to_numpy() if hasattr(_tgt_month < _m, 'to_numpy') else np.asarray(_tgt_month < _m)
        te = (_tgt_month == _m).to_numpy() if hasattr(_tgt_month == _m, 'to_numpy') else np.asarray(_tgt_month == _m)
        if te.sum() < 200 or tr.sum() < 1000:
            continue
        _xtr, _ytr, _xte, _yte, _dte = _x[tr], _y[tr], _x[te], _y[te], _tgt_date[te]
        _vintage = rv_dates[rv_month_idx < _m].max()
        # pooled AR(1)
        _a, _rho = _ols_ar1(_xtr, _ytr)
        # conditional AR(1): per-bucket OLS via sums, shrink to pooled
        _td = pd.DataFrame({'b': _bucket_pair[tr], 'x': _xtr, 'y': _ytr})
        _td['xx'] = _td['x'] * _td['x']; _td['xy'] = _td['x'] * _td['y']
        _ag = _td.groupby('b').agg(n=('x', 'size'), sx=('x', 'sum'), sy=('y', 'sum'),
                                   sxx=('xx', 'sum'), sxy=('xy', 'sum'))
        _den = (_ag['n'] * _ag['sxx'] - _ag['sx'] ** 2)
        _rb = np.where(_den.to_numpy() > 0,
                       (_ag['n'] * _ag['sxy'] - _ag['sx'] * _ag['sy']).to_numpy() / np.where(_den.to_numpy() == 0, np.nan, _den.to_numpy()),
                       0.0)
        _nb = _ag['n'].to_numpy()
        _ab = (_ag['sy'].to_numpy() - _rb * _ag['sx'].to_numpy()) / _nb
        _small = _nb < 200
        _rb = np.where(_small, _rho, _rb); _ab = np.where(_small, _a, _ab)
        _w = _nb / (_nb + OOS_SHRINK_KAPPA)
        _bp_a = pd.Series(_w * _ab + (1 - _w) * _a, index=_ag.index)
        _bp_r = pd.Series(_w * _rb + (1 - _w) * _rho, index=_ag.index)
        _bte = _bucket_pair[te]
        _pred_cond = _bp_a.reindex(_bte).fillna(_a).to_numpy() + _bp_r.reindex(_bte).fillna(_rho).to_numpy() * _xte
        # ridge (chunked normal equations); guard against failure
        try:
            _tri = np.where(tr)[0]
            _XtX = np.zeros((_pr, _pr)); _Xty = np.zeros(_pr)
            for _s0 in range(0, _tri.size, 3_000_000):
                _blk = _tri[_s0:_s0 + 3_000_000]
                _Xb = _Xr[_blk].astype(np.float64)
                _XtX += _Xb.T @ _Xb; _Xty += _Xb.T @ _y[_blk]
            _regm = OOS_RIDGE_LAMBDA * np.eye(_pr); _regm[0, 0] = 0.0
            _beta = np.linalg.solve(_XtX + _regm, _Xty)
            _pred_ridge = _Xr[te].astype(np.float64) @ _beta
        except Exception as _re:
            print(f"  ridge fold {_m} skipped: {_re}")
            _pred_ridge = np.full(_xte.size, np.nan)
        # tally
        _tally('zero', np.zeros_like(_yte), _yte, _dte)
        _tally('pooled_ar1', _a + _rho * _xte, _yte, _dte)
        _tally('cond_ar1', _pred_cond, _yte, _dte)
        _tally('ridge', _pred_ridge, _yte, _dte)
        # persist primary (conditional AR(1)) predictions
        _parts.append(pd.DataFrame({
            'bond_idx': _pi[te].astype(np.int32),
            'target_date': _dte,
            'signal_available_ts': _sig_date[te],
            'model_vintage': np.datetime64(_vintage),
            'peer_bucket': pd.Categorical(_bte, categories=_pb_cats),
            'residual_t': _xte.astype(np.float32),
            'predicted_resid_t1': _pred_cond.astype(np.float32),
            'actual_resid_t1': _yte.astype(np.float32),
            'forecast_sigma': _sigma_pair[te].astype(np.float32),
        }))

oos_predictions = pd.concat(_parts, ignore_index=True)
oos_predictions['cusip'] = pd.Categorical(rv_cusips.values[oos_predictions['bond_idx'].to_numpy()])
try:
    oos_predictions.to_parquet(OOS_PANEL_PATH, index=False); _persisted = str(OOS_PANEL_PATH)
except Exception as _e:
    OOS_PANEL_PATH = Path('oos_predictions_cache.pkl')
    oos_predictions.to_pickle(OOS_PANEL_PATH); _persisted = str(OOS_PANEL_PATH)
print(f"OOS panel: {len(oos_predictions):,} rows | "
      f"{pd.Timestamp(oos_predictions['target_date'].min()).date()} -> "
      f"{pd.Timestamp(oos_predictions['target_date'].max()).date()} | persisted -> {_persisted}")

_rows = []
for _mn in _MODELS:
    a = _acc[_mn]
    _rows.append({'model': _mn,
                  'oos_R2': (1 - a['sse'] / a['sy2']) if a['sy2'] > 0 else np.nan,
                  'pearson_IC': float(np.nanmean(a['pic'])) if a['pic'] else np.nan,
                  'rank_IC': float(np.nanmean(a['ric'])) if a['ric'] else np.nan,
                  'sign_acc': (a['sgc'] / a['sgn']) if a['sgn'] else np.nan,
                  'n_pred': a['n']})
oos_model_summary = pd.DataFrame(_rows).set_index('model')
print("\nOOS forecast model comparison (expanding monthly folds; target = eps_{t+1} level):")
display(oos_model_summary.style.format({'oos_R2': '{:.4f}', 'pearson_IC': '{:.4f}',
                                        'rank_IC': '{:.4f}', 'sign_acc': '{:.3f}', 'n_pred': '{:,.0f}'}))
log_runtime("END 9.10 expanding-window OOS forecasts", _oosfit_start)


In [ ]:
# ── 9.11 OOS PIPELINE (3/4): PEER-RELATIVE SCORE + UNCERTAINTY ───────────
# Do NOT rank raw/averaged residuals. Rank the OOS forecast within comparable
# peers, and standardize by forecast uncertainty: RV = eps_hat / (sigma_hat + delta).
from pathlib import Path
_oosscore_start = time.perf_counter()
log_runtime("START 9.11 peer-relative OOS score")
if 'oos_predictions' not in globals():
    oos_predictions = pd.read_parquet(OOS_PANEL_PATH)
FORECAST_SIGMA_DELTA = float(globals().get('RESID_Z_DELTA', 1e-4))

oos_predictions['rv_score'] = (oos_predictions['predicted_resid_t1']
                               / (oos_predictions['forecast_sigma'] + FORECAST_SIGMA_DELTA)).astype(np.float32)
oos_predictions['rv_rank'] = (oos_predictions
                              .groupby(['target_date', 'peer_bucket'], observed=True)['predicted_resid_t1']
                              .rank(pct=True).astype(np.float32))
# Attach peer dimensions (from bond_idx) for later per-segment validation.
_bi = oos_predictions['bond_idx'].to_numpy()
for _nm, _arr in PEER_DIMS.items():
    oos_predictions[f'peer_{_nm}'] = pd.Categorical(_arr[_bi])

try:
    oos_predictions.to_parquet(OOS_PANEL_PATH, index=False); _persisted = str(OOS_PANEL_PATH)
except Exception as _e:
    OOS_PANEL_PATH = Path('oos_predictions_cache.pkl')
    oos_predictions.to_pickle(OOS_PANEL_PATH); _persisted = str(OOS_PANEL_PATH)

print(f"Enriched OOS score panel -> {_persisted}")
print(f"  rows {len(oos_predictions):,} | columns: {list(oos_predictions.columns)}")
print(f"  rv_rank in [{oos_predictions['rv_rank'].min():.3f}, {oos_predictions['rv_rank'].max():.3f}], "
      f"rv_score p1/p99 = {oos_predictions['rv_score'].quantile(0.01):.2f} / "
      f"{oos_predictions['rv_score'].quantile(0.99):.2f}")
display(oos_predictions[['target_date', 'cusip', 'peer_bucket', 'model_vintage', 'residual_t',
                         'predicted_resid_t1', 'forecast_sigma', 'rv_score', 'rv_rank',
                         'actual_resid_t1']].head(8))
log_runtime("END 9.11 peer-relative OOS score", _oosscore_start)


[2026-09-17 16:05:09] START 9.11 peer-relative OOS score
Enriched OOS score panel -> oos_predictions_cache.parquet
  rows 11,934,597 | columns: ['bond_idx', 'target_date', 'signal_available_ts', 'model_vintage', 'peer_bucket', 'residual_t', 'predicted_resid_t1', 'actual_resid_t1', 'forecast_sigma', 'cusip', 'rv_score', 'rv_rank', 'peer_rating', 'peer_tenor', 'peer_call', 'peer_liq']
  rv_rank in [0.000, 1.000], rv_score p1/p99 = -0.61 / 0.62


,target_date,cusip,peer_bucket,model_vintage,residual_t,predicted_resid_t1,forecast_sigma,rv_score,rv_rank,actual_resid_t1
0,2026-07-01,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,0.000,-0.000,0.000,-0.098,0.240,0.000
1,2026-07-02,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,0.000,-0.000,0.000,-0.100,0.353,-0.000
2,2026-07-06,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,-0.000,-0.000,0.000,-0.090,0.832,-0.000
3,2026-07-07,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,-0.000,-0.000,0.000,-0.093,0.817,0.000
4,2026-07-08,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,0.000,-0.000,0.000,-0.095,0.647,0.001
5,2026-07-09,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,0.001,-0.000,0.000,-0.113,0.090,-0.000
6,2026-07-10,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,-0.000,-0.000,0.000,-0.074,0.764,0.000
7,2026-07-13,0004162A4,AA|3-7Y|noncall|L3,2026-06-30,0.000,-0.000,0.000,-0.075,0.512,0.000


[2026-09-17 16:05:24] END 9.11 peer-relative OOS score | elapsed 15.26s


In [ ]:
# ── 9.12 OOS PIPELINE (4/4): PREDICTIVE VALIDATION ──────────────────────
# Evaluate the OOS forecast (not the contemporaneous residual). Primary test:
# RankIC(eps_hat_{t+1|t}, eps_{t+1}); secondary: mean-reversion RankIC(-eps_t, d eps).
from scipy.stats import rankdata
_oosval_start = time.perf_counter()
log_runtime("START 9.12 OOS predictive validation")
if 'oos_predictions' not in globals():
    oos_predictions = pd.read_parquet(OOS_PANEL_PATH)

_op = oos_predictions
_yv = _op['actual_resid_t1'].to_numpy(np.float64)
_pv = _op['predicted_resid_t1'].to_numpy(np.float64)
_ev = _op['residual_t'].to_numpy(np.float64)
_dt = _op['target_date'].to_numpy()
_chg = _yv - _ev


def _spear(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 20 or np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return float(np.corrcoef(rankdata(a[m]), rankdata(b[m]))[0, 1])


with timed_block("9.12 headline OOS metrics"):
    _ric_date = pd.DataFrame({'d': _dt, 'p': _pv, 'y': _yv}).groupby('d').apply(
        lambda z: _spear(z['p'].to_numpy(), z['y'].to_numpy()))
    _op2 = _op[['target_date', 'peer_bucket']].copy()
    _g = _op.groupby(['target_date', 'peer_bucket'], observed=True)
    _op2['pr'] = _g['predicted_resid_t1'].rank(pct=True).to_numpy()
    _op2['yr'] = _g['actual_resid_t1'].rank(pct=True).to_numpy()
    _within_ic = _op2.groupby('target_date').apply(
        lambda z: z['pr'].corr(z['yr']) if len(z) >= 50 else np.nan)
    _oos_r2 = 1 - np.nansum((_yv - _pv) ** 2) / np.nansum(_yv ** 2)
    _nz = (_pv != 0) & (_yv != 0)
    _sign = float(np.mean(np.sign(_pv[_nz]) == np.sign(_yv[_nz])))
    _mr = pd.DataFrame({'d': _dt, 'x': -_ev, 'c': _chg}).groupby('d').apply(
        lambda z: _spear(z['x'].to_numpy(), z['c'].to_numpy()))

print("=== 9.12 OOS PREDICTIVE VALIDATION (conditional AR(1) primary; target = eps_(t+1) level) ===")
print(f"  pooled per-date rank IC (mean, {int(_ric_date.notna().sum())} dates): {_ric_date.mean():.4f} "
      f"(t {np.sqrt(_ric_date.notna().sum()) * _ric_date.mean() / _ric_date.std():.2f})")
print(f"  WITHIN-PEER per-date rank IC (mean):                {_within_ic.mean():.4f}")
print(f"  pooled OOS R2 (vs zero):                            {_oos_r2:.4f}")
print(f"  sign accuracy:                                      {_sign:.3f}")
print(f"  [secondary] mean-reversion RankIC(-eps_t, d eps):   {_mr.mean():.4f}")

_dec = np.clip((rankdata(_pv) / _pv.size * 10).astype(int), 0, 9)
_dec_tbl = pd.DataFrame({'pred_decile': _dec + 1, 'actual_resid_t1': _yv, 'change_1d': _chg}).groupby(
    'pred_decile').agg(mean_actual=('actual_resid_t1', 'mean'),
                       mean_change_1d=('change_1d', 'mean'), n=('actual_resid_t1', 'size'))
print("\nActual next residual & 1d change by predicted decile (monotone in mean_actual => forecast orders future residual):")
display(_dec_tbl.style.format({'mean_actual': '{:.6f}', 'mean_change_1d': '{:.6f}', 'n': '{:,.0f}'}))

_seg_rows = []
for _dim in ['peer_rating', 'peer_tenor', 'peer_call', 'peer_liq']:
    _sg = _op.groupby([_dim, 'target_date'], observed=True).apply(
        lambda z: _spear(z['predicted_resid_t1'].to_numpy(), z['actual_resid_t1'].to_numpy()))
    for _k, _v in _sg.groupby(level=0, observed=True).mean().items():
        _seg_rows.append({'dimension': _dim.replace('peer_', ''), 'bucket': str(_k), 'mean_rank_IC': _v})
_seg_df = pd.DataFrame(_seg_rows)
print("\nMean per-date rank IC of the OOS forecast within peer segments:")
display(_seg_df.pivot_table(index='dimension', columns='bucket', values='mean_rank_IC').style.format('{:.3f}', na_rep='--'))
log_runtime("END 9.12 OOS predictive validation", _oosval_start)


[2026-09-17 16:05:37] START 9.12 OOS predictive validation
[2026-09-17 16:05:37] START 9.12 headline OOS metrics
[2026-09-17 16:05:56] END 9.12 headline OOS metrics | elapsed 18.86s
=== 9.12 OOS PREDICTIVE VALIDATION (conditional AR(1) primary; target = eps_(t+1) level) ===
  pooled per-date rank IC (mean, 53 dates): 0.1996 (t 3.82)
  WITHIN-PEER per-date rank IC (mean):                0.2187
  pooled OOS R2 (vs zero):                            0.0426
  sign accuracy:                                      0.585
  [secondary] mean-reversion RankIC(-eps_t, d eps):   0.3924

Actual next residual & 1d change by predicted decile (monotone in mean_actual => forecast orders future residual):


<HTML output>

,mean_actual,mean_change_1d,n
pred_decile,,,
1,-0.000490,0.001044,"1,193,459"
2,-0.000205,0.000176,"1,193,460"
3,-0.000108,0.000099,"1,193,460"
4,-0.000044,0.000064,"1,193,458"
5,-0.000016,0.000031,"1,193,461"
6,-0.000014,-0.000045,"1,193,461"
7,0.000023,-0.000069,"1,193,455"
8,0.000048,-0.000088,"1,193,463"
9,0.000147,-0.000223,"1,193,462"


Mean per-date rank IC of the OOS forecast within peer segments:


<HTML output>

bucket,1-3Y,16Y+,3-7Y,8-15Y,A,AA,AAA,BB,BBB,L1,L2,L3,L4,L5,NR,callable,noncall
dimension,,,,,,,,,,,,,,,,,
call,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,0.209,0.194
liq,--,--,--,--,--,--,--,--,--,0.302,0.281,0.203,0.111,0.165,--,--,--
rating,--,--,--,--,0.189,0.183,0.174,0.178,0.181,--,--,--,--,--,0.253,--,--
tenor,0.356,0.112,0.192,0.150,--,--,--,--,--,--,--,--,--,--,--,--,--


[2026-09-17 16:06:16] END 9.12 OOS predictive validation | elapsed 39.42s


## Section 10 (revised) — Residual Predictability, Decay & Robustness

Interprets the **frozen OOS forecast** from §9.9–§9.12: how fast ICE residual persistence decays (half-life), how quickly the forecast's cross-sectional ordering washes out with horizon, and whether the OOS edge is **stable across folds** and **survives de-smoothing** (skip-a-day, active vs stale). No new model is trained here — the raw ICE residual remains the target. The full-sample conditional-AR analysis that follows is the supporting diagnostic.

In [ ]:
# ── 10.1 RESIDUAL PREDICTABILITY & DECAY ────────────────────────────────
# How fast ICE residual persistence decays, and how quickly the frozen OOS
# forecast's cross-sectional ordering washes out with horizon.
from scipy.stats import rankdata
_dec_start = time.perf_counter()
log_runtime("START 10.1 residual predictability & decay")

_H_DECAY = 10
_ac = []
for _k in range(1, _H_DECAY + 1):
    _a = Rp[:, :T_rv - _k]; _b = Rp[:, _k:]
    _m = np.isfinite(_a) & np.isfinite(_b)
    _ac.append(float(np.corrcoef(_a[_m], _b[_m])[0, 1]) if _m.sum() > 1000 else np.nan)
_rho1 = _ac[0]
_halflife = float(np.log(0.5) / np.log(_rho1)) if 0 < _rho1 < 1 else np.nan
decay_df = pd.DataFrame({'lag_k': range(1, _H_DECAY + 1), 'autocorr': _ac}).set_index('lag_k')
print(f"Residual autocorrelation decay: rho_1 = {_rho1:.4f} -> AR(1) half-life ~ {_halflife:.2f} trading days")
display(decay_df.T.style.format('{:.4f}'))

# Forecast-edge decay: actual residual LEVEL at horizon h by predicted decile.
_pos = pd.Index(rv_dates).get_indexer(oos_predictions['target_date'].to_numpy())  # position of eps_{t+1}
_bi = oos_predictions['bond_idx'].to_numpy()
_pdec = np.clip((rankdata(oos_predictions['predicted_resid_t1'].to_numpy()) / len(oos_predictions) * 10).astype(int), 0, 9)
_HORIZ = [1, 2, 3, 5]
_cols = []
for _h in _HORIZ:
    _p2 = _pos + (_h - 1); _ok = _p2 < T_rv
    _fut = np.full(len(oos_predictions), np.nan)
    _fut[_ok] = Rp[_bi[_ok], _p2[_ok]]
    _cols.append(pd.Series(_fut).groupby(_pdec).mean().rename(f'h{_h}'))
conv_df = pd.concat(_cols, axis=1)
conv_df.index = [f'D{i + 1}' for i in conv_df.index]
print("\nMean actual residual LEVEL at horizon h by predicted decile (spread shrinking with h => edge decays):")
display(conv_df.style.format('{:.6f}'))
_spread_h = {f'h{_h}': float(conv_df[f'h{_h}'].iloc[-1] - conv_df[f'h{_h}'].iloc[0]) for _h in _HORIZ}
print("D10-D1 spread by horizon:", {k: round(v, 6) for k, v in _spread_h.items()})
log_runtime("END 10.1 residual predictability & decay", _dec_start)


In [ ]:
# ── 10.2 OOS FORECAST ROBUSTNESS & VERDICT ──────────────────────────────
# Fold-level stability, de-smoothing survival (skip-a-day), active-vs-stale, gate.
from scipy.stats import rankdata
_rob_start = time.perf_counter()
log_runtime("START 10.2 OOS robustness & verdict")
_op = oos_predictions
_pv = _op['predicted_resid_t1'].to_numpy(np.float64)
_yv = _op['actual_resid_t1'].to_numpy(np.float64)
_dt = _op['target_date'].to_numpy()


def _spear(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 20 or np.std(a[m]) == 0 or np.std(b[m]) == 0:
        return np.nan
    return float(np.corrcoef(rankdata(a[m]), rankdata(b[m]))[0, 1])


_ric_date = pd.DataFrame({'d': _dt, 'p': _pv, 'y': _yv}).groupby('d').apply(
    lambda z: _spear(z['p'].to_numpy(), z['y'].to_numpy()))
_ric_month = _ric_date.groupby(pd.PeriodIndex(_ric_date.index, freq='M')).mean()
_sign_stab = float(np.sign(_ric_date.dropna()).mean())
print("OOS rank IC by month (fold-level stability):")
display(_ric_month.to_frame('mean_rank_IC').style.format('{:.4f}'))
print(f"Daily rank IC sign-stability (share of dates same sign): {abs(_sign_stab):.2f}")

# Skip-a-day: does the OOS edge survive predicting eps_{t+2}?
_pos = pd.Index(rv_dates).get_indexer(_op['target_date'].to_numpy()); _bi = _op['bond_idx'].to_numpy()
_p2 = _pos + 1; _ok = _p2 < T_rv
_y2 = np.full(len(_op), np.nan); _y2[_ok] = Rp[_bi[_ok], _p2[_ok]]
_ic_h1 = _spear(_pv, _yv); _ic_skip = _spear(_pv, _y2)
_skip_ret = _ic_skip / _ic_h1 if _ic_h1 else np.nan
print(f"\nSkip-a-day: IC(pred, eps_t+1)={_ic_h1:.4f}, IC(pred, eps_t+2)={_ic_skip:.4f}, retention={_skip_ret:.2f}")

# Active vs stale (liquidity) OOS IC.
_liq = _op['peer_liq'].astype(str).to_numpy()
_ic_active = _spear(_pv[_liq == 'L5'], _yv[_liq == 'L5'])
_ic_stale = _spear(_pv[_liq == 'L1'], _yv[_liq == 'L1'])
print(f"Active(L5) OOS IC={_ic_active:.4f} vs Stale(L1) OOS IC={_ic_stale:.4f} "
      f"(edge in active => tradeable; only in stale => microstructure)")

_mean_ic = float(_ric_date.mean())
_t = float(np.sqrt(_ric_date.notna().sum()) * _mean_ic / _ric_date.std()) if _ric_date.std() > 0 else np.nan
_oos_r2 = 1 - np.nansum((_yv - _pv) ** 2) / np.nansum(_yv ** 2)
_g1 = np.isfinite(_t) and _t >= 2.0
_g2 = abs(_sign_stab) >= 0.6
_g3 = _oos_r2 > 0
_g4 = np.isfinite(_skip_ret) and _skip_ret >= 0.3
print("\n=== 10.2 OOS PREDICTABILITY VERDICT ===")
print(f"  mean daily rank IC {_mean_ic:.4f} (t {_t:.2f}, >=2? {_g1}) | sign-stability {abs(_sign_stab):.2f} (>=0.6? {_g2})")
print(f"  OOS R2 {_oos_r2:.4f} (>0? {_g3}) | skip-a-day retention {_skip_ret:.2f} (>=0.3? {_g4})")
_proceed = _g1 and _g2 and _g3
print(f"  VERDICT: {'PROCEED to MSRB external validation of the frozen OOS score' if _proceed else 'OOS predictability too weak/microstructural for MSRB'}")
if _proceed and not _g4:
    print("  CAUTION: low skip-a-day retention => part of the edge is a 1-day stale-mark echo; MSRB is the decisive test.")
log_runtime("END 10.2 OOS robustness & verdict", _rob_start)


## 10. Where Is AR(1) Strongest? Conditional Residual Persistence

> **Pipeline role (redesign).** This section is a **full-sample diagnostic** of *where* AR(1) persistence concentrates (rating / tenor / liquidity / call / state). The **tradeable** forecast is the fold-specific, strictly out-of-sample conditional AR(1) in **§9.10–§9.12**; the single-split estimates below are retained for interpretation only and are *superseded for signal construction*.

Stage 9 established a robust **global** AR(1) in the residuals (OOS rank IC ≈ 0.27), and Section 9.8 showed that a richer ARX overfits. Before considering anything more complex, this section asks whether that persistence is a **market-wide** effect or **concentrated in specific bond segments** — i.e. whether $\rho$ is constant or a function of bond type.

For each grouping $g$ (rating, tenor, liquidity, call, state) fit a **bucket-specific** AR(1)

$$\varepsilon_{i,t+1} = a_g + \rho_g\,\varepsilon_{i,t} + u_{i,t+1},$$

on the training window, then evaluate each group's **out-of-sample rank IC** (next-day residual). Finally a **conditional** model assigns each bond its bucket's $\rho_{g(i)}$ and is compared head-to-head against the single global $\rho$.

### Liquidity proxy

The panel carries no trade counts, so liquidity is proxied by the **price-update rate** — the fraction of observed days on which the evaluated mark actually moved ($r_{i,t}\neq 0$). A low update rate means a stale / illiquid mark. This makes the liquidity test a direct probe of the Stage 9 suspicion that residual persistence is a **stale-mark microstructure** effect: if $\rho$ rises as liquidity falls, persistence is largely a liquidity phenomenon.

### Success criterion

- **Conditional AR(1) materially beats global** → $\rho \neq \text{const}$, i.e. $\rho = g(\text{rating}, \text{tenor}, \text{liquidity}, \text{call}, \text{state})$ — genuine regime structure.
- **It does not** → muni residual persistence is largely a simple, homogeneous AR(1) effect (still a clean, useful conclusion).

Evaluation matches Stage 5: next-day residual target, pooled OLS, time-based OOS split (first 70% of dates), universe ≥ 60 observed days.

In [ ]:
# ── 10.1 CONDITIONAL AR(1) SETUP: BUCKETS, DATASET, GLOBAL BASELINE ──────
_section10_start = time.perf_counter()
log_runtime("START Section 10: conditional AR(1)")

_split_t10 = int(np.floor(T_rv * 0.7))


def _coarse_rating(r):
    if pd.isna(r):
        return 'NR'
    r = str(r).upper()
    if r == 'NR':           return 'NR'
    if r.startswith('AAA'): return 'AAA'
    if r.startswith('AA'):  return 'AA'
    if r.startswith('A'):   return 'A'
    if r.startswith('BBB'): return 'BBB'
    if r.startswith('BB'):  return 'BB'
    if r.startswith('B'):   return 'B'
    if r.startswith('C') or r.startswith('D'): return 'CCC-'
    return 'NR'


_TENOR_MAP = {'1y': '1-3Y', '2y': '1-3Y', '3y': '1-3Y', '4y': '3-7Y', '5-7y': '3-7Y',
              '8-9y': '8-15Y', '10-15y': '8-15Y', '16-20y': '16Y+', '21y+': '16Y+'}
_RATING_ORDER = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC-', 'NR']
_TENOR_ORDER = ['1-3Y', '3-7Y', '8-15Y', '16Y+', 'MISSING']

# Static rating / tenor group per universe bond (from string buckets, so AAA is retained).
_meta = df.groupby('cusip', sort=False)[['rating_grade', 'tenor_bucket']].first().reindex(rv_cusips)
rating_group = _meta['rating_grade'].map(_coarse_rating).to_numpy()
tenor_group = _meta['tenor_bucket'].map(lambda t: _TENOR_MAP.get(str(t), 'MISSING')).to_numpy()

# Liquidity/activity proxy: realized daily-return volatility of the evaluated mark. Illiquid credit
# marks tend to be smoothed (low vol), so LOW vol ≈ stale/illiquid. (Trade counts / volume are not in
# the current pull; among available fields this is the proxy with real cross-sectional dispersion.)
_obs_full = obs_mask > 0
_odays = np.maximum(_obs_full.sum(axis=1), 1)
_rmean = (R * _obs_full).sum(axis=1) / _odays
_rvol = np.sqrt(np.clip((R * R * _obs_full).sum(axis=1) / _odays - _rmean ** 2, 0, None))
_liq_metric = _rvol[_uni]
_liq_q = pd.qcut(pd.Series(_liq_metric).rank(method='first'), 5, labels=False)
_n_liq = int(_liq_q.max()) + 1
liq_group = np.array([f'L{int(v)+1}' for v in _liq_q], dtype=object)
_LIQ_ORDER = [f'L{i+1}' for i in range(_n_liq)]   # L1 = lowest vol (stale/smoothed) → L{max} = highest vol (active)
print(f"Buckets — rating: {sorted(set(rating_group))} | tenor: {sorted(set(tenor_group))} | "
      f"liquidity quintiles: {_n_liq} (L1 low-vol/stale → L{_n_liq} high-vol/active)")


def _ar1_dataset(Rp):
    b, t, xs, ys = [], [], [], []
    for tt in range(1, T_rv):
        x, y = Rp[:, tt - 1], Rp[:, tt]
        v = np.isfinite(x) & np.isfinite(y)
        idx = np.where(v)[0]
        b.append(idx); t.append(np.full(idx.size, tt)); xs.append(x[v]); ys.append(y[v])
    return np.concatenate(b), np.concatenate(t), np.concatenate(xs), np.concatenate(ys)


def _ols1(x, y):
    beta, *_ = np.linalg.lstsq(np.column_stack([np.ones_like(x), x]), y, rcond=None)
    return float(beta[0]), float(beta[1])


bond_row, tt_arr, x_arr, y_arr = _ar1_dataset(Rp)
_train10 = tt_arr < _split_t10
_te_idx = np.where(~_train10)[0]
_ttt, _yt = tt_arr[_te_idx], y_arr[_te_idx]
print(f"AR(1) pairs: {x_arr.size:,} (train {int(_train10.sum()):,} / test {_te_idx.size:,}); "
      f"OOS split at {rv_dates[_split_t10].date()}")


# Hierarchical shrinkage: shrink each bucket's AR(1) toward the global (parent)
# estimate with weight w = n / (n + kappa). Sparse buckets are pulled to the global
# rho; data-rich buckets keep their own estimate. This prevents a thin
# rating×tenor×liquidity cell from producing an extreme mean-reversion signal.
SHRINK_KAPPA = 500.0


def _fit_conditional(bond_labels, min_train=200, kappa=SHRINK_KAPPA):
    rl = bond_labels[bond_row]
    ga, grho = _ols1(x_arr[_train10], y_arr[_train10])   # global (parent) estimate
    yhat = np.full(x_arr.shape, np.nan)
    params = {}
    for g in pd.unique(rl[_train10]):
        gtr = _train10 & (rl == g)
        n_g = int(gtr.sum())
        a_raw, rho_raw = _ols1(x_arr[gtr], y_arr[gtr]) if n_g >= min_train else (ga, grho)
        w = n_g / (n_g + kappa)                          # James–Stein-style shrinkage weight
        a = w * a_raw + (1.0 - w) * ga
        rho = w * rho_raw + (1.0 - w) * grho
        params[g] = {'rho': rho, 'rho_raw': rho_raw, 'shrink_w': w, 'n_train': n_g}
        yhat[rl == g] = a + rho * x_arr[rl == g]
    _miss = np.isnan(yhat)
    if _miss.any():
        yhat[_miss] = ga + grho * x_arr[_miss]
    return yhat, params


def _eval_full(yhat):
    p = yhat[_te_idx]
    r2 = 1.0 - np.sum((_yt - p) ** 2) / np.sum(_yt ** 2)
    d = pd.DataFrame({'d': _ttt, 'p': p, 'y': _yt})
    grp = d.groupby('d')
    ric = grp.apply(lambda z: z['p'].corr(z['y'], method='spearman') if len(z) >= 20 else np.nan).mean()

    def _sp(z):
        if len(z) < N_DECILES * 5:
            return np.nan
        dd = pd.qcut(z['p'].rank(method='first'), N_DECILES, labels=False).to_numpy()
        yv = z['y'].to_numpy()
        return yv[dd == N_DECILES - 1].mean() - yv[dd == 0].mean()
    sp = grp.apply(_sp)
    sharpe = float(np.sqrt(ANN) * sp.mean() / sp.std()) if sp.std() > 0 else np.nan
    return {'rank_IC': float(ric), 'oos_R2': float(r2), 'spread_sharpe': sharpe}


def _bucket_ar1(bond_labels, order):
    yhat, params = _fit_conditional(bond_labels)
    rl_test = bond_labels[bond_row[_te_idx]]
    d = pd.DataFrame({'d': _ttt, 'g': rl_test, 'p': yhat[_te_idx], 'y': _yt})
    ric = d.groupby(['g', 'd']).apply(lambda z: z['p'].corr(z['y'], method='spearman') if len(z) >= 20 else np.nan)
    ric = ric.groupby(level='g').mean()
    present = set(bond_labels)
    rows = [{'bucket': g, 'rho': params.get(g, {}).get('rho', np.nan),
             'n_train': params.get(g, {}).get('n_train', 0), 'oos_rank_IC': float(ric.get(g, np.nan))}
            for g in order if g in present]
    return pd.DataFrame(rows).set_index('bucket'), yhat


_ag, _rhog = _ols1(x_arr[_train10], y_arr[_train10])
yhat_global = _ag + _rhog * x_arr
cond_models = {'Global AR(1)': yhat_global}

_gstats = _eval_full(yhat_global)

print(f"Global AR(1): rho={_rhog:.4f} | OOS rank IC={_gstats['rank_IC']:.4f} | "      f"OOS R2={_gstats['oos_R2']:.4f} | spread Sharpe={_gstats['spread_sharpe']:.2f}")

[2026-09-17 15:14:57] START Section 10: conditional AR(1)
Buckets — rating: ['A', 'AA', 'AAA', 'B', 'BB', 'BBB', 'CCC-', 'NR'] | tenor: ['1-3Y', '16Y+', '3-7Y', '8-15Y'] | liquidity quintiles: 5 (L1 low-vol/stale → L5 high-vol/active)
AR(1) pairs: 25,798,300 (train 17,963,938 / test 7,834,362); OOS split at 2026-07-28
Global AR(1): rho=0.1565 | OOS rank IC=0.2714 | OOS R2=0.0390 | spread Sharpe=8.85


### 10A Rating-Specific AR(1)

Fit $\rho_g$ separately per coarse credit bucket (AAA…NR). If persistence rises down the credit curve, residual momentum is stronger in lower-rated (typically less liquid, harder-to-value) names.

In [ ]:
# ── 10A RATING-SPECIFIC AR(1) ───────────────────────────────────────────
with timed_block("Stage 10A rating-specific AR(1)"):
    rating_ar1_tbl, _yh_rat = _bucket_ar1(rating_group, _RATING_ORDER)
cond_models['Rating-cond AR(1)'] = _yh_rat
print("Rating-specific AR(1) — persistence ρ and OOS rank IC by credit bucket:")
display(rating_ar1_tbl.style.format({'rho': '{:.4f}', 'n_train': '{:,.0f}', 'oos_rank_IC': '{:.4f}'}))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(rating_ar1_tbl.index, rating_ar1_tbl['rho'], color='tab:blue', alpha=0.8)
ax.axhline(_rhog, color='red', linestyle='--', alpha=0.7, label=f'global ρ={_rhog:.3f}')
ax.set_ylabel('AR(1) ρ'); ax.set_title('Residual persistence ρ by rating')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


[2026-09-17 15:23:06] START Stage 10A rating-specific AR(1)
[2026-09-17 15:23:21] END Stage 10A rating-specific AR(1) | elapsed 15.68s
Rating-specific AR(1) — persistence ρ and OOS rank IC by credit bucket:


<HTML output>

,rho,n_train,oos_rank_IC
bucket,,,
AAA,0.1583,"2,625,742",0.2783
AA,0.1432,"10,141,146",0.2629
A,0.1502,"1,334,952",0.2569
BBB,0.1109,"47,077",0.2833
BB,0.1491,"3,447",0.2254
B,0.0422,"1,422",nan
CCC-,0.1565,79,nan
NR,0.1913,"3,810,073",0.2950


<Figure: muni_characteristic_factor_export_71_2.png (image not included in markdown export)>

    


### 10B Tenor-Specific AR(1)

Fit $\rho_g$ per coarse maturity bucket. Longer-tenor bonds carry more duration/valuation uncertainty, so persistence may differ across the curve.

In [ ]:
# ── 10B TENOR-SPECIFIC AR(1) ────────────────────────────────────────────
with timed_block("Stage 10B tenor-specific AR(1)"):
    tenor_ar1_tbl, _yh_ten = _bucket_ar1(tenor_group, _TENOR_ORDER)
cond_models['Tenor-cond AR(1)'] = _yh_ten
print("Tenor-specific AR(1) — persistence ρ and OOS rank IC by maturity bucket:")
display(tenor_ar1_tbl.style.format({'rho': '{:.4f}', 'n_train': '{:,.0f}', 'oos_rank_IC': '{:.4f}'}))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(tenor_ar1_tbl.index, tenor_ar1_tbl['rho'], color='tab:green', alpha=0.8)
ax.axhline(_rhog, color='red', linestyle='--', alpha=0.7, label=f'global ρ={_rhog:.3f}')
ax.set_ylabel('AR(1) ρ'); ax.set_title('Residual persistence ρ by tenor')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


[2026-09-17 15:23:36] START Stage 10B tenor-specific AR(1)
[2026-09-17 15:23:49] END Stage 10B tenor-specific AR(1) | elapsed 12.36s
Tenor-specific AR(1) — persistence ρ and OOS rank IC by maturity bucket:


<HTML output>

,rho,n_train,oos_rank_IC
bucket,,,
1-3Y,0.2591,"5,332,621",0.3324
3-7Y,0.1671,"5,295,834",0.3052
8-15Y,0.1542,"6,060,631",0.2465
16Y+,0.1222,"1,274,852",0.1864


<Figure: muni_characteristic_factor_export_73_2.png (image not included in markdown export)>

    


### 10C Liquidity-Specific AR(1) — the key test

Fit $\rho_g$ per price-update-rate quintile (L1 = most stale/illiquid → L{max} = most liquid). If $\rho$ **rises monotonically as liquidity falls**, residual persistence is largely a **stale-mark / liquidity phenomenon** rather than tradable alpha — the single most important read in this section.

In [ ]:
# ── 10C LIQUIDITY-SPECIFIC AR(1) ────────────────────────────────────────
with timed_block("Stage 10C liquidity-specific AR(1)"):
    liq_ar1_tbl, _yh_liq = _bucket_ar1(liq_group, _LIQ_ORDER)
cond_models['Liquidity-cond AR(1)'] = _yh_liq
# Median realized return vol per quintile (context for the buckets).
liq_ar1_tbl['median_ret_vol'] = [float(np.nanmedian(_liq_metric[liq_group == g])) for g in liq_ar1_tbl.index]
print("Liquidity-specific AR(1) — persistence ρ and OOS rank IC by return-volatility quintile:")
display(liq_ar1_tbl[['median_ret_vol', 'rho', 'n_train', 'oos_rank_IC']].style.format(
    {'median_ret_vol': '{:.5f}', 'rho': '{:.4f}', 'n_train': '{:,.0f}', 'oos_rank_IC': '{:.4f}'}))

_rho_by_liq = liq_ar1_tbl['rho'].to_numpy()
_liq_trend = float(np.corrcoef(np.arange(len(_rho_by_liq)), _rho_by_liq)[0, 1]) if len(_rho_by_liq) > 2 else np.nan
print(f"\nρ vs liquidity trend (corr of ρ with L1→L{_n_liq}): {_liq_trend:+.3f} "
      f"— negative ⇒ persistence concentrates in stale/illiquid bonds (microstructure).")

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(liq_ar1_tbl.index, liq_ar1_tbl['rho'], color='tab:purple', alpha=0.8)
ax.axhline(_rhog, color='red', linestyle='--', alpha=0.7, label=f'global ρ={_rhog:.3f}')
ax.set_ylabel('AR(1) ρ'); ax.set_xlabel('L1 low-vol/stale → L{} high-vol/active'.format(_n_liq))
ax.set_title('Residual persistence ρ by return volatility (liquidity proxy)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


[2026-09-17 15:23:57] START Stage 10C liquidity-specific AR(1)
[2026-09-17 15:24:12] END Stage 10C liquidity-specific AR(1) | elapsed 14.25s
Liquidity-specific AR(1) — persistence ρ and OOS rank IC by return-volatility quintile:


<HTML output>

,median_ret_vol,rho,n_train,oos_rank_IC
bucket,,,,
L1,0.00018,0.3169,"3,591,537",0.3325
L2,0.00069,0.2566,"3,617,109",0.3049
L3,0.00131,0.1789,"3,610,118",0.2933
L4,0.00198,0.0786,"3,604,223",0.2170
L5,0.00299,0.1449,"3,540,951",0.2025


ρ vs liquidity trend (corr of ρ with L1→L5): -0.883 — negative ⇒ persistence concentrates in stale/illiquid bonds (microstructure).


<Figure: muni_characteristic_factor_export_75_3.png (image not included in markdown export)>

    


### 10E Call-Bucket AR(1)

Fit $\rho$ separately per **IPCA call-window bucket** (`df['call_bucket']`: No Call, 0–3m … 10y — the same definition feeding the IPCA `call_bucket_*` loadings). Tests whether residual persistence depends on call optionality / time-to-call.

In [ ]:
# ── 10E CALL-BUCKET AR(1) ───────────────────────────────────────────────
# Same call-window buckets as the IPCA characteristic (df['call_bucket']).
_CALL_ORDER = ['No Call', '0-3m', '3-6m', '6-9m', '9-12m', '1y', '2y', '3y', '4y', '5-7y', '8-9y', '10y']
call_group = (df.groupby('cusip', sort=False)['call_bucket'].first()
              .reindex(rv_cusips).fillna('No Call').astype(str).to_numpy())
_CALL_ORDER = _CALL_ORDER + [g for g in pd.unique(call_group) if g not in _CALL_ORDER]
with timed_block("Stage 10E call-bucket AR(1)"):
    call_ar1_tbl, _yh_call = _bucket_ar1(call_group, _CALL_ORDER)
cond_models['Call-cond AR(1)'] = _yh_call
print("Call-bucket AR(1) — persistence rho and OOS rank IC by IPCA call-window bucket:")
display(call_ar1_tbl.style.format({'rho': '{:.4f}', 'n_train': '{:,.0f}', 'oos_rank_IC': '{:.4f}'}))

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(call_ar1_tbl.index.astype(str), call_ar1_tbl['rho'], color='tab:green', alpha=0.8)
ax.axhline(_rhog, color='red', linestyle='--', alpha=0.7, label=f'global rho={_rhog:.3f}')
ax.set_ylabel('AR(1) rho'); ax.set_title('Residual persistence rho by call bucket (IPCA definition)')
ax.tick_params(axis='x', rotation=45); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


[2026-09-17 19:29:42] START Stage 10E call-bucket AR(1)
[2026-09-17 19:30:02] END Stage 10E call-bucket AR(1) | elapsed 20.01s
Call-bucket AR(1) — persistence rho and OOS rank IC by IPCA call-window bucket:


<HTML output>

,rho,n_train,oos_rank_IC
bucket,,,
No Call,0.1011,"7,043,859",0.2927
0-3m,0.0775,"3,145",nan
3-6m,0.1658,"212,057",0.3240
6-9m,0.2640,"647,186",0.3115
9-12m,0.2584,"733,364",0.3235
1y,0.2522,"2,050,897",0.3032
2y,0.2296,"954,165",0.3003
3y,0.1750,"950,790",0.2835
4y,0.1531,"905,116",0.2562


<Figure: muni_characteristic_factor_export_77_2.png (image not included in markdown export)>

    


### 10F State-Bucket AR(1)

Fit $\rho$ separately per **IPCA state bucket** (`df['state_bucket']`: CA, NY, TX, IL, Other States — the same definition feeding the IPCA `state_bucket_*` loadings). Tests whether residual persistence is a state / issuer-geography effect.

In [ ]:
# ── 10F STATE-BUCKET AR(1) ──────────────────────────────────────────────
# Same state buckets as the IPCA characteristic (df['state_bucket']).
_STATE_ORDER = ['CA', 'NY', 'TX', 'IL', 'Other States']
state_group = (df.groupby('cusip', sort=False)['state_bucket'].first()
               .reindex(rv_cusips).fillna('Other States').astype(str).to_numpy())
_STATE_ORDER = _STATE_ORDER + [g for g in pd.unique(state_group) if g not in _STATE_ORDER]
with timed_block("Stage 10F state-bucket AR(1)"):
    state_ar1_tbl, _yh_state = _bucket_ar1(state_group, _STATE_ORDER)
cond_models['State-cond AR(1)'] = _yh_state
print("State-bucket AR(1) — persistence rho and OOS rank IC by IPCA state bucket:")
display(state_ar1_tbl.style.format({'rho': '{:.4f}', 'n_train': '{:,.0f}', 'oos_rank_IC': '{:.4f}'}))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(state_ar1_tbl.index.astype(str), state_ar1_tbl['rho'], color='tab:orange', alpha=0.8)
ax.axhline(_rhog, color='red', linestyle='--', alpha=0.7, label=f'global rho={_rhog:.3f}')
ax.set_ylabel('AR(1) rho'); ax.set_title('Residual persistence rho by state bucket')
ax.tick_params(axis='x', rotation=30); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


[2026-09-17 19:30:19] START Stage 10F state-bucket AR(1)
[2026-09-17 19:30:30] END Stage 10F state-bucket AR(1) | elapsed 11.02s
State-bucket AR(1) — persistence rho and OOS rank IC by IPCA state bucket:


<HTML output>

,rho,n_train,oos_rank_IC
bucket,,,
CA,0.1656,"2,296,323",0.2777
NY,0.1359,"1,218,910",0.2554
TX,0.1471,"3,049,857",0.2543
Other States,0.1625,"11,398,848",0.2748


<Figure: muni_characteristic_factor_export_79_2.png (image not included in markdown export)>

    


### 10D Conditional vs Global AR(1)

Assign each bond its bucket-specific $\rho_{g(i)}$ and forecast $\hat\varepsilon_{i,t+1}=a_{g(i)}+\rho_{g(i)}\varepsilon_{i,t}$, then compare each conditional model against the single global $\rho$ on the same out-of-sample rank IC. A **combined** rating×tenor×liquidity conditioning is also tested (with a global fallback for sparse cells). If none materially beats the global AR(1), the strongest conclusion is that muni residual persistence is a simple, homogeneous AR(1) effect.

In [ ]:
# ── 10D CONDITIONAL vs GLOBAL AR(1) ─────────────────────────────────────
_combined = np.array([f'{r}|{t}|{l}' for r, t, l in zip(rating_group, tenor_group, liq_group)], dtype=object)
with timed_block("Stage 10D combined-conditional fit"):
    _yh_comb, _comb_params = _fit_conditional(_combined)
cond_models['Combined-cond AR(1)'] = _yh_comb
print(f"Combined rating×tenor×liquidity cells: {len(_comb_params)} "
      f"(sparse cells fall back to global ρ)")

with timed_block("Stage 10D model comparison"):
    cond_compare = pd.DataFrame(
        {name: _eval_full(yh) for name, yh in cond_models.items()}
    ).T[['rank_IC', 'oos_R2', 'spread_sharpe']]
print("\nConditional AR(1) vs global — out-of-sample (next-day residual):")
display(cond_compare.style.format({'rank_IC': '{:.4f}', 'oos_R2': '{:.4f}', 'spread_sharpe': '{:.2f}'}))

_g_ric = cond_compare.loc['Global AR(1)', 'rank_IC']
_best_name = cond_compare['rank_IC'].idxmax()
_best_ric = cond_compare.loc[_best_name, 'rank_IC']
_improve = _best_ric - _g_ric
print(f"\nBest by rank IC: {_best_name} — {_best_ric:.4f} vs global {_g_ric:.4f} "
      f"(Δ={_improve:+.4f}, {100 * _improve / abs(_g_ric):+.1f}% relative)")
if _improve >= max(0.02, 0.10 * abs(_g_ric)):
    print("VERDICT (rank IC): conditioning materially improves the cross-sectional signal.")
else:
    print("VERDICT (rank IC): conditioning does NOT beat global AR(1) — for a cross-sectional book a "
          "single ρ suffices (AR(1) forecasts are monotone in ε_t, so group-specific ρ is a scale-free rescaling).")

# ρ is strongly regime-dependent even though the cross-sectional ranking is scale-invariant.
_rho_ten, _rho_liq = tenor_ar1_tbl['rho'], liq_ar1_tbl['rho']
print("\nNuance — ρ is NOT constant, even though rank IC is unchanged:")
print(f"  ρ by tenor:      {_rho_ten.min():.3f} (16Y+) -> {_rho_ten.max():.3f} (1-3Y)")
print(f"  ρ by volatility: {_rho_liq.min():.3f} (active) -> {_rho_liq.max():.3f} (stale/low-vol); trend corr={_liq_trend:+.2f}")
print("  => Persistence concentrates in short-tenor and low-vol (stale/smoothed) marks — a microstructure")
print("     signature; conditioning lifts point-prediction R² (calibration) but not cross-sectional rank IC.")
log_runtime("END Section 10: conditional AR(1)", _section10_start)


[2026-09-17 15:24:24] START Stage 10D combined-conditional fit
[2026-09-17 15:26:42] END Stage 10D combined-conditional fit | elapsed 2m 17.26s
Combined rating×tenor×liquidity cells: 115 (sparse cells fall back to global ρ)
[2026-09-17 15:26:42] START Stage 10D model comparison
[2026-09-17 15:27:02] END Stage 10D model comparison | elapsed 20.73s

Conditional AR(1) vs global — out-of-sample (next-day residual):


<HTML output>

,rank_IC,oos_R2,spread_sharpe
Global AR(1),0.2714,0.0390,8.85
Rating-cond AR(1),0.2709,0.0392,8.83
Tenor-cond AR(1),0.2726,0.0401,9.06
Liquidity-cond AR(1),0.2666,0.0437,9.17
Combined-cond AR(1),0.2597,0.0448,9.04


Best by rank IC: Tenor-cond AR(1) — 0.2726 vs global 0.2714 (Δ=+0.0012, +0.4% relative)
VERDICT (rank IC): conditioning does NOT beat global AR(1) — for a cross-sectional book a single ρ suffices (AR(1) forecasts are monotone in ε_t, so group-specific ρ is a scale-free rescaling).

Nuance — ρ is NOT constant, even though rank IC is unchanged:
  ρ by tenor:      0.122 (16Y+) -> 0.259 (1-3Y)
  ρ by volatility: 0.079 (active) -> 0.317 (stale/low-vol); trend corr=-0.88
  => Persistence concentrates in short-tenor and low-vol (stale/smoothed) marks — a microstructure
     signature; conditioning lifts point-prediction R² (calibration) but not cross-sectional rank IC.
[2026-09-17 15:27:02] END Section 10: conditional AR(1) | elapsed 12m 05.64s


## 10.7 AR(1) Evidence — The 5-Question Research Tree

Before adding ARMA/nonlinear complexity, stress-test the simple residual AR(1) along the standard chain: **Real? → Broad? → How fast does it decay? → Incremental to known effects? → Tradable?** All estimates below use the strict time split from §10.1 (train = first 70% of dates, forecast the held-out tail); the AR(1) forecast $\hat\varepsilon_{i,t+1}=a+\phi\,\varepsilon_{i,t}$ is monotone in $\varepsilon_{i,t}$, so a within-slice rank IC is the residual autocorrelation inside that slice.

- **Q1 Real?** — walk-forward $\phi$, t-stat, OOS R², Pearson/rank IC, IC by month (below; the point-in-time expanding version is §11.1).
- **Q2 Broad?** — IC sliced by residual magnitude / vol / liquidity / sign / side / regime / sector.
- **Q3 Decay?** — IC$(h)$ for $h=1,2,3,5,10$ and AR-order increment (also §10.1 half-life, §9.6 AR(p)).
- **Q4 Incremental?** — Fama–MacBeth: does $\phi$ survive reversal / momentum / vol / liquidity / shock / factor-exposure controls?
- **Q5 Tradable?** — IC → spread → turnover → cost-adjusted PnL (§11.3) and the frozen-score MSRB $\beta$ (final MSRB section).

In [ ]:
# ── 10.7 (Q1) IS THE AR(1) PREDICTABILITY REAL? — WALK-FORWARD SUMMARY ────
_q1_start = time.perf_counter()
log_runtime("START Q1 AR(1) reality check")

# Global AR(1) on the training window; forecast the strictly out-of-sample tail.
_a1, _phi = _ols1(x_arr[_train10], y_arr[_train10])
_xt = x_arr[_train10]
_res_tr = y_arr[_train10] - (_a1 + _phi * _xt)
_sxx = float(((_xt - _xt.mean()) ** 2).sum())
_se_phi = float(np.sqrt((_res_tr @ _res_tr) / (_xt.size - 2) / _sxx))
_t_phi_naive = _phi / _se_phi                                  # inflated by cross-sectional corr

_pte = _a1 + _phi * x_arr[_te_idx]; _yte = y_arr[_te_idx]; _dte = tt_arr[_te_idx]
_oos_r2 = 1.0 - float(np.sum((_yte - _pte) ** 2) / np.sum(_yte ** 2))

_pdf = pd.DataFrame({'d': _dte, 'p': _pte, 'y': _yte}).groupby('d')
_ric = _pdf.apply(lambda z: z['p'].corr(z['y'], method='spearman') if len(z) >= 20 else np.nan)
_pic = _pdf.apply(lambda z: z['p'].corr(z['y']) if len(z) >= 20 else np.nan)


def _series_t(s):
    return float(np.sqrt(s.notna().sum()) * s.mean() / s.std()) if s.std() > 0 else np.nan


q1_summary = pd.Series({
    'phi (global AR1)': _phi,
    'phi t (naive OLS; inflated)': _t_phi_naive,
    'OOS R2 (vs zero)': _oos_r2,
    'mean Pearson IC': float(_pic.mean()),
    'Pearson IC t (daily series)': _series_t(_pic),
    'mean rank IC': float(_ric.mean()),
    'rank IC t (daily series)': _series_t(_ric),
    'n test dates': int(_ric.notna().sum()),
})
print(f"Q1 — Is the AR(1) predictability real? (walk-forward split at {rv_dates[_split_t10].date()})")
display(q1_summary.to_frame('value').style.format('{:.4f}'))
print("  NOTE: the naive phi t-stat is massively inflated by cross-sectional correlation;")
print("  the daily-IC t-stat is the honest measure. Point-in-time expanding version: see Section 11.1.")

_ric_m = _ric.copy()
_ric_m.index = pd.PeriodIndex([rv_dates[int(i)] for i in _ric.index], freq='M')
print("\nRank IC by month (OOS tail):")
display(_ric_m.groupby(level=0).mean().to_frame('mean_rank_IC').style.format('{:.4f}'))
log_runtime("END Q1 AR(1) reality check", _q1_start)


[2026-09-17 20:27:37] START Q1 AR(1) reality check



---------------------------------------------------------------------------

NameError                                 Traceback (most recent call last)

Cell In[10], line 6
      2 _q1_start = time.perf_counter()
      3 log_runtime("START Q1 AR(1) reality check")
      4 
      5 # Global AR(1) on the training window; forecast the strictly out-of-sample tail.
----> 6 _a1, _phi = _ols1(x_arr[_train10], y_arr[_train10])
      7 _xt = x_arr[_train10]
      8 _res_tr = y_arr[_train10] - (_a1 + _phi * _xt)
      9 _sxx = float(((_xt - _xt.mean()) ** 2).sum())


NameError: name '_ols1' is not defined


In [ ]:
# ── 10.8 (Q2) WHERE DOES THE IC COME FROM? — SLICE DECOMPOSITION ─────────
_q2_start = time.perf_counter()
log_runtime("START Q2 IC slice decomposition")
_i = bond_row[_te_idx]; _t = tt_arr[_te_idx]
_xte = x_arr[_te_idx]; _yte = y_arr[_te_idx]; _dte = tt_arr[_te_idx]


def _slice_ic(keys, min_n=200):
    d = pd.DataFrame({'k': np.asarray(keys), 'x': _xte, 'y': _yte})
    ic = d.groupby('k').apply(lambda z: z['x'].corr(z['y'], method='spearman') if len(z) >= min_n else np.nan)
    n = d.groupby('k').size()
    return pd.DataFrame({'pooled_rank_IC': ic, 'n_pairs': n})


_state_uni = (df.groupby('cusip', sort=False)['state_bucket'].first()
              .reindex(rv_cusips).fillna('Other States').astype(str).to_numpy())
_mkt_ret = np.array([np.nanmean(R[obs_mask[:, tt] > 0, tt]) if (obs_mask[:, tt] > 0).any() else np.nan
                     for tt in range(T_rv)])
_absmag = pd.qcut(pd.Series(np.abs(_xte)).rank(method='first'), 3,
                  labels=['|res| low', '|res| mid', '|res| high']).astype(str).to_numpy()
_volq = pd.qcut(pd.Series(np.nan_to_num(resid_std_20[_i, _t])).rank(method='first'), 3,
                labels=['vol low', 'vol mid', 'vol high']).astype(str).to_numpy()
_fdec = pd.qcut(pd.Series(_xte).rank(method='first'), 10, labels=False)
_side = np.where(_fdec == 9, 'top decile (long)', np.where(_fdec == 0, 'bottom decile (short)', 'middle'))
_slices = {
    'residual magnitude': _absmag,
    'idiosyncratic vol': _volq,
    'residual sign': np.where(_xte >= 0, 'res >= 0', 'res < 0'),
    'forecast side': _side,
    'liquidity (L1 stale..L5 active)': peer_liq[_i].astype(str),
    'rating': peer_rating[_i].astype(str),
    'state (sector proxy)': _state_uni[_i],
    'market regime': np.where(_mkt_ret[_dte] >= np.nanmedian(_mkt_ret), 'mkt up-day', 'mkt down-day'),
}
print("Q2 — Where does the IC come from? (pooled OOS rank IC of res_t vs res_t+1 within each slice)")
for _name, _keys in _slices.items():
    print(f"\n{_name}:")
    display(_slice_ic(_keys).style.format({'pooled_rank_IC': '{:.4f}', 'n_pairs': '{:,.0f}'}))
log_runtime("END Q2 IC slice decomposition", _q2_start)


In [ ]:
# ── 10.9 (Q3) DECAY STRUCTURE — IC(h) AND AR-ORDER INCREMENT ─────────────
_q3_start = time.perf_counter()
_H = [1, 2, 3, 5, 10]
_rows = []
for _h in _H:
    _a = Rp[:, :T_rv - _h]; _b = Rp[:, _h:]
    _m = np.isfinite(_a) & np.isfinite(_b)
    _rows.append({'h': _h, 'autocorr (pearson)': float(np.corrcoef(_a[_m], _b[_m])[0, 1]) if _m.sum() > 1000 else np.nan})
decay_ic_df = pd.DataFrame(_rows).set_index('h')
_r1 = decay_ic_df.loc[1, 'autocorr (pearson)']
_hl = float(np.log(0.5) / np.log(_r1)) if 0 < _r1 < 1 else np.nan
print(f"Q3 — Decay: residual autocorr by horizon h (rho_1={_r1:.4f} -> AR(1) half-life ~ {_hl:.2f} days):")
display(decay_ic_df.T.style.format('{:.4f}'))
print("\nAR(p) OOS benchmarks (from Section 9.6) — does lag 2+ add over AR(1)?")
_arv = [c for c in ['oos_R2', 'IC', 'rank_IC', 'D10_minus_D1', 'spread_sharpe'] if c in ar_summary.columns]
display(ar_summary[_arv].style.format('{:.4f}'))
print("  Read: autocorr collapses after lag 1 and AR(3)/AR(5) barely move OOS R2/rank IC over AR(1)")
print("  => short (~1-day) memory; higher AR orders add little. Consistent with Section 10.1.")
log_runtime("END Q3 decay", _q3_start)


In [ ]:
# ── 10.10 (Q4) IS AR(1) INCREMENTAL TO KNOWN EFFECTS? — FAMA-MACBETH ─────
# Cross-sectional regression each OOS date: res_{t+1} ~ res_t + controls, with
# controls = lag-2 residual, trailing residual mean (momentum), idiosyncratic vol,
# |res_t| (recent shock), liquidity (return-vol = spread proxy), trailing-5d raw
# return (reversal), and the 3 IPCA factor loadings (factor exposure). X standardized
# per date so betas are per-1-SD effects; honest inference via the FM t-stat.
_q4_start = time.perf_counter()
log_runtime("START Q4 Fama-MacBeth AR(1) survival")
_uni_idx = np.where(_uni)[0]
_Runi = R[_uni]
_Rz4 = np.where(np.isfinite(_Runi), _Runi, 0.0)
_cum4 = np.concatenate([np.zeros((N_uni, 1)), np.cumsum(_Rz4, axis=1)], axis=1)
_lo = np.clip(np.arange(T_rv) - 4, 0, None)
_rev5 = _cum4[:, np.arange(T_rv) + 1] - _cum4[:, _lo]           # trailing 5-day raw return
_liqm = _liq_metric                                            # per-universe-bond return vol

_names = ['res_t', 'res_lag1', 'resid_ma20', 'resid_vol', 'abs_shock', 'liq_vol', 'rev_5d', 'load1', 'load2', 'load3']
_uni_beta1 = []
_full = {nm: [] for nm in _names}
for tt in range(_split_t10, T_rv - 1):
    _obs = np.isfinite(Rp[:, tt]) & np.isfinite(Rp[:, tt + 1])
    if _obs.sum() < 200:
        continue
    _y = Rp[_obs, tt + 1]
    _pos = np.where(_obs)[0]
    _xr = Rp[_obs, tt]
    _xr_s = (_xr - _xr.mean()) / (_xr.std() + 1e-12)
    _uni_beta1.append(float(np.polyfit(_xr_s, _y, 1)[0]))
    _load = Z[_uni_idx[_pos], :, tt] @ ipca_model.Gamma
    _feat = np.column_stack([
        _xr,
        np.nan_to_num(resid_lag1[_obs, tt]),
        np.nan_to_num(resid_mean_20[_obs, tt]),
        np.nan_to_num(resid_std_20[_obs, tt]),
        np.abs(_xr),
        _liqm[_pos],
        _rev5[_obs, tt],
        _load[:, 0], _load[:, 1], _load[:, 2],
    ])
    _feat = (_feat - _feat.mean(0)) / (_feat.std(0) + 1e-12)
    _Xd = np.column_stack([np.ones(_feat.shape[0]), _feat])
    try:
        _beta = np.linalg.lstsq(_Xd, _y, rcond=None)[0]
    except Exception:
        continue
    for _j, _nm in enumerate(_names):
        _full[_nm].append(_beta[_j + 1])


def _fm(lst):
    a = np.asarray(lst, float); a = a[np.isfinite(a)]
    return (float(a.mean()), float(np.sqrt(a.size) * a.mean() / a.std()) if a.std() > 0 else np.nan, a.size)


_rows = []
_m, _tt, _n = _fm(_uni_beta1)
_rows.append({'term': 'res_t (univariate)', 'mean_beta': _m, 'FM_t': _tt, 'n_dates': _n})
for _nm in _names:
    _m, _tt, _n = _fm(_full[_nm])
    _rows.append({'term': f'{_nm} (full)', 'mean_beta': _m, 'FM_t': _tt, 'n_dates': _n})
fm_df = pd.DataFrame(_rows).set_index('term')
print("Q4 — Fama-MacBeth: does the AR(1) coefficient survive controls? (OOS dates; X standardized/date)")
display(fm_df.style.format({'mean_beta': '{:+.4f}', 'FM_t': '{:+.2f}', 'n_dates': '{:.0f}'}))
_bu = fm_df.loc['res_t (univariate)', 'mean_beta']; _bf = fm_df.loc['res_t (full)', 'mean_beta']
_tf = fm_df.loc['res_t (full)', 'FM_t']
print(f"\n  AR(1) beta: univariate {_bu:+.4f} -> with controls {_bf:+.4f} "
      f"({100 * (_bf - _bu) / abs(_bu):+.0f}% change); FM t (full) {_tf:+.2f}")
print("  => AR(1) SURVIVES the controls (still same-signed and significant)."
      if (abs(_tf) >= 2 and np.sign(_bf) == np.sign(_bu))
      else "  => AR(1) does NOT clearly survive the controls.")
log_runtime("END Q4 Fama-MacBeth AR(1) survival", _q4_start)


### 10.11 (Q5) Does it survive implementation?

The tradability chain — **IC → return spread → turnover → cost-adjusted PnL** — is already built:

- **§11.3** forms the residual decile long/short, measures two-sided turnover, and solves for the **break-even round-trip cost** ($c^\star=2\,\overline{\text{PnL}}/\overline{\text{turnover}}$); the edge decays in ~1 day, so $c^\star$ sits well below plausible muni round-trips.
- The **final MSRB section** regresses realized transaction-price error on the frozen OOS score: $\beta<0$ and correctly signed, but ~0 incremental R² and **concentrated entirely in stale/illiquid (L1) names** — the same microstructure signature seen in Q2 (liquidity slice) and §10.2.

**Tree verdict.** Real (Q1) and broad enough to be statistically robust, but it **decays in ~1 day (Q3)**, is **concentrated in stale/low-liquidity marks (Q2)**, and while the AR(1) coefficient tends to **survive the controls (Q4)** it does **not clear realistic costs (Q5)**. The highest-value next work is explaining/stress-testing this simple effect (done here), not ARMA/nonlinear complexity.

## 11. Tradeability Reality Check (Pre-MSRB Gate)

> **Pipeline role (redesign).** These stale-mark, skip-a-day, active-vs-stale, and turnover / break-even checks are the **robustness layer** for the OOS forecast of **§9.9–§9.12** — they test whether the residual edge survives de-smoothing and clears realistic muni round-trip costs before any trading claim.

Sections 9–10 found a robust residual AR(1) (OOS rank IC ≈ 0.26) whose persistence **concentrates in stale / low-volatility marks** — a price-smoothing signature. Before committing to an MSRB trade-data pull and a constrained portfolio build (the proposed Phases 2–4), this section runs the **cheapest experiments that can kill the idea**, all on existing data (no MSRB required):

1. **Point-in-time expanding AR(1)** — the honest walk-forward baseline: $\hat\rho_t$ estimated only from data before $t$, forecasting $\hat\varepsilon_{i,t+1}=\hat a_t+\hat\rho_t\varepsilon_{i,t}$. Confirms the signal is not an artifact of a single train/test split.
2. **Stale-mark decontamination** — two tests that strip the smoothing bleed:
   - **Skip-a-day**: predict $\varepsilon_{t+2}$ from $\varepsilon_t$ (gap the immediate bounce). If the IC collapses, it was a 1-day stale-price echo.
   - **Active vs stale**: recompute the IC inside the highest-volatility (active) quintile vs the lowest (stale). If the IC lives only in stale names, it is microstructure, not tradable alpha.
3. **Turnover & break-even cost** — a residual-space decile long/short with realistic turnover, and the **round-trip cost that zeroes net P&L**. Muni round-trips are ~20–100+ bps; a fast-decaying ~6 bps/day edge may not clear that bar.

> **Scope.** P&L here is **residual-space** (a proxy: weights on the idiosyncratic return). A true portfolio maps residual weights to bond weights via the Section 8 residual-maker $M_t$ and trades at **trade** prices, not evaluated marks — which is exactly what the MSRB validation (Phase 3) would test. This section decides whether that pull is worth it.

### Gate
- **PROCEED** to the MSRB pull + portfolio build only if the IC **survives** skip-a-day and exists in **active** (tradeable) names, **and** the break-even round-trip cost clears a plausible muni level.
- Otherwise the honest conclusion is that the residual signal is largely a **marking/microstructure** effect — still valuable, but as a **fair-value / de-smoothing** signal (validate via MSRB Test B) rather than a daily trading strategy.

In [ ]:
# ── 11.1 POINT-IN-TIME EXPANDING-WINDOW AR(1) ───────────────────────────
from scipy.stats import rankdata

_section11_start = time.perf_counter()
log_runtime("START Section 11: tradeability reality check")


def _pear(a, b):
    if a.size < 10 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _rank_ic(x, y, min_n=20):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < min_n:
        return np.nan
    return _pear(rankdata(x[m]), rankdata(y[m]))


def _tstat(a):
    a = np.asarray(a, float); a = a[np.isfinite(a)]
    return float(np.sqrt(a.size) * a.mean() / a.std()) if a.size > 2 and a.std() > 0 else np.nan


# Expanding pooled AR(1): rho_t uses only pairs whose target date <= t, then forecasts t+1.
_BURN = 20
_Sx = _Sy = _Sxx = _Sxy = _npairs = 0.0
pit_rank_ic = []
pit_rho = []
with timed_block("PIT expanding AR(1)"):
    for t in range(1, T_rv - 1):
        x0, y0 = Rp[:, t - 1], Rp[:, t]
        m0 = np.isfinite(x0) & np.isfinite(y0)
        xx, yy = x0[m0], y0[m0]
        _Sx += xx.sum(); _Sy += yy.sum(); _Sxx += float((xx * xx).sum())
        _Sxy += float((xx * yy).sum()); _npairs += xx.size
        if t < _BURN or _npairs < 1000:
            continue
        den = _npairs * _Sxx - _Sx * _Sx
        if den <= 0:
            continue
        rho = (_npairs * _Sxy - _Sx * _Sy) / den
        a = (_Sy - rho * _Sx) / _npairs
        pit_rho.append(rho)
        yhat = a + rho * Rp[:, t]                       # forecast eps_{t+1} from known eps_t
        pit_rank_ic.append(_rank_ic(yhat, Rp[:, t + 1]))

pit_rank_ic = np.array(pit_rank_ic)
pit_rho = np.array(pit_rho)
print("Point-in-time expanding AR(1) (rho_t from data strictly before the forecast):")
print(f"  forecast dates      : {int(np.isfinite(pit_rank_ic).sum())}")
print(f"  mean daily rank IC  : {np.nanmean(pit_rank_ic):.4f}  (t-stat {_tstat(pit_rank_ic):.2f})")
print(f"  rho_t  first/mean/last : {pit_rho[0]:.4f} / {pit_rho.mean():.4f} / {pit_rho[-1]:.4f}")
print("  => a genuine walk-forward baseline; compare against the single-split AR(1) (rank IC ~0.26).")


[2026-09-15 18:12:45] START Section 11: tradeability reality check
[2026-09-15 18:12:45] START PIT expanding AR(1)
[2026-09-15 18:12:52] END PIT expanding AR(1) | elapsed 6.53s
Point-in-time expanding AR(1) (rho_t from data strictly before the forecast):
  forecast dates      : 93
  mean daily rank IC  : 0.2195  (t-stat 5.18)
  rho_t  first/mean/last : 0.1138 / 0.1304 / 0.1783
  => a genuine walk-forward baseline; compare against the single-split AR(1) (rank IC ~0.26).


### 11.2 Stale-Mark Decontamination

Two tests strip the evaluated-mark smoothing bleed from the residual IC:

- **Skip-a-day**: replace the target $\varepsilon_{t+1}$ with $\varepsilon_{t+2}$, gapping the immediate 1-day echo. If $\operatorname{IC}(\varepsilon_t,\varepsilon_{t+2}) \ll \operatorname{IC}(\varepsilon_t,\varepsilon_{t+1})$, the edge is a one-day stale-price bounce, not persistence.
- **Active vs stale**: recompute the $h{=}1$ IC inside the **highest** residual-volatility quintile (active, least stale) and the **lowest** (stale). Alpha should live in *active* names; if it lives in *stale* names, it is microstructure.

Retention ratios (skip-IC / baseline-IC, active-IC / stale-IC) near 1 support a real signal; near 0 means smoothing.

In [ ]:
# ── 11.2 STALE-MARK DECONTAMINATION ─────────────────────────────────────
with timed_block("stale-mark decontamination"):
    ic_h1 = np.array([_rank_ic(Rp[:, t], Rp[:, t + 1]) for t in range(T_rv - 1)])
    ic_skip = np.array([_rank_ic(Rp[:, t], Rp[:, t + 2]) for t in range(T_rv - 2)])

    # Per-bond residual volatility quintiles (self-contained staleness proxy).
    _bvol = np.nanstd(Rp, axis=1)
    _vq = pd.qcut(pd.Series(_bvol).rank(method='first'), 5, labels=False).to_numpy()
    _active = _vq == 4      # highest resid vol = most active / least stale
    _stale = _vq == 0       # lowest resid vol = most stale / smoothed
    ic_active = np.array([_rank_ic(Rp[_active, t], Rp[_active, t + 1]) for t in range(T_rv - 1)])
    ic_stale = np.array([_rank_ic(Rp[_stale, t], Rp[_stale, t + 1]) for t in range(T_rv - 1)])

decon_tbl = pd.DataFrame(
    {'mean_rank_IC': [np.nanmean(ic_h1), np.nanmean(ic_skip), np.nanmean(ic_active), np.nanmean(ic_stale)],
     't_stat': [_tstat(ic_h1), _tstat(ic_skip), _tstat(ic_active), _tstat(ic_stale)]},
    index=['baseline  eps_t -> eps_t+1', 'skip-a-day  eps_t -> eps_t+2',
           'active quintile L5 (h=1)', 'stale quintile L1 (h=1)'],
)
print("Stale-mark decontamination — residual rank IC:")
display(decon_tbl.style.format({'mean_rank_IC': '{:.4f}', 't_stat': '{:.2f}'}))

skip_retention = float(np.nanmean(ic_skip) / np.nanmean(ic_h1)) if np.nanmean(ic_h1) else np.nan
active_retention = float(np.nanmean(ic_active) / np.nanmean(ic_stale)) if np.nanmean(ic_stale) else np.nan
print(f"\nSkip-a-day retention (skip/baseline IC): {skip_retention:.2f}  "
      f"(near 0 => 1-day stale-price echo)")
print(f"Active/stale IC ratio: {active_retention:.2f}  "
      f"(<< 1 => edge concentrated in stale marks = microstructure)")


[2026-09-15 18:12:52] START stale-mark decontamination
[2026-09-15 18:13:10] END stale-mark decontamination | elapsed 17.80s
Stale-mark decontamination — residual rank IC:


<HTML output>

,mean_rank_IC,t_stat
baseline eps_t -> eps_t+1,0.2193,6.07
skip-a-day eps_t -> eps_t+2,0.1132,3.00
active quintile L5 (h=1),0.1788,4.66
stale quintile L1 (h=1),0.2743,7.94


Skip-a-day retention (skip/baseline IC): 0.52  (near 0 => 1-day stale-price echo)
Active/stale IC ratio: 0.65  (<< 1 => edge concentrated in stale marks = microstructure)


### 11.3 Turnover and Break-Even Cost

A residual-space decile long/short: each day, long the top-decile and short the bottom-decile bonds by $\varepsilon_t$ (momentum), dollar-neutral, and earn the next-day residual. Gross daily P&L is the D10−D1 spread; two-sided turnover is $\sum_i|w_{i,t}-w_{i,t-1}|$.

Net of a round-trip cost $c$: $\;\text{P\&L}^{net}_t = \text{P\&L}^{gross}_t - \tfrac{c}{2}\sum_i|w_{i,t}-w_{i,t-1}|$. The **break-even round-trip** is $c^\star = 2\,\overline{\text{P\&L}^{gross}}\,/\,\overline{\text{turnover}}$ — the cost at which the strategy makes zero. Because the edge decays in ~2 days, we also rebalance every $k\in\{1,2,3,5\}$ days to trade turnover against signal staleness. A $c^\star$ below a plausible muni round-trip (≈ 20–100 bps) means the signal is not tradable at this frequency.

In [ ]:
# ── 11.3 TURNOVER AND BREAK-EVEN COST ───────────────────────────────────
def _ls_hold(sig, ret, hold=1, n_bins=N_DECILES):
    """Residual-space decile L/S; reform weights every `hold` days. Returns daily gross P&L and two-sided turnover."""
    Tn, Nn = sig.shape[1], sig.shape[0]
    gross, traded = [], []
    prev = np.zeros(Nn); w = np.zeros(Nn)
    for t in range(Tn - 1):
        if t % hold == 0:
            s = sig[:, t]; m = np.isfinite(s); w = np.zeros(Nn)
            if m.sum() >= n_bins * 5:
                idx = np.where(m)[0]; rk = rankdata(s[m], method='average')
                d = np.clip(((rk - 1) / len(rk) * n_bins).astype(int), 0, n_bins - 1)
                lo, hi = idx[d == 0], idx[d == n_bins - 1]
                if len(lo) and len(hi):
                    w[hi] = 1.0 / len(hi); w[lo] = -1.0 / len(lo)
        r = ret[:, t + 1]; rr = np.where(np.isfinite(r), r, 0.0)
        gross.append(float(w @ rr)); traded.append(float(np.abs(w - prev).sum())); prev = w.copy()
    return np.array(gross), np.array(traded)


def _net_sharpe(g, tr, c_rt):
    net = g - 0.5 * c_rt * tr
    return float(np.sqrt(ANN) * np.nanmean(net) / np.nanstd(net)) if np.nanstd(net) > 0 else np.nan


_REF_5, _REF_20 = 5e-4, 20e-4     # 5 bps and 20 bps round-trip
_bt = {}
_rows = []
with timed_block("decile L/S turnover + cost"):
    for hold in [1, 2, 3, 5]:
        g, tr = _ls_hold(Rp, Rp, hold)
        _bt[hold] = (g, tr)
        mg, mt = np.nanmean(g), np.nanmean(tr)
        be_rt = 2.0 * mg / mt if mt > 0 else np.nan
        _rows.append({'rebal_days': hold, 'gross_bps_day': mg * 1e4, 'turnover_day': mt,
                      'gross_sharpe': (np.sqrt(ANN) * mg / np.nanstd(g)) if np.nanstd(g) > 0 else np.nan,
                      'breakeven_rt_bps': be_rt * 1e4,
                      'net_sharpe_5bps': _net_sharpe(g, tr, _REF_5),
                      'net_sharpe_20bps': _net_sharpe(g, tr, _REF_20)})

cost_tbl = pd.DataFrame(_rows).set_index('rebal_days')
print("Residual decile L/S — gross edge vs turnover and break-even round-trip cost:")
display(cost_tbl.style.format({'gross_bps_day': '{:.2f}', 'turnover_day': '{:.2f}', 'gross_sharpe': '{:.2f}',
                               'breakeven_rt_bps': '{:.1f}', 'net_sharpe_5bps': '{:.2f}', 'net_sharpe_20bps': '{:.2f}'}))

be_rt_h1_bps = float(cost_tbl.loc[1, 'breakeven_rt_bps'])
_g1, _tr1 = _bt[1]
_cgrid = np.array([0, 2, 5, 10, 20, 30, 50, 100]) * 1e-4
_net_ann = np.array([252 * np.nanmean(_g1 - 0.5 * c * _tr1) * 1e4 for c in _cgrid])
print(f"\nDaily rebalance break-even round-trip cost: {be_rt_h1_bps:.1f} bps "
      f"(muni round-trips are ~20-100+ bps).")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(_cgrid * 1e4, _net_ann, 'o-', color='tab:red')
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(be_rt_h1_bps, color='tab:blue', linestyle='--', alpha=0.7, label=f'break-even {be_rt_h1_bps:.1f} bps')
ax.axvspan(20, 100, color='grey', alpha=0.15, label='plausible muni round-trip')
ax.set_xlabel('Assumed round-trip cost (bps)'); ax.set_ylabel('Net annualized return (bps)')
ax.set_title('Daily residual L/S: net P&L vs transaction cost'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


[2026-09-15 18:13:10] START decile L/S turnover + cost
[2026-09-15 18:14:10] END decile L/S turnover + cost | elapsed 59.88s
Residual decile L/S — gross edge vs turnover and break-even round-trip cost:


<HTML output>

,gross_bps_day,turnover_day,gross_sharpe,breakeven_rt_bps,net_sharpe_5bps,net_sharpe_20bps
rebal_days,,,,,,
1,6.50,2.73,7.39,4.8,-0.37,-19.92
2,4.36,1.49,5.02,5.8,0.71,-8.59
3,3.98,0.99,5.35,8.0,1.97,-5.26
5,2.33,0.63,2.89,7.4,0.92,-3.63


Daily rebalance break-even round-trip cost: 4.8 bps (muni round-trips are ~20-100+ bps).


<Figure: muni_characteristic_factor_export_93_3.png (image not included in markdown export)>

    


### 11.4 Verdict

Combine the three tests into a single data-driven recommendation:

- **Signal-is-real gate**: the residual IC must *survive* skip-a-day (skip retention ≥ 0.4) **and** exist in active names (active/stale IC ratio ≥ 0.4).
- **Cost gate**: the daily break-even round-trip must clear a plausible muni floor (≥ 20 bps).

**If both pass →** PROCEED to the MSRB pull (Phase 3): re-price the residual L/S at trade prices and confirm the edge is not a marking artifact.
**If either fails →** do **not** build a daily trading portfolio. Reposition the residual signal as a **fair-value / de-smoothing** product — a mark-correction / stale-price-detection tool — and validate *that* claim with MSRB (do dealer trades print through the model's fair value?).

In [ ]:
# ── 11.4 VERDICT ────────────────────────────────────────────────────────
_SKIP_MIN, _ACTIVE_MIN, _BE_MIN_BPS = 0.40, 0.40, 20.0

gate_signal_real = (np.isfinite(skip_retention) and skip_retention >= _SKIP_MIN and
                    np.isfinite(active_retention) and active_retention >= _ACTIVE_MIN)
gate_cost = np.isfinite(be_rt_h1_bps) and be_rt_h1_bps >= _BE_MIN_BPS

print("=" * 72)
print("SECTION 11 VERDICT — residual AR(1) tradeability")
print("=" * 72)
print(f"  PIT expanding-window mean rank IC : {np.nanmean(pit_rank_ic):.4f} (t {_tstat(pit_rank_ic):.2f})")
print(f"  skip-a-day retention              : {skip_retention:.2f}  "
      f"(>= {_SKIP_MIN:.2f} ? {'PASS' if np.isfinite(skip_retention) and skip_retention >= _SKIP_MIN else 'FAIL'})")
print(f"  active/stale IC ratio             : {active_retention:.2f}  "
      f"(>= {_ACTIVE_MIN:.2f} ? {'PASS' if np.isfinite(active_retention) and active_retention >= _ACTIVE_MIN else 'FAIL'})")
print(f"  daily break-even round-trip       : {be_rt_h1_bps:.1f} bps  "
      f"(>= {_BE_MIN_BPS:.0f} ? {'PASS' if gate_cost else 'FAIL'})")
print("-" * 72)
print(f"  signal-is-real gate : {'PASS' if gate_signal_real else 'FAIL'}")
print(f"  cost gate           : {'PASS' if gate_cost else 'FAIL'}")
print("-" * 72)
if gate_signal_real and gate_cost:
    print("  RECOMMENDATION: PROCEED to MSRB pull (Phase 3).")
    print("    Re-price the residual L/S at trade prices; confirm the edge is not a marking artifact")
    print("    before building the constrained portfolio optimizer.")
else:
    print("  RECOMMENDATION: DO NOT build a daily trading portfolio yet.")
    print("    The residual edge is largely a stale-mark / microstructure effect at this frequency.")
    print("    PIVOT: reposition it as a fair-value / de-smoothing (mark-correction) signal and")
    print("    validate with MSRB Test B (do dealer trades print through the model fair value?).")
print("=" * 72)

log_runtime(f"END Section 11 (elapsed {time.perf_counter() - _section11_start:.1f}s)")
# validated 2026-08-01


SECTION 11 VERDICT — residual AR(1) tradeability
  PIT expanding-window mean rank IC : 0.2195 (t 5.18)
  skip-a-day retention              : 0.52  (>= 0.40 ? PASS)
  active/stale IC ratio             : 0.65  (>= 0.40 ? PASS)
  daily break-even round-trip       : 4.8 bps  (>= 20 ? FAIL)
------------------------------------------------------------------------
  signal-is-real gate : PASS
  cost gate           : FAIL
------------------------------------------------------------------------
  RECOMMENDATION: DO NOT build a daily trading portfolio yet.
    The residual edge is largely a stale-mark / microstructure effect at this frequency.
    PIVOT: reposition it as a fair-value / de-smoothing (mark-correction) signal and
    validate with MSRB Test B (do dealer trades print through the model fair value?).
[2026-09-15 18:14:10] END Section 11 (elapsed 84.6s)


## 12. MSRB Trade Data (Phase 3 — Pull & Cache)

> **Pipeline role (redesign).** This is the deferred **next-trade validation** for the OOS score panel. Once `oos_predictions_cache.parquet` (**§9.10–§9.11**) exists, join each signal to MSRB prints occurring **after `signal_available_ts`**, conditioning on trade side / size / signal-to-trade lag / liquidity / rating / tenor / callability, to confirm the edge at real trade prices.

Section 11 showed the residual signal is measured on **evaluated marks**, so its "edge" may be a stale-price artifact. The decisive test needs **actual dealer trade prints** — `muni.MSRB` (RTRS). This section **pulls MSRB trades for the algo universe and caches them locally** so the data is ready before any strategy build, following the same incremental-cache pattern as the price panel in Section 1.

### What MSRB gives us
Per the source contract, `muni.MSRB` carries `date`, `cusip`, `quantity`, `tradetime`, `tradetype` (some feeds also expose a trade price / yield). Trade `tradetype` distinguishes customer-buy / customer-sell / inter-dealer prints. We pull the **raw** prints (no `tradetype` filtering) so the local validation can choose how to use them.

### What it enables (built next, once the cache exists)
- **Test A — re-price the residual L/S at trade levels.** Replace evaluated-mark residual returns with returns measured between real prints. If the Section 11 edge survives at trade prices, it is tradable; if it collapses, it was a marking effect.
- **Test B — fair-value / de-smoothing.** Do subsequent dealer trades print **through** the model's fair value (i.e., does a rich/cheap residual predict the direction of the next real trade)? This validates the signal as a mark-correction tool even if it is not a daily trading strategy.

### Caching
Raw prints are cached to `msrb_trades_cache.pkl`. On rerun the Deephaven query starts from the latest cached date (re-pulling that overlap day to catch late prints), then concatenates and de-duplicates so only new prints are appended.

> **Schema note.** The exact column set of `muni.MSRB` in this environment is discovered at pull time and printed; the query keeps the documented keys plus any price/yield fields that exist. If no trade-price column is present, Test A/B need a priced RTRS feed — the coverage cell flags this explicitly.

In [ ]:
# ── 12.1 INCREMENTAL MSRB PULL WINDOW ───────────────────────────────────
# Mirror the panel_slim pattern: derive the pull start from the local MSRB cache
# and inject it into the remote Deephaven session for the %%deephaven_cell query.
base_msrb_cutoff = '2026-04-01'
MSRB_CACHE_PATH = Path('msrb_trades_cache.pkl')
msrb_cutoff_date = base_msrb_cutoff

if MSRB_CACHE_PATH.exists():
    try:
        _c = pd.read_pickle(MSRB_CACHE_PATH)
        if isinstance(_c, pd.DataFrame) and not _c.empty and 'date' in _c.columns:
            _d = pd.to_datetime(_c['date'], errors='coerce').dropna()
            if not _d.empty:
                _from = max(_d.max().normalize() - pd.Timedelta(days=1), pd.Timestamp(base_msrb_cutoff))
                msrb_cutoff_date = str(_from.date())
                print(f"Found local MSRB cache through {_d.max().date()}; refreshing from {msrb_cutoff_date}")
            else:
                print("Local MSRB cache has no parseable dates; using base cutoff")
        else:
            print("Local MSRB cache is empty or invalid; using base cutoff")
    except Exception as exc:
        print(f"WARNING: could not inspect {MSRB_CACHE_PATH}: {exc}; using base cutoff")
    finally:
        globals().pop('_c', None)
        globals().pop('_d', None)
else:
    print(f"No local MSRB cache found; pulling from {msrb_cutoff_date}")

try:
    DeephavenSession.session.run_script(f"msrb_cutoff_date = '{msrb_cutoff_date}'")
    print(f"Injected msrb_cutoff_date='{msrb_cutoff_date}' into the Deephaven session.")
except Exception as exc:
    print(f"Deephaven session not ready to receive msrb_cutoff_date ({exc}); "
          f"MSRB cell will default to the full window until the session is live.")


No local MSRB cache found; pulling from 2026-04-01
Injected msrb_cutoff_date='2026-04-01' into the Deephaven session.


In [ ]:
%%deephaven_cell

import time as _time

# Runs in the remote Deephaven session. msrb_cutoff_date may have been injected
# by the preceding local cell; default to the full research window otherwise.
try:
    msrb_cutoff_date
except NameError:
    msrb_cutoff_date = '2026-04-01'

_cell_start = _time.perf_counter()
print("[runtime] START Deephaven MSRB ingestion: RTRS trade prints, algo universe")
print(f"msrb_cutoff_date: {msrb_cutoff_date}")

# Same universe restriction as the price panel keeps the pull tractable.
algo_universe = (
    db.historical_table("muni_offline", "muni_algo_trade_track")
      .select_distinct(['cusip'])
)

msrb_raw = (
    db.historical_table("muni", "MSRB")
      .where(f"date >= `{msrb_cutoff_date}`")
      .where_in(algo_universe, cols=['cusip'])
)


def _dh_cols(tbl):
    """Best-effort column-name introspection across Deephaven API flavors."""
    try:
        return [str(n) for n in tbl.column_names]
    except Exception:
        pass
    try:
        return [c.name for c in tbl.columns]
    except Exception:
        pass
    try:
        return [str(n) for n in tbl.getDefinition().getColumnNames()]
    except Exception:
        return []


# Keep the documented keys plus any price/yield/size fields that exist, so the
# local validation can re-price at trade levels when the feed carries them.
_avail = _dh_cols(msrb_raw)
print("MSRB available columns:", _avail)
_base_cols = ['date', 'cusip', 'quantity', 'tradetime', 'tradetype']
_extra_candidates = [
    'price', 'dollar_price', 'trade_price', 'tradeprice', 'clean_price', 'par_price',
    'yield', 'trade_yield', 'tradeyield', 'yld',
    'par_traded', 'par', 'side', 'settlement_date', 'settledate', 'pricetype',
]
if _avail:
    _keep = [c for c in _base_cols + _extra_candidates if c in _avail]
    if not _keep:
        _keep = _base_cols          # names differ entirely; fall back to documented set
else:
    _keep = _base_cols              # introspection unavailable; use documented set
print("MSRB columns kept:", _keep)

msrb_slim = msrb_raw.view(_keep)
print("msrb_slim incremental size:", msrb_slim.size)
print(f"[runtime] END Deephaven MSRB ingestion | elapsed {_time.perf_counter() - _cell_start:.2f}s")
DeephavenWidget(msrb_slim)


[runtime] START Deephaven MSRB ingestion: RTRS trade prints, algo universe
msrb_cutoff_date: 2026-04-01
('MSRB available columns:', ['date', 'atsindicator', 'bondtype', 'brokersbroker', 'coupon', 'cusip', 'dated_date', 'feed_id', 'listofrngpxind', 'maturity_date', 'message_time', 'muni_tradedate', 'ntbcindicator', 'parweightedavgpxind', 'price', 'quantity', 'security_description', 'settlement_date', 'tradetime', 'tradetype', 'trade_id', 'trantype_indicator', 'whenissued', 'yield', 'log_timestamp', 'TailerTxTime', 'DisRxTime'])
('MSRB columns kept:', ['date', 'cusip', 'quantity', 'tradetime', 'tradetype', 'price', 'yield', 'settlement_date'])
('msrb_slim incremental size:', 4687318)
[runtime] END Deephaven MSRB ingestion | elapsed 8.96s



DeephavenWidget(height=600, iframe_url='https://nppra52a0006.wellsfargo.com:24781/iframe/widget/?name=_dw_var_…



Completed on FICC Deephaven worker


In [ ]:
# ── 12.3 PULL MSRB TO PANDAS + INCREMENTAL CACHE ────────────────────────
MSRB_CACHE_PATH = Path('msrb_trades_cache.pkl')

try:
    with timed_block("Pull incremental MSRB to pandas"):
        msrb_incremental = DeephavenSession.session.open_table('msrb_slim').to_arrow().to_pandas()
except Exception as exc:
    print(f"WARNING: could not pull msrb_slim from Deephaven ({exc}); using local cache if available.")
    if not MSRB_CACHE_PATH.exists():
        raise RuntimeError(
            f"MSRB Deephaven pull failed and local cache {MSRB_CACHE_PATH} is not available in the notebook kernel cwd."
        ) from exc
    with timed_block("Load local MSRB cache without remote refresh"):
        msrb = pd.read_pickle(MSRB_CACHE_PATH)
    if 'date' in msrb.columns:
        msrb['date'] = pd.to_datetime(msrb['date'], errors='coerce')
    print(f"Loaded local MSRB cache without remote refresh: {len(msrb):,} rows")
    if 'date' in msrb.columns and msrb['date'].notna().any():
        print(f"MSRB cache date range: {msrb['date'].min().date()} → {msrb['date'].max().date()}")
    print(f"columns: {list(msrb.columns)}")
    display(msrb.head(3))
else:
    msrb_incremental['date'] = pd.to_datetime(msrb_incremental['date'], errors='coerce')
    print(f"MSRB incremental pull: {msrb_incremental.shape}; columns: {list(msrb_incremental.columns)}")

    if MSRB_CACHE_PATH.exists():
        with timed_block("Load local MSRB cache"):
            try:
                cached_msrb = pd.read_pickle(MSRB_CACHE_PATH)
            except Exception as exc:
                print(f"WARNING: could not load {MSRB_CACHE_PATH}: {exc}; rebuilding from incremental pull")
                cached_msrb = pd.DataFrame(columns=msrb_incremental.columns)
        if not cached_msrb.empty and 'date' in cached_msrb.columns:
            cached_msrb['date'] = pd.to_datetime(cached_msrb['date'], errors='coerce')
    else:
        cached_msrb = pd.DataFrame(columns=msrb_incremental.columns)

    with timed_block("Merge incremental MSRB into local cache"):
        before_rows = len(cached_msrb)
        msrb = pd.concat([cached_msrb, msrb_incremental], ignore_index=True)
        _sort_keys = [c for c in ['date', 'cusip'] if c in msrb.columns]
        if _sort_keys:
            msrb = msrb.dropna(subset=_sort_keys)
            msrb = msrb.drop_duplicates(keep='last').sort_values(_sort_keys).reset_index(drop=True)
        else:
            msrb = msrb.drop_duplicates(keep='last').reset_index(drop=True)
        msrb.to_pickle(MSRB_CACHE_PATH)

    new_rows = len(msrb) - before_rows
    print(f"local MSRB cache: {before_rows:,} → {len(msrb):,} rows ({new_rows:+,} net)")
    if 'date' in msrb.columns and msrb['date'].notna().any():
        print(f"MSRB cache date range: {msrb['date'].min().date()} → {msrb['date'].max().date()}")
    print(f"columns: {list(msrb.columns)}")
    display(msrb.head(3))
    del msrb_incremental


### 12.4 Coverage & Schema Check

Confirms the cache is usable for the Phase 3 tests: how many residual-universe bonds actually trade in MSRB, what fraction of observed residual bond-days carry a real print, and — critically — whether a **trade price / yield** column is present (required for re-pricing in Test A/B). CUSIP formats are compared as-is; a low overlap may reflect a check-digit / 8-vs-9-char mismatch rather than missing bonds.

In [ ]:
# ── 12.4 MSRB COVERAGE + SCHEMA CHECK ───────────────────────────────────
_price_cols = [c for c in ['price', 'dollar_price', 'trade_price', 'tradeprice', 'clean_price', 'par_price']
               if c in msrb.columns]
_yield_cols = [c for c in ['yield', 'trade_yield', 'tradeyield', 'yld'] if c in msrb.columns]
print("MSRB columns cached          :", list(msrb.columns))
print("price-like columns present   :", _price_cols or "NONE")
print("yield-like columns present   :", _yield_cols or "NONE")

_required_msr_cols = {'date', 'cusip'}
_missing_msr_cols = sorted(_required_msr_cols - set(msrb.columns))
_has_residual_context = all(name in globals() for name in ['rv_cusips', 'rv_dates', 'Rp'])

if _missing_msr_cols:
    print(f"\nCoverage check skipped: MSRB cache is missing required columns {_missing_msr_cols}.")
elif not _has_residual_context:
    print("\nResidual-universe coverage skipped: run Sections 9-11 first to create rv_cusips, rv_dates, and Rp.")
    print("MSRB pull/cache succeeded; rerun this cell after the residual panel exists.")
else:
    # Overlap of MSRB-traded bonds with the residual modeling universe (Section 9).
    _msrb_cusips = set(msrb['cusip'].dropna().astype(str).unique())
    _resid_cusips = set(map(str, rv_cusips))
    _overlap = _msrb_cusips & _resid_cusips
    print(f"\nMSRB distinct cusips         : {len(_msrb_cusips):,}")
    print(f"residual-universe cusips     : {len(_resid_cusips):,}")
    print(f"overlap (tradeable & modeled): {len(_overlap):,} "
          f"({100 * len(_overlap) / max(len(_resid_cusips), 1):.1f}% of residual universe)")

    # How many observed residual bond-days carry a real print.
    _rv_set = set(pd.DatetimeIndex(pd.to_datetime(rv_dates, errors='coerce')).normalize())
    _md = msrb[['date', 'cusip']].copy()
    _md['date'] = pd.to_datetime(_md['date'], errors='coerce')
    _md = _md.dropna(subset=['date', 'cusip'])
    _md['cusip'] = _md['cusip'].astype(str)
    _md['_d'] = _md['date'].dt.normalize()
    _md = _md[_md['_d'].isin(_rv_set) & _md['cusip'].isin(_overlap)]
    _bond_days_with_trade = _md.drop_duplicates(['cusip', '_d']).shape[0]
    _total_obs = int(np.isfinite(Rp).sum())
    print(f"\nresidual bond-days (observed): {_total_obs:,}")
    print(f"bond-days with an MSRB print : {_bond_days_with_trade:,} "
          f"({100 * _bond_days_with_trade / max(_total_obs, 1):.1f}% of observed residual cells)")
    del _md

if _price_cols or _yield_cols:
    print("\n=> Trade price/yield IS available: Test A (re-price the residual L/S at trade")
    print("   levels) and Test B (fair-value convergence) can be built on this cache.")
else:
    print("\n=> muni.MSRB here exposes counts/timestamps but NO trade price/yield column.")
    print("   Test A/B need a priced RTRS feed; with this table we can still test whether the")
    print("   residual signal predicts trade *activity/direction*, not trade P&L.")


MSRB columns cached          : ['date', 'cusip', 'quantity', 'tradetime', 'tradetype', 'price', 'yield', 'settlement_date']
price-like columns present   : ['price']
yield-like columns present   : ['yield']

MSRB distinct cusips         : 215,422
residual-universe cusips     : 229,857
overlap (tradeable & modeled): 209,529 (91.2% of residual universe)

residual bond-days (observed): 25,804,752
bond-days with an MSRB print : 2,000,204 (7.8% of observed residual cells)

=> Trade price/yield IS available: Test A (re-price the residual L/S at trade
   levels) and Test B (fair-value convergence) can be built on this cache.


## 13. MSRB Trade-Price Validation: Test A and Test B

Section 12 confirmed the MSRB cache has `price` and `yield`, so we can now test whether the evaluated-mark residual signal transfers to real trade prints.

### Test B — fair-value / de-smoothing validation

This is the test most aligned with the evidence so far. For each MSRB trade day, take the **previous model date** $t$ (strictly before the trade date), read the residual $\varepsilon_{i,t}$, and compare the next trade price to the evaluated mark at $t$:

$$r^{\text{trade-from-mark}}_{i,t\to\tau}=\frac{P^{\text{MSRB}}_{i,\tau}}{P^{\text{eval}}_{i,t}}-1.$$

If a positive residual means the evaluated mark is rich versus model fair value, then the fair-value signal is $-\varepsilon_{i,t}$ and should predict the next trade-from-mark return. A pass here means the signal is useful as a **mark-correction / fair-value** tool even if it is not directly tradeable.

### Test A — real trade-price P&L validation

This is the harsher trading test. Enter at the first MSRB trade day after the residual signal date and exit at the next MSRB trade day for the same CUSIP. Then test whether the original residual momentum signal $\varepsilon_{i,t}$ predicts trade-to-trade returns:

$$r^{\text{trade-to-trade}}_{i,\tau\to\tau'}=\frac{P^{\text{MSRB}}_{i,\tau'}}{P^{\text{MSRB}}_{i,	au}}-1.$$

A pass here means the residual AR(1) survives **after** paying the price-discovery entry print. A fail here, combined with a pass in Test B, means the edge is mostly de-smoothing rather than a daily trading strategy.

In [ ]:
# ── 13.1 BUILD MSRB VALIDATION PANEL ────────────────────────────────────
_section13_start = time.perf_counter()
log_runtime("START Section 13: MSRB trade-price validation")

_required_13 = ['msrb', 'rv_cusips', 'rv_dates', 'Rp', 'R_panel', 'df']
_missing_13 = [name for name in _required_13 if name not in globals()]
if _missing_13:
    raise RuntimeError(f"Run Sections 9-12 first; missing required variables: {_missing_13}")

_price_col13 = next((c for c in ['price', 'dollar_price', 'trade_price', 'tradeprice', 'clean_price', 'par_price']
                     if c in msrb.columns), None)
if _price_col13 is None:
    raise RuntimeError("MSRB cache has no trade-price column; cannot run Test A/B trade-price validation.")

with timed_block("prepare MSRB daily trade panel"):
    _rv_cusip_set = set(map(str, rv_cusips))
    _m = msrb[['date', 'cusip', 'quantity', 'tradetype', _price_col13]].copy()
    _m = _m.rename(columns={_price_col13: 'trade_price'})
    _m['date'] = pd.to_datetime(_m['date'], errors='coerce').dt.normalize()
    _m['cusip'] = _m['cusip'].astype(str)
    _m['quantity'] = pd.to_numeric(_m['quantity'], errors='coerce') if 'quantity' in _m else np.nan
    _m['trade_price'] = pd.to_numeric(_m['trade_price'], errors='coerce')
    _m = _m.dropna(subset=['date', 'cusip', 'trade_price'])
    _m = _m[(_m['trade_price'] > 0) & (_m['cusip'].isin(_rv_cusip_set))]
    _m['_qty_w'] = _m['quantity'].fillna(0.0).clip(lower=0.0)
    _m['_px_qty'] = _m['trade_price'] * _m['_qty_w']

    _g = (_m.groupby(['cusip', 'date'], sort=False)
            .agg(price_mean=('trade_price', 'mean'),
                 px_qty=('_px_qty', 'sum'),
                 par_traded=('_qty_w', 'sum'),
                 n_prints=('trade_price', 'size'))
            .reset_index())
    _g['trade_price'] = np.where(_g['par_traded'] > 0, _g['px_qty'] / _g['par_traded'], _g['price_mean'])
    msrb_daily = (_g[['cusip', 'date', 'trade_price', 'n_prints', 'par_traded']]
                  .sort_values(['cusip', 'date'])
                  .reset_index(drop=True))

    msrb_daily['next_trade_date'] = msrb_daily.groupby('cusip')['date'].shift(-1)
    msrb_daily['next_trade_price'] = msrb_daily.groupby('cusip')['trade_price'].shift(-1)
    msrb_daily['trade_to_trade_ret'] = msrb_daily['next_trade_price'] / msrb_daily['trade_price'] - 1.0
    msrb_daily['hold_days'] = (msrb_daily['next_trade_date'] - msrb_daily['date']).dt.days

with timed_block("map MSRB trades to prior residual date"):
    _rv_dates = pd.DatetimeIndex(pd.to_datetime(rv_dates, errors='coerce')).normalize()
    _rv_ns = _rv_dates.values.astype('datetime64[ns]')
    _trade_ns = msrb_daily['date'].to_numpy(dtype='datetime64[ns]')
    _t_idx_all = np.searchsorted(_rv_ns, _trade_ns, side='left') - 1

    _row_map = pd.Series(np.arange(len(rv_cusips), dtype=np.int64), index=np.asarray(rv_cusips, dtype=str))
    _row_idx_all = msrb_daily['cusip'].map(_row_map).to_numpy()
    _valid = (_t_idx_all >= 0) & pd.notna(_row_idx_all)
    _row_idx = _row_idx_all[_valid].astype(np.int64)
    _t_idx = _t_idx_all[_valid].astype(np.int64)

    msrb_validation = msrb_daily.loc[_valid].copy().reset_index(drop=True)
    msrb_validation['signal_date'] = _rv_dates[_t_idx].to_numpy()
    msrb_validation['eps_signal'] = Rp[_row_idx, _t_idx]
    msrb_validation['resid_row'] = _row_idx
    msrb_validation['resid_t'] = _t_idx
    msrb_validation['lag_days'] = (msrb_validation['date'] - msrb_validation['signal_date']).dt.days

    _eval_px = (df[['cusip', 'date', 'price']]
                .assign(cusip=lambda x: x['cusip'].astype(str),
                        date=lambda x: pd.to_datetime(x['date'], errors='coerce').dt.normalize(),
                        eval_price=lambda x: pd.to_numeric(x['price'], errors='coerce'))
                [['cusip', 'date', 'eval_price']]
                .dropna(subset=['cusip', 'date', 'eval_price'])
                .drop_duplicates(['cusip', 'date'], keep='last')
                .rename(columns={'date': 'signal_date'}))
    msrb_validation = msrb_validation.merge(_eval_px, on=['cusip', 'signal_date'], how='left')
    msrb_validation['trade_from_mark_ret'] = msrb_validation['trade_price'] / msrb_validation['eval_price'] - 1.0

msrb_validation = msrb_validation.dropna(subset=['eps_signal', 'trade_price', 'eval_price']).reset_index(drop=True)
msrb_validation = msrb_validation[(msrb_validation['lag_days'] >= 1) & (msrb_validation['lag_days'] <= 20)]
msrb_validation = msrb_validation[np.isfinite(msrb_validation['trade_from_mark_ret'])].reset_index(drop=True)

print(f"MSRB daily trade rows          : {len(msrb_daily):,}")
print(f"MSRB validation rows          : {len(msrb_validation):,}")
print(f"price column used             : {_price_col13}")
if msrb_validation.empty:
    print("No validation rows available after filtering; relax lag/date filters or inspect CUSIP/date alignment.")
else:
    print(f"signal-date lag days min/med/max: {msrb_validation['lag_days'].min():.0f} / "
          f"{msrb_validation['lag_days'].median():.0f} / {msrb_validation['lag_days'].max():.0f}")
    print(f"trade-to-trade rows available : {int(msrb_validation['trade_to_trade_ret'].notna().sum()):,}")
    display(msrb_validation.head(3))

del _m, _g, _eval_px, _row_map, _row_idx, _t_idx, _row_idx_all, _t_idx_all, _valid


[2026-09-15 19:15:27] START Section 13: MSRB trade-price validation
[2026-09-15 19:15:27] START prepare MSRB daily trade panel
[2026-09-15 19:15:40] END prepare MSRB daily trade panel | elapsed 13.12s
[2026-09-15 19:15:40] START map MSRB trades to prior residual date
[2026-09-15 19:15:56] END map MSRB trades to prior residual date | elapsed 15.51s
MSRB daily trade rows          : 2,017,388
MSRB validation rows          : 1,972,662
price column used             : price
signal-date lag days min/med/max: 1 / 1 / 4
trade-to-trade rows available : 1,762,693


,cusip,date,trade_price,n_prints,par_traded,next_trade_date,next_trade_price,trade_to_trade_ret,hold_days,signal_date,eps_signal,resid_row,resid_t,lag_days,eval_price,trade_from_mark_ret
0,0004162A4,2026-04-27,112.721,2,20000.000,NaT,NaN,NaN,NaN,2026-04-24,-0.000,0,16,3,113.355,-0.006
1,0004162D8,2026-07-22,115.621,2,60000.000,2026-07-27,114.613,-0.009,5.000,2026-07-21,0.000,3,75,1,115.864,-0.002
2,0004162D8,2026-07-27,114.613,1,125000.000,2026-08-14,115.977,0.012,18.000,2026-07-24,0.000,3,78,3,114.522,0.001


In [ ]:
# ── 13.2 TEST B: FAIR-VALUE / DE-SMOOTHING ──────────────────────────────
def _safe_corr(a, b, min_n=50, method='pearson'):
    d = pd.DataFrame({'a': a, 'b': b}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < min_n or d['a'].std() == 0 or d['b'].std() == 0:
        return np.nan
    return float(d['a'].corr(d['b'], method=method))


def _daily_rank_ic(frame, signal_col, target_col, date_col='date', min_n=50):
    vals = []
    for _, g in frame.groupby(date_col, sort=False):
        vals.append(_safe_corr(g[signal_col], g[target_col], min_n=min_n, method='spearman'))
    return np.asarray(vals, dtype=float)


def _ann_tstat(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.sqrt(len(x)) * x.mean() / x.std()) if len(x) > 2 and x.std() > 0 else np.nan


def _decile_spread(frame, signal_col, target_col, date_col='date', n_bins=10, min_n=50):
    spreads = []
    rows = []
    for d, g in frame.groupby(date_col, sort=False):
        g = g[[signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(g) < n_bins * min_n:
            continue
        dec = pd.qcut(g[signal_col].rank(method='first'), n_bins, labels=False)
        profile = g.groupby(dec, observed=False)[target_col].mean()
        if 0 in profile.index and n_bins - 1 in profile.index:
            spreads.append(float(profile.loc[n_bins - 1] - profile.loc[0]))
            rows.extend({'date': d, 'decile': int(k) + 1, 'target_mean': float(v)} for k, v in profile.items())
    prof = pd.DataFrame(rows).groupby('decile')['target_mean'].mean() if rows else pd.Series(dtype=float)
    return prof, np.asarray(spreads, dtype=float)

with timed_block("Test B fair-value / de-smoothing"):
    test_b = msrb_validation.copy()
    test_b['fv_signal'] = -test_b['eps_signal']
    test_b = test_b[np.isfinite(test_b['fv_signal']) & np.isfinite(test_b['trade_from_mark_ret'])]
    _b_daily_ric = _daily_rank_ic(test_b, 'fv_signal', 'trade_from_mark_ret', min_n=100)
    _b_prof, _b_spread = _decile_spread(test_b, 'fv_signal', 'trade_from_mark_ret', min_n=20)

msrb_test_b_summary = pd.Series({
    'n_rows': len(test_b),
    'n_dates': test_b['date'].nunique(),
    'mean_trade_from_mark_bps': 1e4 * test_b['trade_from_mark_ret'].mean(),
    'pearson_IC': _safe_corr(test_b['fv_signal'], test_b['trade_from_mark_ret'], method='pearson'),
    'rank_IC': _safe_corr(test_b['fv_signal'], test_b['trade_from_mark_ret'], method='spearman'),
    'daily_rank_IC_mean': np.nanmean(_b_daily_ric),
    'daily_rank_IC_t': _ann_tstat(_b_daily_ric),
    'D10_minus_D1_bps': 1e4 * np.nanmean(_b_spread),
    'spread_t': _ann_tstat(_b_spread),
})
print("Test B — fair-value / de-smoothing: does -residual predict next trade vs evaluated mark?")
display(msrb_test_b_summary.to_frame('value').style.format('{:,.4f}'))
print("\nDecile profile: fair-value signal (-eps) vs next trade-from-mark return (bps)")
display((1e4 * _b_prof).rename('trade_from_mark_bps').to_frame().style.format('{:.2f}'))

fig, ax = plt.subplots(figsize=(8, 4))
if not _b_prof.empty:
    ax.plot(_b_prof.index, 1e4 * _b_prof.values, marker='o', color='tab:blue')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Fair-value signal decile: D1 low (-eps) -> D10 high (-eps)')
ax.set_ylabel('Next MSRB trade vs eval mark (bps)')
ax.set_title('Test B: residual as fair-value / de-smoothing signal')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


[2026-09-15 19:16:22] START Test B fair-value / de-smoothing
[2026-09-15 19:16:25] END Test B fair-value / de-smoothing | elapsed 2.51s
Test B — fair-value / de-smoothing: does -residual predict next trade vs evaluated mark?


<HTML output>

,value
n_rows,"1,972,662.0000"
n_dates,112.0000
mean_trade_from_mark_bps,-28.3261
pearson_IC,0.0211
rank_IC,0.0198
daily_rank_IC_mean,0.0233
daily_rank_IC_t,4.1277
D10_minus_D1_bps,5.6181
spread_t,3.9355


Decile profile: fair-value signal (-eps) vs next trade-from-mark return (bps)


<HTML output>

,trade_from_mark_bps
decile,
1,-30.62
2,-28.33
3,-29.19
4,-29.13
5,-28.39
6,-27.58
7,-27.77
8,-28.61
9,-27.77


<Figure: muni_characteristic_factor_export_104_4.png (image not included in markdown export)>

In [ ]:
# ── 13.3 TEST A: TRADE-PRICE P&L AFTER ENTRY PRINT ──────────────────────
with timed_block("Test A trade-to-trade P&L"):
    test_a = msrb_validation.dropna(subset=['trade_to_trade_ret', 'hold_days']).copy()
    test_a = test_a[(test_a['hold_days'] >= 1) & (test_a['hold_days'] <= 20)]
    test_a['alpha_signal'] = test_a['eps_signal']
    test_a = test_a[np.isfinite(test_a['alpha_signal']) & np.isfinite(test_a['trade_to_trade_ret'])]
    _a_daily_ric = _daily_rank_ic(test_a, 'alpha_signal', 'trade_to_trade_ret', min_n=100)
    _a_prof, _a_spread = _decile_spread(test_a, 'alpha_signal', 'trade_to_trade_ret', min_n=20)

msrb_test_a_summary = pd.Series({
    'n_rows': len(test_a),
    'n_dates': test_a['date'].nunique(),
    'median_hold_days': test_a['hold_days'].median(),
    'mean_trade_to_trade_bps': 1e4 * test_a['trade_to_trade_ret'].mean(),
    'pearson_IC': _safe_corr(test_a['alpha_signal'], test_a['trade_to_trade_ret'], method='pearson'),
    'rank_IC': _safe_corr(test_a['alpha_signal'], test_a['trade_to_trade_ret'], method='spearman'),
    'daily_rank_IC_mean': np.nanmean(_a_daily_ric),
    'daily_rank_IC_t': _ann_tstat(_a_daily_ric),
    'D10_minus_D1_bps': 1e4 * np.nanmean(_a_spread),
    'spread_t': _ann_tstat(_a_spread),
})
print("Test A — trade-price P&L: does residual predict next trade-to-trade return?")
display(msrb_test_a_summary.to_frame('value').style.format('{:,.4f}'))
print("\nDecile profile: residual signal (eps) vs next trade-to-trade return (bps)")
display((1e4 * _a_prof).rename('trade_to_trade_bps').to_frame().style.format('{:.2f}'))

fig, ax = plt.subplots(figsize=(8, 4))
if not _a_prof.empty:
    ax.plot(_a_prof.index, 1e4 * _a_prof.values, marker='o', color='tab:red')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Residual signal decile: D1 low eps -> D10 high eps')
ax.set_ylabel('Next MSRB trade-to-trade return (bps)')
ax.set_title('Test A: residual signal as real trade-price P&L')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


[2026-09-15 19:17:05] START Test A trade-to-trade P&L
[2026-09-15 19:17:07] END Test A trade-to-trade P&L | elapsed 2.23s
Test A — trade-price P&L: does residual predict next trade-to-trade return?


<HTML output>

,value
n_rows,"1,512,649.0000"
n_dates,111.0000
median_hold_days,4.0000
mean_trade_to_trade_bps,-6.1328
pearson_IC,0.0068
rank_IC,0.0151
daily_rank_IC_mean,0.0114
daily_rank_IC_t,2.2296
D10_minus_D1_bps,4.8384
spread_t,4.0116


Decile profile: residual signal (eps) vs next trade-to-trade return (bps)


<HTML output>

,trade_to_trade_bps
decile,
1,-9.75
2,-7.52
3,-7.00
4,-6.62
5,-5.88
6,-6.50
7,-7.04
8,-5.94
9,-6.06


<Figure: muni_characteristic_factor_export_105_4.png (image not included in markdown export)>

In [ ]:
# ── 13.4 MSRB TEST A/B VERDICT ──────────────────────────────────────────
_B_RIC_MIN, _B_SPREAD_MIN_BPS = 0.01, 2.0
_A_RIC_MIN, _A_SPREAD_MIN_BPS = 0.01, 10.0
_A_COST_FLOOR_BPS = 20.0

b_pass = (msrb_test_b_summary['daily_rank_IC_mean'] >= _B_RIC_MIN and
          msrb_test_b_summary['D10_minus_D1_bps'] >= _B_SPREAD_MIN_BPS)
a_stat_pass = (msrb_test_a_summary['daily_rank_IC_mean'] >= _A_RIC_MIN and
               msrb_test_a_summary['D10_minus_D1_bps'] >= 2.0)
a_econ_pass = (msrb_test_a_summary['D10_minus_D1_bps'] >= _A_SPREAD_MIN_BPS and
               msrb_test_a_summary['D10_minus_D1_bps'] >= _A_COST_FLOOR_BPS)

msrb_ab_verdict = pd.Series({
    'Test B daily rank IC': msrb_test_b_summary['daily_rank_IC_mean'],
    'Test B D10-D1 bps': msrb_test_b_summary['D10_minus_D1_bps'],
    'Test A daily rank IC': msrb_test_a_summary['daily_rank_IC_mean'],
    'Test A D10-D1 bps': msrb_test_a_summary['D10_minus_D1_bps'],
    'Test A median hold days': msrb_test_a_summary['median_hold_days'],
    'Test B pass': b_pass,
    'Test A stat pass': a_stat_pass,
    'Test A economic pass': a_econ_pass,
})

print("=" * 76)
print("SECTION 13 VERDICT — MSRB validation of residual signal")
print("=" * 76)
display(msrb_ab_verdict.to_frame('value').style.format('{:,.4f}'))
print("-" * 76)
if b_pass and not a_econ_pass:
    print("RECOMMENDATION: PIVOT toward fair-value / de-smoothing first.")
    print("  Test B passes: residuals forecast where real MSRB trades print vs evaluated marks.")
    print("  Test A has a weak positive spread, but it is too small versus plausible muni round-trip costs.")
    print("  Build the product as mark-correction / rich-cheap validation; keep trade execution as secondary.")
elif b_pass and a_econ_pass:
    print("RECOMMENDATION: PROCEED to a constrained MSRB-backed portfolio build.")
    print("  Both trade-price P&L and fair-value convergence clear the initial gates.")
elif a_stat_pass and not b_pass:
    print("RECOMMENDATION: Investigate trade-price alpha, but diagnose why fair-value convergence failed.")
else:
    print("RECOMMENDATION: STOP strategy build for now; neither MSRB test clears the initial gates.")
print("=" * 76)
log_runtime(f"END Section 13 (elapsed {time.perf_counter() - _section13_start:.1f}s)")


SECTION 13 VERDICT — MSRB validation of residual signal


<HTML output>

,value
Test B daily rank IC,0.0233
Test B D10-D1 bps,5.6181
Test A daily rank IC,0.0114
Test A D10-D1 bps,4.8384
Test A median hold days,4.0000
Test B pass,1.0000
Test A stat pass,1.0000
Test A economic pass,0.0000


----------------------------------------------------------------------------
RECOMMENDATION: PIVOT toward fair-value / de-smoothing first.
  Test B passes: residuals forecast where real MSRB trades print vs evaluated marks.
  Test A has a weak positive spread, but it is too small versus plausible muni round-trip costs.
  Build the product as mark-correction / rich-cheap validation; keep trade execution as secondary.
[2026-09-15 19:18:00] END Section 13 (elapsed 152.9s)


## 14. Robustness: Within-Bucket and Characteristic-Neutral Residual Rankings

The **headline ranking remains the overall combined IPCA residual signal**: rank bonds by $\varepsilon_{i,t}$ (or a global AR(1) forecast, which is a monotone transform when $\hat\rho>0$). We do **not** rank by individual IPCA factor scores for the alpha test.

This section adds two robustness views:

1. **Within-bucket ranking.** Re-rank the same combined residual signal inside comparable groups: rating, tenor, rating $\times$ tenor, and rating $\times$ tenor $\times$ call status. Then average the daily within-group rank ICs and decile spreads. This tests whether the signal is bond-level relative value among close comparables rather than broad segment separation.
2. **Characteristic-neutralized ranking.** Demean the combined residual signal by date and rating $\times$ tenor $\times$ call-status group, then rank the neutralized residual across the whole traded sample. This is a compact fixed-effect check: does the residual still work after removing static-characteristic bucket means?

Because MSRB prints are sparse inside fine cells, the rating $\times$ tenor $\times$ call-status specification is included only where groups have enough observations on a given date.

In [ ]:
# ── 14.1 SETUP: COMPARABLE BUCKETS + ROBUSTNESS SCORING ────────────────
_section14_start = time.perf_counter()
log_runtime("START Section 14: within-bucket robustness")

_required_14 = ['msrb_validation', 'df', 'test_a', 'test_b']
_missing_14 = [name for name in _required_14 if name not in globals()]
if _missing_14:
    raise RuntimeError(f"Run Sections 13.1-13.3 first; missing required variables: {_missing_14}")


def _coarse_tenor_label(x):
    m = {'1y': '1-3Y', '2y': '1-3Y', '3y': '1-3Y', '4y': '3-7Y', '5-7y': '3-7Y',
         '8-9y': '8-15Y', '10-15y': '8-15Y', '16-20y': '16Y+', '21y+': '16Y+'}
    return m.get(str(x), 'MISSING')


def _call_status_label(x):
    return 'noncall' if str(x) == 'No Call' else 'callable'

with timed_block("attach rating/tenor/call buckets"):
    _bucket_ref = (df[['cusip', 'rating_grade', 'tenor_bucket', 'call_bucket']]
                   .assign(cusip=lambda x: x['cusip'].astype(str))
                   .drop_duplicates('cusip', keep='last')
                   .copy())
    _bucket_ref['rating_coarse'] = _bucket_ref['rating_grade'].map(_coarse_rating)
    _bucket_ref['tenor_coarse'] = _bucket_ref['tenor_bucket'].map(_coarse_tenor_label)
    _bucket_ref['call_status'] = _bucket_ref['call_bucket'].map(_call_status_label)

    msrb_validation_buckets = msrb_validation.merge(
        _bucket_ref[['cusip', 'rating_coarse', 'tenor_coarse', 'call_status']],
        on='cusip', how='left',
    )
    for _c in ['rating_coarse', 'tenor_coarse', 'call_status']:
        msrb_validation_buckets[_c] = msrb_validation_buckets[_c].fillna('MISSING')
    msrb_validation_buckets['rating_tenor'] = (msrb_validation_buckets['rating_coarse'].astype(str)
                                              + '|' + msrb_validation_buckets['tenor_coarse'].astype(str))
    msrb_validation_buckets['rating_tenor_call'] = (msrb_validation_buckets['rating_tenor']
                                                   + '|' + msrb_validation_buckets['call_status'].astype(str))

    _key_cols = ['cusip', 'date', 'signal_date', 'trade_price']
    _bucket_cols = ['rating_coarse', 'tenor_coarse', 'call_status', 'rating_tenor', 'rating_tenor_call']
    test_a_rb = test_a.merge(msrb_validation_buckets[_key_cols + _bucket_cols], on=_key_cols, how='left')
    test_b_rb = test_b.merge(msrb_validation_buckets[_key_cols + _bucket_cols], on=_key_cols, how='left')


def _within_bucket_stats(frame, signal_col, target_col, bucket_col, min_group_n=50, n_bins=10):
    daily_ics, daily_spreads = [], []
    used_groups, used_rows = 0, 0
    for _, day in frame.groupby('date', sort=False):
        group_ics, group_spreads, group_sizes = [], [], []
        for _, g in day.groupby(bucket_col, sort=False):
            g = g[[signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(g) < min_group_n or g[signal_col].std() == 0 or g[target_col].std() == 0:
                continue
            group_ics.append(g[signal_col].corr(g[target_col], method='spearman'))
            group_sizes.append(len(g))
            if len(g) >= n_bins * max(5, min_group_n // 5):
                dec = pd.qcut(g[signal_col].rank(method='first'), n_bins, labels=False)
                prof = g.groupby(dec, observed=False)[target_col].mean()
                if 0 in prof.index and n_bins - 1 in prof.index:
                    group_spreads.append(float(prof.loc[n_bins - 1] - prof.loc[0]))
            used_groups += 1
            used_rows += len(g)
        if group_ics:
            w = np.asarray(group_sizes, dtype=float)
            daily_ics.append(float(np.average(group_ics, weights=w)))
        if group_spreads:
            daily_spreads.append(float(np.mean(group_spreads)))
    daily_ics = np.asarray(daily_ics, dtype=float)
    daily_spreads = np.asarray(daily_spreads, dtype=float)
    return {
        'bucket': bucket_col,
        'used_groups': used_groups,
        'used_rows': used_rows,
        'daily_rank_IC': np.nanmean(daily_ics),
        'rank_IC_t': _ann_tstat(daily_ics),
        'D10_minus_D1_bps': 1e4 * np.nanmean(daily_spreads),
        'spread_t': _ann_tstat(daily_spreads),
        'n_ic_dates': int(np.isfinite(daily_ics).sum()),
        'n_spread_dates': int(np.isfinite(daily_spreads).sum()),
    }

print(f"Section 14 input rows — Test B: {len(test_b_rb):,}; Test A: {len(test_a_rb):,}")
print("Comparable buckets attached:", _bucket_cols)
display(msrb_validation_buckets[_bucket_cols].describe().T[['unique', 'top', 'freq']])


[2026-09-15 19:51:14] START Section 14: within-bucket robustness
[2026-09-15 19:51:14] START attach rating/tenor/call buckets
[2026-09-15 19:51:22] END attach rating/tenor/call buckets | elapsed 7.99s
Section 14 input rows — Test B: 1,972,662; Test A: 1,512,649
Comparable buckets attached: ['rating_coarse', 'tenor_coarse', 'call_status', 'rating_tenor', 'rating_tenor_call']


,unique,top,freq
rating_coarse,8,AA,1269456
tenor_coarse,4,8-15Y,689407
call_status,2,callable,1148013
rating_tenor,28,AA|8-15Y,461567
rating_tenor_call,49,AA|8-15Y|callable,410238


In [ ]:
# ── 14.2 WITHIN-BUCKET RANKING ROBUSTNESS ───────────────────────────────
_BUCKET_SPECS = ['rating_coarse', 'tenor_coarse', 'rating_tenor', 'rating_tenor_call']

with timed_block("within-bucket robustness scoring"):
    _wb_rows = []
    for _bucket in _BUCKET_SPECS:
        _b = _within_bucket_stats(test_b_rb, 'fv_signal', 'trade_from_mark_ret', _bucket, min_group_n=50)
        _b.update({'test': 'B fair-value', 'signal': '-eps', 'target': 'trade_from_mark_ret'})
        _wb_rows.append(_b)

        _a = _within_bucket_stats(test_a_rb, 'alpha_signal', 'trade_to_trade_ret', _bucket, min_group_n=50)
        _a.update({'test': 'A trade P&L', 'signal': 'eps', 'target': 'trade_to_trade_ret'})
        _wb_rows.append(_a)

within_bucket_summary = (pd.DataFrame(_wb_rows)
                         [['test', 'bucket', 'signal', 'target', 'daily_rank_IC', 'rank_IC_t',
                           'D10_minus_D1_bps', 'spread_t', 'used_groups', 'used_rows',
                           'n_ic_dates', 'n_spread_dates']])
print("Within-bucket robustness: same combined residual signal, re-ranked inside comparable groups")
display(within_bucket_summary.style.format({
    'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}', 'D10_minus_D1_bps': '{:.2f}',
    'spread_t': '{:.2f}', 'used_groups': '{:,.0f}', 'used_rows': '{:,.0f}',
    'n_ic_dates': '{:,.0f}', 'n_spread_dates': '{:,.0f}',
}))

fig, ax = plt.subplots(figsize=(10, 4.5))
_plot = within_bucket_summary.pivot(index='bucket', columns='test', values='daily_rank_IC').reindex(_BUCKET_SPECS)
_plot.plot(kind='bar', ax=ax, color=['tab:red', 'tab:blue'])
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Daily within-bucket rank IC')
ax.set_title('Robustness: residual signal ranked within comparable buckets')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()


[2026-09-15 19:51:39] START within-bucket robustness scoring
[2026-09-15 19:52:49] END within-bucket robustness scoring | elapsed 1m 09.80s
Within-bucket robustness: same combined residual signal, re-ranked inside comparable groups


<HTML output>

,test,bucket,signal,target,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,used_groups,used_rows,n_ic_dates,n_spread_dates
0,B fair-value,rating_coarse,-eps,trade_from_mark_ret,0.0199,3.72,5.37,3.38,452,"1,968,710",112,112
1,A trade P&L,rating_coarse,eps,trade_to_trade_ret,0.0115,2.32,4.83,3.35,444,"1,509,719",111,111
2,B fair-value,tenor_coarse,-eps,trade_from_mark_ret,0.0176,3.10,6.01,5.01,448,"1,972,662",112,112
3,A trade P&L,tenor_coarse,eps,trade_to_trade_ret,0.0174,3.31,4.52,3.83,444,"1,512,649",111,111
4,B fair-value,rating_tenor,-eps,trade_from_mark_ret,0.0136,2.84,4.67,3.59,"1,783","1,968,134",112,112
5,A trade P&L,rating_tenor,eps,trade_to_trade_ret,0.0168,3.41,4.27,3.30,"1,716","1,507,535",111,111
6,B fair-value,rating_tenor_call,-eps,trade_from_mark_ret,0.0117,2.34,4.00,4.63,"2,947","1,963,330",112,112
7,A trade P&L,rating_tenor_call,eps,trade_to_trade_ret,0.0183,3.70,3.36,3.15,"2,741","1,499,351",111,111


<Figure: muni_characteristic_factor_export_109_2.png (image not included in markdown export)>

In [ ]:
# ── 14.3 CHARACTERISTIC-NEUTRALIZED RESIDUAL RANKING ────────────────────
def _neutralized_stats(frame, signal_col, target_col, neutral_col, min_group_n=50):
    d = frame[['date', neutral_col, signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    _counts = d.groupby(['date', neutral_col])[signal_col].transform('size')
    d = d[_counts >= min_group_n].copy()
    d['signal_neutral'] = d[signal_col] - d.groupby(['date', neutral_col])[signal_col].transform('mean')
    daily_ric = _daily_rank_ic(d, 'signal_neutral', target_col, min_n=100)
    prof, spread = _decile_spread(d, 'signal_neutral', target_col, min_n=20)
    return {
        'neutralized_by': neutral_col,
        'used_rows': len(d),
        'daily_rank_IC': np.nanmean(daily_ric),
        'rank_IC_t': _ann_tstat(daily_ric),
        'D10_minus_D1_bps': 1e4 * np.nanmean(spread),
        'spread_t': _ann_tstat(spread),
        'n_ic_dates': int(np.isfinite(daily_ric).sum()),
        'n_spread_dates': int(np.isfinite(spread).sum()),
    }

with timed_block("characteristic-neutralized robustness"):
    _neutral_col = 'rating_tenor_call'
    _nb = _neutralized_stats(test_b_rb, 'fv_signal', 'trade_from_mark_ret', _neutral_col)
    _nb.update({'test': 'B fair-value', 'signal': '-eps demeaned'})
    _na = _neutralized_stats(test_a_rb, 'alpha_signal', 'trade_to_trade_ret', _neutral_col)
    _na.update({'test': 'A trade P&L', 'signal': 'eps demeaned'})
    neutralized_summary = pd.DataFrame([_nb, _na])[
        ['test', 'neutralized_by', 'signal', 'daily_rank_IC', 'rank_IC_t',
         'D10_minus_D1_bps', 'spread_t', 'used_rows', 'n_ic_dates', 'n_spread_dates']]

print("Characteristic-neutralized robustness: signal demeaned by date × rating×tenor×call")
display(neutralized_summary.style.format({
    'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}', 'D10_minus_D1_bps': '{:.2f}',
    'spread_t': '{:.2f}', 'used_rows': '{:,.0f}', 'n_ic_dates': '{:,.0f}', 'n_spread_dates': '{:,.0f}',
}))


[2026-09-15 19:53:30] START characteristic-neutralized robustness
[2026-09-15 19:53:34] END characteristic-neutralized robustness | elapsed 4.44s
Characteristic-neutralized robustness: signal demeaned by date × rating×tenor×call


<HTML output>

,test,neutralized_by,signal,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,used_rows,n_ic_dates,n_spread_dates
0,B fair-value,rating_tenor_call,-eps demeaned,0.0262,5.43,5.55,3.61,"1,963,330",112,112
1,A trade P&L,rating_tenor_call,eps demeaned,0.0095,2.37,5.62,3.75,"1,499,351",111,111


In [ ]:
# ── 14.4 ROBUSTNESS VERDICT ─────────────────────────────────────────────
_b_fine = within_bucket_summary[(within_bucket_summary['test'] == 'B fair-value') &
                                (within_bucket_summary['bucket'] == 'rating_tenor_call')].iloc[0]
_a_fine = within_bucket_summary[(within_bucket_summary['test'] == 'A trade P&L') &
                                (within_bucket_summary['bucket'] == 'rating_tenor_call')].iloc[0]
_b_neut = neutralized_summary[neutralized_summary['test'] == 'B fair-value'].iloc[0]
_a_neut = neutralized_summary[neutralized_summary['test'] == 'A trade P&L'].iloc[0]

print("=" * 76)
print("SECTION 14 VERDICT — residual-ranking robustness")
print("=" * 76)
print("Primary ranking remains the overall combined IPCA residual signal (eps / global AR(1));")
print("no alpha test ranks by individual IPCA factors.")
print("-" * 76)
print(f"Within rating×tenor×call — Test B fair-value: rank IC={_b_fine['daily_rank_IC']:.4f}, "
      f"D10-D1={_b_fine['D10_minus_D1_bps']:.2f} bps")
print(f"Within rating×tenor×call — Test A trade P&L : rank IC={_a_fine['daily_rank_IC']:.4f}, "
      f"D10-D1={_a_fine['D10_minus_D1_bps']:.2f} bps")
print(f"Neutralized by rating×tenor×call — Test B   : rank IC={_b_neut['daily_rank_IC']:.4f}, "
      f"D10-D1={_b_neut['D10_minus_D1_bps']:.2f} bps")
print(f"Neutralized by rating×tenor×call — Test A   : rank IC={_a_neut['daily_rank_IC']:.4f}, "
      f"D10-D1={_a_neut['D10_minus_D1_bps']:.2f} bps")
print("-" * 76)
if _b_neut['daily_rank_IC'] > 0 and _b_neut['D10_minus_D1_bps'] > 0:
    print("ROBUSTNESS READ: the fair-value / de-smoothing result is not just broad bucket separation;")
    print("it survives close-comparable ranking and characteristic-neutralization.")
else:
    print("ROBUSTNESS READ: the fair-value result weakens after characteristic controls; treat as segment-driven.")
if _a_neut['D10_minus_D1_bps'] > 0:
    print("Trade-price P&L remains statistically visible but still economically small versus muni costs.")
else:
    print("Trade-price P&L does not survive the neutralized robustness check.")
print("=" * 76)
log_runtime(f"END Section 14 (elapsed {time.perf_counter() - _section14_start:.1f}s)")


SECTION 14 VERDICT — residual-ranking robustness
Primary ranking remains the overall combined IPCA residual signal (eps / global AR(1));
no alpha test ranks by individual IPCA factors.
----------------------------------------------------------------------------
Within rating×tenor×call — Test B fair-value: rank IC=0.0117, D10-D1=4.00 bps
Within rating×tenor×call — Test A trade P&L : rank IC=0.0183, D10-D1=3.36 bps
Neutralized by rating×tenor×call — Test B   : rank IC=0.0262, D10-D1=5.55 bps
Neutralized by rating×tenor×call — Test A   : rank IC=0.0095, D10-D1=5.62 bps
----------------------------------------------------------------------------
ROBUSTNESS READ: the fair-value / de-smoothing result is not just broad bucket separation;
it survives close-comparable ranking and characteristic-neutralization.
Trade-price P&L remains statistically visible but still economically small versus muni costs.
[2026-09-15 19:54:07] END Section 14 (elapsed 172.5s)


## 15. Executable-Opportunity MSRB Validation

Section 13's Test A is a **descriptive trade-to-trade diagnostic**: it asks whether the residual signal predicts the next observed MSRB trade-price change. That is useful, but it does **not** establish that both prices were executable from the strategy's side.

This section imposes the stricter tradeability condition: score only when a genuine side-specific opportunity exists.

For a **buy** opportunity, the strategy pays the current customer offer / sell-to-customer print and exits at a future executable bid:

$$y^{buy}_{i,t}=\frac{P^{future\ executable\ bid}_{i}}{P^{current\ offer}_{i}}-1.$$

For a **sell** opportunity, the strategy sells at the current customer bid / buy-from-customer print and covers at a future executable offer:

$$y^{sell}_{i,t}=\frac{P^{current\ bid}_{i}}{P^{future\ executable\ offer}_{i}}-1.$$

At minimum, results are separated by:

- customer buy / customer sell / interdealer side labels,
- trade size bucket,
- first vs subsequent print for the CUSIP-day-side,
- signal-to-trade lag.

This is now the binding alpha/tradeability test. The overall combined residual ranking remains the primary signal; within-bucket and neutralized rankings remain robustness checks; no alpha test ranks by individual IPCA factors.

In [ ]:
# ── 15.1 SIDE-AWARE EXECUTABLE OPPORTUNITY PANEL ────────────────────────
_section15_start = time.perf_counter()
log_runtime("START Section 15: executable-opportunity MSRB validation")

_required_15 = ['msrb', 'rv_cusips', 'rv_dates', 'Rp', 'df']
_missing_15 = [name for name in _required_15 if name not in globals()]
if _missing_15:
    raise RuntimeError(f"Run Sections 9-12 first; missing required variables: {_missing_15}")

_price_col15 = next((c for c in ['price', 'dollar_price', 'trade_price', 'tradeprice', 'clean_price', 'par_price']
                     if c in msrb.columns), None)
if _price_col15 is None:
    raise RuntimeError("MSRB cache has no trade-price column; cannot build executable opportunity targets.")


def _msrb_side_label(x):
    # MSRB P/S are dealer-perspective: P = dealer purchase from customer (bid), S = dealer sale to customer (offer).
    s = str(x).strip().upper()
    if s in {'S', 'SELL', 'SALE', 'CUSTOMER_BUY', 'CUSTOMER BUY', 'CUST_BUY', 'CUST BUY'}:
        return 'customer_buy'
    if s in {'P', 'PURCHASE', 'BUY', 'CUSTOMER_SELL', 'CUSTOMER SELL', 'CUST_SELL', 'CUST SELL'}:
        return 'customer_sell'
    if s in {'D', 'I', 'ID', 'INTERDEALER', 'INTER-DEALER', 'INTER_DEALER', 'DEALER'}:
        return 'interdealer'
    return 'unknown'


def _trade_size_bucket(q):
    if pd.isna(q):
        return 'unknown'
    q = float(q)
    if q < 25_000: return '<25k'
    if q < 100_000: return '25-100k'
    if q < 250_000: return '100-250k'
    if q < 1_000_000: return '250k-1mm'
    return '1mm+'

with timed_block("prepare side-aware MSRB print panel"):
    _rv_cusip_set = set(map(str, rv_cusips))
    _cols15 = ['date', 'cusip', 'quantity', 'tradetype', _price_col15]
    if 'tradetime' in msrb.columns:
        _cols15.append('tradetime')
    msrb_prints = msrb[_cols15].copy().rename(columns={_price_col15: 'trade_price'})
    msrb_prints['date'] = pd.to_datetime(msrb_prints['date'], errors='coerce').dt.normalize()
    msrb_prints['cusip'] = msrb_prints['cusip'].astype(str)
    msrb_prints['quantity'] = pd.to_numeric(msrb_prints['quantity'], errors='coerce')
    msrb_prints['trade_price'] = pd.to_numeric(msrb_prints['trade_price'], errors='coerce')
    msrb_prints['side'] = msrb_prints['tradetype'].map(_msrb_side_label)
    msrb_prints = msrb_prints.dropna(subset=['date', 'cusip', 'trade_price'])
    msrb_prints = msrb_prints[(msrb_prints['trade_price'] > 0) & msrb_prints['cusip'].isin(_rv_cusip_set)]
    msrb_prints['size_bucket'] = msrb_prints['quantity'].map(_trade_size_bucket)
    if 'tradetime' in msrb_prints.columns:
        msrb_prints['tradetime_sort'] = pd.to_datetime(msrb_prints['tradetime'], errors='coerce')
        msrb_prints = msrb_prints.sort_values(['cusip', 'date', 'side', 'tradetime_sort', 'trade_price'])
    else:
        msrb_prints = msrb_prints.sort_values(['cusip', 'date', 'side', 'trade_price'])
    msrb_prints['side_print_seq'] = msrb_prints.groupby(['cusip', 'date', 'side']).cumcount() + 1
    msrb_prints['print_order'] = np.where(msrb_prints['side_print_seq'] == 1, 'first', 'subsequent')
    msrb_prints['_qty_w'] = msrb_prints['quantity'].fillna(0.0).clip(lower=0.0)
    msrb_prints['_px_qty'] = msrb_prints['trade_price'] * msrb_prints['_qty_w']

print("MSRB tradetype -> side mapping (raw counts):")
display(msrb_prints.groupby(['tradetype', 'side']).size().rename('prints').reset_index().sort_values('prints', ascending=False))
print("Side / size / first-subsequent counts:")
display(msrb_prints.groupby(['side', 'size_bucket', 'print_order']).size().rename('prints').reset_index().head(30))

with timed_block("aggregate daily executable exits by side"):
    _side_daily_raw = (msrb_prints.groupby(['cusip', 'date', 'side'], sort=False)
                       .agg(price_mean=('trade_price', 'mean'),
                            px_qty=('_px_qty', 'sum'),
                            par_traded=('_qty_w', 'sum'),
                            n_prints=('trade_price', 'size'))
                       .reset_index())
    _side_daily_raw['exec_price'] = np.where(_side_daily_raw['par_traded'] > 0,
                                             _side_daily_raw['px_qty'] / _side_daily_raw['par_traded'],
                                             _side_daily_raw['price_mean'])
    side_daily = (_side_daily_raw[['cusip', 'date', 'side', 'exec_price', 'n_prints', 'par_traded']]
                  .sort_values(['cusip', 'side', 'date'])
                  .reset_index(drop=True))

with timed_block("attach residual signal to executable entry prints"):
    _rv_dates15 = pd.DatetimeIndex(pd.to_datetime(rv_dates, errors='coerce')).normalize()
    _rv_ns15 = _rv_dates15.values.astype('datetime64[ns]')
    _trade_ns15 = msrb_prints['date'].to_numpy(dtype='datetime64[ns]')
    _t_idx_all15 = np.searchsorted(_rv_ns15, _trade_ns15, side='left') - 1
    _row_map15 = pd.Series(np.arange(len(rv_cusips), dtype=np.int64), index=np.asarray(rv_cusips, dtype=str))
    _row_idx_all15 = msrb_prints['cusip'].map(_row_map15).to_numpy()
    _valid15 = (_t_idx_all15 >= 0) & pd.notna(_row_idx_all15)
    _row_idx15 = _row_idx_all15[_valid15].astype(np.int64)
    _t_idx15 = _t_idx_all15[_valid15].astype(np.int64)

    executable_entries = msrb_prints.loc[_valid15].copy().reset_index(drop=True)
    executable_entries['entry_price'] = executable_entries['trade_price']
    executable_entries['signal_date'] = _rv_dates15[_t_idx15].to_numpy()
    executable_entries['eps_signal'] = Rp[_row_idx15, _t_idx15]
    executable_entries['signal_to_trade_lag'] = (executable_entries['date'] - executable_entries['signal_date']).dt.days
    executable_entries = executable_entries.dropna(subset=['eps_signal'])
    executable_entries = executable_entries[(executable_entries['signal_to_trade_lag'] >= 1) &
                                            (executable_entries['signal_to_trade_lag'] <= 20)].reset_index(drop=True)


def _future_side_lookup(entries, future_side):
    fut = side_daily[side_daily['side'] == future_side][['cusip', 'date', 'exec_price']].copy()
    fut_map = {c: (g['date'].to_numpy(dtype='datetime64[ns]'), g['exec_price'].to_numpy(dtype=float))
               for c, g in fut.groupby('cusip', sort=False)}
    future_dates = np.full(len(entries), np.datetime64('NaT'), dtype='datetime64[ns]')
    future_prices = np.full(len(entries), np.nan, dtype=float)
    entry_dates = entries['date'].to_numpy(dtype='datetime64[ns]')
    for c, loc in entries.groupby('cusip', sort=False).indices.items():
        if c not in fut_map:
            continue
        fd, fp = fut_map[c]
        loc = np.asarray(loc, dtype=np.int64)
        pos = np.searchsorted(fd, entry_dates[loc], side='right')
        ok = pos < len(fd)
        if ok.any():
            hit = loc[ok]
            future_dates[hit] = fd[pos[ok]]
            future_prices[hit] = fp[pos[ok]]
    return pd.to_datetime(future_dates), future_prices

with timed_block("build side-consistent future executable targets"):
    buy_entries = executable_entries[executable_entries['side'] == 'customer_buy'].copy().reset_index(drop=True)
    sell_entries = executable_entries[executable_entries['side'] == 'customer_sell'].copy().reset_index(drop=True)
    interdealer_entries = executable_entries[executable_entries['side'] == 'interdealer'].copy().reset_index(drop=True)

    buy_opps = buy_entries.copy()
    buy_opps['future_exec_date'], buy_opps['future_exec_price'] = _future_side_lookup(buy_opps, 'customer_sell')
    buy_opps['opportunity_side'] = 'buy'
    buy_opps['executable_ret'] = buy_opps['future_exec_price'] / buy_opps['entry_price'] - 1.0
    buy_opps['alpha_signal'] = buy_opps['eps_signal']

    sell_opps = sell_entries.copy()
    sell_opps['future_exec_date'], sell_opps['future_exec_price'] = _future_side_lookup(sell_opps, 'customer_buy')
    sell_opps['opportunity_side'] = 'sell'
    sell_opps['executable_ret'] = sell_opps['entry_price'] / sell_opps['future_exec_price'] - 1.0
    sell_opps['alpha_signal'] = -sell_opps['eps_signal']

    executable_opps = pd.concat([buy_opps, sell_opps], ignore_index=True)
    executable_opps['exec_hold_days'] = (executable_opps['future_exec_date'] - executable_opps['date']).dt.days
    executable_opps = executable_opps.dropna(subset=['future_exec_price', 'executable_ret', 'alpha_signal'])
    executable_opps = executable_opps[(executable_opps['exec_hold_days'] >= 1) & (executable_opps['exec_hold_days'] <= 30)]
    executable_opps = executable_opps[np.isfinite(executable_opps['executable_ret'])].reset_index(drop=True)

print(f"side-aware entry prints          : {len(executable_entries):,}")
print(f"buy executable opportunities     : {int((executable_opps['opportunity_side'] == 'buy').sum()):,}")
print(f"sell executable opportunities    : {int((executable_opps['opportunity_side'] == 'sell').sum()):,}")
print(f"interdealer entry rows tracked   : {len(interdealer_entries):,} (reported separately; not used for buy/sell P&L)")
if not executable_opps.empty:
    print(f"signal-to-trade lag min/med/max  : {executable_opps['signal_to_trade_lag'].min():.0f} / "
          f"{executable_opps['signal_to_trade_lag'].median():.0f} / {executable_opps['signal_to_trade_lag'].max():.0f}")
    print(f"exec hold days min/med/max       : {executable_opps['exec_hold_days'].min():.0f} / "
          f"{executable_opps['exec_hold_days'].median():.0f} / {executable_opps['exec_hold_days'].max():.0f}")
    display(executable_opps[['cusip', 'date', 'side', 'opportunity_side', 'entry_price', 'future_exec_date',
                             'future_exec_price', 'executable_ret', 'eps_signal', 'alpha_signal',
                             'signal_to_trade_lag', 'exec_hold_days', 'size_bucket', 'print_order',
                             'quantity']].head(5))

del _side_daily_raw, _row_map15, _row_idx_all15, _t_idx_all15, _valid15, _row_idx15, _t_idx15


[2026-09-15 20:13:12] START Section 15: executable-opportunity MSRB validation
[2026-09-15 20:13:12] START prepare side-aware MSRB print panel
[2026-09-15 20:13:43] END prepare side-aware MSRB print panel | elapsed 30.87s
MSRB tradetype -> side mapping (raw counts):


,tradetype,side,prints
0,D,interdealer,1512136
2,S,customer_buy,1465464
1,P,customer_sell,1387756


Side / size / first-subsequent counts:


,side,size_bucket,print_order,prints
0,customer_buy,100-250k,first,130465
1,customer_buy,100-250k,subsequent,33451
2,customer_buy,1mm+,first,16786
3,customer_buy,1mm+,subsequent,10977
4,customer_buy,25-100k,first,470325
5,customer_buy,25-100k,subsequent,87064
6,customer_buy,250k-1mm,first,46732
7,customer_buy,250k-1mm,subsequent,17966
8,customer_buy,<25k,first,575040
9,customer_buy,<25k,subsequent,76658


[2026-09-15 20:13:44] START aggregate daily executable exits by side
[2026-09-15 20:13:46] END aggregate daily executable exits by side | elapsed 1.84s
[2026-09-15 20:13:46] START attach residual signal to executable entry prints
[2026-09-15 20:13:48] END attach residual signal to executable entry prints | elapsed 2.87s
[2026-09-15 20:13:48] START build side-consistent future executable targets
[2026-09-15 20:14:58] END build side-consistent future executable targets | elapsed 1m 09.35s
side-aware entry prints          : 4,256,243
buy executable opportunities     : 991,405
sell executable opportunities    : 1,119,292
interdealer entry rows tracked   : 1,480,282 (reported separately; not used for buy/sell P&L)
signal-to-trade lag min/med/max  : 1 / 1 / 4
exec hold days min/med/max       : 1 / 5 / 30


,cusip,date,side,opportunity_side,entry_price,future_exec_date,future_exec_price,executable_ret,eps_signal,alpha_signal,signal_to_trade_lag,exec_hold_days,size_bucket,print_order,quantity
0,0004162D8,2026-07-22,customer_buy,buy,115.637,2026-07-27,114.613,-0.009,0.000,0.000,1,5.000,25-100k,first,30000.000
1,0004162H9,2026-08-04,customer_buy,buy,110.853,2026-08-14,111.753,0.008,0.000,0.000,1,10.000,25-100k,first,25000.000
2,0004162H9,2026-08-14,customer_buy,buy,111.818,2026-09-01,109.228,-0.023,0.000,0.000,1,18.000,25-100k,first,25000.000
3,0004162H9,2026-08-19,customer_buy,buy,112.019,2026-09-01,109.228,-0.025,-0.000,-0.000,1,13.000,<25k,first,20000.000
4,0004162L0,2026-04-20,customer_buy,buy,112.568,2026-04-27,112.086,-0.004,-0.000,-0.000,3,7.000,<25k,first,10000.000


In [ ]:
# ── 15.2 EXECUTABLE-OPPORTUNITY SCORING BY SIDE / SIZE / PRINT / LAG ─────
def _exec_score(frame, label, min_n=500, n_bins=10):
    d = frame[['date', 'alpha_signal', 'executable_ret']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < min_n:
        return {'slice': label, 'n_rows': len(d), 'n_dates': d['date'].nunique(),
                'daily_rank_IC': np.nan, 'rank_IC_t': np.nan,
                'D10_minus_D1_bps': np.nan, 'spread_t': np.nan,
                'mean_ret_bps': np.nan}
    daily_ric = _daily_rank_ic(d, 'alpha_signal', 'executable_ret', min_n=max(50, min_n // 20))
    _, spread = _decile_spread(d, 'alpha_signal', 'executable_ret', min_n=max(10, min_n // 100), n_bins=n_bins)
    return {'slice': label, 'n_rows': len(d), 'n_dates': d['date'].nunique(),
            'daily_rank_IC': np.nanmean(daily_ric), 'rank_IC_t': _ann_tstat(daily_ric),
            'D10_minus_D1_bps': 1e4 * np.nanmean(spread), 'spread_t': _ann_tstat(spread),
            'mean_ret_bps': 1e4 * d['executable_ret'].mean()}

with timed_block("score executable opportunities"):
    _score_rows = [_exec_score(executable_opps, 'ALL executable opportunities')]
    for _side in ['buy', 'sell']:
        _score_rows.append(_exec_score(executable_opps[executable_opps['opportunity_side'] == _side], f'side={_side}'))
    for _bucket in ['<25k', '25-100k', '100-250k', '250k-1mm', '1mm+']:
        _score_rows.append(_exec_score(executable_opps[executable_opps['size_bucket'] == _bucket], f'size={_bucket}'))
    for _po in ['first', 'subsequent']:
        _score_rows.append(_exec_score(executable_opps[executable_opps['print_order'] == _po], f'print={_po}'))
    _lag_bins = pd.cut(executable_opps['signal_to_trade_lag'], bins=[0, 1, 2, 5, 20], labels=['lag=1d', 'lag=2d', 'lag=3-5d', 'lag=6-20d'])
    for _lag in ['lag=1d', 'lag=2d', 'lag=3-5d', 'lag=6-20d']:
        _score_rows.append(_exec_score(executable_opps[_lag_bins.astype(str) == _lag], _lag))

executable_score_summary = pd.DataFrame(_score_rows)
print("Executable-opportunity alpha scoring: side-consistent entry/exit only")
display(executable_score_summary.style.format({
    'n_rows': '{:,.0f}', 'n_dates': '{:,.0f}', 'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}',
    'D10_minus_D1_bps': '{:.2f}', 'spread_t': '{:.2f}', 'mean_ret_bps': '{:.2f}',
}))

fig, ax = plt.subplots(figsize=(11, 4.5))
_plot_exec = executable_score_summary.set_index('slice').loc[
    ['ALL executable opportunities', 'side=buy', 'side=sell', 'print=first', 'print=subsequent'],
    'D10_minus_D1_bps'
]
_plot_exec.plot(kind='bar', ax=ax, color='tab:orange')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(20, color='red', linestyle='--', alpha=0.6, label='20 bps reference')
ax.set_ylabel('D10-D1 executable return (bps)')
ax.set_title('Executable opportunity spread by side / print order')
ax.grid(True, alpha=0.3, axis='y'); ax.legend()
plt.xticks(rotation=25, ha='right')
plt.tight_layout(); plt.show()


[2026-09-15 20:15:28] START score executable opportunities
[2026-09-15 20:15:42] END score executable opportunities | elapsed 14.67s
Executable-opportunity alpha scoring: side-consistent entry/exit only


<HTML output>

,slice,n_rows,n_dates,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,mean_ret_bps
0,ALL executable opportunities,"2,110,697",111,0.0323,5.01,9.98,1.81,-45.87
1,side=buy,"991,405",111,-0.0028,-0.36,14.47,1.28,-42.54
2,side=sell,"1,119,292",111,0.0258,4.34,6.68,1.52,-48.83
3,size=<25k,"965,350",111,0.0209,3.00,3.60,0.60,-44.65
4,size=25-100k,"790,955",111,0.0393,5.90,12.53,1.32,-46.47
5,size=100-250k,"232,759",111,0.0473,6.69,4.94,0.45,-38.92
6,size=250k-1mm,"89,806",111,0.0486,5.52,-0.11,-0.01,-33.68
7,size=1mm+,"31,827",111,0.0506,4.77,141.97,1.37,-153.55
8,print=first,"1,824,294",111,0.0339,5.16,8.07,2.03,-44.09
9,print=subsequent,"286,403",111,0.0229,3.27,23.75,0.82,-57.23


<Figure: muni_characteristic_factor_export_114_2.png (image not included in markdown export)>

In [ ]:
# ── 15.3 EXECUTABLE-OPPORTUNITY VERDICT ─────────────────────────────────
_exec_all = executable_score_summary[executable_score_summary['slice'] == 'ALL executable opportunities'].iloc[0]
_exec_buy = executable_score_summary[executable_score_summary['slice'] == 'side=buy'].iloc[0]
_exec_sell = executable_score_summary[executable_score_summary['slice'] == 'side=sell'].iloc[0]
_exec_first = executable_score_summary[executable_score_summary['slice'] == 'print=first'].iloc[0]
_exec_subseq = executable_score_summary[executable_score_summary['slice'] == 'print=subsequent'].iloc[0]

_EXEC_COST_FLOOR_BPS = 20.0
_EXEC_T_MIN = 2.0
exec_stat_pass = np.isfinite(_exec_all['daily_rank_IC']) and _exec_all['daily_rank_IC'] > 0 and _exec_all['rank_IC_t'] >= _EXEC_T_MIN
exec_econ_pass = (_exec_all['D10_minus_D1_bps'] >= _EXEC_COST_FLOOR_BPS and
                  _exec_all['spread_t'] >= _EXEC_T_MIN)

print("=" * 78)
print("SECTION 15 VERDICT — executable-opportunity tradeability")
print("=" * 78)
print("Section 13 Test A is descriptive trade-to-trade; Section 15 is the binding")
print("side-consistent executable-opportunity test.")
print("-" * 78)
print(f"All executable opportunities : rank IC={_exec_all['daily_rank_IC']:.4f} (t={_exec_all['rank_IC_t']:.2f}), "
      f"D10-D1={_exec_all['D10_minus_D1_bps']:.2f} bps (spread t={_exec_all['spread_t']:.2f})")
print(f"Buy opportunities            : rank IC={_exec_buy['daily_rank_IC']:.4f}, "
      f"D10-D1={_exec_buy['D10_minus_D1_bps']:.2f} bps")
print(f"Sell opportunities           : rank IC={_exec_sell['daily_rank_IC']:.4f}, "
      f"D10-D1={_exec_sell['D10_minus_D1_bps']:.2f} bps")
print(f"First prints                 : D10-D1={_exec_first['D10_minus_D1_bps']:.2f} bps; "
      f"subsequent prints={_exec_subseq['D10_minus_D1_bps']:.2f} bps")
print("-" * 78)
print(f"Statistical gate: {'PASS' if exec_stat_pass else 'FAIL'}")
print(f"Economic gate   : {'PASS' if exec_econ_pass else 'FAIL'} "
      f"(needs >= {_EXEC_COST_FLOOR_BPS:.0f} bps and spread t >= {_EXEC_T_MIN:.1f})")
print("-" * 78)
if exec_stat_pass and exec_econ_pass:
    print("RECOMMENDATION: proceed to constrained executable portfolio construction, but keep side/size filters.")
elif exec_stat_pass:
    print("RECOMMENDATION: do not call this tradeable alpha yet.")
    print("  The signal ranks executable opportunities, but the all-in side-consistent spread is below")
    print("  a conservative muni cost floor and the spread t-stat is weak. Keep it as fair-value /")
    print("  de-smoothing first; investigate narrower execution filters only as secondary research.")
else:
    print("RECOMMENDATION: stop trade-strategy build; opportunity-gated alpha does not survive.")
print("=" * 78)
log_runtime(f"END Section 15 (elapsed {time.perf_counter() - _section15_start:.1f}s)")


SECTION 15 VERDICT — executable-opportunity tradeability
Section 13 Test A is descriptive trade-to-trade; Section 15 is the binding
side-consistent executable-opportunity test.
------------------------------------------------------------------------------
All executable opportunities : rank IC=0.0323 (t=5.01), D10-D1=9.98 bps (spread t=1.81)
Buy opportunities            : rank IC=-0.0028, D10-D1=14.47 bps
Sell opportunities           : rank IC=0.0258, D10-D1=6.68 bps
First prints                 : D10-D1=8.07 bps; subsequent prints=23.75 bps
------------------------------------------------------------------------------
Statistical gate: PASS
Economic gate   : FAIL (needs >= 20 bps and spread t >= 2.0)
------------------------------------------------------------------------------
RECOMMENDATION: do not call this tradeable alpha yet.
  The signal ranks executable opportunities, but the all-in side-consistent spread is below
  a conservative muni cost floor and the spread t-stat is weak. 

## 16. Fair-Value Bps Calibration and Calibrated Ranking

The raw residual ranking remains useful, but an RFQ workflow also needs a **calibrated fair-value move in price bps**:

$$\widehat{\Delta P}_{i,t}=E\left[P^{next\ trade}_{i}-P^{base}_{i,t}\mid\mathcal F_t\right].$$

Here the target is the side-consistent executable return from Section 15, in bps. We estimate separate monotonic mappings for:

- opportunity side: buy vs sell,
- trade size bucket,
- first vs subsequent print.

The calibrated score is not an individual IPCA factor rank. It is a monotonic transformation of the **combined residual signal** into expected executable-return bps, learned separately inside the execution context. This lets us compare:

1. **Raw rank** — rank by the combined residual opportunity signal.
2. **Calibrated rank** — rank by $\widehat{\Delta P}$, which accounts for side, size, and print-order response curves.

Interpretation discipline: a larger calibrated D10-D1 spread does not by itself prove positive expected P&L. The reliability table must show whether the top predicted bucket clears zero. If the top bucket is still negative, the score is an **avoid-worst-trades / quote-bias** signal, not standalone trade alpha.

To keep the timestamp logic honest, the calibration below is expanding by trade date: each date's mapping uses only earlier executable opportunities, with a global historical fallback when a slice is sparse.

In [ ]:
# ── 16.1 EXPANDING MONOTONIC FAIR-VALUE BPS CALIBRATION ────────────────
from sklearn.isotonic import IsotonicRegression

_section16_start = time.perf_counter()
log_runtime("START Section 16: fair-value bps calibration")

_required_16 = ['executable_opps']
_missing_16 = [name for name in _required_16 if name not in globals()]
if _missing_16:
    raise RuntimeError(f"Run Section 15 first; missing required variables: {_missing_16}")

CAL_MIN_SLICE_OBS = 2_000
CAL_MIN_GLOBAL_OBS = 20_000
CAL_N_BINS = 10


def _fit_monotone_curve(hist, signal_col='alpha_signal', target_col='executable_ret'):
    h = hist[[signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(h) < CAL_MIN_SLICE_OBS or h[signal_col].nunique() < CAL_N_BINS:
        return None
    q = pd.qcut(h[signal_col].rank(method='first'), CAL_N_BINS, labels=False, duplicates='drop')
    curve = (h.assign(bin=q)
             .groupby('bin', observed=False)
             .agg(x=(signal_col, 'mean'), y=(target_col, 'mean'), n=(target_col, 'size'))
             .dropna())
    if len(curve) < 4:
        return None
    iso = IsotonicRegression(increasing=True, out_of_bounds='clip')
    iso.fit(curve['x'].to_numpy(dtype=float), curve['y'].to_numpy(dtype=float), sample_weight=curve['n'].to_numpy(dtype=float))
    return iso


def _predict_with_curve(curve, signal):
    if curve is None:
        return np.full(len(signal), np.nan)
    return curve.predict(np.asarray(signal, dtype=float))

with timed_block("expanding monotonic calibration by side/size/print"):
    cal_base = executable_opps.copy()
    cal_base['cal_key'] = (cal_base['opportunity_side'].astype(str) + '|'
                           + cal_base['size_bucket'].astype(str) + '|'
                           + cal_base['print_order'].astype(str))
    cal_base['target_bps'] = 1e4 * cal_base['executable_ret']
    cal_base = cal_base.sort_values(['date', 'cal_key']).reset_index(drop=True)
    cal_base['delta_p_hat_bps'] = np.nan
    cal_base['calibration_scope'] = 'none'

    _dates = pd.DatetimeIndex(cal_base['date'].dropna().sort_values().unique())
    _keys = sorted(cal_base['cal_key'].dropna().unique())
    for _d in _dates:
        _hist_all = cal_base[cal_base['date'] < _d]
        _today_mask = cal_base['date'] == _d
        if len(_hist_all) < CAL_MIN_GLOBAL_OBS:
            continue
        _global_curve = _fit_monotone_curve(_hist_all)
        for _key in _keys:
            _mask = _today_mask & (cal_base['cal_key'] == _key)
            if not _mask.any():
                continue
            _hist_key = _hist_all[_hist_all['cal_key'] == _key]
            _curve = _fit_monotone_curve(_hist_key) if len(_hist_key) >= CAL_MIN_SLICE_OBS else None
            if _curve is not None:
                cal_base.loc[_mask, 'delta_p_hat_bps'] = 1e4 * _predict_with_curve(_curve, cal_base.loc[_mask, 'alpha_signal'])
                cal_base.loc[_mask, 'calibration_scope'] = 'side_size_print'
            elif _global_curve is not None:
                cal_base.loc[_mask, 'delta_p_hat_bps'] = 1e4 * _predict_with_curve(_global_curve, cal_base.loc[_mask, 'alpha_signal'])
                cal_base.loc[_mask, 'calibration_scope'] = 'global_fallback'

calibrated_opps = cal_base.dropna(subset=['delta_p_hat_bps']).reset_index(drop=True)
calibration_coverage = calibrated_opps['calibration_scope'].value_counts().rename('rows').to_frame()
print(f"calibrated opportunities: {len(calibrated_opps):,} / {len(cal_base):,} "
      f"({100 * len(calibrated_opps) / max(len(cal_base), 1):.1f}%)")
print("Calibration scope coverage:")
display(calibration_coverage)
print("Calibration keys with most rows:")
display(calibrated_opps['cal_key'].value_counts().head(10).rename('rows').to_frame())


[2026-09-15 20:46:46] START Section 16: fair-value bps calibration
[2026-09-15 20:46:46] START expanding monotonic calibration by side/size/print
[2026-09-15 20:51:13] END expanding monotonic calibration by side/size/print | elapsed 4m 26.65s
calibrated opportunities: 2,088,257 / 2,110,697 (98.9%)
Calibration scope coverage:


,rows
calibration_scope,
side_size_print,2059473
global_fallback,28784


Calibration keys with most rows:


,rows
cal_key,
sell|<25k|first,469110
buy|<25k|first,371366
sell|25-100k|first,359820
buy|25-100k|first,313875
sell|100-250k|first,100092
buy|100-250k|first,92779
buy|25-100k|subsequent,63317
buy|<25k|subsequent,57267
sell|<25k|subsequent,57007


In [ ]:
# ── 16.2 RAW VS CALIBRATED EXPECTED-BPS RANKING ─────────────────────────
def _score_rank_model(frame, signal_col, target_col='executable_ret', label='model'):
    d = frame[['date', signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    daily_ric = _daily_rank_ic(d, signal_col, target_col, min_n=100)
    prof, spread = _decile_spread(d, signal_col, target_col, min_n=20)
    return {'model': label, 'n_rows': len(d), 'n_dates': d['date'].nunique(),
            'daily_rank_IC': np.nanmean(daily_ric), 'rank_IC_t': _ann_tstat(daily_ric),
            'D10_minus_D1_bps': 1e4 * np.nanmean(spread), 'spread_t': _ann_tstat(spread),
            'mean_target_bps': 1e4 * d[target_col].mean()}

with timed_block("score raw vs calibrated ranking"):
    calibrated_opps['delta_p_hat_ret'] = calibrated_opps['delta_p_hat_bps'] / 1e4
    raw_vs_calibrated = pd.DataFrame([
        _score_rank_model(calibrated_opps, 'alpha_signal', label='raw residual opportunity signal'),
        _score_rank_model(calibrated_opps, 'delta_p_hat_bps', label='calibrated expected ΔP bps'),
    ])
    _cal_slice_rows = []
    for _side in ['buy', 'sell']:
        _cal_slice_rows.append(_score_rank_model(calibrated_opps[calibrated_opps['opportunity_side'] == _side],
                                                 'delta_p_hat_bps', label=f'calibrated side={_side}'))
    for _po in ['first', 'subsequent']:
        _cal_slice_rows.append(_score_rank_model(calibrated_opps[calibrated_opps['print_order'] == _po],
                                                 'delta_p_hat_bps', label=f'calibrated print={_po}'))
    calibrated_slice_summary = pd.DataFrame(_cal_slice_rows)

print("Raw residual rank vs calibrated expected-price-move rank")
display(raw_vs_calibrated.style.format({
    'n_rows': '{:,.0f}', 'n_dates': '{:,.0f}', 'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}',
    'D10_minus_D1_bps': '{:.2f}', 'spread_t': '{:.2f}', 'mean_target_bps': '{:.2f}',
}))
print("\nCalibrated ranking by opportunity side and print order")
display(calibrated_slice_summary.style.format({
    'n_rows': '{:,.0f}', 'n_dates': '{:,.0f}', 'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}',
    'D10_minus_D1_bps': '{:.2f}', 'spread_t': '{:.2f}', 'mean_target_bps': '{:.2f}',
}))
print("\nRead this table conservatively: positive D10-D1 means ordering lift; positive-EV needs the top predicted bucket to clear zero in Section 16.3.")

fig, ax = plt.subplots(figsize=(9, 4.5))
raw_vs_calibrated.set_index('model')['D10_minus_D1_bps'].plot(kind='bar', ax=ax, color=['tab:gray', 'tab:green'])
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(20, color='red', linestyle='--', alpha=0.6, label='20 bps reference')
ax.set_ylabel('D10-D1 executable return (bps)')
ax.set_title('Ordering lift only: raw residual rank vs calibrated expected dP rank')
ax.grid(True, alpha=0.3, axis='y'); ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()


[2026-09-15 20:51:56] START score raw vs calibrated ranking
[2026-09-15 20:52:04] END score raw vs calibrated ranking | elapsed 8.34s
Raw residual rank vs calibrated expected-price-move rank


<HTML output>

,model,n_rows,n_dates,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,mean_target_bps
0,raw residual opportunity signal,"2,088,257",110,0.0341,5.46,10.84,1.97,-45.83
1,calibrated expected ΔP bps,"2,088,257",110,-0.0185,-0.99,43.81,6.13,-45.83


Calibrated ranking by opportunity side and print order


<HTML output>

,model,n_rows,n_dates,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,mean_target_bps
0,calibrated side=buy,"980,218",110,0.0513,8.84,10.02,0.41,-42.85
1,calibrated side=sell,"1,108,039",110,0.0887,20.15,54.93,10.63,-48.48
2,calibrated print=first,"1,804,855",110,-0.0049,-0.25,38.29,5.85,-44.05
3,calibrated print=subsequent,"283,402",110,-0.1015,-5.04,71.79,2.59,-57.17


<Figure: muni_characteristic_factor_export_118_4.png (image not included in markdown export)>

In [ ]:
# ── 16.3 CONTEXT-NEUTRAL CALIBRATED-RANKING CHECK ──────────────────────
def _within_context_model_stats(frame, signal_col, target_col='executable_ret', context_col='cal_key', min_group_n=100, n_bins=10):
    daily_ics, daily_spreads = [], []
    used_groups, used_rows = 0, 0
    for _, day in frame.groupby('date', sort=False):
        group_ics, group_weights, group_spreads = [], [], []
        for _, g in day.groupby(context_col, sort=False):
            d = g[[signal_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(d) < min_group_n or d[signal_col].std() == 0 or d[target_col].std() == 0:
                continue
            group_ics.append(d[signal_col].corr(d[target_col], method='spearman'))
            group_weights.append(len(d))
            if len(d) >= n_bins * 10:
                dec = pd.qcut(d[signal_col].rank(method='first'), n_bins, labels=False)
                prof = d.groupby(dec, observed=False)[target_col].mean()
                if 0 in prof.index and n_bins - 1 in prof.index:
                    group_spreads.append(float(prof.loc[n_bins - 1] - prof.loc[0]))
            used_groups += 1
            used_rows += len(d)
        if group_ics:
            daily_ics.append(float(np.average(group_ics, weights=np.asarray(group_weights, dtype=float))))
        if group_spreads:
            daily_spreads.append(float(np.mean(group_spreads)))
    daily_ics = np.asarray(daily_ics, dtype=float)
    daily_spreads = np.asarray(daily_spreads, dtype=float)
    return {'model': signal_col, 'context': context_col, 'used_groups': used_groups, 'used_rows': used_rows,
            'daily_rank_IC': np.nanmean(daily_ics), 'rank_IC_t': _ann_tstat(daily_ics),
            'D10_minus_D1_bps': 1e4 * np.nanmean(daily_spreads), 'spread_t': _ann_tstat(daily_spreads),
            'n_ic_dates': int(np.isfinite(daily_ics).sum()), 'n_spread_dates': int(np.isfinite(daily_spreads).sum())}

with timed_block("context-neutral calibrated ranking check"):
    context_neutral_rank_summary = pd.DataFrame([
        _within_context_model_stats(calibrated_opps, 'alpha_signal'),
        _within_context_model_stats(calibrated_opps, 'delta_p_hat_bps'),
    ])
    context_neutral_rank_summary['model'] = ['raw residual within side/size/print',
                                             'calibrated ΔP bps within side/size/print']

    _cal_d = calibrated_opps[['delta_p_hat_bps', 'target_bps']].replace([np.inf, -np.inf], np.nan).dropna()
    _cal_d['pred_decile'] = pd.qcut(_cal_d['delta_p_hat_bps'].rank(method='first'), 10, labels=False) + 1
    calibration_reliability = (_cal_d.groupby('pred_decile', observed=False)
                               .agg(pred_bps=('delta_p_hat_bps', 'mean'),
                                    realized_bps=('target_bps', 'mean'),
                                    n=('target_bps', 'size')))

print("Context-neutral ranking check: evaluate inside side×size×print calibration keys")
display(context_neutral_rank_summary.style.format({
    'used_groups': '{:,.0f}', 'used_rows': '{:,.0f}', 'daily_rank_IC': '{:.4f}',
    'rank_IC_t': '{:.2f}', 'D10_minus_D1_bps': '{:.2f}', 'spread_t': '{:.2f}',
    'n_ic_dates': '{:,.0f}', 'n_spread_dates': '{:,.0f}',
}))
print("\nCalibration reliability by predicted ΔP decile (pooled across contexts)")
display(calibration_reliability.style.format({'pred_bps': '{:.2f}', 'realized_bps': '{:.2f}', 'n': '{:,.0f}'}))


[2026-09-15 20:53:01] START context-neutral calibrated ranking check
[2026-09-15 20:53:21] END context-neutral calibrated ranking check | elapsed 19.23s
Context-neutral ranking check: evaluate inside side×size×print calibration keys


<HTML output>

,model,context,used_groups,used_rows,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,n_ic_dates,n_spread_dates
0,raw residual within side/size/print,cal_key,"1,689","2,060,671",0.0102,1.88,29.94,1.70,110,110
1,calibrated ΔP bps within side/size/print,cal_key,"1,688","2,060,544",0.0515,10.18,62.24,3.17,76,110


Calibration reliability by predicted ΔP decile (pooled across contexts)


<HTML output>

,pred_bps,realized_bps,n
pred_decile,,,
1,-96.16,-90.10,"208,826"
2,-61.17,-58.42,"208,826"
3,-51.64,-48.97,"208,825"
4,-46.70,-36.93,"208,826"
5,-43.10,-27.76,"208,826"
6,-40.34,-26.88,"208,825"
7,-37.23,-44.34,"208,826"
8,-32.59,-40.85,"208,825"
9,-26.20,-38.03,"208,826"


In [ ]:
# ── 16.4 CALIBRATED-RANKING VERDICT ─────────────────────────────────────
_raw_pooled = raw_vs_calibrated[raw_vs_calibrated['model'] == 'raw residual opportunity signal'].iloc[0]
_cal_pooled = raw_vs_calibrated[raw_vs_calibrated['model'] == 'calibrated expected ΔP bps'].iloc[0]
_raw_ctx = context_neutral_rank_summary[context_neutral_rank_summary['model'] == 'raw residual within side/size/print'].iloc[0]
_cal_ctx = context_neutral_rank_summary[context_neutral_rank_summary['model'] == 'calibrated ΔP bps within side/size/print'].iloc[0]
_cal_pooled_lift = _cal_pooled['D10_minus_D1_bps'] - _raw_pooled['D10_minus_D1_bps']
_cal_ctx_lift = _cal_ctx['D10_minus_D1_bps'] - _raw_ctx['D10_minus_D1_bps']
_top_realized_bps = float(calibration_reliability.loc[10, 'realized_bps']) if 10 in calibration_reliability.index else np.nan
_bottom_realized_bps = float(calibration_reliability.loc[1, 'realized_bps']) if 1 in calibration_reliability.index else np.nan

rank_pass = (_cal_ctx['daily_rank_IC'] > 0 and _cal_ctx['rank_IC_t'] >= 2.0 and
             _cal_ctx['D10_minus_D1_bps'] > _raw_ctx['D10_minus_D1_bps'])
level_pass = np.isfinite(_top_realized_bps) and _top_realized_bps > 0

print("=" * 78)
print("SECTION 16 VERDICT — calibrated fair-value bps ranking")
print("=" * 78)
print("Raw residual rank remains the transparent diagnostic. Calibrated ΔP bps is the")
print("execution-context-aware RFQ score, estimated by side × size × first/sub print.")
print("-" * 78)
print(f"Pooled raw residual rank      : rank IC={_raw_pooled['daily_rank_IC']:.4f}, "
      f"D10-D1={_raw_pooled['D10_minus_D1_bps']:.2f} bps")
print(f"Pooled calibrated ΔP rank     : rank IC={_cal_pooled['daily_rank_IC']:.4f}, "
      f"D10-D1={_cal_pooled['D10_minus_D1_bps']:.2f} bps "
      f"(lift={_cal_pooled_lift:+.2f} bps)")
print(f"Within side×size×print raw    : rank IC={_raw_ctx['daily_rank_IC']:.4f}, "
      f"D10-D1={_raw_ctx['D10_minus_D1_bps']:.2f} bps")
print(f"Within side×size×print calib  : rank IC={_cal_ctx['daily_rank_IC']:.4f}, "
      f"D10-D1={_cal_ctx['D10_minus_D1_bps']:.2f} bps "
      f"(lift={_cal_ctx_lift:+.2f} bps)")
print(f"Reliability D1 realized       : {_bottom_realized_bps:.2f} bps")
print(f"Reliability D10 realized      : {_top_realized_bps:.2f} bps")
print("-" * 78)
print(f"Ranking gate: {'PASS' if rank_pass else 'FAIL'}")
print(f"Positive-level gate: {'PASS' if level_pass else 'FAIL'}")
print("-" * 78)
if rank_pass and not level_pass:
    print("READ: calibrated ΔP materially improves ordering inside execution contexts, but the")
    print("top predicted decile is still negative after executable bid/offer mechanics.")
    print("Use calibrated ΔP bps as fair-value / quote-bias / avoid-worst-trades signal, not")
    print("as standalone positive-EV trade alpha yet.")
elif rank_pass and level_pass:
    print("READ: calibrated ΔP improves ranking and the top predicted bucket clears zero; this")
    print("is the first version that deserves constrained execution research.")
else:
    print("READ: calibration is not yet improving within-context rank quality; keep raw residual")
    print("rank primary and revisit calibration design.")
print("=" * 78)
log_runtime(f"END Section 16 (elapsed {time.perf_counter() - _section16_start:.1f}s)")


SECTION 16 VERDICT — calibrated fair-value bps ranking
Raw residual rank remains the transparent diagnostic. Calibrated ΔP bps is the
execution-context-aware RFQ score, estimated by side × size × first/sub print.
------------------------------------------------------------------------------
Pooled raw residual rank      : rank IC=0.0341, D10-D1=10.84 bps
Pooled calibrated ΔP rank     : rank IC=-0.0185, D10-D1=43.81 bps (lift=+32.97 bps)
Within side×size×print raw    : rank IC=0.0102, D10-D1=29.94 bps
Within side×size×print calib  : rank IC=0.0515, D10-D1=62.24 bps (lift=+32.30 bps)
Reliability D1 realized       : -90.10 bps
Reliability D10 realized      : -46.06 bps
------------------------------------------------------------------------------
Ranking gate: PASS
Positive-level gate: FAIL
------------------------------------------------------------------------------
READ: calibrated ΔP materially improves ordering inside execution contexts, but the
top predicted decile is still negative

## 17. Attention-Factor Challenger Roadmap (Post-IPCA Baseline)

Sections 13–16 now define the production target: **MSRB-calibrated fair-value / quote-bias bps**, not standalone daily trade alpha. The current IPCA residual remains the baseline. The attention model should be introduced only as a **challenger peer representation**, while keeping the temporal layer fixed as AR(1) at first.

### 17.1 Timing Discipline

Lagged characteristics alone do **not** eliminate look-ahead bias. A clean RFQ feature table must respect every timestamp:

- $\Gamma$ / model parameters trained before the residual signal date.
- $\varepsilon_t$ computed only after the evaluated mark and same-day cross-sectional factor are available.
- AR(1) coefficient estimated only from earlier residual pairs.
- Peer trades and liquidity features observed before the RFQ timestamp.
- Target MSRB print strictly after the RFQ timestamp.

The current full-sample residual panel is acceptable for exploration and diagnostics. For a final RFQ backtest, residuals must be regenerated with walk-forward model versions:

$$\widehat\Gamma^{(<t)} \rightarrow \widehat\varepsilon_t \rightarrow \widehat\rho^{(<t)} \rightarrow \widehat\varepsilon_{t+1|t}.$$

### 17.2 Attention Challenger

Current IPCA imposes a linear characteristic-to-loading map:

$$\beta^{IPCA}_{i,t-1}=z_{i,t-1}'\Gamma.$$

The attention extension first embeds point-in-time bond state:

$$h_{i,t-1}=\phi_\theta(x_{i,t-1}),$$

then learns $K$ attention heads that define factor portfolios:

$$a_{k,i,t-1}=\operatorname{softmax}_i\left(\frac{q_k'h_{i,t-1}}{\sqrt d}\right),\qquad
F_{k,t}=\sum_i a_{k,i,t-1}r_{i,t}.$$

The implied attention loadings produce residuals:

$$\varepsilon^{Attn}_{i,t}=r_{i,t}-(\beta^{Attn}_{i,t-1})'F_t.$$

Use static features already in the IPCA model, plus dynamic MSRB / mark-state features where available:

- duration / spread duration proxies, tenor, call structure, coupon, premium/discount, rating, sector, state / tax fields;
- days since last MSRB trade, recent trade count, volume, side mix, size mix;
- evaluated-price update rate, recent return volatility, recent residual behavior;
- trade-side and trade-size statistics.

### 17.3 Training Objective

Do **not** begin with the full paper-style net-Sharpe objective. For this muni RFQ use case, train for factor quality plus RFQ fair-value usefulness:

$$\mathcal L=\lambda_{fit}\mathcal L_{factor}+\lambda_{rv}\mathcal L_{MSRB}+\lambda_{stable}\mathcal L_{stability}.$$

Factor loss preserves a sensible systematic decomposition:

$$\mathcal L_{factor}=\sum_{i,t}\left(r_{i,t}-\beta_{i,t-1}'F_t\right)^2.$$

MSRB loss targets side-consistent next-trade-versus-base fair value:

$$\mathcal L_{MSRB}=\sum_{i,\tau}\ell\left(\widehat b^{RV}_{i,\tau},\;P^{next\ eligible\ trade}_{i,\tau^+}-P^{base}_{i,\tau}\right).$$

Stability regularization should penalize excessive day-to-day attention-weight drift, concentration in a few bonds, stale-observation reliance, and unstable peer membership.

### 17.4 Keep AR(1) Fixed Initially

The first attention experiment should isolate whether attention improves the **cross-sectional peer representation**. Keep the temporal layer fixed:

$$\widehat\varepsilon^{Attn}_{i,t+1|t}=\widehat a_t+\widehat\rho_t\varepsilon^{Attn}_{i,t}.$$

Then calibrate RFQ bias:

$$b^{RV}_{i,\tau}=g\left(\widehat\varepsilon^{Attn}_{i,t+1|t},\;\text{attention-peer trades},\;\text{side},\;\text{size},\;\text{freshness}\right).$$

Do not introduce attention and LongConv at the same time; otherwise we cannot attribute any improvement.

### 17.5 Required Comparison

Evaluate identical point-in-time folds:

1. **IPCA + AR(1)** baseline.
2. **Attention factors + AR(1)** challenger.
3. **IPCA + attention ensemble + AR(1)**.

Judge them on next-trade-versus-base calibration, within rating $\times$ tenor $\times$ call rank IC, side-specific performance, calibration error in bps, peer stability, economic coherence, and eventual RFQ utility.

**Research endpoint:**

$$\boxed{\text{Attention-derived peer representation}\rightarrow\text{factor-adjusted residual}\rightarrow\text{AR(1) continuation}\rightarrow\text{MSRB-calibrated RV bias}}.$$

In [ ]:
# ── 17.6 POINT-IN-TIME RFQ FEATURE TABLE AUDIT SPEC ─────────────────────
rfq_feature_audit_columns = [
    'rfq_timestamp', 'cusip', 'side', 'size',
    'model_family', 'model_train_end', 'gamma_version',
    'residual_date', 'residual_asof_timestamp',
    'ar_train_end', 'peer_data_asof_timestamp',
    'rv_bias_bps', 'raw_residual_rank', 'calibrated_delta_p_bps',
    'target_trade_timestamp', 'target_trade_price', 'target_vs_base_bps',
]

audit_rules = pd.DataFrame([
    {'rule': 'model trained before residual', 'assertion': 'model_train_end < residual_date'},
    {'rule': 'residual available before RFQ', 'assertion': 'residual_asof_timestamp <= rfq_timestamp'},
    {'rule': 'AR fit historical at RFQ', 'assertion': 'ar_train_end < rfq_timestamp'},
    {'rule': 'peer data historical at RFQ', 'assertion': 'peer_data_asof_timestamp <= rfq_timestamp'},
    {'rule': 'target after RFQ', 'assertion': 'target_trade_timestamp > rfq_timestamp'},
    {'rule': 'no individual factor alpha rank', 'assertion': 'ranking_input in {combined_residual, ar_forecast, calibrated_delta_p_bps}'},
])

model_comparison_spec = pd.DataFrame([
    {'model': 'IPCA + AR(1)', 'role': 'baseline', 'temporal_layer': 'fixed PIT AR(1)',
     'peer_representation': 'linear characteristic loadings', 'status': 'current benchmark'},
    {'model': 'Attention factors + AR(1)', 'role': 'challenger', 'temporal_layer': 'fixed PIT AR(1)',
     'peer_representation': 'learned attention embeddings / heads', 'status': 'next experiment'},
    {'model': 'IPCA + attention ensemble + AR(1)', 'role': 'ensemble', 'temporal_layer': 'fixed PIT AR(1)',
     'peer_representation': 'baseline + attention residual blend', 'status': 'only after attention standalone'},
])

print("Required point-in-time RFQ feature table columns:")
display(pd.Series(rfq_feature_audit_columns, name='column').to_frame())
print("\nTimestamp audit rules:")
display(audit_rules)
print("\nRequired model comparison:")
display(model_comparison_spec)


Required point-in-time RFQ feature table columns:


,column
0,rfq_timestamp
1,cusip
2,side
3,size
4,model_family
5,model_train_end
6,gamma_version
7,residual_date
8,residual_asof_timestamp
9,ar_train_end


Timestamp audit rules:


,rule,assertion
0,model trained before residual,model_train_end < residual_date
1,residual available before RFQ,residual_asof_timestamp <= rfq_timestamp
2,AR fit historical at RFQ,ar_train_end < rfq_timestamp
3,peer data historical at RFQ,peer_data_asof_timestamp <= rfq_timestamp
4,target after RFQ,target_trade_timestamp > rfq_timestamp
5,no individual factor alpha rank,"ranking_input in {combined_residual, ar_foreca..."


Required model comparison:


,model,role,temporal_layer,peer_representation,status
0,IPCA + AR(1),baseline,fixed PIT AR(1),linear characteristic loadings,current benchmark
1,Attention factors + AR(1),challenger,fixed PIT AR(1),learned attention embeddings / heads,next experiment
2,IPCA + attention ensemble + AR(1),ensemble,fixed PIT AR(1),baseline + attention residual blend,only after attention standalone


## 18. Attention Factor v0 — Runnable Challenger to IPCA

This section builds the first attention-factor challenger inside the current notebook. It is intentionally conservative:

- **No LongConv yet.** Keep the temporal layer fixed as point-in-time AR(1), so any lift comes from the cross-sectional peer representation.
- **No full joint neural RFQ objective yet.** This v0 uses train-window IPCA loadings as attention query seeds, then converts characteristic scores into softmax attention portfolios.
- **No individual IPCA-factor alpha ranking.** Alpha signals are still combined residual / AR forecasts.
- **No full-sample parameter leakage for evaluation.** Query seeds come from the first weekly walk-forward model when available; evaluation is restricted to signal dates after that model's train end.

### Construction

For each date $t$, use lagged bond characteristics $Z_{i,t-1}$ and train-only query heads $Q$ to compute attention weights:

$$a_{k,i,t-1}=\operatorname{softmax}_i\left(s\,Z_{i,t-1}'q_k\right),\qquad
F^{Attn}_{k,t}=\sum_i a_{k,i,t-1}r_{i,t}.$$

Then fit a characteristic-to-loading map on the train window only, holding the attention factor returns fixed:

$$r_{i,t}\approx Z_{i,t-1}'\Gamma^{Attn}F^{Attn}_t.$$

This yields attention residuals $\varepsilon^{Attn}_{i,t}$, point-in-time AR forecasts, and an apples-to-apples MSRB executable-opportunity comparison versus the IPCA + AR(1) baseline.

In [ ]:
# ── 18.1 ATTENTION FACTOR v0 CONSTRUCTION ───────────────────────────────
_section18_start = time.perf_counter()
log_runtime("START Section 18: attention factor v0")

_required_18 = ['R', 'Z', 'obs_mask', 'panel_dates', 'R_panel', 'characteristics', 'weekly_folds', 'fold_cache', 'fold_cache_key']
_missing_18 = [name for name in _required_18 if name not in globals()]
if _missing_18:
    raise RuntimeError(f"Run Sections 5-13 first; missing required variables: {_missing_18}")

ATTN_K = int(globals().get('selected_K', 3))
ATTN_TEMP = 4.0
ATTN_RIDGE = 1e-6
ATTN_MIN_OBS = 60

with timed_block("attention query seed selection"):
    _fold0 = weekly_folds[0] if weekly_folds else None
    if _fold0 is not None:
        _key0 = fold_cache_key(_fold0['fold_kind'], _fold0, ATTN_K, CV_MAX_ITER, 1e-5)
        _entry0 = fold_cache.get(_key0)
    else:
        _entry0 = None
    if _entry0 is not None and _entry0.get('Gamma') is not None:
        attention_query_seed = np.asarray(_entry0['Gamma'], dtype=float)
        attention_train_end = pd.Timestamp(_fold0['train_end'])
        attention_source = f"weekly walk-forward fold 1 train_end={attention_train_end.date()}"
    else:
        attention_query_seed = np.asarray(ipca_model.Gamma, dtype=float)
        attention_train_end = pd.Timestamp(panel_dates[min(int(np.floor(len(panel_dates) * 0.7)), len(panel_dates) - 1)])
        attention_source = "fallback full-sample Gamma seed (diagnostic only)"
    attention_query_seed = attention_query_seed[:, :ATTN_K]
    attention_query_seed = attention_query_seed / np.maximum(np.linalg.norm(attention_query_seed, axis=0, keepdims=True), 1e-12)

print(f"Attention query source: {attention_source}")
print(f"Attention evaluation starts strictly after {attention_train_end.date()}")


def _softmax_stable(x):
    z = x - np.nanmax(x)
    ez = np.exp(z)
    s = ez.sum()
    return ez / s if s > 0 else np.full_like(x, 1.0 / len(x))

with timed_block("attention factor returns"):
    N_attn, T_attn = R.shape
    F_attn = np.full((T_attn, ATTN_K), np.nan, dtype=np.float64)
    attn_effective_n = np.full((T_attn, ATTN_K), np.nan, dtype=np.float64)
    attn_top10_share = np.full((T_attn, ATTN_K), np.nan, dtype=np.float64)
    for _t in range(T_attn):
        _obs = obs_mask[:, _t] > 0
        if int(_obs.sum()) < ATTN_K * 10:
            continue
        _Zt = Z[_obs, :, _t].astype(float)
        _Rt = R[_obs, _t].astype(float)
        _scores = _Zt @ attention_query_seed
        for _k in range(ATTN_K):
            _w = _softmax_stable(ATTN_TEMP * _scores[:, _k])
            F_attn[_t, _k] = float(_w @ _Rt)
            attn_effective_n[_t, _k] = 1.0 / float(np.sum(_w * _w))
            attn_top10_share[_t, _k] = float(np.sort(_w)[-10:].sum())

with timed_block("fit attention loadings on pre-evaluation window"):
    _train_cols_attn = np.where(pd.DatetimeIndex(panel_dates) <= attention_train_end)[0]
    _L = Z.shape[1]
    _lhs = np.zeros((_L * ATTN_K, _L * ATTN_K), dtype=np.float64)
    _rhs = np.zeros(_L * ATTN_K, dtype=np.float64)
    _n_fit_rows = 0
    for _t in _train_cols_attn:
        _ft = F_attn[_t]
        if not np.all(np.isfinite(_ft)):
            continue
        _obs = obs_mask[:, _t] > 0
        if not _obs.any():
            continue
        _Zt = Z[_obs, :, _t].astype(float)
        _Rt = R[_obs, _t].astype(float)
        _ZtZ = _Zt.T @ _Zt
        _ZtR = _Zt.T @ _Rt
        _lhs += np.kron(np.outer(_ft, _ft), _ZtZ)
        _rhs += np.kron(_ft, _ZtR)
        _n_fit_rows += int(_obs.sum())
    gamma_attn_vec = np.linalg.solve(_lhs + ATTN_RIDGE * np.eye(_L * ATTN_K), _rhs)
    Gamma_attn = gamma_attn_vec.reshape((_L, ATTN_K), order='F')

with timed_block("attention residual panel"):
    R_hat_attn = np.full_like(R, np.nan, dtype=np.float64)
    resid_attn_panel = np.full_like(R, np.nan, dtype=np.float64)
    for _t in range(T_attn):
        _ft = F_attn[_t]
        if not np.all(np.isfinite(_ft)):
            continue
        _obs = obs_mask[:, _t] > 0
        if not _obs.any():
            continue
        _pred = (Z[_obs, :, _t].astype(float) @ Gamma_attn) @ _ft
        R_hat_attn[_obs, _t] = _pred
        resid_attn_panel[_obs, _t] = R[_obs, _t].astype(float) - _pred

attn_fit_metrics = ipca_performance_metrics(R, np.nan_to_num(R_hat_attn), obs_mask)
attn_gamma_df = pd.DataFrame(Gamma_attn, index=characteristics, columns=[f'Attn_{k+1}' for k in range(ATTN_K)])
attn_factor_df = pd.DataFrame(F_attn, index=panel_dates, columns=[f'Attn_{k+1}' for k in range(ATTN_K)])
attn_concentration = pd.DataFrame({
    'median_effective_n': np.nanmedian(attn_effective_n, axis=0),
    'median_top10_share': np.nanmedian(attn_top10_share, axis=0),
}, index=attn_factor_df.columns)

print(f"Attention loading fit rows: {_n_fit_rows:,}")
print("Attention factor fit metrics (diagnostic; v0 attention factors, train-seeded queries):")
display(pd.Series(attn_fit_metrics).to_frame('value').style.format('{:.4f}'))
print("Attention concentration diagnostics:")
display(attn_concentration.style.format({'median_effective_n': '{:,.1f}', 'median_top10_share': '{:.4f}'}))

# Build post-train attention residual universe and point-in-time AR(1) forecasts.
_eval_start_idx_attn = int(np.searchsorted(pd.DatetimeIndex(panel_dates), attention_train_end, side='right'))
_obs_attn = np.isfinite(resid_attn_panel)
_uni_attn = _obs_attn.sum(axis=1) >= ATTN_MIN_OBS
attn_cusips = R_panel.index[_uni_attn]
Rp_attn = resid_attn_panel[_uni_attn]
N_attn_uni, T_attn_rv = Rp_attn.shape

with timed_block("PIT AR(1) forecasts for attention residuals"):
    _Sx = _Sy = _Sxx = _Sxy = _npairs = 0.0
    attn_ar_pred = np.full_like(Rp_attn, np.nan, dtype=np.float64)
    attn_ar_rho_path = []
    for _t in range(1, T_attn_rv - 1):
        _x0, _y0 = Rp_attn[:, _t - 1], Rp_attn[:, _t]
        _m0 = np.isfinite(_x0) & np.isfinite(_y0)
        _xx, _yy = _x0[_m0], _y0[_m0]
        _Sx += _xx.sum(); _Sy += _yy.sum(); _Sxx += float((_xx * _xx).sum())
        _Sxy += float((_xx * _yy).sum()); _npairs += _xx.size
        if _t < max(20, _eval_start_idx_attn) or _npairs < 1000:
            continue
        _den = _npairs * _Sxx - _Sx * _Sx
        if _den <= 0:
            continue
        _rho = (_npairs * _Sxy - _Sx * _Sy) / _den
        _a = (_Sy - _rho * _Sx) / _npairs
        attn_ar_rho_path.append(_rho)
        attn_ar_pred[:, _t] = _a + _rho * Rp_attn[:, _t]

print(f"Attention residual universe: {N_attn_uni:,} bonds x {T_attn_rv} dates")
print(f"PIT attention AR forecasts available: {int(np.isfinite(attn_ar_pred).sum()):,} bond-days")
print(f"attention AR rho mean/last: {np.nanmean(attn_ar_rho_path):.4f} / {attn_ar_rho_path[-1]:.4f}")


[2026-09-16 13:10:30] START Section 18: attention factor v0
[2026-09-16 13:10:30] START attention query seed selection
[2026-09-16 13:10:30] END attention query seed selection | elapsed 0.00s
Attention query source: weekly walk-forward fold 1 train_end=2026-07-31
Attention evaluation starts strictly after 2026-07-31
[2026-09-16 13:10:30] START attention factor returns
[2026-09-16 13:11:34] END attention factor returns | elapsed 1m 03.52s
[2026-09-16 13:11:34] START fit attention loadings on pre-evaluation window
[2026-09-16 13:11:48] END fit attention loadings on pre-evaluation window | elapsed 13.77s
[2026-09-16 13:11:48] START attention residual panel
[2026-09-16 13:12:22] END attention residual panel | elapsed 34.93s
Attention loading fit rows: 19,280,231
Attention factor fit metrics (diagnostic; v0 attention factors, train-seeded queries):


<HTML output>

,value
Total R2,0.7602
Time Series R2,0.7861
Cross Section R2,0.4587
Relative Pricing Error,0.1323


Attention concentration diagnostics:


<HTML output>

,median_effective_n,median_top10_share
Attn_1,"181,146.8",0.0002
Attn_2,"118,271.6",0.0003
Attn_3,"145,137.5",0.0002


[2026-09-16 13:12:34] START PIT AR(1) forecasts for attention residuals
[2026-09-16 13:12:35] END PIT AR(1) forecasts for attention residuals | elapsed 0.53s
Attention residual universe: 229,857 bonds x 114 dates
PIT attention AR forecasts available: 6,480,823 bond-days
attention AR rho mean/last: 0.1645 / 0.1795


In [ ]:
# ── 18.2 ATTENTION vs IPCA ON EXECUTABLE OPPORTUNITIES ──────────────────
_required_18b = ['executable_opps', 'attn_cusips', 'attn_ar_pred', 'panel_dates']
_missing_18b = [name for name in _required_18b if name not in globals()]
if _missing_18b:
    raise RuntimeError(f"Run Sections 15 and 18.1 first; missing required variables: {_missing_18b}")

with timed_block("attach attention forecasts to executable opportunities"):
    _attn_row_map = pd.Series(np.arange(len(attn_cusips), dtype=np.int64), index=np.asarray(attn_cusips, dtype=str))
    _opp = executable_opps.copy()
    _row = _opp['cusip'].astype(str).map(_attn_row_map).to_numpy()
    _t = _opp['resid_t'].to_numpy(dtype=np.int64) if 'resid_t' in _opp.columns else None
    if _t is None:
        _rv_dates18 = pd.DatetimeIndex(pd.to_datetime(rv_dates, errors='coerce')).normalize()
        _sig_ns18 = pd.to_datetime(_opp['signal_date'], errors='coerce').to_numpy(dtype='datetime64[ns]')
        _t = np.searchsorted(_rv_dates18.values.astype('datetime64[ns]'), _sig_ns18, side='left')
    _valid = pd.notna(_row) & (_t >= 0) & (_t < attn_ar_pred.shape[1])
    _attn_forecast = np.full(len(_opp), np.nan, dtype=float)
    _attn_forecast[_valid] = attn_ar_pred[_row[_valid].astype(np.int64), _t[_valid].astype(np.int64)]
    _opp['attn_ar_signal'] = _attn_forecast
    _opp['ipca_ar_signal'] = _opp['alpha_signal']
    _opp['attn_opportunity_signal'] = np.where(_opp['opportunity_side'] == 'sell', -_opp['attn_ar_signal'], _opp['attn_ar_signal'])
    _opp['ipca_opportunity_signal'] = _opp['ipca_ar_signal']
    _z_ipca = _opp.groupby('date')['ipca_opportunity_signal'].transform(lambda s: (s - s.mean()) / s.std() if s.std() > 0 else np.nan)
    _z_attn = _opp.groupby('date')['attn_opportunity_signal'].transform(lambda s: (s - s.mean()) / s.std() if s.std() > 0 else np.nan)
    _opp['ensemble_signal'] = 0.5 * _z_ipca + 0.5 * _z_attn
    attention_exec_opps = _opp[_opp['date'] > attention_train_end].copy()

with timed_block("score attention challenger on executable opportunities"):
    attention_exec_compare = pd.DataFrame([
        _score_rank_model(attention_exec_opps, 'ipca_opportunity_signal', label='IPCA + AR(1) baseline'),
        _score_rank_model(attention_exec_opps, 'attn_opportunity_signal', label='Attention factors + AR(1)'),
        _score_rank_model(attention_exec_opps, 'ensemble_signal', label='IPCA + attention ensemble + AR(1)'),
    ])
    _attn_side_rows = []
    for _side in ['buy', 'sell']:
        _side_df = attention_exec_opps[attention_exec_opps['opportunity_side'] == _side]
        _attn_side_rows.append(_score_rank_model(_side_df, 'attn_opportunity_signal', label=f'attention side={_side}'))
    attention_side_summary = pd.DataFrame(_attn_side_rows)

print(f"Attention executable comparison rows: {len(attention_exec_opps):,} "
      f"(dates > {attention_train_end.date()})")
print("Model comparison on identical executable opportunities:")
display(attention_exec_compare.style.format({
    'n_rows': '{:,.0f}', 'n_dates': '{:,.0f}', 'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}',
    'D10_minus_D1_bps': '{:.2f}', 'spread_t': '{:.2f}', 'mean_target_bps': '{:.2f}',
}))
print("\nAttention side-specific scoring:")
display(attention_side_summary.style.format({
    'n_rows': '{:,.0f}', 'n_dates': '{:,.0f}', 'daily_rank_IC': '{:.4f}', 'rank_IC_t': '{:.2f}',
    'D10_minus_D1_bps': '{:.2f}', 'spread_t': '{:.2f}', 'mean_target_bps': '{:.2f}',
}))

fig, ax = plt.subplots(figsize=(10, 4.5))
attention_exec_compare.set_index('model')['D10_minus_D1_bps'].plot(kind='bar', ax=ax, color=['tab:gray', 'tab:blue', 'tab:green'])
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(20, color='red', linestyle='--', alpha=0.6, label='20 bps reference')
ax.set_ylabel('D10-D1 executable return (bps)')
ax.set_title('Attention factor challenger vs IPCA baseline')
ax.grid(True, alpha=0.3, axis='y'); ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

del _attn_row_map, _row, _t, _valid, _attn_forecast


[2026-09-16 13:13:14] START attach attention forecasts to executable opportunities
[2026-09-16 13:13:16] END attach attention forecasts to executable opportunities | elapsed 1.74s
[2026-09-16 13:13:16] START score attention challenger on executable opportunities
[2026-09-16 13:13:18] END score attention challenger on executable opportunities | elapsed 1.87s
Attention executable comparison rows: 543,877 (dates > 2026-07-31)
Model comparison on identical executable opportunities:


<HTML output>

,model,n_rows,n_dates,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,mean_target_bps
0,IPCA + AR(1) baseline,"543,877",29,0.0485,3.93,16.65,2.12,-57.26
1,Attention factors + AR(1),"518,882",28,-0.0495,-2.43,12.54,1.34,-57.13
2,IPCA + attention ensemble + AR(1),"518,882",28,-0.0013,-0.11,16.15,1.85,-57.13


Attention side-specific scoring:


<HTML output>

,model,n_rows,n_dates,daily_rank_IC,rank_IC_t,D10_minus_D1_bps,spread_t,mean_target_bps
0,attention side=buy,"243,660",28,0.0234,1.64,28.62,2.24,-81.00
1,attention side=sell,"275,222",28,0.0137,1.23,7.53,0.66,-36.00


<Figure: muni_characteristic_factor_export_125_4.png (image not included in markdown export)>

In [ ]:
# ── 18.3 ATTENTION CHALLENGER VERDICT ──────────────────────────────────
_ipca_row = attention_exec_compare[attention_exec_compare['model'] == 'IPCA + AR(1) baseline'].iloc[0]
_attn_row = attention_exec_compare[attention_exec_compare['model'] == 'Attention factors + AR(1)'].iloc[0]
_ens_row = attention_exec_compare[attention_exec_compare['model'] == 'IPCA + attention ensemble + AR(1)'].iloc[0]
_attn_lift = _attn_row['D10_minus_D1_bps'] - _ipca_row['D10_minus_D1_bps']
_ens_lift = _ens_row['D10_minus_D1_bps'] - _ipca_row['D10_minus_D1_bps']
_attn_weight_too_diffuse = bool(np.nanmedian(attn_concentration['median_effective_n']) > 0.25 * np.nanmedian((obs_mask > 0).sum(axis=0)))

attention_challenger_verdict = pd.Series({
    'IPCA D10-D1 bps': _ipca_row['D10_minus_D1_bps'],
    'Attention D10-D1 bps': _attn_row['D10_minus_D1_bps'],
    'Ensemble D10-D1 bps': _ens_row['D10_minus_D1_bps'],
    'Attention lift bps': _attn_lift,
    'Ensemble lift bps': _ens_lift,
    'Attention rank IC': _attn_row['daily_rank_IC'],
    'IPCA rank IC': _ipca_row['daily_rank_IC'],
    'Median attention effective N': np.nanmedian(attn_concentration['median_effective_n']),
    'Diffuse attention weights': _attn_weight_too_diffuse,
})

challenger_win = (_attn_lift > 2.0 and _attn_row['daily_rank_IC'] > _ipca_row['daily_rank_IC'])
ensemble_win = (_ens_lift > 2.0 and _ens_row['daily_rank_IC'] >= _ipca_row['daily_rank_IC'])

print("=" * 78)
print("SECTION 18 VERDICT — attention factor v0 challenger")
print("=" * 78)
display(attention_challenger_verdict.to_frame('value').style.format('{:,.4f}'))
print("-" * 78)
if challenger_win:
    print("READ: attention v0 beats the IPCA baseline on executable opportunity ranking.")
    print("Next: calibrate attention ΔP bps and run the same within-context checks as Section 16.")
elif ensemble_win:
    print("READ: standalone attention does not win, but the ensemble adds incremental value.")
    print("Next: inspect which CUSIP/date contexts the ensemble improves before expanding model complexity.")
else:
    print("READ: attention v0 is not a challenger win yet.")
    print("  The IPCA baseline remains stronger on the headline executable spread and rank IC.")
    print("  Attention weights are very diffuse, so this v0 behaves more like smooth characteristic")
    print("  pooling than a sharp learned peer model.")
    print("Next: only improve attention if we add real dynamic features (trade recency, side/size flow,")
    print("  price-update state) or a supervised RFQ calibration objective; do not add LongConv yet.")
print("=" * 78)
log_runtime(f"END Section 18 (elapsed {time.perf_counter() - _section18_start:.1f}s)")


SECTION 18 VERDICT — attention factor v0 challenger


<HTML output>

,value
IPCA D10-D1 bps,16.6483
Attention D10-D1 bps,12.5398
Ensemble D10-D1 bps,16.1499
Attention lift bps,-4.1085
Ensemble lift bps,-0.4984
Attention rank IC,-0.0495
IPCA rank IC,0.0485
Median attention effective N,"145,137.4945"
Diffuse attention weights,1.0000


------------------------------------------------------------------------------
READ: attention v0 is not a challenger win yet.
  The IPCA baseline remains stronger on the headline executable spread and rank IC.
  Attention weights are very diffuse, so this v0 behaves more like smooth characteristic
  pooling than a sharp learned peer model.
Next: only improve attention if we add real dynamic features (trade recency, side/size flow,
  price-update state) or a supervised RFQ calibration objective; do not add LongConv yet.
[2026-09-16 13:13:57] END Section 18 (elapsed 207.1s)


## 19. Final Production Assessment: Conditional RV Score, Not Standalone Stat-Arb

The strongest description of this research is:

$$\text{IPCA risk decomposition}\rightarrow\text{residual forecast}\rightarrow\text{calibrated RV adjustment}\rightarrow\text{quote and inventory optimization}.$$

Do **not** describe the raw residual as unqualified "mispricing." A meaningful share of the residual structure is linked to evaluated-price smoothing, liquidity, stale-mark propagation, and side-specific execution mechanics. The production artifact should be a:

> **Conditional relative-value and mark-correction score**

That framing is economically accurate and commercially useful for market making.

### What We Keep

- **IPCA** as the interpretable systematic fair-value decomposition.
- **$K=3$** as the current parsimonious baseline.
- **Point-in-time AR(1)** as the temporal benchmark every challenger must beat.
- **Side-aware MSRB executable validation** as the binding tradeability test.
- **Calibrated $\Delta P$ bps** as the RFQ-facing fair-value / quote-bias output.

### What We Revise Next

1. Generate a complete **point-in-time residual and calibration chain**; separate diagnostic residuals from production PIT residuals.
2. Add dynamic liquidity and price-discovery features: days since last customer/interdealer trade, recent trade count/volume, side imbalance, size mix, price-update state, RFQ side/size, issuer/obligor liquidity, callable-adjusted duration, years-to-call, and curve residuals.
3. Forecast **decision-specific executable markouts**, not generic future residuals.
4. Separate customer-buy and customer-sell models.
5. Treat attention as a constrained challenger, not the production model.
6. Output both expected edge and uncertainty.
7. Measure incremental optimizer value, not only standalone IC or decile spread.

### Production Architecture

```text
Point-in-time bond characteristics
    -> IPCA systematic fair-value model
    -> point-in-time residual
        -> AR(1) persistence baseline
        -> dynamic microstructure model
        -> constrained sparse peer-attention challenger
    -> side/size/liquidity-conditioned executable markout
    -> calibrated distribution: mean + uncertainty
    -> market-making optimizer
        -> fill probability
        -> inventory cost
        -> hedge/liquidation cost
        -> risk constraints
        -> quote offset
```

The model should not hard-code a trade decision. It should provide a calibrated distributional input that the optimizer combines with fill probability, inventory, liquidity, and adverse-selection terms.

In [ ]:
# ── 19.1 PRODUCTION OUTPUT SPEC: RV DISTRIBUTION + OPTIMIZER INPUTS ─────
rv_distribution_spec = pd.DataFrame([
    {'field': 'rv_mu_bps', 'definition': 'conditional expected executable markout / fair-value adjustment', 'consumer': 'quote offset'},
    {'field': 'rv_sigma_bps', 'definition': 'uncertainty of RV markout conditional on side/size/liquidity/freshness', 'consumer': 'confidence haircut'},
    {'field': 'rv_ce_bps', 'definition': 'certainty-equivalent RV adjustment: mu - eta*sigma', 'consumer': 'optimizer objective'},
    {'field': 'rv_rank_pct', 'definition': 'cross-sectional percentile of calibrated RV score within comparable context', 'consumer': 'relative-value diagnostics'},
    {'field': 'rv_signal_age_days', 'definition': 'age of residual / model state used for the RFQ', 'consumer': 'staleness control'},
    {'field': 'model_vintage', 'definition': 'model version and training end timestamp', 'consumer': 'PIT audit'},
    {'field': 'target_definition', 'definition': 'side-consistent next eligible post-RFQ MSRB markout', 'consumer': 'backtest audit'},
])

dynamic_feature_backlog = pd.DataFrame([
    {'feature_family': 'trade recency', 'examples': 'days since last customer trade; days since last interdealer trade', 'priority': 'P0'},
    {'feature_family': 'trade activity', 'examples': 'recent customer/interdealer count, volume, side imbalance, size mix', 'priority': 'P0'},
    {'feature_family': 'mark state', 'examples': 'evaluated-price age, update magnitude, update rate, mark volatility', 'priority': 'P0'},
    {'feature_family': 'curve RV', 'examples': 'rating-state-tenor curve residual; issuer/obligor curve residual', 'priority': 'P1'},
    {'feature_family': 'structure', 'examples': 'callable-adjusted duration, years-to-call, premium/discount status', 'priority': 'P1'},
    {'feature_family': 'RFQ context', 'examples': 'RFQ side, size, quote revision state, recent RFQ side/size pressure', 'priority': 'P1'},
    {'feature_family': 'attention constraints', 'examples': 'issuer/sector/rating/duration candidate peers, no-self attention, sparse top-k', 'priority': 'P2'},
])

next_research_steps = pd.DataFrame([
    {'step': 1, 'work': 'Build production PIT residual table', 'success_metric': 'all audit inequalities pass; no full-sample residuals in claimed OOS'},
    {'step': 2, 'work': 'Add P0 dynamic MSRB/mark-state features', 'success_metric': 'improves executable markout calibration error and within-context rank IC'},
    {'step': 3, 'work': 'Fit side-separated calibrated markout distributions', 'success_metric': 'positive CE lift in quote/inventory simulation, not just IC'},
    {'step': 4, 'work': 'Constrained sparse attention challenger', 'success_metric': 'beats IPCA+AR(1) on identical PIT folds and peer-stability diagnostics'},
    {'step': 5, 'work': 'Optimizer integration test', 'success_metric': 'incremental RFQ utility after fill/inventory/liquidity costs'},
])

print("RV distribution / optimizer-facing output fields:")
display(rv_distribution_spec)
print("\nDynamic feature backlog:")
display(dynamic_feature_backlog)
print("\nPrioritized next research steps:")
display(next_research_steps)


RV distribution / optimizer-facing output fields:


,field,definition,consumer
0,rv_mu_bps,conditional expected executable markout / fair...,quote offset
1,rv_sigma_bps,uncertainty of RV markout conditional on side/...,confidence haircut
2,rv_ce_bps,certainty-equivalent RV adjustment: mu - eta*s...,optimizer objective
3,rv_rank_pct,cross-sectional percentile of calibrated RV sc...,relative-value diagnostics
4,rv_signal_age_days,age of residual / model state used for the RFQ,staleness control
5,model_vintage,model version and training end timestamp,PIT audit
6,target_definition,side-consistent next eligible post-RFQ MSRB ma...,backtest audit


Dynamic feature backlog:


,feature_family,examples,priority
0,trade recency,days since last customer trade; days since las...,P0
1,trade activity,"recent customer/interdealer count, volume, sid...",P0
2,mark state,"evaluated-price age, update magnitude, update ...",P0
3,curve RV,rating-state-tenor curve residual; issuer/obli...,P1
4,structure,"callable-adjusted duration, years-to-call, pre...",P1
5,RFQ context,"RFQ side, size, quote revision state, recent R...",P1
6,attention constraints,"issuer/sector/rating/duration candidate peers,...",P2


Prioritized next research steps:


,step,work,success_metric
0,1,Build production PIT residual table,all audit inequalities pass; no full-sample re...
1,2,Add P0 dynamic MSRB/mark-state features,improves executable markout calibration error ...
2,3,Fit side-separated calibrated markout distribu...,positive CE lift in quote/inventory simulation...
3,4,Constrained sparse attention challenger,beats IPCA+AR(1) on identical PIT folds and pe...
4,5,Optimizer integration test,incremental RFQ utility after fill/inventory/l...


## 20. AlgoSignal-MSRB Match Pull & Cache

`muni_offline.algosignal_msrb_match` is the missing bridge from model research to a market-making charge model. It joins an algo signal / quote snapshot to subsequent MSRB trade prints, with fields such as:

- `signal_ts`, `signal_side`, `algo_signal_yield`, `last_value`
- `msrb_trade_id`, `msrb_tradetime`, `msrb_side`, `msrb_price`, `msrb_yield`, `msrb_quantity`, `size_bin`
- `MinuteFromSignal`, `MmdYld`, `dMmdSprdSide`, `algo_minus_msrb_yield_bp`

The first charge target is yield error in bps:

$$y^{yield}_{i,\tau}=\text{MSRB yield}_{i,\tau^+}-\text{algo signal yield}_{i,\tau}.$$

Because the sample shows `algo_minus_msrb_yield_bp = (algo_signal_yield - msrb_yield) * 100`, the canonical target is:

$$y^{yield}_{i,\tau}=-\text{algo\_minus\_msrb\_yield\_bp}.$$

This section pulls the matched table, caches it to `algosignal_msrb_match_cache.pkl`, and prints schema/coverage checks before any modeling.

In [ ]:
# ── 20.1 INCREMENTAL ALGOSIGNAL-MSRB MATCH PULL WINDOW ──────────────────
base_match_cutoff = '2026-04-01'
ALGOSIGNAL_MSRB_CACHE_PATH = Path('algosignal_msrb_match_cache.pkl')
match_cutoff_date = base_match_cutoff

if ALGOSIGNAL_MSRB_CACHE_PATH.exists():
    try:
        _cached_match = pd.read_pickle(ALGOSIGNAL_MSRB_CACHE_PATH)
        if isinstance(_cached_match, pd.DataFrame) and not _cached_match.empty and 'date' in _cached_match.columns:
            _dates = pd.to_datetime(_cached_match['date'], errors='coerce').dropna()
            if not _dates.empty:
                _from = max(_dates.max().normalize() - pd.Timedelta(days=1), pd.Timestamp(base_match_cutoff))
                match_cutoff_date = str(_from.date())
                print(f"Found local algosignal/MSRB cache through {_dates.max().date()}; refreshing from {match_cutoff_date}")
            else:
                print("Local algosignal/MSRB cache has no parseable dates; using base cutoff")
        else:
            print("Local algosignal/MSRB cache is empty or invalid; using base cutoff")
    except Exception as exc:
        print(f"WARNING: could not inspect {ALGOSIGNAL_MSRB_CACHE_PATH}: {exc}; using base cutoff")
    finally:
        globals().pop('_cached_match', None)
        globals().pop('_dates', None)
else:
    print(f"No local algosignal/MSRB cache found; pulling from {match_cutoff_date}")

try:
    DeephavenSession.session.run_script(f"match_cutoff_date = '{match_cutoff_date}'")
    print(f"Injected match_cutoff_date='{match_cutoff_date}' into the Deephaven session.")
except Exception as exc:
    print(f"Deephaven session not ready to receive match_cutoff_date ({exc}); "
          f"match query will default to the full window until the session is live.")


In [ ]:
%%deephaven_cell

import time as _time

try:
    match_cutoff_date
except NameError:
    match_cutoff_date = '2026-04-01'

_cell_start = _time.perf_counter()
print("[runtime] START Deephaven ingestion: muni_offline.algosignal_msrb_match")
print(f"match_cutoff_date: {match_cutoff_date}")

algosignal_msrb_raw = (
    db.historical_table("muni_offline", "algosignal_msrb_match")
      .where(f"date >= `{match_cutoff_date}`")
)


def _dh_cols(tbl):
    try:
        return [str(n) for n in tbl.column_names]
    except Exception:
        pass
    try:
        return [c.name for c in tbl.columns]
    except Exception:
        pass
    try:
        return [str(n) for n in tbl.getDefinition().getColumnNames()]
    except Exception:
        return []


_avail = _dh_cols(algosignal_msrb_raw)
print("algosignal_msrb_match available columns:", _avail)
_want = [
    'date', 'msrb_trade_id', 'cusip', 'msrb_quantity', 'size_bin',
    'msrb_side', 'signal_side', 'msrb_tradetime', 'msrb_price', 'msrb_yield',
    'has_algosignal_match', 'signal_ts', 'MinuteFromSignal', 'MmdYld',
    'dMmdSprdSide', 'last_value', 'algo_signal_yield', 'algo_minus_msrb_yield_bp', 'Flg',
]
_keep = [c for c in _want if c in _avail] if _avail else _want
print("algosignal_msrb_match columns kept:", _keep)

algosignal_msrb_slim = algosignal_msrb_raw.view(_keep)
print("algosignal_msrb_slim incremental size:", algosignal_msrb_slim.size)
print(f"[runtime] END algosignal/MSRB ingestion | elapsed {_time.perf_counter() - _cell_start:.2f}s")
DeephavenWidget(algosignal_msrb_slim)


In [ ]:
# ── 20.3 PULL ALGOSIGNAL-MSRB MATCH TO PANDAS + CACHE ───────────────────
ALGOSIGNAL_MSRB_CACHE_PATH = Path('algosignal_msrb_match_cache.pkl')

try:
    with timed_block("Pull incremental algosignal/MSRB match to pandas"):
        match_incremental = DeephavenSession.session.open_table('algosignal_msrb_slim').to_arrow().to_pandas()
except Exception as exc:
    print(f"WARNING: could not pull algosignal_msrb_slim from Deephaven ({exc}); using local cache if available.")
    if not ALGOSIGNAL_MSRB_CACHE_PATH.exists():
        raise RuntimeError(
            f"AlgoSignal/MSRB Deephaven pull failed and local cache {ALGOSIGNAL_MSRB_CACHE_PATH} is not available in the notebook kernel cwd."
        ) from exc
    with timed_block("Load local algosignal/MSRB match cache without remote refresh"):
        algosignal_msrb = pd.read_pickle(ALGOSIGNAL_MSRB_CACHE_PATH)
    for _col in ['date', 'msrb_tradetime', 'signal_ts']:
        if _col in algosignal_msrb.columns:
            algosignal_msrb[_col] = pd.to_datetime(algosignal_msrb[_col], errors='coerce')
    print(f"Loaded local algosignal/MSRB cache without remote refresh: {len(algosignal_msrb):,} rows")
    if 'date' in algosignal_msrb.columns and algosignal_msrb['date'].notna().any():
        print(f"cache date range: {algosignal_msrb['date'].min().date()} → {algosignal_msrb['date'].max().date()}")
    print(f"columns: {list(algosignal_msrb.columns)}")
    display(algosignal_msrb.head(3))
else:
    print(f"algosignal/MSRB incremental pull: {match_incremental.shape}; columns: {list(match_incremental.columns)}")
    for _col in ['date', 'msrb_tradetime', 'signal_ts']:
        if _col in match_incremental.columns:
            match_incremental[_col] = pd.to_datetime(match_incremental[_col], errors='coerce')
    for _col in ['msrb_quantity', 'size_bin', 'msrb_price', 'msrb_yield', 'MinuteFromSignal',
                 'MmdYld', 'dMmdSprdSide', 'last_value', 'algo_signal_yield',
                 'algo_minus_msrb_yield_bp', 'Flg']:
        if _col in match_incremental.columns:
            match_incremental[_col] = pd.to_numeric(match_incremental[_col], errors='coerce')

    if ALGOSIGNAL_MSRB_CACHE_PATH.exists():
        with timed_block("Load local algosignal/MSRB match cache"):
            try:
                cached_match = pd.read_pickle(ALGOSIGNAL_MSRB_CACHE_PATH)
            except Exception as exc:
                print(f"WARNING: could not load {ALGOSIGNAL_MSRB_CACHE_PATH}: {exc}; rebuilding from incremental pull")
                cached_match = pd.DataFrame(columns=match_incremental.columns)
        for _col in ['date', 'msrb_tradetime', 'signal_ts']:
            if _col in cached_match.columns:
                cached_match[_col] = pd.to_datetime(cached_match[_col], errors='coerce')
    else:
        cached_match = pd.DataFrame(columns=match_incremental.columns)

    with timed_block("Merge incremental algosignal/MSRB into local cache"):
        before_rows = len(cached_match)
        _cols = list(dict.fromkeys(list(cached_match.columns) + list(match_incremental.columns)))
        algosignal_msrb = pd.concat([
            cached_match.reindex(columns=_cols),
            match_incremental.reindex(columns=_cols),
        ], ignore_index=True)
        _sort_keys = [c for c in ['date', 'cusip', 'signal_ts', 'msrb_trade_id'] if c in algosignal_msrb.columns]
        if _sort_keys:
            algosignal_msrb = algosignal_msrb.dropna(subset=[_sort_keys[0]])
        algosignal_msrb = algosignal_msrb.drop_duplicates(keep='last')
        if _sort_keys:
            algosignal_msrb = algosignal_msrb.sort_values(_sort_keys)
        algosignal_msrb = algosignal_msrb.reset_index(drop=True)
        algosignal_msrb.to_pickle(ALGOSIGNAL_MSRB_CACHE_PATH)

    print(f"local algosignal/MSRB cache: {before_rows:,} → {len(algosignal_msrb):,} rows ({len(algosignal_msrb)-before_rows:+,} net)")
    if 'date' in algosignal_msrb.columns and algosignal_msrb['date'].notna().any():
        print(f"cache date range: {algosignal_msrb['date'].min().date()} → {algosignal_msrb['date'].max().date()}")
    print(f"columns: {list(algosignal_msrb.columns)}")
    display(algosignal_msrb.head(3))
    del match_incremental


[2026-09-17 13:39:58] START Pull incremental algosignal/MSRB match to pandas
[2026-09-17 13:39:58] END Pull incremental algosignal/MSRB match to pandas | elapsed 0.00s
[2026-09-17 13:39:58] START Load local algosignal/MSRB match cache without remote refresh
[2026-09-17 13:40:52] END Load local algosignal/MSRB match cache without remote refresh | elapsed 53.32s
Loaded local algosignal/MSRB cache without remote refresh: 8,436,844 rows
cache date range: 2026-04-01 → 2026-09-15
columns: ['date', 'msrb_trade_id', 'cusip', 'msrb_quantity', 'size_bin', 'msrb_side', 'signal_side', 'msrb_tradetime', 'msrb_price', 'msrb_yield', 'has_algosignal_match', 'signal_ts', 'MinuteFromSignal', 'MmdYld', 'dMmdSprdSide', 'last_value', 'algo_signal_yield', 'algo_minus_msrb_yield_bp', 'Flg']


,date,msrb_trade_id,cusip,msrb_quantity,size_bin,msrb_side,signal_side,msrb_tradetime,msrb_price,msrb_yield,has_algosignal_match,signal_ts,MinuteFromSignal,MmdYld,dMmdSprdSide,last_value,algo_signal_yield,algo_minus_msrb_yield_bp,Flg
0,2026-04-01,2026040106328900,00037CWZ9,150000.000,100000.000,P,P,2026-04-01 15:56:52+00:00,102.624,3.091,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-01,2026040110667000,0004162L0,100000.000,100000.000,D,D,2026-04-01 17:44:32+00:00,111.006,3.467,1,2026-04-01 17:40:34.775118+00:00,3.954,3.580,1.440,-12.400,3.470,0.340,0.000
2,2026-04-01,2026040110677800,0004162L0,100000.000,100000.000,P,P,2026-04-01 17:44:32+00:00,110.946,3.475,1,2026-04-01 17:40:34.775118+00:00,3.954,3.580,1.930,-12.400,3.475,0.030,0.000


In [ ]:
# ── 20.4 ALGOSIGNAL-MSRB MATCH SCHEMA + TARGET CHECK ────────────────────
if 'algosignal_msrb' not in globals():
    if ALGOSIGNAL_MSRB_CACHE_PATH.exists():
        algosignal_msrb = pd.read_pickle(ALGOSIGNAL_MSRB_CACHE_PATH)
    else:
        raise RuntimeError("Run Section 20.3 first or create algosignal_msrb_match_cache.pkl")

_required_match = ['cusip', 'signal_ts', 'msrb_tradetime', 'msrb_yield', 'algo_signal_yield', 'algo_minus_msrb_yield_bp']
_missing_match = [c for c in _required_match if c not in algosignal_msrb.columns]
print("algosignal/MSRB columns:", list(algosignal_msrb.columns))
print("missing charge-critical columns:", _missing_match or "NONE")

_match = algosignal_msrb.copy()
for _col in ['date', 'msrb_tradetime', 'signal_ts']:
    if _col in _match.columns:
        _match[_col] = pd.to_datetime(_match[_col], errors='coerce')
for _col in ['msrb_yield', 'algo_signal_yield', 'algo_minus_msrb_yield_bp', 'MinuteFromSignal', 'msrb_quantity', 'size_bin']:
    if _col in _match.columns:
        _match[_col] = pd.to_numeric(_match[_col], errors='coerce')

if 'has_algosignal_match' in _match.columns:
    _has = pd.to_numeric(_match['has_algosignal_match'], errors='coerce').fillna(0).astype(int)
else:
    _has = _match['signal_ts'].notna().astype(int) if 'signal_ts' in _match.columns else pd.Series(0, index=_match.index)

_match['yield_error_bp'] = -_match['algo_minus_msrb_yield_bp'] if 'algo_minus_msrb_yield_bp' in _match.columns else np.nan
if {'msrb_yield', 'algo_signal_yield'}.issubset(_match.columns):
    _match['yield_error_bp_direct'] = (_match['msrb_yield'] - _match['algo_signal_yield']) * 100.0
    _target_diff = (_match['yield_error_bp'] - _match['yield_error_bp_direct']).abs()
else:
    _match['yield_error_bp_direct'] = np.nan
    _target_diff = pd.Series(dtype=float)

matched_algosignal_msrb = _match[_has == 1].copy()
matched_algosignal_msrb = matched_algosignal_msrb.dropna(subset=[c for c in ['yield_error_bp', 'signal_ts', 'msrb_tradetime'] if c in matched_algosignal_msrb.columns])
print(f"rows total / matched usable: {len(_match):,} / {len(matched_algosignal_msrb):,}")
if not _target_diff.empty:
    print(f"target sign check: median |(-algo_minus_msrb) - direct| = {_target_diff.median():.4f} bp")
if 'MinuteFromSignal' in matched_algosignal_msrb.columns:
    print("MinuteFromSignal summary:")
    display(matched_algosignal_msrb['MinuteFromSignal'].describe().to_frame('MinuteFromSignal'))
for _col in ['signal_side', 'msrb_side', 'size_bin', 'Flg']:
    if _col in matched_algosignal_msrb.columns:
        print(f"\n{_col} distribution:")
        display(matched_algosignal_msrb[_col].value_counts(dropna=False).head(20).to_frame('rows'))
display(matched_algosignal_msrb.head(5))


algosignal/MSRB columns: ['date', 'msrb_trade_id', 'cusip', 'msrb_quantity', 'size_bin', 'msrb_side', 'signal_side', 'msrb_tradetime', 'msrb_price', 'msrb_yield', 'has_algosignal_match', 'signal_ts', 'MinuteFromSignal', 'MmdYld', 'dMmdSprdSide', 'last_value', 'algo_signal_yield', 'algo_minus_msrb_yield_bp', 'Flg']
missing charge-critical columns: NONE
rows total / matched usable: 8,436,844 / 5,818,372
target sign check: median |(-algo_minus_msrb) - direct| = 0.0000 bp
MinuteFromSignal summary:


,MinuteFromSignal
count,5818372.000
mean,8.059
std,5.346
min,0.000
25%,3.788
50%,7.879
75%,12.074
max,479.840


signal_side distribution:


,rows
signal_side,
D,2311278
S,1868444
P,1638650


msrb_side distribution:


...

size_bin distribution:


,rows
size_bin,
25000.000,1540693
10000.000,991268
50000.000,947378
100000.000,732080
5000.000,591530
15000.000,548600
250000.000,272013
500000.000,96152
1000000.000,73223


Flg distribution:


,rows
Flg,
0.000,5818372


,date,msrb_trade_id,cusip,msrb_quantity,size_bin,msrb_side,signal_side,msrb_tradetime,msrb_price,msrb_yield,has_algosignal_match,signal_ts,MinuteFromSignal,MmdYld,dMmdSprdSide,last_value,algo_signal_yield,algo_minus_msrb_yield_bp,Flg,yield_error_bp,yield_error_bp_direct
1,2026-04-01,2026040110667000,0004162L0,100000.000,100000.000,D,D,2026-04-01 17:44:32+00:00,111.006,3.467,1,2026-04-01 17:40:34.775118+00:00,3.954,3.580,1.440,-12.400,3.470,0.340,0.000,-0.340,-0.340
2,2026-04-01,2026040110677800,0004162L0,100000.000,100000.000,P,P,2026-04-01 17:44:32+00:00,110.946,3.475,1,2026-04-01 17:40:34.775118+00:00,3.954,3.580,1.930,-12.400,3.475,0.030,0.000,-0.030,-0.030
3,2026-04-01,2026040114120500,0004162L0,100000.000,100000.000,S,S,2026-04-01 19:12:05+00:00,111.482,3.405,1,2026-04-01 19:05:39.482903+00:00,6.425,3.580,-8.490,-10.500,3.390,-1.490,0.000,1.490,1.490
4,2026-04-01,2026040100138800,000416Y43,25000.000,25000.000,D,D,2026-04-01 12:07:35+00:00,109.566,3.312,1,2026-04-01 11:56:12.347152+00:00,11.378,3.460,-15.350,-15.700,3.149,-16.250,0.000,16.250,16.250
5,2026-04-01,2026040100143100,000416Y43,25000.000,25000.000,D,D,2026-04-01 12:07:35+00:00,109.566,3.312,1,2026-04-01 11:56:12.347152+00:00,11.378,3.460,-15.350,-15.700,3.149,-16.250,0.000,16.250,16.250


## 21. Charge Calibration Dataset

This section converts the matched algo-signal/MSRB table into a point-in-time charge modeling dataset.

Target:

$$y^{yield}_{i,\tau}=\text{MSRB yield}_{i,\tau^+}-\text{algo signal yield}_{i,\tau}=-\text{algo\_minus\_msrb\_yield\_bp}.$$

Features:

- current IPCA residual from the latest available residual date before `signal_ts`,
- point-in-time AR(1) residual forecast from that residual date,
- liquidity / freshness context (`MinuteFromSignal`, residual volatility),
- market-making context (`signal_side`, `msrb_side`, `size_bin`, quantity, `Flg`),
- optional MMD/spread context (`MmdYld`, `dMmdSprdSide`).

The first version models yield error in bps. Price charge conversion is deferred until `last_value` is confirmed to be the relevant signal/base price or a reliable DV01/duration is added.

In [ ]:
# -- 21.1 BUILD CHARGE CALIBRATION DATASET -------------------------------
if 'matched_algosignal_msrb' not in globals():
    if 'algosignal_msrb' not in globals() and ALGOSIGNAL_MSRB_CACHE_PATH.exists():
        algosignal_msrb = pd.read_pickle(ALGOSIGNAL_MSRB_CACHE_PATH)
    if 'algosignal_msrb' not in globals():
        raise RuntimeError("Run Section 20 first to create matched_algosignal_msrb")
    _tmp = algosignal_msrb.copy()
    _tmp['yield_error_bp'] = -pd.to_numeric(_tmp['algo_minus_msrb_yield_bp'], errors='coerce')
    _has = pd.to_numeric(_tmp.get('has_algosignal_match', 0), errors='coerce').fillna(0).astype(int)
    matched_algosignal_msrb = _tmp[_has == 1].dropna(subset=['signal_ts', 'msrb_tradetime', 'yield_error_bp']).copy()

with timed_block("build IPCA AR forecast matrix for charge features"):
    if 'ipca_ar_pred' not in globals():
        _Sx = _Sy = _Sxx = _Sxy = _npairs = 0.0
        ipca_ar_pred = np.full_like(Rp, np.nan, dtype=np.float64)
        ipca_ar_rho_path = []
        for _t in range(1, T_rv - 1):
            _x0, _y0 = Rp[:, _t - 1], Rp[:, _t]
            _m0 = np.isfinite(_x0) & np.isfinite(_y0)
            _xx, _yy = _x0[_m0], _y0[_m0]
            _Sx += _xx.sum(); _Sy += _yy.sum(); _Sxx += float((_xx * _xx).sum())
            _Sxy += float((_xx * _yy).sum()); _npairs += _xx.size
            if _t < 20 or _npairs < 1000:
                continue
            _den = _npairs * _Sxx - _Sx * _Sx
            if _den <= 0:
                continue
            _rho = (_npairs * _Sxy - _Sx * _Sy) / _den
            _a = (_Sy - _rho * _Sx) / _npairs
            ipca_ar_rho_path.append(_rho)
            ipca_ar_pred[:, _t] = _a + _rho * Rp[:, _t]

with timed_block("join charge rows to latest PIT residual/AR feature"):
    charge_df = matched_algosignal_msrb.copy()
    charge_df['cusip'] = charge_df['cusip'].astype(str)
    charge_df['signal_ts'] = pd.to_datetime(charge_df['signal_ts'], errors='coerce', utc=True)
    charge_df['msrb_tradetime'] = pd.to_datetime(charge_df['msrb_tradetime'], errors='coerce', utc=True)
    charge_df['signal_date'] = charge_df['signal_ts'].dt.tz_convert(None).dt.normalize()
    charge_df['msrb_trade_date'] = charge_df['msrb_tradetime'].dt.tz_convert(None).dt.normalize()
    charge_df['yield_error_bp'] = pd.to_numeric(charge_df['yield_error_bp'], errors='coerce')
    charge_df['abs_yield_error_bp'] = charge_df['yield_error_bp'].abs()

    _rv_dates_charge = pd.DatetimeIndex(pd.to_datetime(rv_dates, errors='coerce')).normalize()
    _rv_ns = _rv_dates_charge.values.astype('datetime64[ns]')
    _sig_ns = charge_df['signal_date'].to_numpy(dtype='datetime64[ns]')
    _resid_t = np.searchsorted(_rv_ns, _sig_ns, side='left') - 1
    _row_map = pd.Series(np.arange(len(rv_cusips), dtype=np.int64), index=np.asarray(rv_cusips, dtype=str))
    _row = charge_df['cusip'].map(_row_map).to_numpy()
    _valid = (_resid_t >= 0) & pd.notna(_row)
    charge_df = charge_df.loc[_valid].copy().reset_index(drop=True)
    _row = _row[_valid].astype(np.int64)
    _resid_t = _resid_t[_valid].astype(np.int64)
    charge_df['resid_t'] = _resid_t
    charge_df['residual_date'] = pd.to_datetime(_rv_dates_charge[_resid_t]).to_numpy(dtype='datetime64[ns]')
    charge_df['rv_signal'] = Rp[_row, _resid_t]
    charge_df['ar_forecast'] = ipca_ar_pred[_row, _resid_t]
    charge_df['resid_vol'] = np.nanstd(Rp, axis=1)[_row]
    _signal_days = charge_df['signal_date'].to_numpy(dtype='datetime64[D]')
    _residual_days = charge_df['residual_date'].to_numpy(dtype='datetime64[D]')
    charge_df['residual_age_days'] = (_signal_days - _residual_days).astype('timedelta64[D]').astype(int)

    for _col in ['MinuteFromSignal', 'msrb_quantity', 'size_bin', 'MmdYld', 'dMmdSprdSide', 'last_value', 'Flg']:
        if _col in charge_df.columns:
            charge_df[_col] = pd.to_numeric(charge_df[_col], errors='coerce')
    for _col in ['signal_side', 'msrb_side']:
        if _col in charge_df.columns:
            charge_df[_col] = charge_df[_col].astype('string').fillna('MISSING').str.upper()

CHARGE_TARGET_CLIP_BPS = 500.0
charge_model_df = charge_df.dropna(subset=['yield_error_bp', 'rv_signal']).copy()
charge_model_df = charge_model_df[charge_model_df['abs_yield_error_bp'] <= CHARGE_TARGET_CLIP_BPS].copy()
charge_model_df['size_bin_label'] = charge_model_df['size_bin'].astype('string').fillna('MISSING') if 'size_bin' in charge_model_df.columns else 'MISSING'
charge_model_df['minute_bucket'] = pd.cut(pd.to_numeric(charge_model_df.get('MinuteFromSignal', np.nan), errors='coerce'),
                                          bins=[-np.inf, 1, 5, 30, 240, np.inf],
                                          labels=['<=1m', '1-5m', '5-30m', '30m-4h', '>4h'])

print(f"charge rows after PIT residual join/filter: {len(charge_model_df):,}")
print(f"signal date range: {charge_model_df['signal_date'].min().date()} -> {charge_model_df['signal_date'].max().date()}")
print(f"target yield error bp: mean={charge_model_df['yield_error_bp'].mean():.2f}, "
      f"median={charge_model_df['yield_error_bp'].median():.2f}, MAE={charge_model_df['yield_error_bp'].abs().mean():.2f}")
print(f"non-null AR forecast coverage: {charge_model_df['ar_forecast'].notna().mean():.1%}")
display(charge_model_df[['cusip', 'signal_ts', 'msrb_tradetime', 'signal_side', 'msrb_side',
                         'algo_signal_yield', 'msrb_yield', 'yield_error_bp',
                         'rv_signal', 'ar_forecast', 'MinuteFromSignal', 'size_bin']].head(5))


[2026-09-17 13:42:28] START build IPCA AR forecast matrix for charge features
[2026-09-17 13:42:28] END build IPCA AR forecast matrix for charge features | elapsed 0.64s
[2026-09-17 13:42:28] START join charge rows to latest PIT residual/AR feature
[2026-09-17 13:42:36] END join charge rows to latest PIT residual/AR feature | elapsed 8.05s
charge rows after PIT residual join/filter: 3,970,218
signal date range: 2026-04-06 -> 2026-09-15
target yield error bp: mean=-1.49, median=-0.06, MAE=10.18
non-null AR forecast coverage: 83.7%


,cusip,signal_ts,msrb_tradetime,signal_side,msrb_side,algo_signal_yield,msrb_yield,yield_error_bp,rv_signal,ar_forecast,MinuteFromSignal,size_bin
0,0004162P1,2026-04-06 15:28:08.912094+00:00,2026-04-06 15:36:19+00:00,P,P,4.080,4.094,1.350,-0.000,NaN,8.168,25000.000
1,0004162P1,2026-04-06 15:28:08.912094+00:00,2026-04-06 15:36:19+00:00,S,S,3.993,4.082,8.910,-0.000,NaN,8.168,25000.000
2,0004162R7,2026-04-06 15:12:40.075052+00:00,2026-04-06 15:13:12+00:00,D,D,4.230,4.274,4.390,-0.000,NaN,0.532,15000.000
3,0004162R7,2026-04-06 15:12:40.075052+00:00,2026-04-06 15:13:12+00:00,P,P,4.267,4.278,1.070,-0.000,NaN,0.532,15000.000
4,000416U62,2026-04-06 14:56:49.055808+00:00,2026-04-06 15:02:03+00:00,S,S,2.568,2.646,7.810,-0.000,NaN,5.232,25000.000


## 22. Charge Models: RV, AR, and Market-Making Context

Fit the first yield-bps charge models on a time split:

1. **RV only:** current IPCA residual.
2. **RV + AR:** residual plus point-in-time AR(1) forecast.
3. **RV + AR + MM context:** add side, size, freshness/liquidity, MMD/spread context.

Evaluation is quote-error reduction, not IC:

$$\text{Baseline MAE}=|\text{MSRB yield}-\text{algo signal yield}|,$$

$$\text{Adjusted MAE}=|\text{MSRB yield}-(\text{algo signal yield}+\widehat y/100)| = |y-\widehat y|.$$

In [ ]:
# -- 22.1 FIT CHARGE MODELS AND EVALUATE QUOTE-ERROR REDUCTION ----------
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

if 'charge_model_df' not in globals():
    raise RuntimeError("Run Section 21 first to create charge_model_df")

_charge = charge_model_df.copy()
_charge = _charge.dropna(subset=['yield_error_bp', 'signal_date'])
_unique_dates = pd.DatetimeIndex(sorted(_charge['signal_date'].dropna().unique()))
if len(_unique_dates) < 5:
    raise RuntimeError("Not enough signal dates for time-split charge model evaluation")
_split_date = _unique_dates[int(np.floor(len(_unique_dates) * 0.7))]
_charge_train = _charge[_charge['signal_date'] < _split_date].copy()
_charge_test = _charge[_charge['signal_date'] >= _split_date].copy()
if _charge_train.empty or _charge_test.empty:
    raise RuntimeError("Charge model time split produced an empty train or test set")
print(f"Charge model split: train < {_split_date.date()} | train={len(_charge_train):,}, test={len(_charge_test):,}")


def _format_charge_col_list(cols, max_items=8):
    cols = list(cols)
    shown = ', '.join(map(str, cols[:max_items]))
    extra = f", ... +{len(cols) - max_items} more" if len(cols) > max_items else ""
    return f"{len(cols)} ({shown}{extra})"


def _zero_benchmark_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.sum(y_true ** 2)
    return 1.0 - np.sum((y_true - y_pred) ** 2) / denom if denom > 0 else np.nan


def _design_matrices(train, test, num_cols, cat_cols):
    num_cols = [c for c in num_cols if c in train.columns]
    cat_cols = [c for c in cat_cols if c in train.columns]
    if num_cols:
        train_num = train[num_cols].apply(pd.to_numeric, errors='coerce')
        test_num = test[num_cols].apply(pd.to_numeric, errors='coerce')
        med = train_num.median()
        scale = train_num.std().replace(0, 1.0)
        Xtr_num = ((train_num - med) / scale).fillna(0.0)
        Xte_num = ((test_num - med) / scale).fillna(0.0)
    else:
        Xtr_num = pd.DataFrame(index=train.index)
        Xte_num = pd.DataFrame(index=test.index)
    if cat_cols:
        train_cat = train[cat_cols].astype('string').fillna('MISSING')
        test_cat = test[cat_cols].astype('string').fillna('MISSING')
        Xtr_cat = pd.get_dummies(train_cat, prefix=cat_cols, dummy_na=False)
        Xte_cat = pd.get_dummies(test_cat, prefix=cat_cols, dummy_na=False)
        cat_columns = Xtr_cat.columns
        unseen_test_columns = Xte_cat.columns.difference(cat_columns)
        if len(unseen_test_columns):
            print(f"  {_format_charge_col_list(unseen_test_columns)} unseen test categories mapped to all-zero train-fitted dummies")
        Xtr_cat = Xtr_cat.reindex(columns=cat_columns, fill_value=0.0)
        Xte_cat = Xte_cat.reindex(columns=cat_columns, fill_value=0.0)
    else:
        Xtr_cat = pd.DataFrame(index=train.index)
        Xte_cat = pd.DataFrame(index=test.index)
    Xtr = pd.concat([Xtr_num, Xtr_cat], axis=1)
    Xte = pd.concat([Xte_num, Xte_cat], axis=1)
    return Xtr.astype(float), Xte.astype(float)


def _charge_prediction_result(name, train, test, pred, model_obj=None):
    yte = test['yield_error_bp'].to_numpy(dtype=float)
    pred = np.asarray(pred, dtype=float)
    baseline_mae = mean_absolute_error(yte, np.zeros_like(yte))
    adjusted_mae = mean_absolute_error(yte, pred)
    rmse = float(np.sqrt(mean_squared_error(yte, pred)))
    corr = _safe_corr(pd.Series(pred), pd.Series(yte), method='spearman') if '_safe_corr' in globals() else np.nan
    out = test.copy()
    out['pred_yield_error_bp'] = pred
    out['adjusted_error_bp'] = out['yield_error_bp'] - out['pred_yield_error_bp']
    out['adjusted_signal_yield'] = out['algo_signal_yield'] + out['pred_yield_error_bp'] / 100.0
    return {
        'model': name,
        'n_train': len(train),
        'n_test': len(test),
        'baseline_mae_bp': baseline_mae,
        'adjusted_mae_bp': adjusted_mae,
        'mae_improvement_bp': baseline_mae - adjusted_mae,
        'mae_improvement_pct': (baseline_mae - adjusted_mae) / baseline_mae if baseline_mae else np.nan,
        'rmse_bp': rmse,
        'zero_benchmark_r2': _zero_benchmark_r2(yte, pred),
        'rank_corr': corr,
        'model_obj': model_obj,
        'test_predictions': out,
    }


def _fit_charge_model(name, num_cols, cat_cols, alpha=10.0):
    train = _charge_train.dropna(subset=['yield_error_bp']).copy()
    test = _charge_test.dropna(subset=['yield_error_bp']).copy()
    Xtr, Xte = _design_matrices(train, test, num_cols, cat_cols)
    ytr = train['yield_error_bp'].to_numpy(dtype=float)
    model = Ridge(alpha=alpha)
    model.fit(Xtr, ytr)
    pred = np.clip(model.predict(Xte), -CHARGE_TARGET_CLIP_BPS, CHARGE_TARGET_CLIP_BPS)
    return _charge_prediction_result(name, train, test, pred, model_obj=model)

_no_adjust_test = _charge_test.dropna(subset=['yield_error_bp']).copy()
_charge_models = [
    _charge_prediction_result('No adjustment', _charge_train, _no_adjust_test, np.zeros(len(_no_adjust_test))),
    _fit_charge_model('RV only', ['rv_signal'], []),
    _fit_charge_model('RV + AR', ['rv_signal', 'ar_forecast'], []),
    _fit_charge_model('RV + AR + MM context',
                      ['rv_signal', 'ar_forecast', 'resid_vol', 'residual_age_days', 'MinuteFromSignal',
                       'msrb_quantity', 'MmdYld', 'dMmdSprdSide', 'Flg'],
                      ['signal_side', 'msrb_side', 'size_bin_label', 'minute_bucket']),
]
charge_model_summary = pd.DataFrame([{k: v for k, v in m.items() if k not in ['model_obj', 'test_predictions']} for m in _charge_models])
charge_model_predictions = {m['model']: m['test_predictions'] for m in _charge_models}
print("Charge model evaluation: yield-error MAE before vs after predicted charge")
print("Categorical encoders are fit on train only; unseen test levels are all-zero in the trained dummy basis.")
display(charge_model_summary.style.format({
    'n_train': '{:,.0f}', 'n_test': '{:,.0f}', 'baseline_mae_bp': '{:.2f}',
    'adjusted_mae_bp': '{:.2f}', 'mae_improvement_bp': '{:+.2f}',
    'mae_improvement_pct': '{:+.1%}', 'rmse_bp': '{:.2f}',
    'zero_benchmark_r2': '{:.4f}', 'rank_corr': '{:.4f}',
}))
_no_adjust_mae = charge_model_summary.loc[charge_model_summary['model'] == 'No adjustment', 'adjusted_mae_bp'].iloc[0]
_best_learned = charge_model_summary[charge_model_summary['model'] != 'No adjustment'].sort_values('adjusted_mae_bp').iloc[0]
_best_learned_charge_model_name = str(_best_learned['model'])
_best_charge_model_name = _best_learned_charge_model_name if _best_learned['adjusted_mae_bp'] < _no_adjust_mae else 'No adjustment'
_best_charge_pred = charge_model_predictions[_best_charge_model_name]
_best_learned_charge_pred = charge_model_predictions[_best_learned_charge_model_name]
print(f"Best learned charge model by adjusted MAE: {_best_learned_charge_model_name}")
print(f"Best overall charge model by adjusted MAE: {_best_charge_model_name}")

Charge model split: train < 2026-07-29 | train=2,636,632, test=1,333,586
Charge model evaluation: yield-error MAE before vs after predicted charge
Categorical encoders are fit on train only; unseen test levels are all-zero in the trained dummy basis.


<HTML output>

,model,n_train,n_test,baseline_mae_bp,adjusted_mae_bp,mae_improvement_bp,mae_improvement_pct,rmse_bp,zero_benchmark_r2,rank_corr
0,No adjustment,"2,636,632","1,333,586",10.13,10.13,+0.00,+0.0%,31.47,0.0000,nan
1,RV only,"2,636,632","1,333,586",10.13,10.25,-0.11,-1.1%,31.42,0.0034,-0.0048
2,RV + AR,"2,636,632","1,333,586",10.13,10.25,-0.11,-1.1%,31.42,0.0034,0.0079
3,RV + AR + MM context,"2,636,632","1,333,586",10.13,10.65,-0.51,-5.1%,31.32,0.0093,0.0933


Best learned charge model by adjusted MAE: RV only
Best overall charge model by adjusted MAE: No adjustment


## 23. Charge Evaluation by Side, Size, and Freshness

The business question is whether the predicted charge reduces future quote error:

$$|	ext{MSRB yield}-	ext{algo signal yield}| \quad 	ext{vs} \quad |	ext{MSRB yield}-	ext{adjusted signal yield}|.$$

If **No adjustment** wins overall, slicing only the best overall model is not informative because every slice has exactly zero improvement. This section therefore reports both the headline best-overall model and the best learned model sliced by signal side, MSRB side, size, and signal-to-trade freshness.

In [ ]:
# -- 23.1 CHARGE MODEL SLICE EVALUATION + VERDICT ------------------------
def _prep_charge_eval(pred_df):
    out = pred_df.copy()
    out['baseline_abs_error_bp'] = out['yield_error_bp'].abs()
    out['adjusted_abs_error_bp'] = out['adjusted_error_bp'].abs()
    out['improvement_bp'] = out['baseline_abs_error_bp'] - out['adjusted_abs_error_bp']
    return out


def _slice_eval(df, group_col):
    if group_col not in df.columns:
        return pd.DataFrame()
    rows = []
    for g, d in df.groupby(group_col, dropna=False):
        if len(d) < 100:
            continue
        baseline_mae = d['baseline_abs_error_bp'].mean()
        improvement = d['improvement_bp'].mean()
        rows.append({
            group_col: g,
            'n': len(d),
            'baseline_mae_bp': baseline_mae,
            'adjusted_mae_bp': d['adjusted_abs_error_bp'].mean(),
            'improvement_bp': improvement,
            'improvement_pct': improvement / baseline_mae if baseline_mae else np.nan,
        })
    return pd.DataFrame(rows).sort_values('improvement_bp', ascending=False)


_eval_overall_best = _prep_charge_eval(_best_charge_pred)
_eval_learned = _prep_charge_eval(_best_learned_charge_pred) if '_best_learned_charge_pred' in globals() else _eval_overall_best.copy()
_learned_slice_name = _best_learned_charge_model_name if '_best_learned_charge_model_name' in globals() else _best_charge_model_name
charge_slice_tables = {}

print(f"Best overall model for headline decision: {_best_charge_model_name}")
print(f"Best learned model for diagnostic slicing: {_learned_slice_name}")
if _best_charge_model_name == 'No adjustment':
    print("No adjustment wins overall, so slice diagnostics below use the best learned model to show where the learned charge helps or hurts.")

for _col in ['signal_side', 'msrb_side', 'size_bin_label', 'minute_bucket', 'Flg']:
    _tbl = _slice_eval(_eval_learned, _col)
    if not _tbl.empty:
        charge_slice_tables[_col] = _tbl
        print(f"\nBest learned charge-model improvement by {_col}:")
        display(_tbl.style.format({'n': '{:,.0f}', 'baseline_mae_bp': '{:.2f}', 'adjusted_mae_bp': '{:.2f}',
                                   'improvement_bp': '{:+.2f}', 'improvement_pct': '{:+.1%}'}))

_best = charge_model_summary[charge_model_summary['model'] == _best_charge_model_name].iloc[0]
_best_learned_row = charge_model_summary[charge_model_summary['model'] == _learned_slice_name].iloc[0]
print("=" * 78)
print("SECTION 23 VERDICT — first yield-bps charge model")
print("=" * 78)
print(f"Best overall model: {_best_charge_model_name}")
print(f"Baseline quote-error MAE : {_best['baseline_mae_bp']:.2f} bps")
print(f"Adjusted quote-error MAE : {_best['adjusted_mae_bp']:.2f} bps")
print(f"MAE improvement          : {_best['mae_improvement_bp']:+.2f} bps ({_best['mae_improvement_pct']:+.1%})")
print("-" * 78)
print(f"Best learned model       : {_learned_slice_name}")
print(f"Learned adjusted MAE     : {_best_learned_row['adjusted_mae_bp']:.2f} bps")
print(f"Learned MAE improvement  : {_best_learned_row['mae_improvement_bp']:+.2f} bps ({_best_learned_row['mae_improvement_pct']:+.1%})")
print("-" * 78)
if _best['mae_improvement_bp'] > 0:
    print("READ: predicted yield-bps charge reduces held-out quote error. This is the first")
    print("direct bridge from IPCA residual research into market-making quote adjustment.")
else:
    print("READ: first charge model does not reduce held-out quote error yet. Keep the dataset,")
    print("but improve PIT features, target definition, side-specific calibration, and baselines")
    print("before using the output as a charge.")
print("=" * 78)

Best overall model for headline decision: No adjustment
Best learned model for diagnostic slicing: RV only
No adjustment wins overall, so slice diagnostics below use the best learned model to show where the learned charge helps or hurts.

Best learned charge-model improvement by signal_side:


<HTML output>

,signal_side,n,baseline_mae_bp,adjusted_mae_bp,improvement_bp,improvement_pct
0,D,"504,774",10.47,10.36,+0.11,+1.1%
2,S,"426,342",10.37,10.51,-0.14,-1.4%
1,P,"402,470",9.46,9.83,-0.37,-3.9%


Best learned charge-model improvement by msrb_side:


<HTML output>

...

,minute_bucket,n,baseline_mae_bp,adjusted_mae_bp,improvement_bp,improvement_pct
2,5-30m,"916,083",10.26,10.37,-0.11,-1.1%
0,<=1m,"90,567",9.61,9.72,-0.12,-1.2%
1,1-5m,"326,106",9.92,10.04,-0.12,-1.2%
3,30m-4h,830,11.82,12.03,-0.20,-1.7%


Best learned charge-model improvement by Flg:


<HTML output>

,Flg,n,baseline_mae_bp,adjusted_mae_bp,improvement_bp,improvement_pct
0,0.000000,"1,333,586",10.13,10.25,-0.11,-1.1%


SECTION 23 VERDICT — first yield-bps charge model
Best overall model: No adjustment
Baseline quote-error MAE : 10.13 bps
Adjusted quote-error MAE : 10.13 bps
MAE improvement          : +0.00 bps (+0.0%)
------------------------------------------------------------------------------
Best learned model       : RV only
Learned adjusted MAE     : 10.25 bps
Learned MAE improvement  : -0.11 bps (-1.1%)
------------------------------------------------------------------------------
READ: first charge model does not reduce held-out quote error yet. Keep the dataset,
but improve PIT features, target definition, side-specific calibration, and baselines
before using the output as a charge.


## Final MSRB Validation — Frozen OOS RV Score vs Transaction-Price Error

The **independent external test**: does the *frozen* OOS relative-value score (from §9.9–§9.12) explain subsequent MSRB transaction-price errors, controlling for side / size / signal-to-trade lag / liquidity / intervening market move? Each matched trade is joined to the most recent RV signal available **strictly before** the trade (`signal_available_ts < tradetime`) — no look-ahead, no retraining.

$$e^{MSRB}_{i,\tau}\;(\text{bp}) = (y^{MSRB}_{i,\tau} - y^{algo}_{i,\tau})\times 100 = \alpha + \beta\,RV_{i,t} + \gamma^\top X_{i,\tau} + u_{i,\tau}.$$

**Sign convention (locked):** higher RV ⇒ higher predicted residual return (richer) ⇒ trade should print at *lower* yield vs the algo signal ⇒ expect $\beta<0$ in this yield-error space. The key question is whether $\beta$ is stable, correctly signed, and monotonic across RV deciles (by side).

In [ ]:
# ── FINAL MSRB VALIDATION: FROZEN OOS RV SCORE vs TRANSACTION-PRICE ERROR ─
# e^MSRB (bp) = (msrb_yield - algo_signal_yield)*100 ; test e = a + b*RV + gamma'X.
# RV is joined as-of the last signal_available_ts strictly before the trade (no look-ahead).
from scipy.stats import rankdata
_msrbrv_start = time.perf_counter()
log_runtime("START final MSRB RV-score validation")
if 'oos_predictions' not in globals():
    oos_predictions = pd.read_parquet(OOS_PANEL_PATH)
assert 'matched_algosignal_msrb' in globals(), "matched_algosignal_msrb not in kernel; run the MSRB match section first"


def _num(s):
    return pd.to_numeric(pd.Series(s).astype(str).str.replace(',', '', regex=False), errors='coerce')


_mm = matched_algosignal_msrb.copy()
_mm.columns = [str(c) for c in _mm.columns]
_mm['cusip'] = _mm['cusip'].astype(str)
_mm['_msrb_yield'] = _num(_mm['msrb_yield'])
_mm['_algo_yield'] = _num(_mm['algo_signal_yield'])
_mm['e_msrb_bp'] = (_mm['_msrb_yield'] - _mm['_algo_yield']) * 100.0        # LHS (user-specified)
_mm['_qty'] = _num(_mm['msrb_quantity']) if 'msrb_quantity' in _mm else _num(_mm.get('size_bin'))
_mm['_lag'] = _num(_mm['MinuteFromSignal']) if 'MinuteFromSignal' in _mm else np.nan
_mm['_mkt'] = _num(_mm['dMmdSprdSide']) if 'dMmdSprdSide' in _mm else 0.0
_mm['_side'] = _mm['msrb_side'].astype(str) if 'msrb_side' in _mm else 'U'
def _to_naive_ns(s):
    s = pd.to_datetime(s, errors='coerce')
    if getattr(s.dtype, 'tz', None) is not None:
        s = s.dt.tz_localize(None)
    return s.astype('datetime64[ns]')

_mm['_tradetime'] = _to_naive_ns(_mm['msrb_tradetime'] if 'msrb_tradetime' in _mm else _mm['date'])
_mm['_tradedate'] = _mm['_tradetime'].dt.normalize()
_mm = _mm.dropna(subset=['e_msrb_bp', 'cusip', '_tradedate', '_tradetime'])
_mm = _mm[_mm['_algo_yield'].notna()]

# As-of join: most recent frozen RV signal known strictly before the trade.
_sig = (oos_predictions[['cusip', 'target_date', 'signal_available_ts', 'rv_rank', 'rv_score', 'peer_liq']]
        .assign(cusip=lambda d: d['cusip'].astype(str)))
_sig['target_date'] = _to_naive_ns(_sig['target_date'])
_sig['signal_available_ts'] = _to_naive_ns(_sig['signal_available_ts'])
_sig = _sig.sort_values('target_date')
_mm = _mm.sort_values('_tradedate')
_j = pd.merge_asof(_mm, _sig, left_on='_tradedate', right_on='target_date', by='cusip',
                   direction='backward', allow_exact_matches=True)
_j = _j[_j['signal_available_ts'] < _j['_tradetime']]
_j = _j.dropna(subset=['rv_rank', 'rv_score', 'e_msrb_bp'])
print(f"MSRB trades matched to a frozen RV signal: {len(_j):,} "
      f"(cusips {_j['cusip'].nunique():,}; "
      f"{_j['_tradedate'].min().date() if len(_j) else 'NA'} -> "
      f"{_j['_tradedate'].max().date() if len(_j) else 'NA'})")
print("  trades by side:", _j['_side'].value_counts().to_dict())

if len(_j) < 200:
    print("Insufficient RV-matched MSRB coverage for a stable regression; reporting counts only.")
else:
    def _design(dfj, rv_col):
        _side_d = pd.get_dummies(dfj['_side'], prefix='side', drop_first=True).astype(float)
        _liq_d = pd.get_dummies(dfj['peer_liq'].astype(str), prefix='liq', drop_first=True).astype(float)
        _logsz = np.log1p(dfj['_qty'].clip(lower=0).fillna(0.0)).rename('log_size')
        _lag = dfj['_lag'].fillna(dfj['_lag'].median()).rename('lag_min')
        _mkt = dfj['_mkt'].fillna(0.0).rename('mkt')
        X = pd.concat([pd.Series(1.0, index=dfj.index, name='const'), dfj[rv_col].rename('RV'),
                       _logsz, _lag, _mkt, _side_d, _liq_d], axis=1)
        return X.to_numpy(float), list(X.columns)

    def _ols(X, y):
        XtX = X.T @ X
        beta = np.linalg.solve(XtX, X.T @ y)
        resid = y - X @ beta
        n, k = X.shape
        s2 = (resid @ resid) / max(n - k, 1)
        se = np.sqrt(np.clip(np.diag(s2 * np.linalg.pinv(XtX)), 0, None))
        sst = ((y - y.mean()) ** 2).sum()
        return beta, se, (1 - (resid @ resid) / sst if sst > 0 else np.nan)

    _y = _j['e_msrb_bp'].to_numpy(float)
    print("\n=== e^MSRB (bp) = a + b*RV + gamma'X  (side/log-size/lag/market/liquidity controls) ===")
    print("  sign convention: higher RV = richer => expect beta < 0 (trade prints at lower yield vs algo)")
    for rv_col in ['rv_rank', 'rv_score']:
        X, names = _design(_j, rv_col)
        beta, se, r2 = _ols(X, _y)
        _i = names.index('RV')
        X0 = np.delete(X, _i, axis=1); _, _, r20 = _ols(X0, _y)
        _t = beta[_i] / se[_i] if se[_i] > 0 else np.nan
        print(f"  [{rv_col:8s}] beta={beta[_i]:+.3f} bp  t={_t:+.2f}  incR2={r2 - r20:+.4f}  "
              f"(fullR2={r2:.4f}, n={len(_y):,})")

    # beta(rv_rank) stability across OOS folds (trade month).
    _rows = []
    for _mth, _z in _j.groupby(_j['_tradedate'].dt.to_period('M')):
        if len(_z) < 100:
            _rows.append({'month': str(_mth), 'n': len(_z), 'beta_rv_rank_bp': np.nan}); continue
        X, names = _design(_z, 'rv_rank'); b, s, _ = _ols(X, _z['e_msrb_bp'].to_numpy(float))
        _i = names.index('RV')
        _rows.append({'month': str(_mth), 'n': len(_z), 'beta_rv_rank_bp': b[_i], 't': b[_i] / s[_i] if s[_i] > 0 else np.nan})
    print("\nbeta(rv_rank) stability by trade month:")
    display(pd.DataFrame(_rows).set_index('month').style.format({'n': '{:,.0f}', 'beta_rv_rank_bp': '{:+.3f}', 't': '{:+.2f}'}, na_rep='--'))

    # e^MSRB by RV decile, split by side (monotonicity).
    _j['_rvdec'] = np.clip((rankdata(_j['rv_rank'].to_numpy()) / len(_j) * 10).astype(int) + 1, 1, 10)
    _dec_side = _j.pivot_table(index='_rvdec', columns='_side', values='e_msrb_bp', aggfunc='mean')
    _dec_n = _j.pivot_table(index='_rvdec', columns='_side', values='e_msrb_bp', aggfunc='size')
    print("\nMean e^MSRB (bp) by RV decile x side (monotone in RV => signal transfers to trades):")
    display(_dec_side.style.format('{:+.2f}', na_rep='--'))
    print("coverage (n) by RV decile x side:")
    display(_dec_n.style.format('{:,.0f}', na_rep='--'))

    # Robustness: beta(rv_rank) within liquidity buckets.
    _rob = []
    for _lq, _z in _j.groupby(_j['peer_liq'].astype(str)):
        if len(_z) < 100:
            _rob.append({'liq': _lq, 'n': len(_z), 'beta_rv_rank_bp': np.nan}); continue
        X, names = _design(_z, 'rv_rank'); b, s, _ = _ols(X, _z['e_msrb_bp'].to_numpy(float))
        _i = names.index('RV')
        _rob.append({'liq': _lq, 'n': len(_z), 'beta_rv_rank_bp': b[_i], 't': b[_i] / s[_i] if s[_i] > 0 else np.nan})
    print("\nbeta(rv_rank) by liquidity bucket (robustness):")
    display(pd.DataFrame(_rob).set_index('liq').style.format({'n': '{:,.0f}', 'beta_rv_rank_bp': '{:+.3f}', 't': '{:+.2f}'}, na_rep='--'))

msrb_rv_validation = _j if 'e_msrb_bp' in _j.columns else None
log_runtime("END final MSRB RV-score validation", _msrbrv_start)


[2026-09-17 18:54:53] START final MSRB RV-score validation
MSRB trades matched to a frozen RV signal: 1,971,945 (cusips 155,685; 2026-07-01 -> 2026-09-15)
  trades by side: {'D': 745554, 'S': 636115, 'P': 590276}

=== e^MSRB (bp) = a + b*RV + gamma'X  (side/log-size/lag/market/liquidity controls) ===
  sign convention: higher RV = richer => expect beta < 0 (trade prints at lower yield vs algo)
  [rv_rank ] beta=-2.501 bp  t=-13.57  incR2=+0.0001  (fullR2=0.0107, n=1,971,945)
  [rv_score] beta=-1.932 bp  t=-7.10  incR2=+0.0000  (fullR2=0.0107, n=1,971,945)

beta(rv_rank) stability by trade month:


<HTML output>

,n,beta_rv_rank_bp,t
month,,,
2026-07,"735,912",-2.934,-9.76
2026-08,"794,395",-1.816,-6.06
2026-09,"441,638",-2.693,-7.29


Mean e^MSRB (bp) by RV decile x side (monotone in RV => signal transfers to trades):


<HTML output>

...

_side,D,P,S
_rvdec,,,
1,"75,244","59,252","62,698"
2,"74,703","59,190","63,301"
3,"73,578","60,138","63,478"
4,"73,084","60,007","64,101"
5,"73,265","59,935","63,998"
6,"73,466","60,034","63,695"
7,"73,461","60,119","63,615"
8,"73,790","59,644","63,759"
9,"76,482","57,537","63,174"


beta(rv_rank) by liquidity bucket (robustness):


<HTML output>

,n,beta_rv_rank_bp,t
liq,,,
L1,"179,023",-26.487,-16.28
L2,"386,110",-0.458,-1.48
L3,"382,432",+0.617,+2.16
L4,"459,512",+0.071,+0.38
L5,"564,868",-0.423,-2.00


[2026-09-17 18:55:50] END final MSRB RV-score validation | elapsed 56.50s
